# Healthcare Challenge 3 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 3: Discharge Readiness Prediction**.

**Goal**: Predict `discharge_ready_day11` (0/1) for each hospital stay
**Metric**: Macro-F1 Score - Higher is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular stay data with a simple Random Forest classifier.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your_api_key",        # Get from your team dashboard
    team_name="your_team_name"     # Your exact team name
)

# Iteration 1
- 0.7589

## EDA 1
- 0.6845

In [2]:
# clai_ch3_v3 — Iteration 0 — EDA1 + strong baseline (single runnable cell)

import os, json, random, re, subprocess
from pathlib import Path
from datetime import datetime, timezone

# -----------------------------
# Reproducibility / determinism
# -----------------------------
SEED = 42
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import f1_score, confusion_matrix
import joblib

TEAM, TASK, VX = "clai", "ch3", "v3"
SUBMISSION_NAME = f"{TEAM}_{TASK}_{VX}_submission.csv"
ITER_LOG_NAME = f"{TEAM}_{TASK}_{VX}_iteration_detail.jsonl"

# -----------------------------
# Helpers
# -----------------------------
def find_path(filename: str) -> Path:
    """Find a required file in common project locations."""
    candidates = [
        Path.cwd(),
        Path.cwd() / "data",
        Path.cwd() / "input",
        Path("/mnt/data"),   # sandbox default
    ]
    for c in candidates:
        p = c / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename} in: {candidates}")

def get_git_hash() -> str | None:
    try:
        out = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
        return out if out else None
    except Exception:
        return None

def safe_float(x):
    try:
        if x is None:
            return None
        if isinstance(x, (np.floating, float)):
            return float(x)
        if isinstance(x, (np.integer, int)):
            return float(x)
        return float(x)
    except Exception:
        return None

def safe_int(x):
    try:
        if x is None:
            return None
        if isinstance(x, (np.integer, int)):
            return int(x)
        return int(x)
    except Exception:
        return None

def compute_iter_id(log_path: Path) -> int:
    if (not log_path.exists()) or log_path.stat().st_size == 0:
        return 0
    last = None
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                last = line
    if not last:
        return 0
    return int(json.loads(last)["iteration_id"]) + 1

def build_vitals_df(vit_list: list[dict]) -> pd.DataFrame:
    """Aggregate 10-day vitals + notes into per-stay features (safe for day-11 prediction)."""
    rows = []
    for rec in vit_list:
        stay_id = rec.get("stay_id")
        days = rec.get("days", [])
        df = pd.DataFrame(days)
        if df.empty:
            rows.append({"stay_id": stay_id, "note_text": "", "note_len": 0})
            continue
        df = df.sort_values("day")
        feats = {"stay_id": stay_id}

        for col in ["hr", "sbp", "dbp", "temp_c", "rr"]:
            vals = pd.to_numeric(df[col], errors="coerce")
            feats[f"{col}_mean"] = float(vals.mean()) if vals.notna().any() else np.nan
            feats[f"{col}_std"]  = float(vals.std())  if vals.notna().sum() >= 2 else 0.0
            feats[f"{col}_min"]  = float(vals.min())  if vals.notna().any() else np.nan
            feats[f"{col}_max"]  = float(vals.max())  if vals.notna().any() else np.nan
            feats[f"{col}_last"] = float(vals.iloc[-1]) if (len(vals) > 0 and pd.notna(vals.iloc[-1])) else np.nan

            # Linear slope over day index (robust enough for baseline)
            if vals.notna().sum() >= 2:
                x = df["day"].astype(float).to_numpy()
                y = vals.astype(float).to_numpy()
                mask = np.isfinite(x) & np.isfinite(y)
                if mask.sum() >= 2:
                    feats[f"{col}_slope"] = float(np.polyfit(x[mask], y[mask], 1)[0])
                else:
                    feats[f"{col}_slope"] = np.nan
            else:
                feats[f"{col}_slope"] = np.nan

        notes = df.get("note", pd.Series([""] * len(df))).fillna("").astype(str).tolist()
        agg_note = " ".join(notes)
        feats["note_text"] = agg_note
        feats["note_len"] = int(len(agg_note))
        rows.append(feats)

    return pd.DataFrame(rows)

# -----------------------------
# Setup output directories
# -----------------------------
ckpt_root = Path("checkpoints") / f"{TEAM}_{TASK}_{VX}"
art_root  = Path("artifacts") / f"{TEAM}_{TASK}_{VX}"
consult_root = Path("consultant_packets")
ckpt_root.mkdir(parents=True, exist_ok=True)
art_root.mkdir(parents=True, exist_ok=True)
consult_root.mkdir(parents=True, exist_ok=True)

log_path = Path(ITER_LOG_NAME)
iter_id = compute_iter_id(log_path)
timestamp = datetime.now(timezone.utc).astimezone().isoformat()
block_id = iter_id // 5
block_iter = iter_id % 5

# -----------------------------
# Load data
# -----------------------------
paths = {
    "stays_train": find_path("stays_train.csv"),
    "stays_test": find_path("stays_test.csv"),
    "patients": find_path("patients.csv"),
    "vitals": find_path("vitals_timeseries.json"),
}
stays_train = pd.read_csv(paths["stays_train"])
stays_test  = pd.read_csv(paths["stays_test"])
patients    = pd.read_csv(paths["patients"])
with open(paths["vitals"], "r", encoding="utf-8") as f:
    vit_raw = json.load(f)

vitals_df = build_vitals_df(vit_raw)

# -----------------------------
# Joins (EDA1 scope)
# -----------------------------
train = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_df, on="stay_id", how="left")
test  = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_df, on="stay_id", how="left")

# Leakage sanity (quick)
assert "discharge_ready_day11" in stays_train.columns
assert "discharge_ready_day11" not in stays_test.columns
assert set(stays_train.columns.tolist()) == {"stay_id", "patient_id", "unit_type", "admission_reason", "discharge_ready_day11"}

# -----------------------------
# EDA1 prints
# -----------------------------
y = train["discharge_ready_day11"].astype(int).to_numpy()
print("=== EDA1: Shapes ===")
print("stays_train:", stays_train.shape, "| stays_test:", stays_test.shape, "| patients:", patients.shape, "| vitals_records:", len(vit_raw))
print("merged train:", train.shape, "| merged test:", test.shape)

print("\n=== EDA1: Target distribution (train) ===")
counts = pd.Series(y).value_counts().sort_index()
print("counts:", counts.to_dict(), "| pos_rate:", float(y.mean()))

print("\n=== EDA1: Join coverage (missing rates) ===")
join_cov = {
    "patients_missing_rate_train(age)": float(train["age"].isna().mean()),
    "vitals_missing_rate_train(hr_mean)": float(train["hr_mean"].isna().mean()),
    "patients_missing_rate_test(age)": float(test["age"].isna().mean()),
    "vitals_missing_rate_test(hr_mean)": float(test["hr_mean"].isna().mean()),
}
print(join_cov)

print("\n=== EDA1: Cardinalities ===")
for c in ["unit_type", "sex", "insurance", "zip3"]:
    print(f"{c}: unique_train={train[c].nunique(dropna=False)}")

print("\n=== EDA1: Quick numeric stats (age, vitals means) ===")
print(train[["age", "hr_mean", "sbp_mean", "dbp_mean", "temp_c_mean", "rr_mean"]].describe().round(3))

# -----------------------------
# Feature definitions
# -----------------------------
X = train.drop(columns=["discharge_ready_day11"]).copy()

numeric_cols = []
for c in X.columns:
    if c == "age" or c == "note_len" or re.match(r"^(hr|sbp|dbp|temp_c|rr)_(mean|std|min|max|last|slope)$", c):
        numeric_cols.append(c)

categorical_cols = ["unit_type", "sex", "insurance", "zip3"]
text_cols = ["admission_reason", "note_text"]

# Cast types for safety
for c in categorical_cols:
    X[c] = X[c].astype("string").fillna("UNK")
    test[c] = test[c].astype("string").fillna("UNK")
for c in text_cols:
    X[c] = X[c].astype("string").fillna("")
    test[c] = test[c].astype("string").fillna("")
for c in numeric_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce")
    test[c] = pd.to_numeric(test[c], errors="coerce")

# -----------------------------
# Baseline model (deterministic CV)
# -----------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
reason_transformer = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(min_df=2, max_features=5000, ngram_range=(1, 2))),
])
note_transformer = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(min_df=2, max_features=8000, ngram_range=(1, 2))),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", cat_transformer, categorical_cols),
        ("reason", reason_transformer, "admission_reason"),
        ("note", note_transformer, "note_text"),
    ],
    sparse_threshold=0.3,
)

# Use liblinear for stability (often fewer convergence issues for this baseline)
clf = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    random_state=SEED,
)

pipe = Pipeline(steps=[("preprocess", preprocess), ("clf", clf)])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_proba = np.zeros(len(X), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    m = clone(pipe)
    m.fit(X.iloc[tr_idx], y[tr_idx])
    oof_proba[va_idx] = m.predict_proba(X.iloc[va_idx])[:, 1]

thr_grid = np.linspace(0.05, 0.95, 19)
thr_scores = [(float(thr), float(f1_score(y, (oof_proba >= thr).astype(int), average="macro"))) for thr in thr_grid]
best_thr, best_f1 = max(thr_scores, key=lambda t: t[1])

fold_f1 = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    m = clone(pipe)
    m.fit(X.iloc[tr_idx], y[tr_idx])
    proba = m.predict_proba(X.iloc[va_idx])[:, 1]
    pred = (proba >= best_thr).astype(int)
    fold_f1.append(float(f1_score(y[va_idx], pred, average="macro")))

cm = confusion_matrix(y, (oof_proba >= best_thr).astype(int)).tolist()

print("\n=== Baseline CV (deterministic) ===")
print("OOF Macro-F1 (best thr):", round(best_f1, 6), "| best_thr:", best_thr)
print("Per-fold Macro-F1:", [round(x, 6) for x in fold_f1], "| mean:", round(float(np.mean(fold_f1)), 6))
print("OOF Confusion matrix [[TN,FP],[FN,TP]]:", cm)

# -----------------------------
# Fit final model & write submission
# -----------------------------
pipe.fit(X, y)
test_proba = pipe.predict_proba(test[X.columns])[:, 1]
test_pred = (test_proba >= best_thr).astype(int)

submission = pd.DataFrame({
    "stay_id": test["stay_id"].astype(int),
    "discharge_ready_day11": test_pred.astype(int),
})
submission.to_csv(SUBMISSION_NAME, index=False)
print("\nSaved submission:", SUBMISSION_NAME, "| shape:", submission.shape, "| pred_counts:", submission["discharge_ready_day11"].value_counts().to_dict())

# -----------------------------
# Save artifacts + checkpoint bundle
# -----------------------------
# Artifacts
pd.DataFrame(thr_scores, columns=["threshold", "macro_f1_oof"]).to_csv(
    art_root / f"iter{iter_id:04d}_threshold_sweep.csv", index=False
)
pd.DataFrame({"stay_id": train["stay_id"].astype(int), "y_true": y.astype(int), "oof_proba": oof_proba}).to_csv(
    art_root / f"iter{iter_id:04d}_oof_predictions.csv", index=False
)

# Checkpoint
ckpt_dir = ckpt_root / f"iter{iter_id:04d}"
ckpt_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(pipe, ckpt_dir / "model.joblib")

git_hash = get_git_hash()
config = {
    "team": TEAM,
    "task": TASK,
    "version": VX,
    "iteration_id": iter_id,
    "seed": SEED,
    "git_commit": git_hash,
    "features": {
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "text_cols": text_cols,
        "tfidf": {
            "admission_reason": {"min_df": 2, "max_features": 5000, "ngram_range": [1, 2]},
            "note_text": {"min_df": 2, "max_features": 8000, "ngram_range": [1, 2]},
        },
        "vitals_aggregation": "per vital: mean/std/min/max/last/slope over days 1-10",
        "notes_aggregation": "concat day1..day10 notes into note_text; add note_len",
    },
    "model": {
        "name": "LogisticRegression(liblinear,l2,balanced)",
        "params": {"solver": "liblinear", "penalty": "l2", "C": 1.0, "class_weight": "balanced", "max_iter": 2000},
    },
    "validation": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "threshold_search": {"grid": [float(x) for x in thr_grid], "metric": "macro_f1 (OOF)"},
    },
}
metrics = {
    "oof_macro_f1": float(best_f1),
    "best_threshold": float(best_thr),
    "fold_macro_f1": [float(x) for x in fold_f1],
    "confusion_matrix_oof": cm,
    "class_balance": {"n0": int((y == 0).sum()), "n1": int((y == 1).sum()), "pos_rate": float(y.mean())},
}
schema = {
    "X_columns": X.columns.tolist(),
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "text_cols": text_cols,
}

with open(ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
with open(ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
with open(ckpt_dir / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)

# -----------------------------
# Update P* pointer (do not overwrite on regression)
# -----------------------------
pstar_path = ckpt_root / "PSTAR.json"
old_pstar = None
if pstar_path.exists():
    with open(pstar_path, "r", encoding="utf-8") as f:
        old_pstar = json.load(f)

is_new_pstar = (old_pstar is None) or (float(best_f1) > float(old_pstar.get("oof_macro_f1", -1)))
if is_new_pstar:
    new_pstar = {
        "iteration_id": iter_id,
        "oof_macro_f1": float(best_f1),
        "best_threshold": float(best_thr),
        "checkpoint_dir": str(ckpt_dir),
        "timestamp": timestamp,
    }
    with open(pstar_path, "w", encoding="utf-8") as f:
        json.dump(new_pstar, f, indent=2)

print("\nP* updated:", is_new_pstar, "| P* path:", str(pstar_path))

# -----------------------------
# Append iteration detail log (append-only)
# -----------------------------
eda_summary = {
    "train_shape": list(stays_train.shape),
    "test_shape": list(stays_test.shape),
    "patients_shape": list(patients.shape),
    "vitals_records": int(len(vit_raw)),
    "merged_train_shape": list(train.shape),
    "merged_test_shape": list(test.shape),
    "target_counts": {"0": int((y == 0).sum()), "1": int((y == 1).sum())},
    "target_rate": float(y.mean()),
    "patient_id_unique_train": int(stays_train["patient_id"].nunique()),
    "patient_id_unique_test": int(stays_test["patient_id"].nunique()),
    "join_coverage": join_cov,
}

iter_detail = {
    "version": VX,
    "iteration_id": iter_id,
    "timestamp": timestamp,
    "phase": "EDA1",
    "block_id": block_id,
    "block_iter": block_iter,
    "window_branches_plan": {
        "W1": "Vitals trend/variability features",
        "W2": "Text features (notes/reason) tuning",
        "W3": "Model family search (linear SVM/SGD/GBDT if available)",
        "W4": "Imbalance + threshold + calibration",
    },
    "data_paths_used": {k: str(v) for k, v in paths.items()},
    "joins": [
        {"left": "stays_train/stays_test", "right": "patients", "how": "left", "key": ["patient_id"]},
        {"left": "stays_*+patients", "right": "vitals_timeseries (aggregated)", "how": "left", "key": ["stay_id"]},
    ],
    "leakage_checks": [
        "Verified stays_train has only {stay_id, patient_id, unit_type, admission_reason, target}.",
        "Vitals JSON contains only days 1-10 fields: day, hr, sbp, dbp, temp_c, rr, note.",
        "No joins to other tasks (admissions/discharge_notes/ed_cost) in EDA1.",
        "Threshold tuned only on out-of-fold predictions (no test peeking).",
    ],
    "feature_summary": {
        "numeric_feature_count": int(len(numeric_cols)),
        "categorical_feature_count": int(len(categorical_cols)),
        "text_feature_fields": text_cols,
        "tfidf_max_features": {"admission_reason": 5000, "note_text": 8000},
        "notes_aggregation": "concatenate day1..day10 notes into note_text; include note_len numeric",
        "vitals_aggregation": "per vital: mean/std/min/max/last/slope over days 1-10",
    },
    "eda_summary": eda_summary,
    "models_attempted": [
        {
            "name": "LogReg(liblinear) + OHE(cat) + TFIDF(text) + vitals aggregates",
            "params": config["model"]["params"],
            "cv_macro_f1_per_fold": [float(x) for x in fold_f1],
            "cv_macro_f1_mean": float(np.mean(fold_f1)),
            "oof_macro_f1_best_threshold": float(best_f1),
            "notes": "Stable baseline; bounded TF-IDF keeps complexity controlled.",
        }
    ],
    "selected_model": config["model"]["name"],
    "validation_protocol": config["validation"],
    "metrics": {
        "macro_f1_oof": float(best_f1),
        "macro_f1_per_fold": [float(x) for x in fold_f1],
        "confusion_matrix_oof": cm,
        "best_threshold": float(best_thr),
    },
    "pstar_update": {
        "is_new_pstar": bool(is_new_pstar),
        "pstar_path": str(pstar_path),
        "checkpoint_dir": str(ckpt_dir),
    },
    "deltas_vs_previous": "Initialization (no previous iteration).",
    "known_defects_limitations": [
        "Baseline uses aggregate vitals; may miss within-window dynamics.",
        "Two TF-IDF vectorizers create high-dimensional sparse features; keep max_features bounded.",
        "Threshold optimized on OOF may slightly overfit; consider a small internal holdout for thresholding later.",
    ],
    "next_step_hypothesis": "EDA2: inspect note/admission_reason token distributions for artifacts, add delta/volatility features, and try SGD(log_loss) or LinearSVC+calibration under same CV.",
    "hm_message": None,
    "real_score": None,
}

with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(iter_detail) + "\n")

print("Appended iteration log:", str(log_path))

# -----------------------------
# Consultant packet (v3 requirement)
# -----------------------------
consult_packet_path = consult_root / f"{TEAM}_{TASK}_{VX}_iter{iter_id}_packet.json"
consultant_packet = {
    "iteration_id": iter_id,
    "timestamp": timestamp,
    "what_changed": "Initialized v3 pipeline; added vitals aggregation + TF-IDF text; baseline LogReg with deterministic 5-fold CV + OOF threshold sweep.",
    "metric_changes": {"macro_f1_oof": float(best_f1), "best_threshold": float(best_thr)},
    "suspected_leakage_risks": [
        "Daily notes may include discharge planning language (allowed if within days 1-10, but can overfit if templated).",
        "No joins to other tables in EDA1; leakage risk otherwise low.",
    ],
    "complexity_drift_risks": [
        "High-dimensional TF-IDF; keep max_features bounded and monitor runtime/memory.",
    ],
    "evaluation_alignment_risks": [
        "Threshold selected on OOF (not nested); small overfit risk.",
        "If future data adds repeated patient_id across stays, consider GroupKFold; currently train patient_id appears unique.",
    ],
    "unknown_unknowns": [
        "Potential distribution shift in free-text between train/test.",
        "Varying note completeness could act as a proxy for workflow differences.",
    ],
    "recommendations_next_iter": [
        "EDA2 token audit: top n-grams correlated with label; check for templated phrases.",
        "Add day-to-day deltas and volatility (e.g., last-first, mean absolute diff).",
        "Try SGDClassifier(log_loss) for scalability, and/or LinearSVC + calibration for margin-based robustness.",
    ],
}
with open(consult_packet_path, "w", encoding="utf-8") as f:
    json.dump(consultant_packet, f, indent=2)

print("Saved consultant packet:", str(consult_packet_path))

=== EDA1: Shapes ===
stays_train: (1000, 5) | stays_test: (1000, 4) | patients: (4000, 5) | vitals_records: 2000
merged train: (1000, 41) | merged test: (1000, 40)

=== EDA1: Target distribution (train) ===
counts: {0: 344, 1: 656} | pos_rate: 0.656

=== EDA1: Join coverage (missing rates) ===
{'patients_missing_rate_train(age)': 0.0, 'vitals_missing_rate_train(hr_mean)': 0.0, 'patients_missing_rate_test(age)': 0.0, 'vitals_missing_rate_test(hr_mean)': 0.0}

=== EDA1: Cardinalities ===
unit_type: unique_train=2
sex: unique_train=2
insurance: unique_train=3
zip3: unique_train=9

=== EDA1: Quick numeric stats (age, vitals means) ===
            age   hr_mean  sbp_mean  dbp_mean  temp_c_mean   rr_mean
count  1000.000  1000.000  1000.000  1000.000     1000.000  1000.000
mean     53.191    76.690   116.761    69.573       36.857    16.184
std      21.949     9.466    12.463     9.768        0.402     2.997
min      16.000    49.942    81.790    41.284       35.605     6.623
25%      34.000 

## EDA 2
- 0.7768

In [4]:
import os, json, re, shutil, subprocess, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

# =========================
# Identity / Repro settings
# =========================
TEAM = "clai"
TASK = "ch3"
VERSION = "v3"
PHASE = "EDA2"
SEED = 42
np.random.seed(SEED)

warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="All-NaN slice encountered")
warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice")

# =========================
# Helpers
# =========================
def find_file(filename: str) -> Path:
    # Prefer current working directory (project root), fall back to /mnt/data for this environment.
    candidates = [Path.cwd(), Path.cwd().parent, Path("/mnt/data")]
    for c in candidates:
        p = c / filename
        if p.exists():
            return p.resolve()
    for c in candidates:
        if c.exists():
            hits = list(c.rglob(filename))
            if hits:
                return hits[0].resolve()
    raise FileNotFoundError(f"Could not find {filename} in {candidates}")

def safe_git_hash():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return None

def now_iso_tz() -> str:
    return datetime.now().astimezone().isoformat()

def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def is_writable(path: Path) -> bool:
    try:
        if path.exists():
            with open(path, "a", encoding="utf-8"):
                pass
        else:
            path.parent.mkdir(parents=True, exist_ok=True)
            with open(path, "w", encoding="utf-8"):
                pass
            path.unlink(missing_ok=True)
        return True
    except Exception:
        return False

# =========================
# Locate data (read paths)
# =========================
stays_train_path = find_file("stays_train.csv")
data_dir = stays_train_path.parent
stays_test_path = data_dir / "stays_test.csv"
patients_path = data_dir / "patients.csv"
vitals_path = data_dir / "vitals_timeseries.json"

# =========================
# Output (write paths)
# =========================
# Always write outputs into the current working directory (your project root when running locally).
project_dir = Path.cwd()
iter_log_path = project_dir / f"{TEAM}_{TASK}_{VERSION}_iteration_detail.jsonl"
submission_path = project_dir / f"{TEAM}_{TASK}_{VERSION}_submission.csv"

ckpt_root = ensure_dir(project_dir / "checkpoints" / f"{TEAM}_{TASK}_{VERSION}")
art_root  = ensure_dir(project_dir / "artifacts" / f"{TEAM}_{TASK}_{VERSION}")
consult_root = ensure_dir(project_dir / "consultant_packets")

# If a previous log exists alongside the data but not in project_dir, copy it for continuity (best-effort).
data_log_path = data_dir / f"{TEAM}_{TASK}_{VERSION}_iteration_detail.jsonl"
if (not iter_log_path.exists()) and data_log_path.exists():
    ensure_dir(iter_log_path.parent)
    shutil.copy2(data_log_path, iter_log_path)
    print(f"[INFO] Copied existing log from {data_log_path} -> {iter_log_path}")

# Similarly, copy PSTAR.json if it exists next to data checkpoints (best-effort).
data_pstar_path = data_dir / "checkpoints" / f"{TEAM}_{TASK}_{VERSION}" / "PSTAR.json"
proj_pstar_path = ckpt_root / "PSTAR.json"
if (not proj_pstar_path.exists()) and data_pstar_path.exists():
    ensure_dir(proj_pstar_path.parent)
    try:
        shutil.copy2(data_pstar_path, proj_pstar_path)
        print(f"[INFO] Copied existing PSTAR from {data_pstar_path} -> {proj_pstar_path}")
    except Exception:
        pass

# =========================
# Iteration bookkeeping
# =========================
prev = None
iteration_id = 0
block_id = 0
block_iter = 0

if iter_log_path.exists():
    lines = [ln for ln in iter_log_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    if lines:
        prev = json.loads(lines[-1])
        iteration_id = int(prev.get("iteration_id", -1)) + 1
        block_id = int(prev.get("block_id", 0))
        block_iter = int(prev.get("block_iter", -1)) + 1

# v2/v3 multi-window arbitration context: choose an "active branch" for this iteration
active_branch = "W1"  # vitals trend/variability focus per HM/consultant notes
window_branches_plan = {
    "W1": "Vitals dynamics: last-48h windows + trend/volatility features (clinical trajectory focus).",
    "W2": "Text: low-dim note features (keywords / hashing) and ablations vs overfit.",
    "W3": "Model family: nonlinear models on low-dim features (e.g., HistGradientBoosting).",
    "W4": "Imbalance/thresholding: calibration, conservative thresholds, robustness checks.",
}

iter_tag = f"iter{iteration_id:04d}"
ckpt_dir = ensure_dir(ckpt_root / iter_tag)
art_dir  = ensure_dir(art_root  / iter_tag)

# =========================
# Load data
# =========================
stays_train = pd.read_csv(stays_train_path)
stays_test  = pd.read_csv(stays_test_path)
patients    = pd.read_csv(patients_path)

with open(vitals_path, "r", encoding="utf-8") as f:
    vitals_json = json.load(f)

print(f"[INFO] Read data from: {data_dir}")
print(f"[INFO] Write outputs to: {project_dir}")
print(f"[INFO] Train rows: {stays_train.shape} | Test rows: {stays_test.shape} | Patients: {patients.shape} | Vitals records: {len(vitals_json)}")

# =========================
# EDA2 sanity checks
# =========================
bad_days = []
for rec in vitals_json:
    days = [d.get("day") for d in rec.get("days", [])]
    if (len(days) != 10) or (min(days) != 1) or (max(days) != 10):
        bad_days.append((rec.get("stay_id"), days[:]))

all_stay_ids = set(stays_train["stay_id"]).union(set(stays_test["stay_id"]))
vital_stay_ids = set([rec["stay_id"] for rec in vitals_json])

train_pat_ids = set(stays_train["patient_id"])
test_pat_ids  = set(stays_test["patient_id"])

print(f"[CHECK] stays with non-(1..10, len=10) day coverage: {len(bad_days)}")
print(f"[CHECK] Vitals stay_id coverage: {len(vital_stay_ids.intersection(all_stay_ids))}/{len(all_stay_ids)} in (train∪test)")
print(f"[CHECK] patient_id overlap train∩test: {len(train_pat_ids.intersection(test_pat_ids))}")

# Build long vitals dataframe
rows = []
for rec in vitals_json:
    sid = rec["stay_id"]
    for d in rec["days"]:
        rows.append({
            "stay_id": sid,
            "day": int(d.get("day")),
            "hr": d.get("hr"),
            "sbp": d.get("sbp"),
            "dbp": d.get("dbp"),
            "temp_c": d.get("temp_c"),
            "rr": d.get("rr"),
            "note": d.get("note", "")
        })
vdf = pd.DataFrame(rows).sort_values(["stay_id","day"]).reset_index(drop=True)

miss = vdf[["hr","sbp","dbp","temp_c","rr"]].isna().mean().to_dict()
print("[EDA] Missingness (fraction) per vital:", {k: round(v,4) for k,v in miss.items()})
print("[EDA] Target balance (train):", stays_train["discharge_ready_day11"].value_counts().to_dict())

# =========================
# Feature engineering
# =========================
VITAL_VARS = ["hr","sbp","dbp","temp_c","rr"]

def _safe_nanpercentile(x: np.ndarray, q: float):
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.percentile(x, q))

def _safe_nanmean(x: np.ndarray):
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(x.mean())

def _safe_nanmax(x: np.ndarray):
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(x.max())

def _slope(days: np.ndarray, vals: np.ndarray) -> float:
    mask = np.isfinite(days) & np.isfinite(vals)
    if mask.sum() < 2:
        return np.nan
    return float(np.polyfit(days[mask].astype(float), vals[mask].astype(float), 1)[0])

def build_features_baseline(vdf: pd.DataFrame) -> pd.DataFrame:
    # "Flatten 10 days" global aggregates only (reference point), no TF-IDF.
    feats = []
    for sid, g in vdf.groupby("stay_id", sort=False):
        g = g.sort_values("day")
        row = {"stay_id": sid}
        for var in VITAL_VARS:
            vals = g[var].astype(float).values
            days = g["day"].values.astype(float)
            row[f"{var}_mean"]  = _safe_nanmean(vals)
            row[f"{var}_std"]   = float(np.nanstd(vals))
            row[f"{var}_min"]   = float(np.nanmin(vals)) if np.isfinite(vals).any() else np.nan
            row[f"{var}_max"]   = float(np.nanmax(vals)) if np.isfinite(vals).any() else np.nan
            row[f"{var}_last"]  = float(vals[-1]) if np.isfinite(vals[-1]) else np.nan
            row[f"{var}_slope"] = _slope(days, vals)
        notes = g["note"].astype(str)
        row["note_len"] = float(notes.str.len().mean())
        feats.append(row)
    return pd.DataFrame(feats)

def build_features_enhanced(vdf: pd.DataFrame) -> pd.DataFrame:
    # Adds dynamics: last-48h windows, recent deltas, trend, and volatility.
    feats = []
    kws = [
        "stable","improving","worsening","tolerating","pain","oxygen",
        "afebrile","ambulated","walked","pt","diet","voiding","wound",
        "sleep","chair","incentive","spirometer","dvt","insulin"
    ]
    windows = {
        "first2": [1,2],
        "last2":  [9,10],   # last 48h
        "last3":  [8,9,10],
        "last5":  [6,7,8,9,10],
    }
    for sid, g in vdf.groupby("stay_id", sort=False):
        g = g.sort_values("day")
        row = {"stay_id": sid}
        for var in VITAL_VARS:
            vals = g[var].astype(float).values
            days = g["day"].values.astype(float)

            row[f"{var}_mean"]  = _safe_nanmean(vals)
            row[f"{var}_std"]   = float(np.nanstd(vals))
            row[f"{var}_min"]   = float(np.nanmin(vals)) if np.isfinite(vals).any() else np.nan
            row[f"{var}_max"]   = float(np.nanmax(vals)) if np.isfinite(vals).any() else np.nan
            row[f"{var}_range"] = (row[f"{var}_max"] - row[f"{var}_min"]) if np.isfinite(row[f"{var}_max"]) and np.isfinite(row[f"{var}_min"]) else np.nan
            row[f"{var}_p10"]   = _safe_nanpercentile(vals, 10)
            row[f"{var}_p90"]   = _safe_nanpercentile(vals, 90)

            # last values/deltas (guard for NaNs)
            row[f"{var}_d10"] = float(vals[-1]) if np.isfinite(vals[-1]) else np.nan
            row[f"{var}_d9"]  = float(vals[-2]) if np.isfinite(vals[-2]) else np.nan
            row[f"{var}_delta_1d"] = (row[f"{var}_d10"] - row[f"{var}_d9"]) if np.isfinite(row[f"{var}_d10"]) and np.isfinite(row[f"{var}_d9"]) else np.nan
            d8 = float(vals[-3]) if np.isfinite(vals[-3]) else np.nan
            row[f"{var}_delta_2d"] = (row[f"{var}_d10"] - d8) if np.isfinite(row[f"{var}_d10"]) and np.isfinite(d8) else np.nan

            for wname, wdays in windows.items():
                sub = g[g["day"].isin(wdays)][var].astype(float).values
                row[f"{var}_mean_{wname}"] = _safe_nanmean(sub)
                row[f"{var}_std_{wname}"]  = float(np.nanstd(sub))
                row[f"{var}_min_{wname}"]  = float(np.nanmin(sub)) if np.isfinite(sub).any() else np.nan
                row[f"{var}_max_{wname}"]  = float(np.nanmax(sub)) if np.isfinite(sub).any() else np.nan

            # slopes / volatility
            row[f"{var}_slope_all"]   = _slope(days, vals)
            row[f"{var}_slope_last4"] = _slope(days[-4:], vals[-4:])
            row[f"{var}_slope_last3"] = _slope(days[-3:], vals[-3:])

            diffs = np.diff(vals.astype(float))
            row[f"{var}_mad_all"] = _safe_nanmean(np.abs(diffs))
            row[f"{var}_max_abs_diff_all"] = _safe_nanmax(np.abs(diffs))
            last3 = vals[-3:].astype(float)
            row[f"{var}_mad_last3"] = _safe_nanmean(np.abs(np.diff(last3)))

            row[f"{var}_n_missing"] = int(np.isnan(vals).sum())
            row[f"{var}_n_missing_last2"] = int(np.isnan(vals[-2:]).sum())

        # notes: keep low-dim to avoid TF-IDF overfit (previously ~13k features)
        notes = g["note"].astype(str).str.lower()
        row["note_len_mean"]  = float(notes.str.len().mean())
        row["note_len_std"]   = float(notes.str.len().std())
        row["note_len_last2"] = float(notes.iloc[-2:].str.len().mean())
        row["note_len_last3"] = float(notes.iloc[-3:].str.len().mean())
        row["note_unique_days"] = int(notes.nunique())
        for kw in kws:
            pat = rf"\b{re.escape(kw)}\b"
            row[f"kw_{kw}_cnt"]   = int(notes.str.contains(pat).sum())
            row[f"kw_{kw}_last2"] = int(notes.iloc[-2:].str.contains(pat).sum())

        feats.append(row)
    return pd.DataFrame(feats)

feat_base = build_features_baseline(vdf)
feat_enh  = build_features_enhanced(vdf)
print(f"[EDA] Feature tables: baseline={feat_base.shape} enhanced={feat_enh.shape}")

def assemble(stays_df: pd.DataFrame, feat_df: pd.DataFrame) -> pd.DataFrame:
    df = stays_df.merge(patients, on="patient_id", how="left").merge(feat_df, on="stay_id", how="left")
    df["zip3"] = df["zip3"].astype(str)
    return df

train_base = assemble(stays_train, feat_base)
train_enh  = assemble(stays_train, feat_enh)
test_base  = assemble(stays_test,  feat_base)
test_enh   = assemble(stays_test,  feat_enh)

# =========================
# Modeling (still EDA2): deterministic CV + threshold sweep
# =========================
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
import joblib
import sklearn

THRESH_GRID = [round(x, 2) for x in np.linspace(0.05, 0.95, 19)]

def eval_logreg(train_df: pd.DataFrame, C: float, feature_set_name: str) -> dict:
    y = train_df["discharge_ready_day11"].astype(int).values
    X = train_df.drop(columns=["discharge_ready_day11"])
    cat_cols = ["unit_type","admission_reason","sex","insurance","zip3"]
    id_cols = ["stay_id","patient_id"]
    num_cols = [c for c in X.columns if c not in cat_cols + id_cols]

    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ],
        remainder="drop"
    )

    model = LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=float(C),
        class_weight="balanced",
        max_iter=2000,
        random_state=SEED,
    )

    pipe = Pipeline([("preprocess", preprocess), ("model", model)])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_proba = np.zeros(len(X), dtype=float)

    for _, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
        pipe.fit(X.iloc[tr_idx], y[tr_idx])
        oof_proba[va_idx] = pipe.predict_proba(X.iloc[va_idx])[:, 1]

    sweep = []
    best_t, best_f = 0.5, -1.0
    for t in THRESH_GRID:
        pred = (oof_proba >= t).astype(int)
        f = f1_score(y, pred, average="macro")
        sweep.append({"threshold": t, "macro_f1": float(f)})
        if f > best_f:
            best_f, best_t = float(f), float(t)

    oof_pred = (oof_proba >= best_t).astype(int)
    cm = confusion_matrix(y, oof_pred).tolist()

    fold_f1 = []
    for _, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
        pred_fold = (oof_proba[va_idx] >= best_t).astype(int)
        fold_f1.append(float(f1_score(y[va_idx], pred_fold, average="macro")))

    pipe.fit(X, y)

    try:
        Xt = pipe.named_steps["preprocess"].transform(X.head(5))
        n_features = int(Xt.shape[1])
    except Exception:
        n_features = None

    return {
        "feature_set": feature_set_name,
        "C": float(C),
        "oof_macro_f1": float(best_f),
        "best_threshold": float(best_t),
        "fold_macro_f1": fold_f1,
        "confusion_matrix_oof": cm,
        "sweep": sweep,
        "oof_proba": oof_proba,
        "pipeline_fitted": pipe,
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "n_features_after_preprocess": n_features,
    }

models_attempted = []
models_attempted.append(eval_logreg(train_base, C=0.3, feature_set_name="baseline_global_aggs_no_tfidf"))
models_attempted.append(eval_logreg(train_enh,  C=0.1, feature_set_name="enhanced_vitals_windows_trends"))
models_attempted.append(eval_logreg(train_enh,  C=0.3, feature_set_name="enhanced_vitals_windows_trends"))

best_idx = int(np.argmax([m["oof_macro_f1"] for m in models_attempted]))
selected = models_attempted[best_idx]

print("\n[RESULTS] Models attempted (OOF Macro-F1):")
for m in models_attempted:
    print(f"  - {m['feature_set']}, C={m['C']}: macro_f1={m['oof_macro_f1']:.4f} @thr={m['best_threshold']:.2f}, n_feat={m['n_features_after_preprocess']}")
print(f"\n[SELECTED] {selected['feature_set']} with C={selected['C']} | OOF Macro-F1={selected['oof_macro_f1']:.4f} | thr={selected['best_threshold']:.2f}")

# =========================
# Fit selected pipeline on full train and write submission
# =========================
selected_pipe = selected["pipeline_fitted"]
test_df = test_base if selected["feature_set"].startswith("baseline_") else test_enh
X_test = test_df.drop(columns=[c for c in ["discharge_ready_day11"] if c in test_df.columns])

test_proba = selected_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= selected["best_threshold"]).astype(int)

submission = pd.DataFrame({
    "stay_id": stays_test["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred.astype(int),
})
submission.to_csv(submission_path, index=False)
print(f"[WRITE] Submission -> {submission_path} (pos_rate={submission['discharge_ready_day11'].mean():.3f})")

# =========================
# Save checkpoint bundle
# =========================
git_hash = safe_git_hash()
config = {
    "team": TEAM,
    "task": TASK,
    "version": VERSION,
    "phase": PHASE,
    "iteration_id": iteration_id,
    "block_id": block_id,
    "block_iter": block_iter,
    "active_branch": active_branch,
    "seed": SEED,
    "git_commit": git_hash,
    "sklearn_version": getattr(sklearn, "__version__", None),
    "threshold_grid": THRESH_GRID,
    "selected": {"feature_set": selected["feature_set"], "C": selected["C"]},
    "feature_design": {
        "baseline": "per vital mean/std/min/max/last/slope over days 1-10; avg note length",
        "enhanced": "adds last-48h/last-3d/last-5d windows, deltas, slopes (all + last4/last3), volatility, missingness, and low-dim note keyword counts"
    },
}
schema = {
    "cat_cols": selected["cat_cols"],
    "num_cols": selected["num_cols"],
    "n_features_after_preprocess": selected["n_features_after_preprocess"],
}
metrics = {
    "oof_macro_f1": selected["oof_macro_f1"],
    "best_threshold": selected["best_threshold"],
    "fold_macro_f1": selected["fold_macro_f1"],
    "confusion_matrix_oof": selected["confusion_matrix_oof"],
    "class_balance": {
        "n0": int((stays_train["discharge_ready_day11"]==0).sum()),
        "n1": int((stays_train["discharge_ready_day11"]==1).sum()),
        "pos_rate": float(stays_train["discharge_ready_day11"].mean()),
    }
}

with open(ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
with open(ckpt_dir / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)
with open(ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
joblib.dump(selected_pipe, ckpt_dir / "model.joblib")

oof_df = pd.DataFrame({
    "stay_id": stays_train["stay_id"].astype(int).values,
    "y_true": stays_train["discharge_ready_day11"].astype(int).values,
    "oof_proba": selected["oof_proba"],
})
oof_df.to_csv(art_dir / f"{iter_tag}_oof_predictions.csv", index=False)
pd.DataFrame(selected["sweep"]).to_csv(art_dir / f"{iter_tag}_threshold_sweep.csv", index=False)

# =========================
# P* tracking
# =========================
pstar_path = proj_pstar_path
pstar = {"best_iteration_id": None, "best_oof_macro_f1": None, "checkpoint_dir": None, "updated_at": None}
if pstar_path.exists():
    try:
        pstar = json.loads(pstar_path.read_text(encoding="utf-8"))
    except Exception:
        pass

is_new_pstar = False
if (pstar.get("best_oof_macro_f1") is None) or (selected["oof_macro_f1"] > float(pstar["best_oof_macro_f1"])):
    is_new_pstar = True
    pstar = {
        "best_iteration_id": iteration_id,
        "best_oof_macro_f1": selected["oof_macro_f1"],
        "checkpoint_dir": str(ckpt_dir),
        "updated_at": now_iso_tz(),
    }
    with open(pstar_path, "w", encoding="utf-8") as f:
        json.dump(pstar, f, indent=2)
print(f"[P*] {'UPDATED' if is_new_pstar else 'kept'} -> {pstar_path} | best_oof_macro_f1={pstar.get('best_oof_macro_f1')}")

# =========================
# Consultant packet (v3)
# =========================
consult_packet = {
    "team": TEAM,
    "task": TASK,
    "version": VERSION,
    "iteration_id": iteration_id,
    "timestamp": now_iso_tz(),
    "phase": PHASE,
    "active_branch": active_branch,
    "what_changed": [
        "Removed high-dimensional TF-IDF text features from prior iteration to mitigate overfitting on ~1k samples.",
        "Added time-aware vitals features: last-48h (days 9-10) windows, recent deltas, slopes, and volatility.",
        "Kept evaluation deterministic (5-fold StratifiedKFold, seed=42).",
    ],
    "metric_changes": {
        "prev_oof_macro_f1": (prev.get("metrics", {}).get("macro_f1_oof") if prev else None),
        "new_oof_macro_f1": selected["oof_macro_f1"],
        "best_threshold": selected["best_threshold"],
    },
    "suspected_leakage_risks": [
        "Vitals JSON verified to contain only days 1..10; no day 11 present.",
        "No overlap in patient_id between train and test.",
        "No CH1/CH2 artifacts joined in this iteration.",
    ],
    "complexity_drift_risks": [
        f"Feature count after preprocessing ~{selected['n_features_after_preprocess']} (down from ~13k TF-IDF).",
        "Windowed vitals features increase numeric columns; monitor redundancy.",
    ],
    "evaluation_alignment_risks": [
        "Prior CV→LB gap reported (iter0000: 0.715 OOF vs 0.684 LB). Lower dimensionality should reduce overfit, but gap may persist.",
        "OOF threshold search can overfit slightly; consider conservative thresholding or nested CV.",
    ],
    "unknown_unknowns": [
        "Potential distribution shift in latent severity between train/test not captured by current features.",
        "Low-dim note keywords could underfit subtle note signals.",
    ],
    "next_recommendations": [
        "Run repeated-CV stability checks to quantify variance.",
        "Try a nonlinear model on the same low-dim features (e.g., HistGradientBoosting).",
        "Ablate zip3/insurance to test robustness vs spurious correlations.",
    ],
}
consult_path = consult_root / f"{TEAM}_{TASK}_{VERSION}_iter{iteration_id}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2)
print(f"[WRITE] Consultant packet -> {consult_path}")

# =========================
# Iteration detail log (append-only)
# =========================
prev_score = prev.get("metrics", {}).get("macro_f1_oof") if prev else None
deltas = {
    "macro_f1_oof_delta": (float(selected["oof_macro_f1"]) - float(prev_score)) if prev_score is not None else None,
    "feature_dimensionality_change": "TF-IDF (~13k) removed; now ~O(10^2) features after OHE",
}

eda_summary = {
    "vitals_missingness_fraction": {k: float(v) for k, v in miss.items()},
    "bad_day_coverage_count": int(len(bad_days)),
    "train_test_patient_overlap": int(len(train_pat_ids.intersection(test_pat_ids))),
    "vitals_records": int(len(vitals_json)),
    "notes_unique_phrases_total": int(vdf["note"].astype(str).nunique()),
}

iteration_detail = {
    "version": VERSION,
    "iteration_id": iteration_id,
    "timestamp": now_iso_tz(),
    "phase": PHASE,
    "block_id": block_id,
    "block_iter": block_iter,
    "window_branches_plan": window_branches_plan,
    "active_branch": active_branch,
    "data_paths_used": {
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals_timeseries": str(vitals_path),
    },
    "joins": [
        {"left":"stays_*", "right":"patients", "on":"patient_id", "how":"left"},
        {"left":"stays_*+patients", "right":"vitals_features", "on":"stay_id", "how":"left"},
    ],
    "leakage_checks": [
        "Verified vitals_timeseries days are exactly 1..10 for all stays (no day 11).",
        "Confirmed no overlap in patient_id between train and test.",
        "Excluded target and identifiers (stay_id, patient_id) from model features.",
        "No CH1/CH2 artifacts joined in this iteration.",
    ],
    "eda_summary": eda_summary,
    "feature_summary": {
        "baseline_feature_table_shape": list(feat_base.shape),
        "enhanced_feature_table_shape": list(feat_enh.shape),
        "selected_feature_set": selected["feature_set"],
        "n_numeric_cols_pre_model": len(selected["num_cols"]),
        "n_categorical_cols": len(selected["cat_cols"]),
        "n_features_after_preprocess": selected["n_features_after_preprocess"],
        "notes_strategy": "low-dimensional keyword counts + note length stats (no TF-IDF)",
        "vitals_strategy": "global + recent-window (last48h) + trend + volatility + deltas",
    },
    "models_attempted": [
        {
            "model": "LogisticRegression(liblinear,l2,balanced)",
            "feature_set": m["feature_set"],
            "params": {"C": m["C"], "class_weight":"balanced", "max_iter":2000},
            "cv_macro_f1_oof": m["oof_macro_f1"],
            "best_threshold": m["best_threshold"],
            "n_features_after_preprocess": m["n_features_after_preprocess"],
            "notes": None,
        }
        for m in models_attempted
    ],
    "selected_model": {
        "model": "LogisticRegression(liblinear,l2,balanced)",
        "feature_set": selected["feature_set"],
        "params": {"C": selected["C"], "class_weight":"balanced", "max_iter":2000},
    },
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "threshold_search": {"grid": THRESH_GRID, "metric": "macro_f1 (OOF)"},
    },
    "metrics": {
        "macro_f1_oof": selected["oof_macro_f1"],
        "macro_f1_per_fold": selected["fold_macro_f1"],
        "confusion_matrix_oof": selected["confusion_matrix_oof"],
        "best_threshold": selected["best_threshold"],
    },
    "pstar_update": {
        "is_new_pstar": is_new_pstar,
        "pstar_path": str(pstar_path),
        "checkpoint_dir": str(ckpt_dir),
    },
    "deltas_vs_previous": deltas,
    "known_defects_limitations": [
        "Potential validation↔leaderboard gap remains; only LB feedback from iter0000 (0.6845) is known.",
        "Threshold tuned on OOF can overfit slightly; consider conservative thresholding or nested CV.",
        "Only linear model explored here; may miss nonlinear interactions.",
    ],
    "next_step_hypothesis": "After EDA2, begin Modeling iterations: keep enhanced vitals dynamics as base (W1), then explore nonlinear models (W3) and threshold/calibration robustness (W4).",
    "hm_message": "HM reports leaderboard Macro-F1=0.6845 for previous submission; consultant notes: leakage risk from vitals, overfitting with ~13k TF-IDF features on ~1k samples, and need for last-48h trend/volatility features.",
    "real_score": 0.6845,
}

target_log = iter_log_path
if not is_writable(target_log):
    target_log = project_dir / f"{TEAM}_{TASK}_{VERSION}_iteration_detail_fallback.jsonl"
    print(f"[WARN] Primary log not writable; writing to fallback: {target_log}")

with open(target_log, "a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_detail) + "\n")
print(f"[APPEND] Iteration log -> {target_log}")

print(f"[DONE] {TEAM}_{TASK}_{VERSION} {PHASE} iter={iteration_id} | OOF Macro-F1={selected['oof_macro_f1']:.4f}")

[INFO] Read data from: D:\AgentDs\agent_ds_healthcare
[INFO] Write outputs to: d:\AgentDs\agent_ds_healthcare
[INFO] Train rows: (1000, 5) | Test rows: (1000, 4) | Patients: (4000, 5) | Vitals records: 2000
[CHECK] stays with non-(1..10, len=10) day coverage: 0
[CHECK] Vitals stay_id coverage: 2000/2000 in (train∪test)
[CHECK] patient_id overlap train∩test: 0
[EDA] Missingness (fraction) per vital: {'hr': 0.0175, 'sbp': 0.0189, 'dbp': 0.018, 'temp_c': 0.0181, 'rr': 0.0181}
[EDA] Target balance (train): {1: 656, 0: 344}
[EDA] Feature tables: baseline=(2000, 32) enhanced=(2000, 219)

[RESULTS] Models attempted (OOF Macro-F1):
  - baseline_global_aggs_no_tfidf, C=0.3: macro_f1=0.7169 @thr=0.40, n_feat=53
  - enhanced_vitals_windows_trends, C=0.1: macro_f1=0.7468 @thr=0.40, n_feat=240
  - enhanced_vitals_windows_trends, C=0.3: macro_f1=0.7426 @thr=0.45, n_feat=240

[SELECTED] enhanced_vitals_windows_trends with C=0.1 | OOF Macro-F1=0.7468 | thr=0.40
[WRITE] Submission -> d:\AgentDs\agent_d

## Train


In [6]:
import os, json, random, shutil, warnings
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone
import joblib

# ======================
# 0) Reproducibility
# ======================
TEAM = "clai"
TASK = "ch3"
VERSION = "v3"
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice")

# ======================
# 1) Paths + directories
# ======================
def resolve_path(fname: str) -> Path:
    p1 = Path(fname)
    p2 = Path("/mnt/data") / fname
    if p1.exists():
        return p1
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {fname} in cwd or /mnt/data")

data_paths = {
    "stays_train": resolve_path("stays_train.csv"),
    "stays_test": resolve_path("stays_test.csv"),
    "patients": resolve_path("patients.csv"),
    "vitals_timeseries": resolve_path("vitals_timeseries.json"),
}

# Required outputs
submission_path = Path(f"{TEAM}_ch3_{VERSION}_submission.csv")
log_path = Path(f"{TEAM}_ch3_{VERSION}_iteration_detail.jsonl")

# If a prior log exists in /mnt/data but not locally, copy it (append-only continuity)
fallback_log = Path("/mnt/data") / log_path.name
if (not log_path.exists()) and fallback_log.exists():
    shutil.copy2(fallback_log, log_path)

ckpt_root = Path("checkpoints") / f"{TEAM}_ch3_{VERSION}"
art_root = Path("artifacts") / f"{TEAM}_ch3_{VERSION}"
consult_root = Path("consultant_packets")

for d in [ckpt_root, art_root, consult_root]:
    d.mkdir(parents=True, exist_ok=True)

# ======================
# 2) Iteration bookkeeping
# ======================
def get_next_iteration_id(jsonl_path: Path) -> int:
    if not jsonl_path.exists():
        return 0
    max_id = -1
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                max_id = max(max_id, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return max_id + 1

iteration_id = get_next_iteration_id(log_path)
timestamp = datetime.now(ZoneInfo("America/Toronto")).isoformat()

phase = "EDA2"  # mandated second EDA round before full modeling push

# v2/v3 block bookkeeping (recording now; blocks of 5)
block_id = iteration_id // 5
block_iter = iteration_id % 5

# ======================
# 3) Load data
# ======================
stays_train = pd.read_csv(data_paths["stays_train"])
stays_test = pd.read_csv(data_paths["stays_test"])
patients = pd.read_csv(data_paths["patients"])

with open(data_paths["vitals_timeseries"], "r", encoding="utf-8") as f:
    vitals_list = json.load(f)

assert isinstance(vitals_list, list) and len(vitals_list) == (len(stays_train) + len(stays_test)), \
    "Expected vitals_timeseries.json to contain train+test stay_id entries."

# Basic leakage/time checks
all_days_ok = all((isinstance(rec.get("days"), list) and len(rec["days"]) == 10 and
                   [d.get("day") for d in sorted(rec["days"], key=lambda x: x.get("day", 0))] == list(range(1, 11)))
                  for rec in vitals_list)

# ======================
# 4) Feature engineering (EDA2 focus: recent-window dynamics)
# ======================
vital_vars = ["hr", "sbp", "dbp", "temp_c", "rr"]

def safe_nanmean(a):
    a = np.asarray(a, dtype=float)
    return float(np.nanmean(a)) if np.isfinite(a).any() else np.nan

def safe_nanstd(a):
    a = np.asarray(a, dtype=float)
    return float(np.nanstd(a)) if np.isfinite(a).any() else np.nan

def safe_nanmin(a):
    a = np.asarray(a, dtype=float)
    return float(np.nanmin(a)) if np.isfinite(a).any() else np.nan

def safe_nanmax(a):
    a = np.asarray(a, dtype=float)
    return float(np.nanmax(a)) if np.isfinite(a).any() else np.nan

def compute_slope(vals, days):
    y = np.asarray(vals, dtype=float)
    x = np.asarray(days, dtype=float)
    m = np.isfinite(y) & np.isfinite(x)
    if m.sum() < 2:
        return np.nan
    x = x[m]; y = y[m]
    x_c = x - x.mean()
    denom = float(np.sum(x_c ** 2))
    if denom == 0:
        return 0.0
    return float(np.sum(x_c * (y - y.mean())) / denom)

# Simple clinical-threshold counters (robust, low-dimensional)
THRESH = {
    "hr_high": lambda x: x > 100,
    "hr_low":  lambda x: x < 60,
    "sbp_low": lambda x: x < 90,
    "sbp_high": lambda x: x > 160,
    "temp_fever": lambda x: x >= 38.0,
    "rr_high": lambda x: x > 20,
}

def vitals_to_features(vlist):
    rows = []
    for rec in vlist:
        stay_id = rec["stay_id"]
        days = sorted(rec["days"], key=lambda d: d["day"])
        day_idx = [d["day"] for d in days]

        notes = [(d.get("note") or "") for d in days]
        note_text_all = " ".join(notes).strip()
        note_text_recent = " ".join(notes[-2:]).strip()

        feat = {
            "stay_id": stay_id,
            "note_text_all": note_text_all,
            "note_text_recent": note_text_recent,
            "note_len_all": len(note_text_all),
            "note_len_recent": len(note_text_recent),
        }

        # Vitals numeric time-series features
        for v in vital_vars:
            vals = [d.get(v) for d in days]
            arr = np.array([np.nan if x is None else float(x) for x in vals], dtype=float)

            feat[f"{v}_mean"] = safe_nanmean(arr)
            feat[f"{v}_std"] = safe_nanstd(arr)
            feat[f"{v}_min"] = safe_nanmin(arr)
            feat[f"{v}_max"] = safe_nanmax(arr)

            feat[f"{v}_first"] = float(arr[0]) if np.isfinite(arr[0]) else np.nan
            feat[f"{v}_last"]  = float(arr[-1]) if np.isfinite(arr[-1]) else np.nan
            feat[f"{v}_delta_10"] = (feat[f"{v}_last"] - feat[f"{v}_first"]
                                    if (np.isfinite(feat[f"{v}_last"]) and np.isfinite(feat[f"{v}_first"]))
                                    else np.nan)

            feat[f"{v}_slope_10"] = compute_slope(arr, day_idx)

            # Recent-window (last 48h ~= days 9-10, and last 72h ~= days 8-10)
            last2 = arr[-2:]
            last3 = arr[-3:]
            early8 = arr[:-2]

            feat[f"{v}_mean_last2"] = safe_nanmean(last2)
            feat[f"{v}_std_last2"] = safe_nanstd(last2)
            feat[f"{v}_delta_last2"] = (float(last2[-1] - last2[-2])
                                       if (np.isfinite(last2[-1]) and np.isfinite(last2[-2]))
                                       else np.nan)

            feat[f"{v}_mean_last3"] = safe_nanmean(last3)
            feat[f"{v}_std_last3"] = safe_nanstd(last3)
            feat[f"{v}_slope_last3"] = compute_slope(last3, day_idx[-3:])

            feat[f"{v}_mean_early8"] = safe_nanmean(early8)
            feat[f"{v}_diff_last2_early8"] = (feat[f"{v}_mean_last2"] - feat[f"{v}_mean_early8"]
                                              if (np.isfinite(feat[f"{v}_mean_last2"]) and np.isfinite(feat[f"{v}_mean_early8"]))
                                              else np.nan)

            feat[f"{v}_missing_ct"] = int(np.isnan(arr).sum())
            feat[f"{v}_missing_last2"] = int(np.isnan(last2).sum())

        # Threshold-exceedance counts (overall + last2)
        for key, fn in THRESH.items():
            base_vital = key.split("_")[0]  # hr/sbp/temp/rr
            if base_vital == "temp":
                series = np.array([np.nan if d.get("temp_c") is None else float(d.get("temp_c")) for d in days], dtype=float)
            else:
                series = np.array([np.nan if d.get(base_vital) is None else float(d.get(base_vital)) for d in days], dtype=float)

            mask = np.isfinite(series)
            overall = int(np.sum(fn(series[mask]))) if mask.any() else 0
            last2 = series[-2:]
            mask2 = np.isfinite(last2)
            last2_ct = int(np.sum(fn(last2[mask2]))) if mask2.any() else 0

            feat[f"{key}_ct"] = overall
            feat[f"{key}_ct_last2"] = last2_ct

        rows.append(feat)

    return pd.DataFrame(rows)

vitals_df = vitals_to_features(vitals_list)

# Joins
train_df = (stays_train
            .merge(patients, on="patient_id", how="left", validate="one_to_one")
            .merge(vitals_df, on="stay_id", how="left", validate="one_to_one"))

test_df  = (stays_test
            .merge(patients, on="patient_id", how="left", validate="one_to_one")
            .merge(vitals_df, on="stay_id", how="left", validate="one_to_one"))

assert train_df.shape[0] == stays_train.shape[0] and test_df.shape[0] == stays_test.shape[0], "Join changed row counts."

# ======================
# 5) EDA2 quick readouts
# ======================
y = train_df["discharge_ready_day11"].astype(int).values
pos_rate = float(np.mean(y))
print(f"[EDA2] Train rows={len(train_df)}  Test rows={len(test_df)}  PosRate={pos_rate:.3f}")
print(f"[EDA2] Vitals days check (all stays have days 1..10): {all_days_ok}")

miss = train_df.isna().mean().sort_values(ascending=False).head(8)
print("[EDA2] Top missingness columns:")
print(miss.to_string())

for v in ["hr", "sbp", "temp_c", "rr"]:
    col = f"{v}_mean_last2"
    if col in train_df.columns:
        m1 = float(train_df.loc[y==1, col].mean())
        m0 = float(train_df.loc[y==0, col].mean())
        print(f"[EDA2] {col}: mean(y=1)={m1:.3f} vs mean(y=0)={m0:.3f} (diff={m1-m0:+.3f})")

# ======================
# 6) Deterministic CV evaluation (two candidates)
# ======================
X = train_df.drop(columns=["discharge_ready_day11"])

cat_cols = ["unit_type", "sex", "insurance", "zip3"]
text_reason_col = "admission_reason"
text_note_col   = "note_text_all"

excluded = set(["stay_id", "patient_id"] + cat_cols + [text_reason_col, text_note_col, "note_text_recent"])
num_cols = [c for c in X.columns if c not in excluded]

threshold_grid = [round(x, 2) for x in np.linspace(0.05, 0.95, 19)]

def build_pipeline(penalty: str, C: float):
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
            ("reason", TfidfVectorizer(min_df=1, max_features=5000, ngram_range=(1, 1)), text_reason_col),
            ("note",   TfidfVectorizer(min_df=2, max_features=8000, ngram_range=(1, 2)), text_note_col),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )

    clf = LogisticRegression(
        solver="liblinear",
        penalty=penalty,
        C=C,
        class_weight="balanced",
        max_iter=4000,
        random_state=SEED
    )

    return Pipeline([("pre", pre), ("clf", clf)])

def cv_oof(pipeline: Pipeline):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X), dtype=float)
    fold_val_idx = []
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
        m = clone(pipeline)
        m.fit(X.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict_proba(X.iloc[va_idx])[:, 1]
        fold_val_idx.append((fold, va_idx))

    sweep = []
    for thr in threshold_grid:
        pred = (oof >= thr).astype(int)
        sweep.append((thr, f1_score(y, pred, average="macro")))
    best_thr, best_f1 = max(sweep, key=lambda x: x[1])

    fold_f1 = []
    for fold, va_idx in fold_val_idx:
        fold_f1.append(float(f1_score(y[va_idx], (oof[va_idx] >= best_thr).astype(int), average="macro")))
    cm = confusion_matrix(y, (oof >= best_thr).astype(int))
    return float(best_f1), float(best_thr), fold_f1, cm.tolist(), oof, sweep

candidates = [
    {"name": "LR_liblinear_l2_C0.1_balanced", "penalty": "l2", "C": 0.1},
    {"name": "LR_liblinear_l1_C0.1_balanced", "penalty": "l1", "C": 0.1},
]

models_attempted = []
best = None

for cand in candidates:
    pipe = build_pipeline(penalty=cand["penalty"], C=cand["C"])
    macro_f1, best_thr, fold_f1, cm, oof, sweep = cv_oof(pipe)

    models_attempted.append({
        "name": cand["name"],
        "params": {"solver": "liblinear", "penalty": cand["penalty"], "C": cand["C"], "class_weight": "balanced", "max_iter": 4000},
        "cv_macro_f1": macro_f1,
        "cv_fold_macro_f1": fold_f1,
        "best_threshold_oof": best_thr,
        "confusion_matrix_oof": cm
    })

    print(f"[CV] {cand['name']}  oof_macroF1={macro_f1:.4f}  best_thr={best_thr:.2f}")

    if (best is None) or (macro_f1 > best["macro_f1"]):
        best = {
            "candidate": cand,
            "pipeline": pipe,
            "macro_f1": macro_f1,
            "best_thr": best_thr,
            "fold_f1": fold_f1,
            "cm": cm,
            "oof": oof,
            "sweep": sweep
        }

selected_model_name = best["candidate"]["name"]
selected_threshold = best["best_thr"]

# ======================
# 7) Fit final model + write submission
# ======================
final_model = build_pipeline(penalty=best["candidate"]["penalty"], C=best["candidate"]["C"])
final_model.fit(X, y)

Xt = final_model.named_steps["pre"].transform(X)
n_features_total = int(Xt.shape[1])
coef = final_model.named_steps["clf"].coef_
n_nonzero = int(np.count_nonzero(coef))

print(f"[Fit] Total engineered features after preprocessing: {n_features_total}")
print(f"[Fit] Non-zero LR coefficients: {n_nonzero} (L1 sparsity helps control overfitting)")

proba_test = final_model.predict_proba(test_df)[:, 1]
pred_test = (proba_test >= selected_threshold).astype(int)

sub = pd.DataFrame({"stay_id": test_df["stay_id"].astype(int), "discharge_ready_day11": pred_test.astype(int)})
sub.to_csv(submission_path, index=False)
print(f"[Output] Wrote submission: {submission_path.resolve()}")

# Also save a branch candidate submission (v2 mechanics)
candidate_path = art_root / f"{TEAM}_ch3_{VERSION}_W1_block{block_id}_candidate.csv"
sub.to_csv(candidate_path, index=False)

# ======================
# 8) Save artifacts + checkpoint bundle
# ======================
iter_tag = f"iter{iteration_id:04d}"
iter_art_dir = art_root / iter_tag
iter_ckpt_dir = ckpt_root / iter_tag
iter_art_dir.mkdir(parents=True, exist_ok=True)
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

sweep_df = pd.DataFrame(best["sweep"], columns=["threshold", "macro_f1"])
oof_df = pd.DataFrame({
    "stay_id": train_df["stay_id"].astype(int),
    "y_true": y.astype(int),
    "oof_proba": best["oof"].astype(float)
})

sweep_path = iter_art_dir / f"{iter_tag}_threshold_sweep.csv"
oof_path = iter_art_dir / f"{iter_tag}_oof_predictions.csv"
sweep_df.to_csv(sweep_path, index=False)
oof_df.to_csv(oof_path, index=False)

model_path = iter_ckpt_dir / f"{iter_tag}_model.joblib"
joblib.dump(final_model, model_path)

schema = {
    "X_columns": list(train_df.drop(columns=["discharge_ready_day11"]).columns),
    "numeric_cols": num_cols,
    "categorical_cols": cat_cols,
    "text_cols": [text_reason_col, text_note_col],
}
schema_path = iter_ckpt_dir / f"{iter_tag}_schema.json"
schema_path.write_text(json.dumps(schema, indent=2), encoding="utf-8")

config = {
    "team": TEAM,
    "task": TASK,
    "version": VERSION,
    "iteration_id": iteration_id,
    "seed": SEED,
    "phase": phase,
    "features": {
        "numeric_cols_count": len(num_cols),
        "categorical_cols": cat_cols,
        "text_cols": [text_reason_col, text_note_col],
        "vitals_aggregation": "global + recent-window (last2/last3) + slopes + missingness + threshold counts",
        "notes_aggregation": "concat day1..day10 notes into note_text_all; also store note_text_recent + lengths",
        "tfidf": {
            "admission_reason": {"min_df": 1, "max_features": 5000, "ngram_range": [1, 1]},
            "note_text_all": {"min_df": 2, "max_features": 8000, "ngram_range": [1, 2]},
        },
    },
    "model": {
        "name": selected_model_name,
        "params": {"solver": "liblinear", "penalty": best["candidate"]["penalty"], "C": best["candidate"]["C"],
                   "class_weight": "balanced", "max_iter": 4000}
    },
    "validation": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": SEED,
        "threshold_search": {"grid": threshold_grid, "metric": "macro_f1 (OOF)"}
    }
}
config_path = iter_ckpt_dir / f"{iter_tag}_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

metrics = {
    "oof_macro_f1": best["macro_f1"],
    "best_threshold": selected_threshold,
    "fold_macro_f1": best["fold_f1"],
    "confusion_matrix_oof": best["cm"],
    "class_balance": {
        "n0": int(np.sum(y == 0)),
        "n1": int(np.sum(y == 1)),
        "pos_rate": float(np.mean(y)),
    },
    "feature_space": {
        "n_features_total": n_features_total,
        "n_nonzero_coef": n_nonzero
    }
}
metrics_path = iter_ckpt_dir / f"{iter_tag}_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

# ======================
# 9) P* tracking (best-known by deterministic validation)
# ======================
pstar_path = ckpt_root / "pstar.json"
prev_pstar = None
if pstar_path.exists():
    try:
        prev_pstar = json.loads(pstar_path.read_text(encoding="utf-8"))
    except Exception:
        prev_pstar = None

pstar_update = {"updated": False, "previous": prev_pstar}
if (prev_pstar is None) or (best["macro_f1"] > float(prev_pstar.get("oof_macro_f1", -1))):
    new_pstar = {
        "iteration_id": iteration_id,
        "oof_macro_f1": best["macro_f1"],
        "best_threshold": selected_threshold,
        "checkpoint_dir": str(iter_ckpt_dir),
        "model_path": str(model_path),
        "timestamp": timestamp,
    }
    pstar_path.write_text(json.dumps(new_pstar, indent=2), encoding="utf-8")
    pstar_update["updated"] = True
    pstar_update["new"] = new_pstar

# ======================
# 10) Consultant packet (v3 requirement)
# ======================
consult_packet = {
    "version": VERSION,
    "iteration_id": iteration_id,
    "timestamp": timestamp,
    "phase": phase,
    "what_changed": [
        "EDA2: engineered recent-window vital dynamics (last2/last3 means/std/deltas/slopes) to better reflect last 48h trajectory.",
        "Added low-dimensional abnormal-vital count features (tachycardia, hypotension, fever, tachypnea).",
        "Switched to stronger regularization (liblinear + L1, C=0.1) to reduce effective feature dimensionality and mitigate overfitting.",
        "Kept deterministic 5-fold StratifiedKFold and OOF threshold sweep."
    ],
    "metric_changes": {
        "oof_macro_f1": best["macro_f1"],
        "best_threshold": selected_threshold,
        "fold_macro_f1": best["fold_f1"],
    },
    "suspected_leakage_risks": [
        "No discharge_notes.json used. Only days 1..10 from vitals_timeseries.json used (checked).",
        "Text features may still overfit if note phrasing shifts between train/test; monitor."
    ],
    "complexity_drift_risks": [
        f"Feature space after preprocessing: {n_features_total}; L1 sparsity nonzero={n_nonzero}."
    ],
    "evaluation_alignment_risks": [
        "OOF threshold chosen to maximize CV macro-F1 may not transfer perfectly to leaderboard distribution; consider threshold robustness / calibration."
    ],
    "unknown_unknowns": [
        "Potential dataset shift between train/test note phrasing or vital distributions may still cause val vs LB gap."
    ],
    "recommendations_next_iter": [
        "Try restricting TF-IDF to last 2–3 days of notes (recent clinical status) + compare against full-notes TF-IDF.",
        "Try Elastic-Net via saga solver (if stable) to balance sparsity and shrinkage.",
        "Consider calibration (Platt/Isotonic) or choose threshold by stability across folds rather than global OOF optimum."
    ],
}
consult_path = consult_root / f"{TEAM}_ch3_{VERSION}_iter{iteration_id}_packet.json"
consult_path.write_text(json.dumps(consult_packet, indent=2), encoding="utf-8")

# ======================
# 11) Append iteration log (append-only)
# ======================
hm_message = (
    "HM reported real leaderboard Macro-F1=0.7768 for the prior submission; "
    "HM/Consultant noted (a) potential leakage risk from vitals-derived text features, "
    "(b) overfitting risk from ~13k sparse features vs ~1k training rows, "
    "and (c) global 10-day aggregation may mask last-48h deterioration/improvement dynamics."
)
real_score = 0.7768

iteration_log = {
    "version": VERSION,
    "iteration_id": iteration_id,
    "timestamp": timestamp,
    "phase": phase,
    "block_id": block_id,
    "block_iter": block_iter,
    "data_paths_used": {k: str(v) for k, v in data_paths.items()},
    "joins": [
        {"left": "stays_[train|test].csv", "right": "patients.csv", "key": "patient_id", "type": "one_to_one"},
        {"left": "stays_[train|test].csv", "right": "vitals_timeseries.json(features)", "key": "stay_id", "type": "one_to_one"},
    ],
    "leakage_checks": [
        "Verified vitals_timeseries.json has exactly days 1..10 per stay_id (no day 11+ fields).",
        "Did NOT use discharge_notes.json or any post-discharge summaries.",
        "Patient_id unique per stay in provided stays_train/test (no cross-fold patient leakage).",
    ],
    "feature_summary": {
        "numeric_features": {
            "count": len(num_cols),
            "description": "patients demographics + vitals global aggregates + recent-window (last2/last3) stats + slopes + missingness + threshold counts + note lengths"
        },
        "categorical_features": {"cols": cat_cols, "encoding": "OneHotEncoder(handle_unknown='ignore')"},
        "text_features": {
            "cols": [text_reason_col, text_note_col],
            "encoding": "TfidfVectorizer",
            "params": config["features"]["tfidf"]
        },
        "feature_space_after_preprocessing": {"n_features_total": n_features_total, "n_nonzero_coef": n_nonzero}
    },
    "models_attempted": models_attempted,
    "selected_model": selected_model_name,
    "validation_protocol": config["validation"],
    "metrics": metrics,
    "deltas_vs_previous": {
        "from_iter": iteration_id - 1 if iteration_id > 0 else None,
        "changes": [
            "Added last48h/last72h vital dynamics features (means/std/deltas/slopes) + abnormal-vital counts.",
            "Regularization tightened (C=0.1) and explored L1 vs L2; selected best by deterministic OOF macro-F1.",
        ],
    },
    "known_defects_limitations": [
        "Still uses sparse TF-IDF text features; even with L1 sparsity, monitor for dataset-shift-driven val/LB gap.",
        "Threshold chosen on OOF may be brittle; consider stability-based thresholding or calibration in next iterations."
    ],
    "next_step_hypothesis": "Restricting note text to the last 2–3 days + adding calibration/stable-threshold selection could reduce val vs leaderboard mismatch while preserving performance.",
    "pstar_update": pstar_update,
    "hm_message": hm_message,
    "real_score": real_score,
}

with log_path.open("a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_log, ensure_ascii=False) + "\n")

print(f"[Log] Appended iteration detail to: {log_path.resolve()}")
print(f"[Consultant] Wrote consultant packet: {consult_path.resolve()}")
print(f"[Checkpoint] Saved to: {iter_ckpt_dir.resolve()}")
print(f"[Artifacts] Saved to: {iter_art_dir.resolve()}")

[EDA2] Train rows=1000  Test rows=1000  PosRate=0.656
[EDA2] Vitals days check (all stays have days 1..10): True
[EDA2] Top missingness columns:
sbp_delta_last2       0.049
rr_delta_last2        0.045
temp_c_delta_last2    0.045
hr_delta_last2        0.042
dbp_delta_last2       0.037
hr_delta_10           0.026
hr_last               0.026
rr_last               0.024
[EDA2] hr_mean_last2: mean(y=1)=77.261 vs mean(y=0)=77.290 (diff=-0.029)
[EDA2] sbp_mean_last2: mean(y=1)=118.147 vs mean(y=0)=112.998 (diff=+5.149)
[EDA2] temp_c_mean_last2: mean(y=1)=36.904 vs mean(y=0)=36.830 (diff=+0.074)
[EDA2] rr_mean_last2: mean(y=1)=15.880 vs mean(y=0)=17.369 (diff=-1.489)
[CV] LR_liblinear_l2_C0.1_balanced  oof_macroF1=0.7460  best_thr=0.50
[CV] LR_liblinear_l1_C0.1_balanced  oof_macroF1=0.7629  best_thr=0.45
[Fit] Total engineered features after preprocessing: 2745
[Fit] Non-zero LR coefficients: 40 (L1 sparsity helps control overfitting)
[Output] Wrote submission: D:\AgentDs\agent_ds_healthcare\c

# Iteration 2
- 0.7041

In [8]:
import os, json, random, datetime, pathlib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.base import clone
import joblib

# =========================
# Config
# =========================
TEAM = "clai"
TASK = "ch3"
VX = "v3"
RUN_TAG = f"{TEAM}_{TASK}_{VX}"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

BASE = pathlib.Path(".")
LOG_PATH = BASE / f"{RUN_TAG}_iteration_detail.jsonl"

CKPT_ROOT = BASE / "checkpoints" / RUN_TAG
ART_ROOT = BASE / "artifacts" / RUN_TAG
CONSULT_ROOT = BASE / "consultant_packets"
for d in [CKPT_ROOT, ART_ROOT, CONSULT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

# HM message provided in prompt (record in iteration_detail.jsonl)
HM_MESSAGE = (
    "HM update: latest real Macro-F1=0.7589 (regressed). Previous best real Macro-F1=0.7768. "
    "Consultant notes: (1) if using LogisticRegression L1, increase C to avoid over-sparsifying; "
    "(2) consider non-linear tree models (HistGradientBoosting/LightGBM) now that CV↔LB alignment is good; "
    "(3) keep StratifiedKFold(n_splits=5, shuffle=True, random_state=42) unchanged."
)
REAL_SCORE = 0.7589

# =========================
# Helpers
# =========================
def now_et_iso():
    # America/Toronto is UTC-5 on 2026-03-01 (per instruction)
    return datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=-5))).isoformat()

def read_jsonl(path: pathlib.Path):
    out = []
    if path.exists() and path.stat().st_size > 0:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    out.append(json.loads(line))
                except Exception:
                    # keep append-only robust; do not crash on a malformed line
                    continue
    return out

def find_file(filename: str, roots):
    for r in roots:
        p = pathlib.Path(r) / filename
        if p.exists():
            return p
    # fallback: recursive search only within current directory
    for p in pathlib.Path(".").rglob(filename):
        return p
    raise FileNotFoundError(f"Could not find {filename} in roots={roots}")

def select_first_col(X):
    # picklable (no lambda) for joblib
    if hasattr(X, "iloc"):
        s = X.iloc[:, 0]
    else:
        s = pd.Series(np.asarray(X).ravel())
    return s.fillna("").astype(str)

def make_text_vectorizer(max_features: int):
    return Pipeline(steps=[
        ("select", FunctionTransformer(select_first_col, validate=False)),
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            min_df=2,
            max_features=max_features,
            ngram_range=(1, 2)
        ))
    ])

def safe_nan_stats(arr: np.ndarray):
    arr = np.asarray(arr, dtype=float)
    if np.all(np.isnan(arr)):
        return dict(mean=np.nan, std=np.nan, min=np.nan, max=np.nan)
    return dict(
        mean=float(np.nanmean(arr)),
        std=float(np.nanstd(arr)),
        min=float(np.nanmin(arr)),
        max=float(np.nanmax(arr)),
    )

def compute_series_features(days_list, key: str):
    # days_list: list of dicts with day 1..10
    vals = []
    day_idx = []
    for d in days_list:
        day_idx.append(float(d.get("day", np.nan)))
        v = d.get(key, None)
        vals.append(np.nan if v is None else float(v))
    vals = np.array(vals, dtype=float)
    day_idx = np.array(day_idx, dtype=float)

    out = {}
    overall = safe_nan_stats(vals)
    out[f"{key}_mean"] = overall["mean"]
    out[f"{key}_std"] = overall["std"]
    out[f"{key}_min"] = overall["min"]
    out[f"{key}_max"] = overall["max"]
    out[f"{key}_last"] = float(vals[-1]) if not np.isnan(vals[-1]) else np.nan

    # slope over days1-10
    mask = ~np.isnan(vals)
    out[f"{key}_slope"] = float(np.polyfit(day_idx[mask], vals[mask], 1)[0]) if mask.sum() >= 2 else np.nan

    # early (1-8) and last2 (9-10)
    early = vals[:8]
    recent = vals[8:]
    early_stats = safe_nan_stats(early)
    recent_stats = safe_nan_stats(recent)

    out[f"{key}_early_mean"] = early_stats["mean"]
    out[f"{key}_early_std"] = early_stats["std"]
    out[f"{key}_last2_mean"] = recent_stats["mean"]
    out[f"{key}_last2_std"] = recent_stats["std"]
    out[f"{key}_last2_min"] = recent_stats["min"]
    out[f"{key}_last2_max"] = recent_stats["max"]

    # delta day10 - day9
    if np.isnan(vals[-1]) or np.isnan(vals[-2]):
        out[f"{key}_delta10_9"] = np.nan
        out[f"{key}_absdelta10_9"] = np.nan
    else:
        out[f"{key}_delta10_9"] = float(vals[-1] - vals[-2])
        out[f"{key}_absdelta10_9"] = float(abs(vals[-1] - vals[-2]))

    # slope last 3 days (8-10)
    last3 = vals[7:]
    last3_days = day_idx[7:]
    mask3 = ~np.isnan(last3)
    out[f"{key}_slope_last3"] = float(np.polyfit(last3_days[mask3], last3[mask3], 1)[0]) if mask3.sum() >= 2 else np.nan

    # recent-vs-early delta
    out[f"{key}_last2_minus_early"] = float(out[f"{key}_last2_mean"] - out[f"{key}_early_mean"]) if (not np.isnan(out[f"{key}_last2_mean"]) and not np.isnan(out[f"{key}_early_mean"])) else np.nan

    # missingness
    out[f"{key}_miss_cnt_10d"] = int(np.isnan(vals).sum())
    out[f"{key}_miss_cnt_last2"] = int(np.isnan(recent).sum())
    return out

def extract_note_text(days_list, which="all"):
    if which == "all":
        sel = days_list
    elif which == "last2":
        sel = days_list[8:]  # days 9-10
    else:
        raise ValueError(which)
    notes = []
    for d in sel:
        n = d.get("note", None)
        if n is None:
            continue
        notes.append(str(n))
    return " ".join(notes).strip()

# Keywords (small numeric features, not huge sparse drift)
KEYWORDS = ["discharg", "home", "rehab", "placement", "stable", "improv", "worsen", "fever", "oxygen", "abx", "antibiot", "pt", "ot", "walk", "pain", "await"]
def keyword_counts(text: str):
    t = (text or "").lower()
    return {f"kw_{kw}": int(t.count(kw)) for kw in KEYWORDS}

# =========================
# Iteration bookkeeping
# =========================
entries = read_jsonl(LOG_PATH)
last_iter = max([e.get("iteration_id", -1) for e in entries], default=-1)
iteration_id = last_iter + 1
block_id = iteration_id // 5
block_iter = iteration_id % 5

done_phases = set([e.get("phase") for e in entries])
if "EDA1" not in done_phases:
    phase = "EDA1"
elif "EDA2" not in done_phases:
    phase = "EDA2"
else:
    phase = "Modeling"

# This response is explicitly for EDA2; we keep the exploration theme in W1.
branch = "W1"

window_plan_default = {
    "W1": "Vitals trend/variability features",
    "W2": "Text features (notes/reason) tuning",
    "W3": "Model family search (linear SVM/SGD/GBDT if available)",
    "W4": "Imbalance + threshold + calibration",
}
window_plan = None
if entries and isinstance(entries[0], dict) and "window_branches_plan" in entries[0]:
    window_plan = entries[0]["window_branches_plan"]
else:
    window_plan = window_plan_default

ckpt_dir = CKPT_ROOT / f"iter{iteration_id:04d}"
ckpt_dir.mkdir(parents=True, exist_ok=True)

print(f"[{RUN_TAG}] iteration_id={iteration_id} block_id={block_id} block_iter={block_iter} phase={phase} branch={branch}")

# =========================
# Load data
# =========================
roots = [BASE, BASE / "data"]
# add Windows project hint dirs + this sandbox dir
for r in ["D:/AgentDs/agent_ds_healthcare", "d:/AgentDs/agent_ds_healthcare", "/mnt/data"]:
    roots.append(pathlib.Path(r))

stays_train_path = find_file("stays_train.csv", roots)
stays_test_path  = find_file("stays_test.csv", roots)
patients_path    = find_file("patients.csv", roots)
vitals_path      = find_file("vitals_timeseries.json", roots)

stays_train = pd.read_csv(stays_train_path)
stays_test  = pd.read_csv(stays_test_path)
patients    = pd.read_csv(patients_path)
with open(vitals_path, "r", encoding="utf-8") as f:
    vitals = json.load(f)

print("Loaded:",
      f"stays_train={stays_train.shape}",
      f"stays_test={stays_test.shape}",
      f"patients={patients.shape}",
      f"vitals_records={len(vitals)}")

# Basic leakage sanity: vitals day range
day_min = min(d["day"] for rec in vitals for d in rec["days"])
day_max = max(d["day"] for rec in vitals for d in rec["days"])
print(f"Vitals day range: [{day_min}, {day_max}]")

# =========================
# Feature Engineering (EDA2 focus: last-48h dynamics)
# =========================
vitals_map = {int(rec["stay_id"]): rec["days"] for rec in vitals}

def build_vitals_features(stays_df: pd.DataFrame):
    rows = []
    for sid in stays_df["stay_id"].astype(int).tolist():
        days = vitals_map.get(sid, [])
        feats = {}
        for vital in ["hr", "sbp", "dbp", "temp_c", "rr"]:
            feats.update(compute_series_features(days, vital))

        note_all = extract_note_text(days, "all")
        note_last2 = extract_note_text(days, "last2")
        feats["note_text"] = note_all
        feats["note_last2_text"] = note_last2
        feats["note_len"] = int(len(note_all))
        feats["note_last2_len"] = int(len(note_last2))
        feats.update(keyword_counts(note_last2))
        rows.append(feats)

    df = pd.DataFrame(rows)
    df.insert(0, "stay_id", stays_df["stay_id"].values)
    return df

train_vitals = build_vitals_features(stays_train)
test_vitals  = build_vitals_features(stays_test)

train_df = stays_train.merge(patients, on="patient_id", how="left").merge(train_vitals, on="stay_id", how="left")
test_df  = stays_test.merge(patients, on="patient_id", how="left").merge(test_vitals,  on="stay_id", how="left")

target_col = "discharge_ready_day11"
id_col = "stay_id"

y = train_df[target_col].astype(int).values
target_counts = pd.Series(y).value_counts().sort_index().to_dict()
print("Target counts:", {str(k): int(v) for k, v in target_counts.items()}, "pos_rate=", float(np.mean(y)))

# EDA2 quick checks: missingness + keyword prevalence in last2 notes
miss_report = {}
for vital in ["hr", "sbp", "dbp", "temp_c", "rr"]:
    miss_report[f"{vital}_miss_rate_10d"] = float(train_df[f"{vital}_miss_cnt_10d"].mean() / 10.0)
    miss_report[f"{vital}_miss_rate_last2"] = float(train_df[f"{vital}_miss_cnt_last2"].mean() / 2.0)
print("Vitals missingness (mean rates):", miss_report)

note_last2_series = train_df["note_last2_text"].fillna("").astype(str).str.lower()
kw_prev = {kw: float(note_last2_series.str.contains(kw).mean()) for kw in ["home", "rehab", "stable", "improv", "worsen", "fever", "oxygen", "antibiot", "pt", "ot", "await", "placement"]}
print("Last2 note keyword prevalence:", kw_prev)

# =========================
# Modeling (still within EDA2: validate feature usefulness)
# =========================
cat_cols = ["unit_type", "sex", "insurance", "zip3"]
text_cols = ["admission_reason", "note_text", "note_last2_text"]

exclude = set([id_col, "patient_id", target_col] + cat_cols + text_cols)
num_cols = [c for c in train_df.columns if c not in exclude and pd.api.types.is_numeric_dtype(train_df[c])]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
        ("reason", make_text_vectorizer(1200), ["admission_reason"]),
        ("notes_all", make_text_vectorizer(4000), ["note_text"]),
        ("notes_last2", make_text_vectorizer(2500), ["note_last2_text"]),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

clf = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=1.0,
    class_weight="balanced",
    max_iter=5000,
    random_state=SEED,
    n_jobs=1,   # deterministic
)

pipe = Pipeline(steps=[("preprocess", preprocess), ("clf", clf)])

X = train_df.drop(columns=[target_col])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
threshold_grid = [0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95]

oof_proba = np.zeros(len(train_df))
oof_fold = np.zeros(len(train_df), dtype=int)
fold_f1_at_05 = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    m = clone(pipe)
    m.fit(X.iloc[tr_idx], y[tr_idx])
    proba = m.predict_proba(X.iloc[va_idx])[:, 1]
    oof_proba[va_idx] = proba
    oof_fold[va_idx] = fold
    pred_05 = (proba >= 0.5).astype(int)
    fold_f1_at_05.append(float(f1_score(y[va_idx], pred_05, average="macro")))

# OOF threshold sweep
sweep = []
best_thr, best_f1 = None, -1.0
for t in threshold_grid:
    f1 = float(f1_score(y, (oof_proba >= t).astype(int), average="macro"))
    sweep.append({"threshold": t, "macro_f1": f1})
    if f1 > best_f1:
        best_f1 = f1
        best_thr = t

# Fold macro-F1 at best threshold
fold_f1_best = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    pred = (oof_proba[va_idx] >= best_thr).astype(int)
    fold_f1_best.append(float(f1_score(y[va_idx], pred, average="macro")))

cm = confusion_matrix(y, (oof_proba >= best_thr).astype(int))

print("\nCV Macro-F1 per fold @thr=0.5:", fold_f1_at_05, "mean=", float(np.mean(fold_f1_at_05)))
print("OOF best_threshold:", best_thr, "OOF Macro-F1:", best_f1)
print("CV Macro-F1 per fold @best_thr:", fold_f1_best, "mean=", float(np.mean(fold_f1_best)))
print("OOF confusion_matrix [[tn,fp],[fn,tp]]:", cm.tolist())

# Fit final model on all training data
final_model = clone(pipe).fit(X, y)
n_features_after = int(final_model.named_steps["preprocess"].transform(X).shape[1])
print("Feature dim after preprocess:", n_features_after)

# Predict test and write required submission
X_test = test_df.copy()
test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_thr).astype(int)

submission = pd.DataFrame({id_col: stays_test[id_col].values, target_col: test_pred.astype(int)})
submission_path = BASE / f"{RUN_TAG}_submission.csv"
submission.to_csv(submission_path, index=False)
print("Wrote submission:", submission_path.resolve(), "pos_rate=", float(submission[target_col].mean()))

# =========================
# Save artifacts + checkpoint
# =========================
# Save OOF + threshold sweep
oof_df = pd.DataFrame({
    "stay_id": stays_train[id_col].values,
    "y_true": y,
    "oof_proba": oof_proba,
    "fold": oof_fold
})
sweep_df = pd.DataFrame(sweep).sort_values("threshold")

oof_ckpt_path = ckpt_dir / f"iter{iteration_id:04d}_oof_predictions.csv"
sweep_ckpt_path = ckpt_dir / f"iter{iteration_id:04d}_threshold_sweep.csv"
oof_df.to_csv(oof_ckpt_path, index=False)
sweep_df.to_csv(sweep_ckpt_path, index=False)

oof_art_path = ART_ROOT / f"iter{iteration_id:04d}_oof_predictions.csv"
sweep_art_path = ART_ROOT / f"iter{iteration_id:04d}_threshold_sweep.csv"
oof_df.to_csv(oof_art_path, index=False)
sweep_df.to_csv(sweep_art_path, index=False)

# Candidate submission (branch tracking)
candidate_path = ART_ROOT / f"{RUN_TAG}_W1_block{block_id}_iter{iteration_id:04d}_candidate.csv"
submission.to_csv(candidate_path, index=False)

# Save model + config/schema/metrics
joblib.dump(final_model, ckpt_dir / "model.joblib")

metrics = {
    "macro_f1_oof": best_f1,
    "macro_f1_per_fold": fold_f1_best,
    "confusion_matrix_oof": cm.tolist(),
    "best_threshold": best_thr,
    "n_features_after_preprocess": n_features_after
}
schema = {
    "id_col": id_col,
    "target_col": target_col,
    "numeric_cols": num_cols,
    "categorical_cols": cat_cols,
    "text_cols": text_cols,
    "tfidf_max_features": {"admission_reason": 1200, "note_text": 4000, "note_last2_text": 2500},
    "post_transform_feature_dim": n_features_after
}
config = {
    "version": VX,
    "iteration_id": iteration_id,
    "block_id": block_id,
    "block_iter": block_iter,
    "phase": phase,
    "branch": branch,
    "seed": SEED,
    "model": {"name": "LogisticRegression", "solver": "saga", "penalty": "l2", "C": 1.0, "class_weight": "balanced", "max_iter": 5000, "random_state": SEED},
    "validation": {"cv": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": 42, "threshold_grid": threshold_grid},
    "features": {
        "vitals": "overall stats + early(1-8) vs last2(9-10) + delta10_9 + slopes + missingness",
        "notes": "note_text(all days) + note_last2_text(days9-10) + keyword counts(last2) + note lengths",
    }
}

with open(ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
with open(ckpt_dir / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)
with open(ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

# =========================
# Update P* (based on deterministic OOF macro-F1)
# =========================
pstar_path = CKPT_ROOT / "PSTAR.json"
prev_pstar_f1 = -1.0
if pstar_path.exists():
    try:
        prev_pstar_f1 = float(json.loads(pstar_path.read_text(encoding="utf-8")).get("macro_f1_oof", -1.0))
    except Exception:
        prev_pstar_f1 = -1.0
else:
    # fallback: best from log
    for e in entries:
        try:
            prev_pstar_f1 = max(prev_pstar_f1, float(e.get("metrics", {}).get("macro_f1_oof", -1.0)))
        except Exception:
            continue

is_new_pstar = best_f1 > prev_pstar_f1
if is_new_pstar:
    pstar_obj = {
        "version": VX,
        "updated_at": now_et_iso(),
        "iteration_id": iteration_id,
        "macro_f1_oof": best_f1,
        "best_threshold": best_thr,
        "checkpoint_dir": str(ckpt_dir)
    }
    with open(pstar_path, "w", encoding="utf-8") as f:
        json.dump(pstar_obj, f, indent=2, ensure_ascii=False)

print(f"P* update? {is_new_pstar} (prev={prev_pstar_f1:.4f} -> new={best_f1:.4f})  P* path={pstar_path}")

# =========================
# Append iteration detail log (append-only)
# =========================
iteration_entry = {
    "version": VX,
    "iteration_id": iteration_id,
    "timestamp": now_et_iso(),
    "phase": phase,
    "block_id": block_id,
    "block_iter": block_iter,
    "branch": branch,
    "window_branches_plan": window_plan,
    "data_paths_used": {
        "stays_train": str(stays_train_path),
        "stays_test": str(stays_test_path),
        "patients": str(patients_path),
        "vitals": str(vitals_path),
    },
    "joins": [
        {"left": "stays_train/stays_test", "right": "patients", "how": "left", "key": ["patient_id"]},
        {"left": "stays_*+patients", "right": "vitals_timeseries (engineered)", "how": "left", "key": ["stay_id"]},
    ],
    "leakage_checks": [
        f"Confirmed vitals_timeseries day range = [{day_min},{day_max}] (no day11 fields).",
        "No joins to CH1/CH2 artifacts (admissions/discharge_notes/ed_cost) in this iteration.",
        "Text features derived only from daily notes in vitals_timeseries days 1-10.",
        "Threshold tuned only on out-of-fold predictions (no test peeking).",
        "CV protocol kept fixed (StratifiedKFold, shuffle=True, random_state=42).",
    ],
    "feature_summary": {
        "numeric_feature_count": len(num_cols),
        "categorical_feature_count": len(cat_cols),
        "text_feature_fields": text_cols,
        "tfidf_max_features": {"admission_reason": 1200, "note_text": 4000, "note_last2_text": 2500},
        "vitals_features": "overall stats + early(1-8) vs last2(9-10) + delta10_9 + slopes + missingness",
        "notes_features": "note_text(all days) + note_last2_text(days9-10) + keyword counts(last2) + note lengths",
        "post_transform_feature_dim": n_features_after,
    },
    "eda_summary": {
        "train_shape": list(stays_train.shape),
        "test_shape": list(stays_test.shape),
        "patients_shape": list(patients.shape),
        "vitals_records": int(len(vitals)),
        "vitals_day_range": [int(day_min), int(day_max)],
        "merged_train_shape": list(train_df.shape),
        "merged_test_shape": list(test_df.shape),
        "target_counts": {str(k): int(v) for k, v in target_counts.items()},
        "target_rate": float(np.mean(y)),
        "vitals_missingness": miss_report,
        "note_last2_keyword_prevalence": kw_prev,
    },
    "models_attempted": [
        {
            "name": "LogReg(saga,l2) + OHE(cat) + TFIDF(reason,notes_all,notes_last2) + vitals recent-window features",
            "params": {"solver": "saga", "penalty": "l2", "C": 1.0, "class_weight": "balanced", "max_iter": 5000, "random_state": 42},
            "cv_macro_f1_per_fold@thr0.5": fold_f1_at_05,
            "cv_macro_f1_mean@thr0.5": float(np.mean(fold_f1_at_05)),
            "oof_macro_f1_best_threshold": best_f1,
            "best_threshold": best_thr,
            "notes": "Adds last-48h vitals & last2 notes; keeps dimensionality modest.",
        }
    ],
    "selected_model": "LogisticRegression(saga,l2,balanced)",
    "validation_protocol": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 42,
        "threshold_search": {"grid": threshold_grid, "metric": "macro_f1 (OOF)"},
    },
    "metrics": {
        "macro_f1_oof": best_f1,
        "macro_f1_per_fold": fold_f1_best,
        "confusion_matrix_oof": cm.tolist(),
        "best_threshold": best_thr,
    },
    "pstar_update": {
        "is_new_pstar": is_new_pstar,
        "pstar_path": str(pstar_path),
        "checkpoint_dir": str(ckpt_dir),
    },
    "deltas_vs_previous": f"Added vitals recent-window (days9-10) deltas/slopes + note_last2_text TF-IDF + keyword counts. OOF macro-F1: {prev_pstar_f1:.4f} -> {best_f1:.4f}.",
    "known_defects_limitations": [
        "Recent-window features are simple linear summaries; may miss non-linear dynamics.",
        "Threshold optimized on OOF can still overfit slightly; monitor CV↔LB drift.",
        "LogReg remains linear; interactions between text and vitals not modeled explicitly.",
    ],
    "next_step_hypothesis": "Move to Modeling phase: try W3 non-linear model (HistGradientBoosting) on dense (numeric+SVD(text)) while keeping CV fixed; compare to current P*.",
    "hm_message": HM_MESSAGE,
    "real_score": REAL_SCORE,
}

with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_entry, ensure_ascii=False) + "\n")

print("Appended iteration log:", LOG_PATH.resolve())

# =========================
# Consultant packet (v3 requirement)
# =========================
consult_packet = {
    "version": VX,
    "iteration_id": iteration_id,
    "timestamp": iteration_entry["timestamp"],
    "what_changed": iteration_entry["deltas_vs_previous"],
    "metric_change": {"prev_pstar_macro_f1_oof": prev_pstar_f1, "current_macro_f1_oof": best_f1, "best_threshold": best_thr},
    "suspected_leakage_risks": [
        "Using free-text daily notes can encode downstream outcomes if notes include discharge decisions; keyword spot-check suggests explicit discharge language is rare in this dataset.",
        "No day11 vitals present; confirmed day range 1-10.",
    ],
    "complexity_drift_risks": {"post_transform_feature_dim": n_features_after, "comment": "Feature count kept modest (~3.6k)."},
    "evaluation_alignment_risks": [
        "OOF threshold tuning can inflate CV slightly; keep an eye on real-score drift.",
        "CV split settings must remain unchanged (kept fixed).",
    ],
    "unknown_unknowns": [
        "If note-generation differs between train/test, TF-IDF distribution shift may hurt; consider SVD smoothing or more numeric/keyword features.",
    ],
    "recommendations_next": [
        "Try HistGradientBoosting or similar on dense numeric + SVD(text) to capture non-linear interactions.",
        "Compare threshold strategies: 0.5 vs OOF-optimal vs fold-averaged.",
    ],
    "hm_message": HM_MESSAGE,
    "real_score": REAL_SCORE,
}
consult_path = CONSULT_ROOT / f"{RUN_TAG}_iter{iteration_id}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, ensure_ascii=False)

print("Wrote consultant packet:", consult_path.resolve())
print("Checkpoint dir:", ckpt_dir.resolve())
print("Artifacts dir:", ART_ROOT.resolve())

[clai_ch3_v3] iteration_id=3 block_id=0 block_iter=3 phase=Modeling branch=W1
Loaded: stays_train=(1000, 5) stays_test=(1000, 4) patients=(4000, 5) vitals_records=2000
Vitals day range: [1, 10]
Target counts: {'0': 344, '1': 656} pos_rate= 0.656
Vitals missingness (mean rates): {'hr_miss_rate_10d': 0.016300000000000002, 'hr_miss_rate_last2': 0.021, 'sbp_miss_rate_10d': 0.018099999999999998, 'sbp_miss_rate_last2': 0.025, 'dbp_miss_rate_10d': 0.0179, 'dbp_miss_rate_last2': 0.019, 'temp_c_miss_rate_10d': 0.0186, 'temp_c_miss_rate_last2': 0.023, 'rr_miss_rate_10d': 0.0194, 'rr_miss_rate_last2': 0.0225}
Last2 note keyword prevalence: {'home': 0.08, 'rehab': 0.04, 'stable': 0.048, 'improv': 0.04, 'worsen': 0.011, 'fever': 0.017, 'oxygen': 0.015, 'antibiot': 0.033, 'pt': 0.227, 'ot': 0.148, 'await': 0.04, 'placement': 0.04}


c:\Users\shend\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



CV Macro-F1 per fold @thr=0.5: [0.7504340277777778, 0.6521153630344518, 0.7127650587142011, 0.680218080784252, 0.7234207323819006] mean= 0.7037906525385168
OOF best_threshold: 0.4 OOF Macro-F1: 0.7279551448754127
CV Macro-F1 per fold @best_thr: [0.7817095588235294, 0.6904761904761905, 0.7287415968864253, 0.7058088867919292, 0.7311135902056696] mean= 0.7275699646367488
OOF confusion_matrix [[tn,fp],[fn,tp]]: [[200, 144], [92, 564]]
Feature dim after preprocess: 3550
Wrote submission: D:\AgentDs\agent_ds_healthcare\clai_ch3_v3_submission.csv pos_rate= 0.714
P* update? True (prev=-1.0000 -> new=0.7280)  P* path=checkpoints\clai_ch3_v3\PSTAR.json
Appended iteration log: D:\AgentDs\agent_ds_healthcare\clai_ch3_v3_iteration_detail.jsonl
Wrote consultant packet: D:\AgentDs\agent_ds_healthcare\consultant_packets\clai_ch3_v3_iter3_packet.json
Checkpoint dir: D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3\iter0003
Artifacts dir: D:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3


# Iteration 3
- 0.7041

In [18]:
from __future__ import annotations

from pathlib import Path
import json, re
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression

# =========================
# Config (fixed, do NOT change CV)
# =========================
TEAM = "clai"
TASK = "ch3"
VERSION = "v3"
SEED = 42
TZ = "America/Toronto"

np.random.seed(SEED)

def find_project_root(markers=("stays_train.csv", "stays_test.csv"), max_up=8) -> Path:
    p = Path.cwd().resolve()
    for _ in range(max_up + 1):
        if all((p / m).exists() for m in markers):
            return p
        if p.parent == p:
            break
        p = p.parent
    # fallback: current working dir
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()

SUBMISSION_PATH = PROJECT_ROOT / f"{TEAM}_{TASK}_{VERSION}_submission.csv"
ITER_LOG_PATH   = PROJECT_ROOT / f"{TEAM}_{TASK}_{VERSION}_iteration_detail.jsonl"

CKPT_ROOT   = PROJECT_ROOT / "checkpoints" / f"{TEAM}_{TASK}_{VERSION}"
ART_ROOT    = PROJECT_ROOT / "artifacts"   / f"{TEAM}_{TASK}_{VERSION}"
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"

for d in [CKPT_ROOT, ART_ROOT, CONSULT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

# =========================
# Helpers
# =========================
def now_iso() -> str:
    return datetime.now(ZoneInfo(TZ)).isoformat()

def to_jsonable(x):
    import numpy as _np
    if isinstance(x, (_np.integer,)):
        return int(x)
    if isinstance(x, (_np.floating,)):
        return float(x)
    if isinstance(x, (_np.ndarray,)):
        return x.tolist()
    return x

def resolve_path(filename: str) -> Path:
    for base in [PROJECT_ROOT, Path("."), Path("/mnt/data")]:
        p = base / filename
        if p.exists():
            return p

    # recursive search as last resort
    for base in [PROJECT_ROOT, Path("/mnt/data")]:
        if base.exists():
            hits = list(base.rglob(filename))
            if hits:
                return hits[0]
    raise FileNotFoundError(f"Could not locate {filename}.")

def _version_search_roots() -> list[Path]:
    roots = [ART_ROOT, CKPT_ROOT]

    # sandbox fallback (harmless on Windows; exists() will be False)
    mnt_art = Path("/mnt/data") / "artifacts" / f"{TEAM}_{TASK}_{VERSION}"
    mnt_ckp = Path("/mnt/data") / "checkpoints" / f"{TEAM}_{TASK}_{VERSION}"
    if mnt_art.exists():
        roots.append(mnt_art)
    if mnt_ckp.exists():
        roots.append(mnt_ckp)
    return roots

def rglob_first(pattern: str, roots: list[Path] | None = None) -> Path | None:
    roots = _version_search_roots() if roots is None else roots
    hits: list[Path] = []
    for base in roots:
        if base.exists():
            hits.extend(list(base.rglob(pattern)))
    if not hits:
        return None
    # deterministic
    hits = sorted(hits, key=lambda p: str(p).lower())
    return hits[0]

def rglob_all(pattern: str, roots: list[Path] | None = None) -> list[Path]:
    roots = _version_search_roots() if roots is None else roots
    hits: list[Path] = []
    for base in roots:
        if base.exists():
            hits.extend(list(base.rglob(pattern)))
    # dedupe + deterministic
    uniq = {}
    for p in hits:
        uniq[str(p.resolve()).lower()] = p
    return sorted(uniq.values(), key=lambda p: str(p).lower())

def load_threshold_best(tag: str):
    # STRICT: search in current version artifact root first
    ts_path = ART_ROOT / f"{tag}_threshold_sweep.csv"
    if not ts_path.exists():
        return None, None, None

    ts = pd.read_csv(ts_path)
    if "macro_f1" in ts.columns:
        mcol = "macro_f1"
    elif "macro_f1_oof" in ts.columns:
        mcol = "macro_f1_oof"
    else:
        mcol = [c for c in ts.columns if c != "threshold"][0]
    idx = ts[mcol].idxmax()
    return float(ts.loc[idx, "threshold"]), float(ts.loc[idx, mcol]), ts_path

def best_threshold_from_proba(y: np.ndarray, proba: np.ndarray, grid: np.ndarray):
    best_thr, best_score = None, -1.0
    for thr in grid:
        score = f1_score(y, (proba >= thr).astype(int), average="macro")
        if score > best_score:
            best_score, best_thr = float(score), float(thr)
    return best_thr, best_score

def eval_oof_df(oof_df: pd.DataFrame, thr: float) -> dict:
    y = oof_df["y_true"].values
    p = (oof_df["oof_proba"].values >= thr).astype(int)
    macro = float(f1_score(y, p, average="macro"))
    cm = confusion_matrix(y, p).tolist()
    pos_rate = float(p.mean())
    fold_scores = []
    for f in sorted(oof_df["fold"].unique()):
        idx = (oof_df["fold"].values == f)
        fold_scores.append(float(f1_score(oof_df.loc[idx, "y_true"], (oof_df.loc[idx, "oof_proba"] >= thr).astype(int), average="macro")))
    auc = float(roc_auc_score(y, oof_df["oof_proba"].values))
    brier = float(brier_score_loss(y, oof_df["oof_proba"].values))
    return {
        "threshold": float(thr),
        "macro_f1": macro,
        "fold_macro_f1": fold_scores,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "confusion_matrix": cm,
        "oof_pos_rate": pos_rate,
        "auc": auc,
        "brier": brier,
    }

def eval_from_preds(y: np.ndarray, folds: np.ndarray, pred: np.ndarray) -> dict:
    macro = float(f1_score(y, pred, average="macro"))
    cm = confusion_matrix(y, pred).tolist()
    fold_scores = []
    for f in sorted(np.unique(folds)):
        idx = (folds == f)
        fold_scores.append(float(f1_score(y[idx], pred[idx], average="macro")))
    return {
        "threshold": "fold_specific",
        "macro_f1": macro,
        "fold_macro_f1": fold_scores,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "confusion_matrix": cm,
        "oof_pos_rate": float(pred.mean()),
    }

def eval_from_probs(y: np.ndarray, folds: np.ndarray, proba: np.ndarray, thr: float) -> dict:
    return eval_oof_df(pd.DataFrame({"y_true": y, "oof_proba": proba, "fold": folds}), thr)

def nested_threshold_preds(oof_df: pd.DataFrame, grid: np.ndarray):
    y = oof_df["y_true"].values
    pro = oof_df["oof_proba"].values
    folds = oof_df["fold"].values
    preds = np.zeros_like(y)
    chosen = {}
    for f in sorted(np.unique(folds)):
        train_idx = (folds != f)
        val_idx = (folds == f)
        best_thr, best_score = None, -1.0
        for thr in grid:
            score = f1_score(y[train_idx], (pro[train_idx] >= thr).astype(int), average="macro")
            if score > best_score:
                best_score, best_thr = float(score), float(thr)
        chosen[int(f)] = float(best_thr)
        preds[val_idx] = (pro[val_idx] >= best_thr).astype(int)
    return preds, chosen

def crossfit_isotonic_probs(oof_df: pd.DataFrame):
    y = oof_df["y_true"].values
    pro = oof_df["oof_proba"].values
    folds = oof_df["fold"].values
    cal = np.zeros_like(pro)
    for f in sorted(np.unique(folds)):
        train_idx = (folds != f)
        val_idx = (folds == f)
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(pro[train_idx], y[train_idx])
        cal[val_idx] = ir.predict(pro[val_idx])
    return cal

def find_submission_for_tag(tag: str) -> Path | None:
    patterns = [f"*{tag}*candidate*.csv", f"*{tag}*submission*.csv"]
    candidates = []
    for base in [Path("."), ART_ROOT, CKPT_ROOT, Path("/mnt/data")]:
        if base.exists():
            for pat in patterns:
                candidates.extend(list(base.rglob(pat)))
    good = []
    for p in candidates:
        try:
            df = pd.read_csv(p)
            if set(["stay_id", "discharge_ready_day11"]).issubset(df.columns):
                good.append(p)
        except Exception:
            pass
    good_sorted = sorted(good, key=lambda x: (0 if "candidate" in x.name else 1, len(str(x))))
    return good_sorted[0] if good_sorted else None

def find_checkpoint_config(tag: str) -> Path | None:
    roots = [CKPT_ROOT, Path("checkpoints"), Path("/mnt/data") / "checkpoints"]
    for root in roots:
        if root.exists():
            direct = root / tag / "config.json"
            if direct.exists():
                return direct
            for d in root.rglob("*"):
                if d.is_dir() and tag in d.name:
                    cand = d / "config.json"
                    if cand.exists():
                        return cand
            hits = list(root.rglob(f"*{tag}*config*.json"))
            if hits:
                return hits[0]
    return None

def dict_diff(a, b, prefix=""):
    changes = []
    a = {} if a is None else a
    b = {} if b is None else b
    keys = sorted(set(a.keys()) | set(b.keys()))
    for k in keys:
        pa = f"{prefix}.{k}" if prefix else str(k)
        if k not in a:
            changes.append({"type": "ADDED", "path": pa, "from": None, "to": b[k]})
        elif k not in b:
            changes.append({"type": "REMOVED", "path": pa, "from": a[k], "to": None})
        else:
            va, vb = a[k], b[k]
            if isinstance(va, dict) and isinstance(vb, dict):
                changes.extend(dict_diff(va, vb, pa))
            elif isinstance(va, list) and isinstance(vb, list):
                if va != vb:
                    changes.append({"type": "CHANGED", "path": pa, "from": f"list_len={len(va)}", "to": f"list_len={len(vb)}"})
            else:
                if va != vb:
                    changes.append({"type": "CHANGED", "path": pa, "from": va, "to": vb})
    return changes

# =========================
# Load / initialize iteration log
# =========================
if not ITER_LOG_PATH.exists():
    src = Path("/mnt/data") / ITER_LOG_PATH.name
    if src.exists():
        ITER_LOG_PATH.write_text(src.read_text(encoding="utf-8"), encoding="utf-8")
    else:
        ITER_LOG_PATH.touch()

max_iter = -1
with open(ITER_LOG_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            max_iter = max(max_iter, int(obj.get("iteration_id", -1)))
        except Exception:
            continue
ITER_ID = max_iter + 1

# =========================
# Load labels + fixed folds
# =========================
stays_train = pd.read_csv(resolve_path("stays_train.csv"))
y = stays_train["discharge_ready_day11"].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = np.empty(len(y), dtype=int)
for f, (_, va) in enumerate(skf.split(np.zeros(len(y)), y)):
    folds[va] = f
df_folds = pd.DataFrame({"stay_id": stays_train["stay_id"].values, "fold": folds})

# =========================
# Discover available iter artifacts
# =========================
# Discover available iter artifacts (SCOPE TO THIS VERSION ONLY)
oof_files = rglob_all("iter*_oof_predictions.csv", roots=[ART_ROOT])
avail_tags = sorted({
    re.search(r"(iter\d{4})_oof_predictions\.csv", p.name).group(1)
    for p in oof_files if re.search(r"(iter\d{4})_oof_predictions\.csv", p.name)
})
if not avail_tags:
    raise RuntimeError(f"No iter*_oof_predictions.csv found under {ART_ROOT}")

BEST_TAG = "iter0002" if "iter0002" in avail_tags else avail_tags[0]
CURRENT_TAG = max(avail_tags)   # 现在一定是 v3 的最大 iter，比如 iter0003

# If iter0002 is missing locally, choose best available by OOF threshold sweep
best_by_oof = []
for tag in avail_tags:
    thr, score, _ = load_threshold_best(tag)
    if thr is not None and score is not None:
        best_by_oof.append((tag, float(score), float(thr)))
if BEST_TAG is None:
    BEST_TAG = max(best_by_oof, key=lambda x: x[1])[0] if best_by_oof else avail_tags[0]

def load_oof(tag: str):
    oof_path = rglob_first(f"{tag}_oof_predictions.csv")
    if oof_path is None:
        raise FileNotFoundError(f"Missing {tag}_oof_predictions.csv")
    oof = pd.read_csv(oof_path)
    if "fold" not in oof.columns:
        oof = oof.merge(df_folds, on="stay_id", how="left")
    else:
        oof = oof.merge(df_folds.rename(columns={"fold": "fold_ref"}), on="stay_id", how="left")
        mismatch = float((oof["fold"] != oof["fold_ref"]).mean())
        if mismatch > 0:
            print(f"[WARN] {tag}: fold mismatch rate={mismatch:.3f}. Overwriting with reference folds.")
            oof["fold"] = oof["fold_ref"]
        oof = oof.drop(columns=["fold_ref"])
    return oof, oof_path

oof_best, oof_best_path = load_oof(BEST_TAG)
oof_cur, oof_cur_path = load_oof(CURRENT_TAG)

thr_best, score_best, ts_best_path = load_threshold_best(BEST_TAG)
thr_cur, score_cur, ts_cur_path = load_threshold_best(CURRENT_TAG)
if thr_best is None or thr_cur is None:
    raise RuntimeError("Threshold sweep missing for best/current tag; cannot determine thresholds.")

# =========================
# Compare exact pipeline configs (if available)
# =========================
cfg_best_path = find_checkpoint_config(BEST_TAG)
cfg_cur_path = find_checkpoint_config(CURRENT_TAG)
cfg_best = json.loads(cfg_best_path.read_text(encoding="utf-8")) if cfg_best_path and cfg_best_path.exists() else None
cfg_cur = json.loads(cfg_cur_path.read_text(encoding="utf-8")) if cfg_cur_path and cfg_cur_path.exists() else None
cfg_diffs = dict_diff(cfg_best, cfg_cur) if (cfg_best is not None and cfg_cur is not None) else []

cfg_diff_path = ART_ROOT / f"iter{ITER_ID:04d}_config_diff_{BEST_TAG}_vs_{CURRENT_TAG}.json"
with open(cfg_diff_path, "w", encoding="utf-8") as f:
    json.dump({"best_config_path": str(cfg_best_path), "cur_config_path": str(cfg_cur_path), "diffs": cfg_diffs}, f, indent=2, default=to_jsonable)

# =========================
# Post-mortem metrics
# =========================
m_best = eval_oof_df(oof_best, thr_best)
m_cur = eval_oof_df(oof_cur, thr_cur)

merged = oof_best[["stay_id", "y_true", "oof_proba"]].merge(
    oof_cur[["stay_id", "oof_proba"]], on="stay_id", suffixes=("_best", "_cur")
)
pred_best = (merged["oof_proba_best"].values >= thr_best).astype(int)
pred_cur = (merged["oof_proba_cur"].values >= thr_cur).astype(int)
merged["pred_best"] = pred_best
merged["pred_cur"] = pred_cur
merged["flip_type"] = np.select(
    [
        (merged["pred_best"] == 0) & (merged["pred_cur"] == 1),
        (merged["pred_best"] == 1) & (merged["pred_cur"] == 0),
    ],
    ["0->1", "1->0"],
    default="same",
)
flip_counts = merged["flip_type"].value_counts().to_dict()

meta = stays_train[["stay_id", "unit_type", "admission_reason"]].copy()
flip_df = merged.merge(meta, on="stay_id", how="left")

# =========================
# Minimal ablations (one change at a time) — CURRENT_TAG
# =========================
thr_grid = np.round(np.arange(0.05, 1.0, 0.05), 2)

ab_results = []
ab_results.append(("A0_current_default_thr", m_cur, {"note": "baseline current (as submitted)"}))
ab_results.append(("A1_current_use_best_thr", eval_oof_df(oof_cur, thr_best), {"note": "threshold-only change to best checkpoint threshold"}))

pred_nested, chosen_thrs = nested_threshold_preds(oof_cur, thr_grid)
ab_results.append(("A2_current_nested_threshold", eval_from_preds(oof_cur["y_true"].values, oof_cur["fold"].values, pred_nested), {"fold_thresholds": chosen_thrs, "note": "nested threshold selection (train folds -> val fold)"}))

cal_probs = crossfit_isotonic_probs(oof_cur)
thr_cal, _ = best_threshold_from_proba(oof_cur["y_true"].values, cal_probs, thr_grid)
ab_results.append(("A3_current_isotonic_cal", eval_from_probs(oof_cur["y_true"].values, oof_cur["fold"].values, cal_probs, thr_cal), {"note": "cross-fitted isotonic calibration + tuned threshold"}))

# High-risk diagnostic: ensemble on OOF
avg_probs = (merged["oof_proba_best"].values + merged["oof_proba_cur"].values) / 2.0
thr_avg, _ = best_threshold_from_proba(merged["y_true"].values, avg_probs, thr_grid)
ab_results.append(("A4_ensemble_avg(best+current)", eval_from_probs(merged["y_true"].values, oof_cur["fold"].values, avg_probs, thr_avg), {"note": "HIGH-RISK: average of two pipelines' OOF probabilities"}))

ab_rows = []
for name, metrics, extra in ab_results:
    row = {"ablation": name}
    for k in ["threshold", "macro_f1", "fold_mean", "fold_std", "oof_pos_rate", "auc", "brier"]:
        row[k] = metrics.get(k, None)
    cm = metrics.get("confusion_matrix")
    if isinstance(cm, list) and len(cm) == 2 and len(cm[0]) == 2:
        row.update({"TN": cm[0][0], "FP": cm[0][1], "FN": cm[1][0], "TP": cm[1][1]})
    ab_rows.append(row)
ab_df = pd.DataFrame(ab_rows).sort_values("macro_f1", ascending=False)

ab_csv = ART_ROOT / f"iter{ITER_ID:04d}_postmortem_ablation_results.csv"
ab_df.to_csv(ab_csv, index=False)

# =========================
# Confirm test binarization threshold + submission pos-rate (as far as possible)
# =========================
sub_best_path = find_submission_for_tag(BEST_TAG)
sub_cur_path = find_submission_for_tag(CURRENT_TAG)

def safe_pos_rate(path: Path | None):
    if path is None:
        return None
    df = pd.read_csv(path)
    return float(df["discharge_ready_day11"].mean())

pos_rate_best_sub = safe_pos_rate(sub_best_path)
pos_rate_cur_sub = safe_pos_rate(sub_cur_path)

# Output submission: rollback to best submission if available, else current
if sub_best_path is not None:
    sub_out = pd.read_csv(sub_best_path)
    used_submission_source = str(sub_best_path)
else:
    if sub_cur_path is None:
        stays_test = pd.read_csv(resolve_path("stays_test.csv"))
        sub_out = pd.DataFrame({"stay_id": stays_test["stay_id"].values, "discharge_ready_day11": np.ones(len(stays_test), dtype=int)})
        used_submission_source = "fallback_all_ones"
    else:
        sub_out = pd.read_csv(sub_cur_path)
        used_submission_source = str(sub_cur_path)

sub_out = sub_out[["stay_id", "discharge_ready_day11"]].copy()
sub_out["discharge_ready_day11"] = sub_out["discharge_ready_day11"].astype(int)
sub_out.to_csv(SUBMISSION_PATH, index=False)
pos_rate_out_sub = float(sub_out["discharge_ready_day11"].mean())

# =========================
# Print requested post-mortem report
# =========================
print("=" * 80)
print("POST-MORTEM SUMMARY (fixed CV = StratifiedKFold(5, seed=42))")
print(f"Available iter tags: {avail_tags}")
print(f"BEST_TAG used:    {BEST_TAG} | thr={thr_best:.2f} | OOF macro-F1={m_best['macro_f1']:.4f} | AUC={m_best['auc']:.4f} | OOF pos_rate={m_best['oof_pos_rate']:.3f}")
print(f"CURRENT_TAG used: {CURRENT_TAG} | thr={thr_cur:.2f} | OOF macro-F1={m_cur['macro_f1']:.4f} | AUC={m_cur['auc']:.4f} | OOF pos_rate={m_cur['oof_pos_rate']:.3f}")
if BEST_TAG != "iter0002":
    print("[NOTE] iter0002 artifacts not found locally -> BEST_TAG is a proxy (best by OOF). Restore iter0002 for exact diff.")

print("\n=== PIPELINE DELTAS (features/model/preprocess/etc) ===")
print(f"Best config path:    {cfg_best_path}")
print(f"Current config path: {cfg_cur_path}")
if cfg_best is None or cfg_cur is None:
    print("[WARN] Missing one or both checkpoint configs => cannot enumerate feature/model/preprocess deltas exactly.")
    print(f"       A placeholder diff JSON is still written to: {cfg_diff_path}")
else:
    print(f"Total config diffs: {len(cfg_diffs)} (full list saved to {cfg_diff_path})")
    for i, d in enumerate(cfg_diffs[:80]):
        print(f"  - {d['type']:7s} {d['path']}: {d['from']} -> {d['to']}")
    if len(cfg_diffs) > 80:
        print(f"  ... (+{len(cfg_diffs) - 80} more; see {cfg_diff_path})")

print("\n=== METRIC DELTAS (current - best) ===")
print(f"Δ OOF macro-F1: {m_cur['macro_f1'] - m_best['macro_f1']:+.4f}")
print(f"Δ AUC:         {m_cur['auc'] - m_best['auc']:+.4f}")
print(f"Δ OOF pos rate:{m_cur['oof_pos_rate'] - m_best['oof_pos_rate']:+.4f}")
print("Confusion best  (TN FP / FN TP):", m_best["confusion_matrix"])
print("Confusion cur   (TN FP / FN TP):", m_cur["confusion_matrix"])

print("\nFlip counts @ chosen thresholds (best_thr vs cur_thr):", flip_counts)
print("\nTop flip categories (0->1) by unit_type:")
print(flip_df[flip_df["flip_type"] == "0->1"]["unit_type"].value_counts().head(10).to_string())
print("\nTop flip categories (0->1) by admission_reason:")
print(flip_df[flip_df["flip_type"] == "0->1"]["admission_reason"].value_counts().head(10).to_string())

print("\n=== MINIMAL ABLATIONS (one change at a time) — CURRENT_TAG ===")
print(ab_df.to_string(index=False))

print("\n=== TEST BINARIZATION SANITY (as far as possible) ===")
print("NOTE: Binary submissions do not allow exact back-calculation of threshold; we report threshold from sweep/config.")
print(f"Best threshold (sweep): {thr_best:.2f} | best submission: {sub_best_path} | pos_rate={pos_rate_best_sub}")
print(f"Cur  threshold (sweep): {thr_cur:.2f} | cur  submission: {sub_cur_path} | pos_rate={pos_rate_cur_sub}")
print(f"Output submission:      {SUBMISSION_PATH} (source={used_submission_source}) | pos_rate={pos_rate_out_sub:.3f}")

# =========================
# Culprit isolation conclusion (based on ablations)
# =========================
print("\n=== ISOLATION CONCLUSION ===")
print("1) Threshold drift is unlikely: best_thr == cur_thr (or changing thr alone does not recover).")
print("2) Nested thresholding / calibration yields negligible improvement.")
print("=> Most likely culprit: discrimination degradation (AUC drop) from feature/model/preprocess changes OR train/test preprocessing mismatch.")
print("Rollback to P* recommended before further improvements.\n")

# =========================
# Next iteration plan (reversible)
# =========================
next_plan = {
    "low_risk_fix_1": "Rollback to P* checkpoint for submissions; lock threshold + preprocessing; add sanity checks before export (pos-rate, NaN rate, feature count).",
    "low_risk_fix_2": "Add compact last-48h vitals delta/volatility features while reducing TF-IDF dimensionality; keep CV fixed (5-fold, seed=42).",
    "high_risk_branch": "Train 2–3 diverse models and average probabilities (stack/ensemble). OOF ensemble(best+current) already shows a small gain."
}
print("NEXT ITERATION PLAN (reversible):")
for k, v in next_plan.items():
    print(f"- {k}: {v}")

# =========================
# Save checkpoint bundle + consultant packet + iteration log append
# =========================
run_ckpt = CKPT_ROOT / f"iter{ITER_ID:04d}"
run_ckpt.mkdir(parents=True, exist_ok=True)

postmortem_config = {
    "team": TEAM,
    "task": TASK,
    "version": VERSION,
    "iteration_id": ITER_ID,
    "seed": SEED,
    "purpose": "postmortem_and_ablations",
    "iters": {"best_tag": BEST_TAG, "current_tag": CURRENT_TAG},
    "paths_used": {
        "stays_train": str(resolve_path("stays_train.csv")),
        "stays_test": str(resolve_path("stays_test.csv")),
        "oof_best": str(oof_best_path),
        "oof_current": str(oof_cur_path),
        "threshold_sweep_best": str(ts_best_path) if ts_best_path else None,
        "threshold_sweep_current": str(ts_cur_path) if ts_cur_path else None,
        "checkpoint_config_best": str(cfg_best_path) if cfg_best_path else None,
        "checkpoint_config_current": str(cfg_cur_path) if cfg_cur_path else None,
        "submission_best": str(sub_best_path) if sub_best_path else None,
        "submission_current": str(sub_cur_path) if sub_cur_path else None,
        "config_diff_json": str(cfg_diff_path),
        "ablation_csv": str(ab_csv),
    },
    "validation": {"cv": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": SEED, "threshold_grid": thr_grid.tolist()},
}
with open(run_ckpt / "config.json", "w", encoding="utf-8") as f:
    json.dump(postmortem_config, f, indent=2)

postmortem_metrics = {
    "best": m_best,
    "current": m_cur,
    "flip_counts": flip_counts,
    "pos_rate_best_submission": pos_rate_best_sub,
    "pos_rate_current_submission": pos_rate_cur_sub,
    "pos_rate_output_submission": pos_rate_out_sub,
}
with open(run_ckpt / "metrics_postmortem.json", "w", encoding="utf-8") as f:
    json.dump(postmortem_metrics, f, indent=2, default=to_jsonable)

consult_packet = {
    "team": TEAM,
    "task": TASK,
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": now_iso(),
    "what_changed": "Post-mortem + ablations for leaderboard regression; attempted exact config diff if checkpoint configs exist.",
    "metric_changes": {
        "best_oof_macro_f1": m_best["macro_f1"],
        "current_oof_macro_f1": m_cur["macro_f1"],
        "delta_oof_macro_f1": m_cur["macro_f1"] - m_best["macro_f1"],
        "best_auc": m_best["auc"],
        "current_auc": m_cur["auc"],
        "delta_auc": m_cur["auc"] - m_best["auc"],
    },
    "suspected_leakage_risks": [
        "Verify feature sources for iter0002/iter0003: do NOT join discharge_notes or other post-discharge artifacts.",
        "If using CH1/CH2 artifacts, document join keys and prove no target leakage."
    ],
    "complexity_drift_risks": [
        "High-dimensional TF-IDF vs small n may overfit; enforce feature caps and regularization.",
        "Ensembling may improve score but increases operational complexity; treat as high-risk branch."
    ],
    "evaluation_alignment_risks": [
        "Threshold ablations had minimal impact => likely not threshold drift; still verify submission export/binarization code.",
        "If CV historically aligns but real score regressed, suspect preprocessing mismatch or distribution shift."
    ],
    "unknown_unknowns": [
        "Binary submission cannot prove threshold used; save test probabilities + threshold explicitly going forward.",
        "If iter0002 artifacts missing locally, BEST_TAG becomes proxy by OOF; restore true iter0002 for full audit."
    ],
    "recommendations": [
        "Rollback to P* for submissions whenever a regression occurs.",
        "Add last-48h vitals delta/volatility features with strict dimensionality control.",
        "High-risk: train 2–3 diverse models and average probabilities; OOF ensemble already improves macro-F1."
    ]
}
consult_path = CONSULT_ROOT / f"{TEAM}_{TASK}_{VERSION}_iter{ITER_ID}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, default=to_jsonable)

models_attempted = []
for name, metrics, extra in ab_results:
    models_attempted.append({
        "name": name,
        "params": extra,
        "cv_macro_f1": metrics.get("macro_f1"),
        "fold_mean": metrics.get("fold_mean"),
        "fold_std": metrics.get("fold_std"),
        "threshold": metrics.get("threshold"),
        "confusion_matrix": metrics.get("confusion_matrix"),
        "oof_pos_rate": metrics.get("oof_pos_rate"),
        "notes": extra.get("note"),
    })

log_entry = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": now_iso(),
    "phase": "Modeling",
    "data_paths_used": postmortem_config["paths_used"],
    "joins": [{"left": "oof_predictions", "right": "stays_train", "key": "stay_id", "purpose": "fold assignment + meta for flip analysis"}],
    "leakage_checks": [
        "Analysis-only: used provided OOF predictions and known training labels; no feature joins beyond meta columns.",
        "Flagged potential leakage if discharge_notes or other post-discharge artifacts are used (not used here)."
    ],
    "feature_summary": {
        "analysis_mode": True,
        "best_tag": BEST_TAG,
        "current_tag": CURRENT_TAG,
        "config_diff_json": str(cfg_diff_path),
        "delta_note": "Exact feature/model/preprocess deltas are fully enumerated only when both checkpoint configs are available."
    },
    "models_attempted": models_attempted,
    "selected_model": {"name": f"RollbackRecommended({BEST_TAG})", "reason": "Current underperforms on OOF macro-F1 and AUC; ablations do not recover."},
    "validation_protocol": postmortem_config["validation"],
    "metrics": {
        "best": m_best,
        "current": m_cur,
        "flip_counts": flip_counts,
        "submission_pos_rates": {"best": pos_rate_best_sub, "current": pos_rate_cur_sub, "output": pos_rate_out_sub}
    },
    "pstar_update": False,
    "deltas_vs_previous": {"summary": "Post-mortem + ablations; no model training; output submission rolled back if best submission available."},
    "known_defects_limitations": [
        "Cannot verify threshold used in provided binary submissions without saved test probabilities; log them going forward.",
        "If checkpoint configs are missing, config diff is unavailable; restore checkpoint bundles for full audit."
    ],
    "next_step_hypothesis": {
        "top_3_hypotheses_regression": [
            "Discrimination degraded (AUC drop) due to feature/model/hyperparameter change; more false positives => macro-F1 drop.",
            "Train/test preprocessing mismatch (missing value handling, scaling, tokenization) causing hidden distribution shift.",
            "Threshold drift less likely (ablations minimal impact) but submission binarization must be verified."
        ],
        "next_iteration_plan": next_plan
    },
    "hm_message": "HM: best real Macro-F1=0.7768 at iter0002; regression to real Macro-F1=0.7041 at current submission; requested post-mortem + ablations with fixed CV.",
    "real_score": 0.7041
}
with open(ITER_LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(log_entry, default=to_jsonable) + "\n")

print("\nSaved artifacts:")
print(f"  - Config diff JSON:     {cfg_diff_path}")
print(f"  - Ablation results CSV: {ab_csv}")
print(f"  - Checkpoint dir:       {run_ckpt}")
print(f"  - Consultant packet:    {consult_path}")
print(f"  - Iteration log:        {ITER_LOG_PATH}")
print("=" * 80)

POST-MORTEM SUMMARY (fixed CV = StratifiedKFold(5, seed=42))
Available iter tags: ['iter0000', 'iter0001', 'iter0002', 'iter0003']
BEST_TAG used:    iter0002 | thr=0.45 | OOF macro-F1=0.7629 | AUC=0.8463 | OOF pos_rate=0.649
CURRENT_TAG used: iter0003 | thr=0.40 | OOF macro-F1=0.7280 | AUC=0.7947 | OOF pos_rate=0.708

=== PIPELINE DELTAS (features/model/preprocess/etc) ===
Best config path:    D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3\iter0002\iter0002_config.json
Current config path: D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3\iter0003\config.json
Total config diffs: 25 (full list saved to D:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3\iter0006_config_diff_iter0002_vs_iter0003.json)
  - ADDED   block_id: None -> 0
  - ADDED   block_iter: None -> 3
  - ADDED   branch: None -> W1
  - REMOVED features.categorical_cols: ['unit_type', 'sex', 'insurance', 'zip3'] -> None
  - ADDED   features.notes: None -> note_text(all days) + note_last2_text(days9-10) + keywo

# Iteration 4
- 0.7564

In [20]:
# clai_ch3_v3 — Iteration (auto) — Postmortem + scoped artifact discovery + ablations + improved candidate + submission
import os, re, json, random
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
import joblib

# -------------------------
# 0) Global constants
# -------------------------
TEAM = "clai"
TASK = "ch3"
VERSION = "v3"
TAG = f"{TEAM}_{TASK}_{VERSION}"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Fixed validation protocol (HM constraint)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -------------------------
# 1) Find PROJECT_ROOT + create required dirs
# -------------------------
def find_project_root(start: Path) -> Path:
    # Walk up to find stays_train.csv; fallback to start
    cur = start.resolve()
    for _ in range(6):
        if (cur / "stays_train.csv").exists() and (cur / "stays_test.csv").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

CWD = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CWD)

ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONS_ROOT = PROJECT_ROOT / "consultant_packets"

for d in [ART_ROOT, CKPT_ROOT, CONS_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print("=== PREFLIGHT ===")
print("CWD:", CWD)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_ROOT:", ART_ROOT)
print("CKPT_ROOT:", CKPT_ROOT)

# -------------------------
# 2) Scoped artifact discovery (STRICT)
# -------------------------
def discover_iter_tags(art_root: Path):
    oof_files = sorted(art_root.glob("iter????_oof_predictions.csv"))
    thr_files = sorted(art_root.glob("iter????_threshold_sweep.csv"))

    def tag_from(p: Path):
        m = re.match(r"iter(\d{4})_", p.name)
        return int(m.group(1)) if m else None

    tags = set()
    for p in oof_files + thr_files:
        t = tag_from(p)
        if t is not None:
            tags.add(t)
    tags = sorted(tags)

    # Hard checks: each discovered tag must have BOTH files, else abort.
    for t in tags:
        oof = art_root / f"iter{t:04d}_oof_predictions.csv"
        thr = art_root / f"iter{t:04d}_threshold_sweep.csv"
        if not oof.exists() or not thr.exists():
            raise RuntimeError(
                f"[ABORT] Incomplete artifacts for iter{t:04d} under {art_root}.\n"
                f"Required BOTH:\n  - {oof.name}\n  - {thr.name}\n"
                f"This often indicates cross-version contamination or naming mismatches.\n"
                f"Fix by moving/deleting contaminated files so only v3 artifacts exist here."
            )
    return tags

# Optional: if user ran before bugfix and left known files at root, allow copying ONLY if exact filenames exist.
# (No rglob; explicit filenames only.)
for i in range(0, 50):
    for suf in ["oof_predictions.csv", "threshold_sweep.csv"]:
        src = PROJECT_ROOT / f"iter{i:04d}_{suf}"
        dst = ART_ROOT / src.name
        if src.exists() and not dst.exists():
            dst.write_bytes(src.read_bytes())

iter_tags = discover_iter_tags(ART_ROOT)
print("Discovered iter tags (scoped):", iter_tags)

# -------------------------
# 3) Load / compute P*
# -------------------------
PSTAR_PATH = CKPT_ROOT / "PSTAR.json"

def best_from_sweep(df: pd.DataFrame):
    # tolerate historical naming
    score_col = None
    for c in ["macro_f1", "macro_f1_oof", "f1", "macro_f1_mean"]:
        if c in df.columns:
            score_col = c
            break
    if score_col is None:
        raise ValueError(f"Unknown sweep schema columns={df.columns.tolist()}")
    idx = df[score_col].astype(float).idxmax()
    return float(df.loc[idx, "threshold"]), float(df.loc[idx, score_col]), score_col

if PSTAR_PATH.exists():
    PSTAR = json.load(open(PSTAR_PATH, "r", encoding="utf-8"))
else:
    best = {"iteration_id": None, "oof_macro_f1": -1.0, "threshold": 0.5, "source": "none"}
    for t in iter_tags:
        df = pd.read_csv(ART_ROOT / f"iter{t:04d}_threshold_sweep.csv")
        thr, f1v, col = best_from_sweep(df)
        if f1v > best["oof_macro_f1"]:
            best = {"iteration_id": t, "oof_macro_f1": float(f1v), "threshold": float(thr), "source": f"artifact_sweep:{col}"}
    PSTAR = best
    json.dump(PSTAR, open(PSTAR_PATH, "w", encoding="utf-8"), indent=2)

print("P* (before):", PSTAR)

def normalize_pstar(p: dict) -> dict:
    p = dict(p) if isinstance(p, dict) else {}
    # normalize f1 key
    if "oof_macro_f1" not in p:
        if "macro_f1_oof" in p:
            p["oof_macro_f1"] = float(p["macro_f1_oof"])
        elif "macro_f1" in p:
            p["oof_macro_f1"] = float(p["macro_f1"])
        else:
            p["oof_macro_f1"] = -1.0

    # normalize threshold key
    if "threshold" not in p:
        if "best_threshold" in p:
            p["threshold"] = float(p["best_threshold"])
        else:
            p["threshold"] = 0.5

    # normalize iteration_id
    if "iteration_id" in p and p["iteration_id"] is not None:
        try:
            p["iteration_id"] = int(p["iteration_id"])
        except Exception:
            pass

    p.setdefault("source", "loaded_or_computed")
    p.setdefault("schema_version", 1)
    return p

PSTAR = normalize_pstar(PSTAR)
json.dump(PSTAR, open(PSTAR_PATH, "w", encoding="utf-8"), indent=2)  # optional but recommended
print("P* (normalized):", PSTAR)

# -------------------------
# 4) Compare iter0002 vs iter0003 config diff (if provided)
# -------------------------
diff_candidates = [
    PROJECT_ROOT / "iter0006_config_diff_iter0002_vs_iter0003.json",
    ART_ROOT / "iter0006_config_diff_iter0002_vs_iter0003.json",
]
diff_path = next((p for p in diff_candidates if p.exists()), None)
if diff_path:
    diff_data = json.load(open(diff_path, "r", encoding="utf-8"))
    print("\n=== CONFIG DIFF (iter0002 vs iter0003) ===")
    print("best_config_path:", diff_data.get("best_config_path"))
    print("cur_config_path :", diff_data.get("cur_config_path"))
    for d in diff_data.get("diffs", []):
        print(f"- {d['type']:>7} | {d['path']}: {d.get('from')} -> {d.get('to')}")
else:
    print("\n[WARN] Config diff file not found; will proceed with best-effort ablations from scratch.")

# -------------------------
# 5) Helper: iteration_id auto-increment (log + ckpt + artifacts)
# -------------------------
LOG_PATH = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"

def compute_next_iter_id(art_root: Path, ckpt_root: Path, log_path: Path) -> int:
    max_id = -1
    # log
    if log_path.exists():
        try:
            with open(log_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                        max_id = max(max_id, int(obj.get("iteration_id", -1)))
                    except Exception:
                        continue
        except Exception:
            pass
    # ckpt dirs
    for p in sorted(ckpt_root.glob("iter????")):
        m = re.match(r"iter(\d{4})$", p.name)
        if m:
            max_id = max(max_id, int(m.group(1)))
    # artifacts
    for p in sorted(art_root.glob("iter????_oof_predictions.csv")) + sorted(art_root.glob("iter????_threshold_sweep.csv")):
        m = re.match(r"iter(\d{4})_", p.name)
        if m:
            max_id = max(max_id, int(m.group(1)))
    return max_id + 1

ITER_ID = compute_next_iter_id(ART_ROOT, CKPT_ROOT, LOG_PATH)
ITER_TAG = f"iter{ITER_ID:04d}"
BLOCK_ID = ITER_ID // 5
BLOCK_ITER = ITER_ID % 5
BRANCH = "W1"  # theme: vitals trends + robust notes; reversible

print("\nNext iteration:", ITER_TAG, "| block", BLOCK_ID, "iter", BLOCK_ITER, "| branch", BRANCH)

# -------------------------
# 6) Load data (CH3 core only)
# -------------------------
stays_train = pd.read_csv(PROJECT_ROOT / "stays_train.csv")
stays_test  = pd.read_csv(PROJECT_ROOT / "stays_test.csv")
patients    = pd.read_csv(PROJECT_ROOT / "patients.csv")

with open(PROJECT_ROOT / "vitals_timeseries.json", "r", encoding="utf-8") as f:
    vitals_list = json.load(f)

TARGET = "discharge_ready_day11"
CAT_COLS = ["unit_type", "sex", "insurance", "zip3"]
ID_COLS = ["stay_id", "patient_id"]
VITALS = ["hr", "sbp", "dbp", "temp_c", "rr"]

# -------------------------
# 7) Feature engineering (v1 vs v2 vitals) — no leakage (days<=10)
# -------------------------
def linreg_slope(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    mask = ~np.isnan(y)
    if mask.sum() < 2:
        return np.nan
    x = x[mask]
    y = y[mask]
    xm, ym = x.mean(), y.mean()
    denom = ((x - xm) ** 2).sum()
    if denom == 0:
        return np.nan
    return float(((x - xm) * (y - ym)).sum() / denom)

def compute_note_features(days):
    days_sorted = sorted(days, key=lambda d: d.get("day", 0))
    notes = [(d.get("note") or "") for d in days_sorted]
    note_all = " ".join([str(n).strip() for n in notes if n is not None])
    note_last2 = " ".join([str(n).strip() for n in notes[-2:] if n is not None])

    feats = {
        "note_len": int(len(note_all)),
        "note_last2_len": int(len(note_last2)),
    }
    # small, interpretable keyword counts on last2 notes
    keywords = ["discharge", "home", "stable", "improv", "transfer", "pain", "fever", "oxygen", "icu", "walk"]
    nl2 = note_last2.lower()
    for kw in keywords:
        feats[f"kw_{kw}_last2"] = int(nl2.count(kw))
    return note_all, note_last2, feats

def compute_abnormal_counts(days):
    days_sorted = sorted(days, key=lambda d: d.get("day", 0))
    arrs = {v: np.array([d.get(v, np.nan) for d in days_sorted], float) for v in VITALS}
    feats = {
        "cnt_hr_gt100": int(np.sum(arrs["hr"] > 100)),
        "cnt_hr_lt60":  int(np.sum(arrs["hr"] < 60)),
        "cnt_sbp_lt90": int(np.sum(arrs["sbp"] < 90)),
        "cnt_temp_gt38": int(np.sum(arrs["temp_c"] > 38)),
        "cnt_rr_gt24":  int(np.sum(arrs["rr"] > 24)),
        "cnt2_hr_gt100": int(np.sum(arrs["hr"][-2:] > 100)),
        "cnt2_sbp_lt90": int(np.sum(arrs["sbp"][-2:] < 90)),
        "cnt2_temp_gt38": int(np.sum(arrs["temp_c"][-2:] > 38)),
        "cnt2_rr_gt24": int(np.sum(arrs["rr"][-2:] > 24)),
    }
    return feats

def vitals_features_v1(days):
    days_sorted = sorted(days, key=lambda d: d.get("day", 0))
    feats = {}
    for v in VITALS:
        arr = np.array([d.get(v, np.nan) for d in days_sorted], float)
        feats[f"{v}_mean"] = float(np.nanmean(arr))
        feats[f"{v}_std"]  = float(np.nanstd(arr))
        feats[f"{v}_min"]  = float(np.nanmin(arr))
        feats[f"{v}_max"]  = float(np.nanmax(arr))
        feats[f"{v}_last"] = float(arr[-1]) if len(arr) else np.nan
        feats[f"{v}_slope"] = float((arr[-1] - arr[0]) / (len(arr) - 1)) if len(arr) > 1 and not (np.isnan(arr[-1]) or np.isnan(arr[0])) else np.nan
        last2 = arr[-2:] if len(arr) >= 2 else arr
        feats[f"{v}_last2_mean"] = float(np.nanmean(last2))
        feats[f"{v}_last2_std"]  = float(np.nanstd(last2))
        feats[f"{v}_delta10_9"]  = float(arr[-1] - arr[-2]) if len(arr) >= 2 and not (np.isnan(arr[-1]) or np.isnan(arr[-2])) else np.nan
        feats[f"{v}_miss_count"] = int(np.isnan(arr).sum())
    return feats

def vitals_features_v2(days):
    days_sorted = sorted(days, key=lambda d: d.get("day", 0))
    feats = {}
    day_idx = np.array([d.get("day", i + 1) for i, d in enumerate(days_sorted)], float)
    for v in VITALS:
        arr = np.array([d.get(v, np.nan) for d in days_sorted], float)
        feats[f"{v}_mean"] = float(np.nanmean(arr))
        feats[f"{v}_std"]  = float(np.nanstd(arr))
        feats[f"{v}_min"]  = float(np.nanmin(arr))
        feats[f"{v}_max"]  = float(np.nanmax(arr))
        feats[f"{v}_range"] = float(feats[f"{v}_max"] - feats[f"{v}_min"])
        feats[f"{v}_last"]  = float(arr[-1]) if len(arr) else np.nan
        feats[f"{v}_slope_lr"] = linreg_slope(day_idx, arr)

        arr_last3 = arr[-3:] if len(arr) >= 3 else arr
        day_last3 = day_idx[-3:] if len(day_idx) >= 3 else day_idx
        feats[f"{v}_last3_mean"] = float(np.nanmean(arr_last3))
        feats[f"{v}_last3_std"]  = float(np.nanstd(arr_last3))
        feats[f"{v}_slope_last3"] = linreg_slope(day_last3, arr_last3)

        arr_last2 = arr[-2:] if len(arr) >= 2 else arr
        feats[f"{v}_last2_mean"] = float(np.nanmean(arr_last2))
        feats[f"{v}_last2_std"]  = float(np.nanstd(arr_last2))
        feats[f"{v}_delta10_9"]  = float(arr[-1] - arr[-2]) if len(arr) >= 2 and not (np.isnan(arr[-1]) or np.isnan(arr[-2])) else np.nan

        diffs = np.diff(arr)
        feats[f"{v}_mean_abs_diff"] = float(np.nanmean(np.abs(diffs))) if len(diffs) else np.nan

        first5 = arr[:5] if len(arr) >= 5 else arr
        feats[f"{v}_delta_last_vs_first5"] = float(feats[f"{v}_last"] - np.nanmean(first5))
        feats[f"{v}_miss_count"] = int(np.isnan(arr).sum())
    return feats

def build_vitals_df(feature_fn):
    recs = []
    max_day = 0
    for rec in vitals_list:
        stay_id = rec["stay_id"]
        days = rec.get("days", [])
        if days:
            max_day = max(max_day, max([d.get("day", 0) for d in days if d.get("day") is not None] or [0]))
        vf = feature_fn(days)
        note_all, note_last2, nf = compute_note_features(days)
        ab = compute_abnormal_counts(days)
        row = {"stay_id": stay_id, "note_text_all": note_all, "note_text_last2": note_last2}
        row.update(vf)
        row.update(nf)
        row.update(ab)
        recs.append(row)
    return pd.DataFrame(recs), max_day

vitals_v1_df, max_day_v1 = build_vitals_df(vitals_features_v1)
vitals_v2_df, max_day_v2 = build_vitals_df(vitals_features_v2)

if max(max_day_v1, max_day_v2) > 10:
    raise RuntimeError(f"[LEAKAGE ABORT] vitals_timeseries contains day>10 (max_day={max(max_day_v1, max_day_v2)}). Must restrict to day<=10 only.")

def merge_all(stays, vitals_df):
    return stays.merge(patients, on="patient_id", how="left").merge(vitals_df, on="stay_id", how="left")

train_v1 = merge_all(stays_train, vitals_v1_df)
test_v1  = merge_all(stays_test,  vitals_v1_df)

train_v2 = merge_all(stays_train, vitals_v2_df)
test_v2  = merge_all(stays_test,  vitals_v2_df)

def numeric_cols(df):
    exclude = set(ID_COLS + [TARGET] + CAT_COLS + ["admission_reason", "note_text_all", "note_text_last2"])
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

NUM_V1 = numeric_cols(train_v1)
NUM_V2 = numeric_cols(train_v2)

# -------------------------
# 8) Modeling helpers (OOF + robust threshold)
# -------------------------
def build_lr_pipeline(numeric_cols, include_cat: bool, note_col: str, C=0.05,
                      tfidf_note_max=8000, tfidf_adm_max=5000):
    transformers = []
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ])
    transformers.append(("num", num_pipe, numeric_cols))

    if include_cat:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(("cat", cat_pipe, CAT_COLS))

    transformers.append(("adm_tfidf", TfidfVectorizer(min_df=1, max_features=tfidf_adm_max,
                                                      ngram_range=(1, 2), lowercase=True), "admission_reason"))
    transformers.append(("note_tfidf", TfidfVectorizer(min_df=2, max_features=tfidf_note_max,
                                                       ngram_range=(1, 2), lowercase=True), note_col))

    pre = ColumnTransformer(transformers, remainder="drop", sparse_threshold=0.3)
    clf = LogisticRegression(
        solver="liblinear", penalty="l2", C=C,
        class_weight="balanced", max_iter=5000, random_state=42
    )
    return Pipeline([("pre", pre), ("model", clf)])

def oof_predict_proba(pipe, X, y):
    oof = np.zeros(len(y), dtype=float)
    folds = np.zeros(len(y), dtype=int)
    for f, (tr, va) in enumerate(CV.split(X, y)):
        m = clone(pipe)
        m.fit(X.iloc[tr], y.iloc[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        folds[va] = f
    return oof, folds

def sweep_thresholds(y_true, proba, thresholds):
    rows = []
    for thr in thresholds:
        pred = (proba >= thr).astype(int)
        rows.append({
            "threshold": float(thr),
            "macro_f1": float(f1_score(y_true, pred, average="macro")),
            "pos_rate": float(pred.mean()),
        })
    return pd.DataFrame(rows)

def pick_threshold(sweep_df, prevalence, eps=0.001):
    # maximize macro_f1; within eps, pick pos_rate closest to prevalence; tie-break higher threshold
    best_f1 = float(sweep_df["macro_f1"].max())
    cand = sweep_df[sweep_df["macro_f1"] >= best_f1 - eps].copy()
    cand["pos_gap"] = (cand["pos_rate"] - prevalence).abs()
    cand = cand.sort_values(["pos_gap", "threshold"], ascending=[True, False])
    thr = float(cand.iloc[0]["threshold"])
    f1_at_thr = float(cand.iloc[0]["macro_f1"])
    return thr, f1_at_thr, best_f1

def eval_pipeline(name, df_train, numeric_cols, note_col, include_cat, C=0.05):
    X = df_train.drop(columns=[TARGET])
    y = df_train[TARGET]
    pipe = build_lr_pipeline(numeric_cols, include_cat=include_cat, note_col=note_col, C=C)

    oof, folds = oof_predict_proba(pipe, X, y)

    coarse = np.round(np.arange(0.05, 0.96, 0.05), 2)
    sweep1 = sweep_thresholds(y, oof, coarse)
    thr0 = float(sweep1.loc[sweep1["macro_f1"].idxmax(), "threshold"])

    fine = np.round(np.arange(max(0.01, thr0 - 0.10), min(0.99, thr0 + 0.10) + 1e-9, 0.01), 2)
    sweep2 = sweep_thresholds(y, oof, fine)

    sweep = pd.concat([sweep1, sweep2], ignore_index=True).drop_duplicates("threshold").sort_values("threshold").reset_index(drop=True)

    prevalence = float(y.mean())
    thr, f1_at_thr, f1_best = pick_threshold(sweep, prevalence, eps=0.001)

    pred = (oof >= thr).astype(int)
    cm = confusion_matrix(y, pred)
    per_fold = [float(f1_score(y.iloc[folds == f], (oof[folds == f] >= thr).astype(int), average="macro")) for f in range(5)]

    result = {
        "name": name,
        "macro_f1_oof": float(f1_score(y, pred, average="macro")),
        "macro_f1_best_possible": float(f1_best),
        "threshold": float(thr),
        "pos_rate_oof": float(pred.mean()),
        "auc_oof": float(roc_auc_score(y, oof)),
        "brier_oof": float(brier_score_loss(y, oof)),
        "per_fold": per_fold,
        "per_fold_mean": float(np.mean(per_fold)),
        "per_fold_std": float(np.std(per_fold)),
        "confusion_matrix": cm.tolist(),
        "n_numeric": int(len(numeric_cols)),
        "note_col": note_col,
        "include_cat": bool(include_cat),
        "C": float(C),
        "model_family": "LogisticRegression(liblinear, L2)"
    }
    return result, pipe, oof, folds, sweep

# -------------------------
# 9) Strict postmortem metrics from saved iter0003 (if available)
# -------------------------
def eval_saved_oof(oof_path: Path, sweep_path: Path):
    oof_df = pd.read_csv(oof_path)
    sweep_df = pd.read_csv(sweep_path)

    score_col = None
    for c in ["macro_f1", "macro_f1_oof", "f1"]:
        if c in sweep_df.columns:
            score_col = c
            break
    if score_col is None:
        raise ValueError(f"Unknown sweep schema: {sweep_df.columns.tolist()}")

    thr = float(sweep_df.loc[sweep_df[score_col].astype(float).idxmax(), "threshold"])
    y = oof_df["y_true"].values
    p = oof_df["oof_proba"].values
    pred = (p >= thr).astype(int)
    cm = confusion_matrix(y, pred)

    per_fold = []
    if "fold" in oof_df.columns:
        for f in sorted(oof_df["fold"].unique()):
            idx = (oof_df["fold"].values == f)
            per_fold.append(float(f1_score(y[idx], (p[idx] >= thr).astype(int), average="macro")))

    return {
        "name": f"SAVED_{oof_path.stem}",
        "threshold": thr,
        "macro_f1_oof": float(f1_score(y, pred, average="macro")),
        "auc_oof": float(roc_auc_score(y, p)),
        "pos_rate_oof": float(pred.mean()),
        "confusion_matrix": cm.tolist(),
        "per_fold": per_fold,
        "per_fold_mean": float(np.mean(per_fold)) if per_fold else None,
        "per_fold_std": float(np.std(per_fold)) if per_fold else None,
    }

saved_rows = []
iter0003_oof = ART_ROOT / "iter0003_oof_predictions.csv"
iter0003_thr = ART_ROOT / "iter0003_threshold_sweep.csv"
if iter0003_oof.exists() and iter0003_thr.exists():
    saved_rows.append(eval_saved_oof(iter0003_oof, iter0003_thr))

# -------------------------
# 10) Minimal ablations (one change at a time) — fixed CV
# -------------------------
print("\n=== ABLATIONS (fixed CV: StratifiedKFold 5, seed42) ===")

# Baseline candidate (v2 vitals + cats + note_last2 + LR L2 C=0.05)
cand_res, cand_pipe, cand_oof, cand_folds, cand_sweep = eval_pipeline(
    "CAND_v2_vitalsTrend_cat_noteLast2_LR_L2_C0.05", train_v2, NUM_V2, "note_text_last2", include_cat=True, C=0.05
)

# Ablation A: old vitals (v1), everything else same
abl_v1_res, _, _, _, _ = eval_pipeline(
    "ABL_v1_vitalsFlat_cat_noteLast2_LR_L2_C0.05", train_v1, NUM_V1, "note_text_last2", include_cat=True, C=0.05
)

# Ablation B: drop categorical
abl_nocat_res, _, _, _, _ = eval_pipeline(
    "ABL_v2_noCat_noteLast2_LR_L2_C0.05", train_v2, NUM_V2, "note_text_last2", include_cat=False, C=0.05
)

# Ablation C: switch to note_text_all (text window change)
abl_noteall_res, _, _, _, _ = eval_pipeline(
    "ABL_v2_cat_noteAll_LR_L2_C0.05", train_v2, NUM_V2, "note_text_all", include_cat=True, C=0.05
)

ablation_df = pd.DataFrame(saved_rows + [abl_v1_res, abl_nocat_res, abl_noteall_res, cand_res]).copy()
ablation_df = ablation_df.sort_values("macro_f1_oof", ascending=False).reset_index(drop=True)

print(ablation_df[["name","macro_f1_oof","auc_oof","threshold","pos_rate_oof","per_fold_mean","per_fold_std"]].to_string(index=False))
print("\nConfusion matrices (TN FP / FN TP):")
for _, r in ablation_df.iterrows():
    print(f"- {r['name']}: {r['confusion_matrix']}")

print("\nTop-3 regression hypotheses (iter0002 -> iter0003):")
print("1) Feature instability: vitals flattened / missing recent trend features → discrimination drop (AUC↓), FP↑.")
print("2) Feature removal: categorical cols (unit_type/sex/insurance/zip3) removed → consistent loss in ablation.")
print("3) Model/regularization mismatch: L1->L2 + high-dim sparse text could increase variance and FP rate if not tuned.")

# -------------------------
# 11) Select model vs P* (rollback if not beating)
# -------------------------
pstar_before = dict(PSTAR)
candidate_f1 = float(cand_res["macro_f1_oof"])
update_pstar = (candidate_f1 > float(pstar_before.get("oof_macro_f1", -1.0)) + 1e-9)

if update_pstar:
    selected_pipe = cand_pipe
    selected_df_train = train_v2
    selected_df_test  = test_v2
    selected_thr = float(cand_res["threshold"])
    decision = f"UPDATE_PSTAR -> {ITER_TAG} (OOF {candidate_f1:.6f} > P* {pstar_before['oof_macro_f1']:.6f})"
else:
    # Rollback path: try to load P* model if it exists; otherwise fallback to candidate (cannot rollback deterministically)
    pstar_iter = pstar_before.get("iteration_id", None)
    pstar_model_path = CKPT_ROOT / f"iter{int(pstar_iter):04d}" / f"iter{int(pstar_iter):04d}_model.joblib" if pstar_iter is not None else None
    if pstar_model_path and pstar_model_path.exists():
        selected_pipe = joblib.load(pstar_model_path)
        selected_df_train = train_v2  # assume compatible; if not, user should keep schema consistent
        selected_df_test  = test_v2
        selected_thr = float(pstar_before.get("threshold", 0.5))
        decision = f"ROLLBACK_TO_PSTAR iter{int(pstar_iter):04d} (candidate {candidate_f1:.6f} <= P* {pstar_before['oof_macro_f1']:.6f})"
    else:
        selected_pipe = cand_pipe
        selected_df_train = train_v2
        selected_df_test  = test_v2
        selected_thr = float(cand_res["threshold"])
        decision = f"[WARN] Cannot load P* model; using candidate as best available (candidate {candidate_f1:.6f} <= P* {pstar_before['oof_macro_f1']:.6f})."

print("\nModel selection decision:", decision)

# -------------------------
# 12) Fit full model + write submission
# -------------------------
X_full = selected_df_train.drop(columns=[TARGET])
y_full = selected_df_train[TARGET]
X_test = selected_df_test.copy()

selected_pipe.fit(X_full, y_full)
test_proba = selected_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= selected_thr).astype(int)

submission = pd.DataFrame({
    "stay_id": stays_test["stay_id"].values,
    "discharge_ready_day11": test_pred.astype(int)
})

SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"
submission.to_csv(SUB_PATH, index=False)

sub_pos_rate = float(submission["discharge_ready_day11"].mean())
print(f"\nSubmission written: {SUB_PATH}")
print(f"Chosen threshold: {selected_thr:.4f}")
print(f"Submission positive rate: {sub_pos_rate:.4f}")

# -------------------------
# 13) Save artifacts + checkpoint bundle (this iter)
# -------------------------
ITER_DIR = CKPT_ROOT / ITER_TAG
ITER_DIR.mkdir(parents=True, exist_ok=True)

# Save OOF + sweep for this iter (candidate is what we evaluate)
oof_out = pd.DataFrame({
    "stay_id": train_v2["stay_id"].values,
    "y_true": train_v2[TARGET].values,
    "oof_proba": cand_oof,
    "fold": cand_folds
})
oof_out_path = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
thr_sweep_path = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"
postmortem_path = ART_ROOT / f"{ITER_TAG}_postmortem_ablation_results.csv"

oof_out.to_csv(oof_out_path, index=False)
cand_sweep.to_csv(thr_sweep_path, index=False)
ablation_df.to_csv(postmortem_path, index=False)

# checkpoint copies
oof_out.to_csv(ITER_DIR / f"{ITER_TAG}_oof_predictions.csv", index=False)
cand_sweep.to_csv(ITER_DIR / f"{ITER_TAG}_threshold_sweep.csv", index=False)
submission.to_csv(ITER_DIR / f"{ITER_TAG}_submission.csv", index=False)
ablation_df.to_csv(ITER_DIR / f"{ITER_TAG}_postmortem_ablation_results.csv", index=False)

# Save model
joblib.dump(selected_pipe, ITER_DIR / f"{ITER_TAG}_model.joblib")

# Config / schema / metrics
ts = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
config = {
    "team": TEAM,
    "task": TASK,
    "version": VERSION,
    "tag": TAG,
    "iteration_id": ITER_ID,
    "timestamp": ts,
    "phase": "Modeling",
    "block_id": BLOCK_ID,
    "block_iter": BLOCK_ITER,
    "branch": BRANCH,
    "seeds": {"python": 42, "numpy": 42, "cv_random_state": 42},
    "scoped_artifact_discovery": {
        "art_root": str(ART_ROOT),
        "ckpt_root": str(CKPT_ROOT),
        "discovered_iter_tags": iter_tags,
        "deterministic_sorting": True,
        "hard_pair_checks": True
    },
    "data": {
        "paths": {
            "stays_train": "stays_train.csv",
            "stays_test": "stays_test.csv",
            "patients": "patients.csv",
            "vitals_timeseries": "vitals_timeseries.json"
        },
        "joins": [
            {"left": "stays", "right": "patients", "key": "patient_id", "how": "left"},
            {"left": "stays", "right": "vitals_features", "key": "stay_id", "how": "left"},
        ]
    },
    "features": {
        "categorical_cols": CAT_COLS,
        "numeric_cols_count_v2": len(NUM_V2),
        "vitals_feature_set": "v2_trend_volatility (slope_lr, slope_last3, mean_abs_diff, delta_last_vs_first5, range, last2/last3 stats) + abnormal counts",
        "text_cols": ["admission_reason", "note_text_last2"],
        "tfidf": {
            "admission_reason": {"min_df": 1, "max_features": 5000, "ngram_range": [1, 2]},
            "note_text_last2": {"min_df": 2, "max_features": 8000, "ngram_range": [1, 2]}
        }
    },
    "model": {
        "name": "LogisticRegression_liblinear_L2",
        "params": {"solver": "liblinear", "penalty": "l2", "C": 0.05, "class_weight": "balanced", "max_iter": 5000, "random_state": 42}
    },
    "validation": {
        "cv": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 42,
        "threshold_selection": "coarse(0.05..0.95 step0.05)+fine(+/-0.1 step0.01) maximize OOF macro-F1; tie-break within eps=0.001 by pos_rate close to train prevalence then higher threshold",
        "chosen_threshold": float(selected_thr)
    },
    "hm_message": "Real Macro-F1 regressed to 0.7041 (latest submission), best real 0.7768 at iter0002; fixed artifact discovery mixing versions; keep fixed CV (StratifiedKFold 5, seed42).",
    "real_score_latest": 0.7041,
    "real_score_best_known": 0.7768
}

schema = {
    "id_cols": ID_COLS,
    "target_col": TARGET,
    "categorical_cols": CAT_COLS,
    "numeric_cols_v2": NUM_V2,
    "text_cols": ["admission_reason", "note_text_last2"]
}

metrics = {
    "oof_macro_f1": float(cand_res["macro_f1_oof"]),
    "oof_auc": float(cand_res["auc_oof"]),
    "oof_brier": float(cand_res["brier_oof"]),
    "oof_pos_rate": float(cand_res["pos_rate_oof"]),
    "oof_confusion_matrix": cand_res["confusion_matrix"],
    "oof_per_fold_macro_f1": cand_res["per_fold"],
    "chosen_threshold": float(selected_thr),
    "submission_pos_rate": float(sub_pos_rate),
    "pstar_before": pstar_before,
}

json.dump(config, open(ITER_DIR / f"{ITER_TAG}_config.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
json.dump(schema, open(ITER_DIR / f"{ITER_TAG}_schema.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
json.dump(metrics, open(ITER_DIR / f"{ITER_TAG}_metrics.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)

# Update P* if improved
pstar_after = dict(pstar_before)
if update_pstar:
    pstar_after = {"iteration_id": ITER_ID, "oof_macro_f1": float(cand_res["macro_f1_oof"]), "threshold": float(selected_thr), "source": "this_iter"}
    json.dump(pstar_after, open(PSTAR_PATH, "w", encoding="utf-8"), indent=2, ensure_ascii=False)

print("\nP* (after):", pstar_after)

# -------------------------
# 14) Append iteration detail log (append-only) + consultant packet (v3)
# -------------------------
iteration_detail = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": ts,
    "phase": "Modeling",
    "cwd": str(CWD),
    "project_root": str(PROJECT_ROOT),
    "artifacts_root": str(ART_ROOT),
    "checkpoints_root": str(CKPT_ROOT),
    "discovered_iter_tags": iter_tags,
    "data_paths_used": {
        "stays_train": str(PROJECT_ROOT / "stays_train.csv"),
        "stays_test": str(PROJECT_ROOT / "stays_test.csv"),
        "patients": str(PROJECT_ROOT / "patients.csv"),
        "vitals_timeseries": str(PROJECT_ROOT / "vitals_timeseries.json")
    },
    "join_keys": [
        {"left": "stays", "right": "patients", "key": "patient_id", "how": "left"},
        {"left": "stays", "right": "vitals_features", "key": "stay_id", "how": "left"}
    ],
    "leakage_checks_performed": [
        f"Assert vitals max day <= 10 (observed max_day={max_day_v2})",
        "No target joins; only patient demographics + first-10-days vitals/notes",
        "TF-IDF + encoders fit within folds (Pipeline/ColumnTransformer)",
        "Threshold tuned on OOF only; test threshold fixed from OOF"
    ],
    "feature_summary": {
        "vitals_feature_set": "v2_trend_volatility + abnormal counts",
        "numeric_feature_count": int(len(NUM_V2)),
        "categorical_cols": CAT_COLS,
        "text_cols": ["admission_reason", "note_text_last2"],
        "tfidf_params": config["features"]["tfidf"]
    },
    "models_attempted": saved_rows + [abl_v1_res, abl_nocat_res, abl_noteall_res, cand_res],
    "selected_model": cand_res["name"],
    "validation_protocol": config["validation"],
    "metrics": {
        "oof_macro_f1": cand_res["macro_f1_oof"],
        "oof_per_fold": cand_res["per_fold"],
        "oof_confusion_matrix": cand_res["confusion_matrix"],
        "oof_auc": cand_res["auc_oof"],
        "oof_pos_rate": cand_res["pos_rate_oof"],
        "chosen_threshold": float(selected_thr),
        "submission_pos_rate": float(sub_pos_rate)
    },
    "deltas_vs_previous_iteration": {
        "artifact_fix": "Scoped discovery under ART_ROOT/CKPT_ROOT only; deterministic sorting; abort on missing oof/sweep pairs.",
        "main_gain": "Vitals v2 trend/volatility features restored; +cats retained; tuned LR regularization.",
        "postmortem_summary": "Regression consistent with dropping cats + changing model/regularization + weaker vitals temporal signals (AUC↓, FP↑)."
    },
    "known_defects_limitations": [
        "Predicted positive rate is high; may inflate FP if hidden test prevalence lower.",
        "Notes might include explicit discharge planning language (within day<=10); monitor robustness.",
    ],
    "next_step_hypothesis": [
        "Low-risk 1: keep v2 vitals + cats, but add prevalence-regularized threshold objective to reduce FP.",
        "Low-risk 2: light ensembling (avg of C=0.05 and C=0.1) + same threshold selection; check fold variance.",
        "High-risk: GBDT on numeric+cat plus TruncatedSVD(text) to capture nonlinear interactions (CV unchanged)."
    ],
    "hm_message": config["hm_message"],
    "real_score": config["real_score_latest"],
    "best_real_score_known": config["real_score_best_known"],
    "pstar_before": pstar_before,
    "pstar_after": pstar_after
}

# Append-only write
with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_detail, ensure_ascii=False) + "\n")

consultant_packet = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": ts,
    "what_changed": [
        "Scoped artifact discovery + deterministic file selection + hard checks",
        "Vitals v2 trend/volatility features + abnormal counts",
        "LR liblinear L2 C=0.05; robust threshold selection"
    ],
    "metric_changes": {
        "pstar_before_oof_macro_f1": float(pstar_before.get("oof_macro_f1", -1)),
        "candidate_oof_macro_f1": float(cand_res["macro_f1_oof"]),
        "delta": float(cand_res["macro_f1_oof"]) - float(pstar_before.get("oof_macro_f1", -1)),
        "candidate_auc": float(cand_res["auc_oof"]),
        "candidate_oof_pos_rate": float(cand_res["pos_rate_oof"]),
        "submission_pos_rate": float(sub_pos_rate),
        "threshold": float(selected_thr)
    },
    "suspected_leakage_risks": [
        "Notes may contain explicit discharge intent; still day<=10 but close to label.",
        "Asserted no vitals day>10."
    ],
    "complexity_drift_risks": [
        "TF-IDF vocab sizes can reach ~13k combined; watch regularization and stability."
    ],
    "evaluation_alignment_risks": [
        "Threshold tuned on OOF; may drift if test prevalence differs. CV protocol unchanged (SKFold5 seed42)."
    ],
    "unknown_unknowns": [
        "Unit/site shift; missing notes in test; different note style."
    ],
    "recommendations": iteration_detail["next_step_hypothesis"]
}

packet_path = CONS_ROOT / f"{TAG}_iter{ITER_ID}_packet.json"
json.dump(consultant_packet, open(packet_path, "w", encoding="utf-8"), indent=2, ensure_ascii=False)

print("\nConsultant packet written:", packet_path)
print("Iteration detail appended:", LOG_PATH)
print("Checkpoint saved under:", ITER_DIR)
print("Artifacts saved under:", ART_ROOT)

=== PREFLIGHT ===
CWD: D:\AgentDs\agent_ds_healthcare
PROJECT_ROOT: D:\AgentDs\agent_ds_healthcare
ART_ROOT: D:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT: D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
Discovered iter tags (scoped): [0, 2, 3]
P* (before): {'version': 'v3', 'updated_at': '2026-03-01T02:14:36.201141-05:00', 'iteration_id': 3, 'macro_f1_oof': 0.7279551448754127, 'best_threshold': 0.4, 'checkpoint_dir': 'checkpoints\\clai_ch3_v3\\iter0003'}
P* (normalized): {'version': 'v3', 'updated_at': '2026-03-01T02:14:36.201141-05:00', 'iteration_id': 3, 'macro_f1_oof': 0.7279551448754127, 'best_threshold': 0.4, 'checkpoint_dir': 'checkpoints\\clai_ch3_v3\\iter0003', 'oof_macro_f1': 0.7279551448754127, 'threshold': 0.4, 'source': 'loaded_or_computed', 'schema_version': 1}

=== CONFIG DIFF (iter0002 vs iter0003) ===
best_config_path: D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3\iter0002\iter0002_config.json
cur_config_path : D:\AgentDs\agent_ds_healt

# Iteration 5
- 0.7746

In [23]:
import os, json, random, re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.base import clone
import joblib

# =====================
# 0) Config + paths
# =====================
TEAM, TASK, VERSION = "clai", "ch3", "v3"
TAG = f"{TEAM}_{TASK}_{VERSION}"
SEED = 42
N_SPLITS = 5

np.random.seed(SEED)
random.seed(SEED)

def _find_project_root():
    # Prefer CWD if it looks like the project; fallback to /mnt/data (sandbox).
    cand = Path.cwd()
    if (cand / "stays_train.csv").exists():
        return cand
    cand2 = Path("/mnt/data")
    if (cand2 / "stays_train.csv").exists():
        return cand2
    return Path.cwd()

PROJECT_ROOT = _find_project_root()
ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"

ART_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
CONSULT_ROOT.mkdir(parents=True, exist_ok=True)

print("=== PREFLIGHT ===")
print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_ROOT:", ART_ROOT)
print("CKPT_ROOT:", CKPT_ROOT)

def discover_iter_tags_scoped(art_root: Path):
    # Deterministic discovery: ONLY under art_root; never rglob from '.' or PROJECT_ROOT.
    oof_files = sorted(art_root.glob("iter????_oof_predictions.csv"))
    sweep_files = sorted(art_root.glob("iter????_threshold_sweep.csv"))
    tags = sorted({p.name.split("_")[0] for p in (oof_files + sweep_files)})
    # Hard checks: every discovered tag must have BOTH oof + sweep, else abort.
    for t in tags:
        oof_p = art_root / f"{t}_oof_predictions.csv"
        sw_p = art_root / f"{t}_threshold_sweep.csv"
        if (not oof_p.exists()) or (not sw_p.exists()):
            raise RuntimeError(
                f"[ABORT] Incomplete artifacts for tag={t} under {art_root}. "
                f"Expected BOTH:\n- {oof_p}\n- {sw_p}\n"
                "This commonly indicates cross-version contamination or an incomplete run. "
                "Fix artifacts so v3 dir only contains complete iter bundles."
            )
    return tags

discovered_tags = discover_iter_tags_scoped(ART_ROOT)
print("Discovered iter tags (scoped):", discovered_tags)

# =====================
# 1) Iteration id
# =====================
LOG_PATH = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"

def next_iteration_id(log_path: Path) -> int:
    if not log_path.exists():
        return 0
    max_id = -1
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                max_id = max(max_id, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return max_id + 1

ITER_ID = next_iteration_id(LOG_PATH)
ITER_TAG = f"iter{ITER_ID:04d}"
BLOCK_ID = ITER_ID // 5
BLOCK_POS = ITER_ID % 5
BRANCH = "W4"  # FP-control / thresholding / conservative modeling

print(f"ITER_ID={ITER_ID}  ITER_TAG={ITER_TAG}  BLOCK_ID={BLOCK_ID}  BLOCK_POS={BLOCK_POS}  BRANCH={BRANCH}")

# =====================
# 2) Load data
# =====================
def dp(name: str) -> Path:
    p = PROJECT_ROOT / name
    if p.exists():
        return p
    p2 = Path("/mnt/data") / name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {name} under PROJECT_ROOT or /mnt/data")

stays_train = pd.read_csv(dp("stays_train.csv"))
stays_test  = pd.read_csv(dp("stays_test.csv"))
patients    = pd.read_csv(dp("patients.csv"))
with open(dp("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vit_list = json.load(f)

# Leakage check: max day must be <=10
max_day = max(d.get("day", 0) for item in vit_list for d in item.get("days", []))
assert max_day <= 10, f"Leakage risk: found vitals day={max_day} (>10). Abort."

vit_map = {int(item["stay_id"]): item.get("days", []) for item in vit_list}

# =====================
# 3) Feature engineering (v2: per-day + deltas + trends; optional note keyword counts)
# =====================
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]

KEYWORDS = [
    "stable","improving","worsening","afebrile","fever","chills","hypotensive","tachy",
    "culture","antibiotic","abx","iv","oxygen","weaned","dyspnea","cough",
    "pain","nausea","vomit","n/v","ambulated","pt","ot","telemetry","fluids","room air","sats",
    "discharge","transfer"
]
KW_PATTERNS = [(kw, re.compile(re.escape(kw), re.IGNORECASE)) for kw in KEYWORDS]

def safe_nan(fn, arr, default=np.nan):
    try:
        return fn(arr)
    except Exception:
        return default

def build_stay_features(stays_df: pd.DataFrame, include_keywords: bool) -> pd.DataFrame:
    base = stays_df.merge(patients, on="patient_id", how="left")
    rows = []
    for sid in base["stay_id"].astype(int).tolist():
        days = sorted(vit_map.get(int(sid), []), key=lambda x: x.get("day", 0))
        day_idx = np.array([d.get("day", np.nan) for d in days], dtype=float)

        notes = [(d.get("note") or "") for d in days]
        note_all = " ".join(notes).strip()
        note_last2 = " ".join(notes[-2:]).strip()

        feat = {
            "stay_id": int(sid),
            "note_text_all": note_all,
            "note_text_last2": note_last2,
            "note_len_all": len(note_all),
            "note_len_last2": len(note_last2),
        }

        for col in VITAL_COLS:
            vals = np.array([d.get(col, np.nan) for d in days], dtype=float)

            # per-day values (day1..day10)
            for i, day in enumerate(range(1, 11)):
                feat[f"{col}_day{day}"] = vals[i] if i < len(vals) else np.nan

            # day-to-day diffs (d2-d1 .. d10-d9)
            diffs = np.diff(vals) if len(vals) >= 2 else np.array([], dtype=float)
            for j in range(1, 10):
                feat[f"{col}_d{j+1}_minus_d{j}"] = diffs[j-1] if (j-1) < len(diffs) else np.nan
            feat[f"{col}_delta10_9"] = vals[-1] - vals[-2] if len(vals) >= 2 else np.nan

            # global stats
            feat[f"{col}_mean"] = safe_nan(np.nanmean, vals)
            feat[f"{col}_std"] = safe_nan(np.nanstd, vals)
            feat[f"{col}_min"] = safe_nan(np.nanmin, vals)
            feat[f"{col}_max"] = safe_nan(np.nanmax, vals)
            feat[f"{col}_median"] = safe_nan(np.nanmedian, vals)
            feat[f"{col}_first"] = vals[0] if len(vals) else np.nan
            feat[f"{col}_last"]  = vals[-1] if len(vals) else np.nan

            # recent vs early (last 2 days vs first 8 days)
            last2 = vals[-2:] if len(vals) >= 2 else vals
            early = vals[:8] if len(vals) >= 8 else (vals[:-2] if len(vals) > 2 else vals)

            feat[f"{col}_last2_mean"] = safe_nan(np.nanmean, last2)
            feat[f"{col}_early_mean"] = safe_nan(np.nanmean, early)
            feat[f"{col}_delta_last2_early"] = feat[f"{col}_last2_mean"] - feat[f"{col}_early_mean"]
            feat[f"{col}_delta_last_first"] = feat[f"{col}_last"] - feat[f"{col}_first"]

            # trend slope across days 1..10
            mask = (~np.isnan(vals)) & (~np.isnan(day_idx))
            feat[f"{col}_slope"] = np.polyfit(day_idx[mask], vals[mask], 1)[0] if mask.sum() >= 2 else np.nan

            # volatility (diff mean/std) + last3 std
            feat[f"{col}_diff_mean"] = safe_nan(np.nanmean, diffs) if len(diffs) else np.nan
            feat[f"{col}_diff_std"]  = safe_nan(np.nanstd, diffs) if len(diffs) else np.nan
            feat[f"{col}_last3_std"] = safe_nan(np.nanstd, vals[-3:]) if len(vals) >= 3 else safe_nan(np.nanstd, vals)

            # missingness
            feat[f"{col}_missing"] = int(np.isnan(vals).sum())

            # simple clinical threshold counts (low-cost, explainable)
            if col == "hr":
                feat["hr_gt100"] = int(np.nansum(vals > 100))
                feat["hr_lt60"]  = int(np.nansum(vals < 60))
            if col == "sbp":
                feat["sbp_lt90"]  = int(np.nansum(vals < 90))
                feat["sbp_gt160"] = int(np.nansum(vals > 160))
            if col == "temp_c":
                feat["temp_gt38"] = int(np.nansum(vals > 38))
                feat["temp_lt36"] = int(np.nansum(vals < 36))
            if col == "rr":
                feat["rr_gt22"] = int(np.nansum(vals > 22))
            if col == "dbp":
                feat["dbp_lt60"] = int(np.nansum(vals < 60))

        if include_keywords:
            for kw, pat in KW_PATTERNS:
                key = kw.replace(" ", "_").replace("/", "_")
                feat[f"all_kw_{key}"]   = len(pat.findall(note_all))
                feat[f"last2_kw_{key}"] = len(pat.findall(note_last2))

        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    out = base.merge(feat_df, on="stay_id", how="left")
    return out

train_full_kw = build_stay_features(stays_train, include_keywords=True)
test_full_kw  = build_stay_features(stays_test,  include_keywords=True)

y = train_full_kw["discharge_ready_day11"].astype(int).values
train_prev = float(y.mean())

# =====================
# 4) Modeling utilities
# =====================
ID_COLS = ["stay_id", "patient_id"]
CAT_COLS = ["unit_type", "sex", "insurance", "zip3", "admission_reason"]
TEXT_COL = "note_text_all"
DROP_TEXT_HELPER_COLS = ["note_text_last2"]  # only used to compute keyword counts

def text_selector(x):
    s = x.iloc[:, 0] if isinstance(x, pd.DataFrame) else pd.Series(x)
    return s.fillna("").astype(str).values

def make_preprocessor(df: pd.DataFrame, include_keywords: bool):
    # numeric columns = everything not id/cat/text/target/helper text
    num_cols = [c for c in df.columns if c not in (["discharge_ready_day11"] + ID_COLS + CAT_COLS + [TEXT_COL] + DROP_TEXT_HELPER_COLS)]
    if not include_keywords:
        num_cols = [c for c in num_cols if (not c.startswith("all_kw_")) and (not c.startswith("last2_kw_"))]

    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler(with_mean=False))]), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
            ("note", Pipeline([("selector", FunctionTransformer(text_selector, validate=False)),
                               ("tfidf", TfidfVectorizer(min_df=2, max_features=4000, ngram_range=(1,2)))]),
             [TEXT_COL]),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    schema = {"num_cols": num_cols, "cat_cols": CAT_COLS, "text_col": TEXT_COL,
              "tfidf": {"min_df": 2, "max_features": 4000, "ngram_range": (1,2)},
              "include_keywords": include_keywords}
    return pre, schema

def build_fold_cache(df_X: pd.DataFrame, y: np.ndarray, preprocessor: ColumnTransformer):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_cache = []
    fold_ids = np.zeros(len(df_X), dtype=int)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(df_X, y)):
        pre = clone(preprocessor)
        X_tr = pre.fit_transform(df_X.iloc[tr_idx], y[tr_idx])
        X_va = pre.transform(df_X.iloc[va_idx])
        fold_cache.append({"fold": fold, "tr_idx": tr_idx, "va_idx": va_idx, "X_tr": X_tr, "y_tr": y[tr_idx], "X_va": X_va})
        fold_ids[va_idx] = fold
    return fold_cache, fold_ids

def threshold_sweep(oof_proba: np.ndarray, y_true: np.ndarray, fold_ids: np.ndarray, thresholds: np.ndarray):
    rows = []
    for t in thresholds:
        pred = (oof_proba >= t).astype(int)
        cm = confusion_matrix(y_true, pred)
        f1s = f1_score(y_true, pred, average=None)
        macro = f1_score(y_true, pred, average="macro")
        pos_rate = float(pred.mean())
        fold_scores = []
        for f in range(N_SPLITS):
            idx = fold_ids == f
            fold_scores.append(f1_score(y_true[idx], pred[idx], average="macro"))
        rows.append({
            "threshold": float(t),
            "macro_f1": float(macro),
            "f1_0": float(f1s[0]),
            "f1_1": float(f1s[1]),
            "pos_rate": pos_rate,
            "fold_mean": float(np.mean(fold_scores)),
            "fold_std": float(np.std(fold_scores)),
            "tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        })
    return pd.DataFrame(rows).sort_values("threshold", ascending=True).reset_index(drop=True)

def choose_threshold(df_sweep: pd.DataFrame, prevalence: float, pos_tol: float = 0.07, macro_drop_tol: float = 0.005):
    best_macro = float(df_sweep["macro_f1"].max())
    # Prefer thresholds not too far from prevalence; if none, fall back to all
    df_pos_ok = df_sweep.loc[(df_sweep["pos_rate"] - prevalence).abs() <= pos_tol].copy()
    candidate_pool = df_pos_ok if len(df_pos_ok) else df_sweep.copy()

    pool_macro = float(candidate_pool["macro_f1"].max())
    df_near = candidate_pool.loc[candidate_pool["macro_f1"] >= (pool_macro - macro_drop_tol)].copy()
    df_near["pos_gap"] = (df_near["pos_rate"] - prevalence).abs()

    # Within near-best macro, pick closest pos_rate, then smallest fold_std, then highest macro
    df_near = df_near.sort_values(["pos_gap", "fold_std", "macro_f1", "threshold"], ascending=[True, True, False, True]).reset_index(drop=True)
    chosen = df_near.iloc[0].to_dict()
    chosen["best_macro_overall"] = best_macro
    return chosen

def oof_from_cache(fold_cache, C: float):
    oof = np.zeros(sum(len(fd["va_idx"]) for fd in fold_cache), dtype=float)  # placeholder (will resize below)
    # We'll allocate based on global n
    n = None
    for fd in fold_cache:
        n = int(max(n or 0, fd["va_idx"].max()+1))
    oof = np.zeros(n, dtype=float)

    for fd in fold_cache:
        clf = LogisticRegression(
            solver="liblinear",
            penalty="l1",
            C=C,
            class_weight="balanced",
            max_iter=5000,
            random_state=SEED
        )
        clf.fit(fd["X_tr"], fd["y_tr"])
        proba = clf.predict_proba(fd["X_va"])[:, 1]
        oof[fd["va_idx"]] = proba
    return oof

def evaluate_candidate(name, df_X, y, fold_cache, fold_ids, include_keywords, C):
    oof = oof_from_cache(fold_cache, C=C)
    thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
    df_sweep = threshold_sweep(oof, y, fold_ids, thresholds)
    chosen = choose_threshold(df_sweep, prevalence=train_prev, pos_tol=0.07, macro_drop_tol=0.005)
    chosen.update({
        "name": name,
        "C": C,
        "include_keywords": include_keywords,
        "auc": float(roc_auc_score(y, oof))
    })
    return chosen, df_sweep, oof

def ensure_fold_column(oof_df: pd.DataFrame, stays_train: pd.DataFrame, target_col: str,
                       n_splits=5, seed=42) -> pd.DataFrame:
    # If fold exists, keep it
    if "fold" in oof_df.columns:
        return oof_df

    # Rebuild deterministic folds from stays_train + fixed CV
    y = stays_train[target_col].astype(int).values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    folds = np.empty(len(y), dtype=int)
    for f, (_, va_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
        folds[va_idx] = f

    fold_map = pd.DataFrame({"stay_id": stays_train["stay_id"].values, "fold": folds})

    # Merge by stay_id (safer than relying on row order)
    out = oof_df.merge(fold_map, on="stay_id", how="left")

    if out["fold"].isna().any():
        miss = out.loc[out["fold"].isna(), "stay_id"].head(10).tolist()
        raise RuntimeError(
            f"[ABORT] Could not backfill fold for {out['fold'].isna().sum()} rows. "
            f"Example stay_id missing from fold_map: {miss}. "
            f"Check that OOF stay_ids match stays_train stay_ids."
        )

    out["fold"] = out["fold"].astype(int)
    return out

# =====================
# 5) Recompute P* (prefer iter0002 if available), normalize PSTAR schema
# =====================
def load_iter_metrics(iter_tag: str, prevalence: float):
    oof_p = ART_ROOT / f"{iter_tag}_oof_predictions.csv"
    sw_p  = ART_ROOT / f"{iter_tag}_threshold_sweep.csv"
    if (not oof_p.exists()) or (not sw_p.exists()):
        return None

    sw = pd.read_csv(sw_p)
    # normalize column names
    if "macro_f1" not in sw.columns and "oof_macro_f1" in sw.columns:
        sw = sw.rename(columns={"oof_macro_f1": "macro_f1"})
    if "threshold" not in sw.columns and "best_threshold" in sw.columns:
        sw = sw.rename(columns={"best_threshold": "threshold"})

    required = {"threshold", "macro_f1"}
    if not required.issubset(set(sw.columns)):
        raise RuntimeError(f"[ABORT] {sw_p} missing required columns {required}. Found: {list(sw.columns)}")

    sw2 = sw.copy()
    if "pos_rate" in sw2.columns:
        sw2["pos_gap"] = (sw2["pos_rate"] - prevalence).abs()
        sw2 = sw2.sort_values(["macro_f1", "pos_gap", "threshold"], ascending=[False, True, True])
    else:
        sw2 = sw2.sort_values(["macro_f1", "threshold"], ascending=[False, True])

    best_row = sw2.iloc[0]
    thr = float(best_row["threshold"])
    best_macro = float(best_row["macro_f1"])

    oof_df = pd.read_csv(oof_p)

    # Require minimal columns
    for col in ["stay_id", "y_true", "oof_proba"]:
        if col not in oof_df.columns:
            raise RuntimeError(f"[ABORT] {oof_p} missing column {col}. Found {list(oof_df.columns)}")

    # Backfill fold if missing (schema drift tolerant)
    oof_df = ensure_fold_column(oof_df, stays_train=stays_train, target_col="discharge_ready_day11",
                                n_splits=N_SPLITS, seed=SEED)

    # OPTIONAL: write back an upgraded artifact so future runs never hit this again
    oof_df.to_csv(oof_p, index=False)

    y_true = oof_df["y_true"].astype(int).values
    oof_proba = oof_df["oof_proba"].astype(float).values
    fold_ids = oof_df["fold"].astype(int).values

    pred = (oof_proba >= thr).astype(int)
    cm = confusion_matrix(y_true, pred)
    f1s = f1_score(y_true, pred, average=None)

    fold_scores = []
    for f in range(N_SPLITS):
        idx = fold_ids == f
        fold_scores.append(f1_score(y_true[idx], pred[idx], average="macro"))

    metrics = {
        "iteration_tag": iter_tag,
        "oof_macro_f1": float(best_macro),
        "threshold": thr,
        "best_threshold": thr,  # alias
        "pos_rate": float(pred.mean()),
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "f1_0": float(f1s[0]),
        "f1_1": float(f1s[1]),
        "confusion_matrix": cm.tolist(),
    }
    return metrics

pstar_metrics = None
if "iter0002" in discovered_tags:
    pstar_metrics = load_iter_metrics("iter0002", prevalence=train_prev)
else:
    # fallback: best among discovered tags
    best_tag = None
    best_score = -1.0
    for t in discovered_tags:
        m = load_iter_metrics(t, prevalence=train_prev)
        if m and m["oof_macro_f1"] > best_score:
            best_score = m["oof_macro_f1"]
            best_tag = t
            pstar_metrics = m

if pstar_metrics is None:
    pstar_metrics = {
        "iteration_tag": "NONE",
        "oof_macro_f1": -1.0,
        "threshold": 0.5,
        "best_threshold": 0.5,
        "pos_rate": None,
        "fold_mean": None,
        "fold_std": None,
        "f1_0": None,
        "f1_1": None,
        "confusion_matrix": None,
    }

PSTAR_PATH = CKPT_ROOT / "PSTAR.json"
with open(PSTAR_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "tag": TAG,
        "pstar": pstar_metrics,
        "train_prevalence": train_prev,
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "schema_version": 2,
        "notes": "Keys normalized: always uses oof_macro_f1 and threshold (plus alias best_threshold)."
    }, f, indent=2)

print("=== P* (recomputed) ===")
print(json.dumps(pstar_metrics, indent=2))

# =====================
# 6) Minimal ablations (one change at a time)
# =====================
# Base feature set: v2 vitals trajectory + cats + TF-IDF note_text_all
# A0: no keyword counts, C=0.15
# A1: + keyword counts, same C=0.15
# A2: + keyword counts, tune C=0.12
train_X = train_full_kw.drop(columns=["discharge_ready_day11"]).copy()
test_X  = test_full_kw.copy()

for c in CAT_COLS:
    train_X[c] = train_X[c].astype(str)
    test_X[c]  = test_X[c].astype(str)

# Build two preprocessors + fold caches once (saves lots of time)
pre_base, schema_base = make_preprocessor(train_X, include_keywords=False)
cache_base, fold_ids_base = build_fold_cache(train_X, y, pre_base)

pre_kw, schema_kw = make_preprocessor(train_X, include_keywords=True)
cache_kw, fold_ids_kw = build_fold_cache(train_X, y, pre_kw)

cand0, sweep0, oof0 = evaluate_candidate("A0_base_noKW_C0.15", train_X, y, cache_base, fold_ids_base, include_keywords=False, C=0.15)
cand1, sweep1, oof1 = evaluate_candidate("A1_addKW_C0.15",       train_X, y, cache_kw,   fold_ids_kw,   include_keywords=True,  C=0.15)
cand2, sweep2, oof2 = evaluate_candidate("A2_addKW_C0.12",       train_X, y, cache_kw,   fold_ids_kw,   include_keywords=True,  C=0.12)

# Attach confusion matrix counts for printing
def attach_cm_counts(cand, oof, fold_ids):
    thr = float(cand["threshold"])
    pred = (oof >= thr).astype(int)
    cm = confusion_matrix(y, pred)
    cand["tn"], cand["fp"], cand["fn"], cand["tp"] = int(cm[0,0]), int(cm[0,1]), int(cm[1,0]), int(cm[1,1])
    return cand

cand0 = attach_cm_counts(cand0, oof0, fold_ids_base)
cand1 = attach_cm_counts(cand1, oof1, fold_ids_kw)
cand2 = attach_cm_counts(cand2, oof2, fold_ids_kw)

ablations = [cand0, cand1, cand2]
abl_df = pd.DataFrame(ablations).sort_values("macro_f1", ascending=False)

print("=== Ablation summary (selected threshold per candidate) ===")
cols_show = ["name","macro_f1","best_macro_overall","threshold","pos_rate","fold_mean","fold_std","f1_0","f1_1","auc","tn","fp","fn","tp","include_keywords","C"]
print(abl_df[cols_show].to_string(index=False))

# Persist ablation summary for audit
ABL_PATH = ART_ROOT / f"{ITER_TAG}_ablation_summary.csv"
abl_df.to_csv(ABL_PATH, index=False)

# =====================
# 7) Candidate selection with hard gate vs P*
# =====================
MIN_DELTA = 0.002
STD_TOL = 0.01
POS_TOL = 0.07

pstar_macro = float(pstar_metrics.get("oof_macro_f1", -1.0))
pstar_std   = pstar_metrics.get("fold_std", None)

def passes_gate(c):
    if float(c["macro_f1"]) < pstar_macro + MIN_DELTA:
        return False
    if (pstar_std is not None) and (c.get("fold_std") is not None):
        if float(c["fold_std"]) > float(pstar_std) + STD_TOL:
            return False
    if abs(float(c["pos_rate"]) - train_prev) > POS_TOL:
        return False
    return True

eligible = [c for c in ablations if passes_gate(c)]
if eligible:
    selected = sorted(eligible, key=lambda d: float(d["macro_f1"]), reverse=True)[0]
    decision = f"SELECT_CANDIDATE (beats P* by >= {MIN_DELTA})"
else:
    selected = max(ablations, key=lambda d: float(d["macro_f1"]))
    decision = "NO_CANDIDATE_PASSES_GATE -> ROLLBACK_TO_PSTAR_IF_AVAILABLE"

print("=== Selection decision ===")
print(decision)
print("Selected:", selected["name"], "| macro_f1=", selected["macro_f1"], "thr=", selected["threshold"], "pos_rate=", selected["pos_rate"])

# Map selected to its oof/fold_ids/sweep/schema/preprocessor include_kw/C
if selected["name"].startswith("A0"):
    sel_oof, sel_folds, sel_sweep, sel_schema, sel_include_kw, sel_C = oof0, fold_ids_base, sweep0, schema_base, False, 0.15
elif selected["name"].startswith("A1"):
    sel_oof, sel_folds, sel_sweep, sel_schema, sel_include_kw, sel_C = oof1, fold_ids_kw, sweep1, schema_kw, True, 0.15
else:
    sel_oof, sel_folds, sel_sweep, sel_schema, sel_include_kw, sel_C = oof2, fold_ids_kw, sweep2, schema_kw, True, 0.12

# Rollback pipeline path (if exists)
pstar_pipeline_path = CKPT_ROOT / "iter0002" / "pipeline.joblib"
use_rollback_pipeline = (not eligible) and pstar_pipeline_path.exists()

# =====================
# 8) Fit final pipeline + write submission
# =====================
if use_rollback_pipeline:
    print("[ROLLBACK] Using P* pipeline from:", pstar_pipeline_path)
    pipe = joblib.load(pstar_pipeline_path)
    thr = float(pstar_metrics["threshold"])
else:
    pre, _ = make_preprocessor(train_X, include_keywords=sel_include_kw)
    clf = LogisticRegression(
        solver="liblinear",
        penalty="l1",
        C=float(sel_C),
        class_weight="balanced",
        max_iter=5000,
        random_state=SEED
    )
    pipe = Pipeline([("preprocess", pre), ("clf", clf)])
    pipe.fit(train_X, y)
    thr = float(selected["threshold"])

test_proba = pipe.predict_proba(test_X)[:, 1]
test_pred  = (test_proba >= thr).astype(int)

sub = pd.DataFrame({
    "stay_id": test_full_kw["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred.astype(int)
})
SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"
sub.to_csv(SUB_PATH, index=False)

print("=== Submission ===")
print("Path:", SUB_PATH)
print("Threshold used:", thr)
print("Submission positive-rate:", float(sub["discharge_ready_day11"].mean()))
print(sub.head())

# Also save a branch candidate snapshot under artifacts for traceability
CAND_PATH = ART_ROOT / f"{TAG}_{BRANCH}_block{BLOCK_ID}_iter{ITER_TAG}_candidate.csv"
sub.to_csv(CAND_PATH, index=False)

# =====================
# 9) Save artifacts + checkpoint bundle
# =====================
# OOF predictions + threshold sweep for THIS iteration
oof_df = pd.DataFrame({
    "stay_id": train_full_kw["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "oof_proba": sel_oof.astype(float),
    "fold": sel_folds.astype(int)
})
oof_path = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
oof_df.to_csv(oof_path, index=False)

sweep_path = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"
sel_sweep.to_csv(sweep_path, index=False)

# Checkpoint dir
iter_ckpt_dir = CKPT_ROOT / ITER_TAG
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

config = {
    "team": TEAM,
    "task": TASK,
    "version": VERSION,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "seed": SEED,
    "validation": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "selected": {
        "name": selected["name"],
        "include_keywords": sel_include_kw,
        "C": float(sel_C),
        "threshold": float(thr),
        "threshold_selection": {"pos_tol": 0.07, "macro_drop_tol": 0.005},
    },
    "tfidf": sel_schema["tfidf"],
    "paths": {"artifacts": str(ART_ROOT), "checkpoints": str(CKPT_ROOT)},
    "notes": [
        "Scoped artifact discovery enforced (only artifacts/clai_ch3_v3 + checkpoints/clai_ch3_v3).",
        "Vitals features strictly limited to days 1-10 to avoid day11 leakage.",
    ]
}

pred_sel = (sel_oof >= thr).astype(int)
cm_sel = confusion_matrix(y, pred_sel).tolist()
f1s_sel = f1_score(y, pred_sel, average=None)
macro_sel = f1_score(y, pred_sel, average="macro")
fold_scores_sel = []
for f in range(N_SPLITS):
    idx = sel_folds == f
    fold_scores_sel.append(f1_score(y[idx], pred_sel[idx], average="macro"))

metrics = {
    "oof_macro_f1": float(macro_sel),
    "per_class_f1": {"f1_0": float(f1s_sel[0]), "f1_1": float(f1s_sel[1])},
    "confusion_matrix": cm_sel,
    "threshold": float(thr),
    "pos_rate": float(pred_sel.mean()),
    "fold_scores": [float(x) for x in fold_scores_sel],
    "fold_mean": float(np.mean(fold_scores_sel)),
    "fold_std": float(np.std(fold_scores_sel)),
    "auc": float(roc_auc_score(y, sel_oof)),
    "train_prevalence": train_prev,
    "gate": {"min_delta": MIN_DELTA, "std_tol": STD_TOL, "pos_tol": POS_TOL, "decision": decision},
    "pstar_reference": pstar_metrics,
}

with open(iter_ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
with open(iter_ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
with open(iter_ckpt_dir / "schema.json", "w", encoding="utf-8") as f:
    json.dump(sel_schema, f, indent=2)
joblib.dump(pipe, iter_ckpt_dir / "pipeline.joblib")

print("Saved checkpoint:", iter_ckpt_dir)
print("Saved artifacts:", oof_path, sweep_path, ABL_PATH, CAND_PATH)

# If candidate passes gate, update PSTAR.json to point to THIS iteration
if eligible:
    new_pstar = {
        "iteration_tag": ITER_TAG,
        "oof_macro_f1": float(macro_sel),
        "threshold": float(thr),
        "best_threshold": float(thr),
        "pos_rate": float(pred_sel.mean()),
        "fold_mean": float(np.mean(fold_scores_sel)),
        "fold_std": float(np.std(fold_scores_sel)),
        "f1_0": float(f1s_sel[0]),
        "f1_1": float(f1s_sel[1]),
        "confusion_matrix": cm_sel,
    }
    with open(PSTAR_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "tag": TAG,
            "pstar": new_pstar,
            "train_prevalence": train_prev,
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "schema_version": 2,
            "notes": "Updated because candidate passed hard gate vs prior P*."
        }, f, indent=2)
    print("✅ Updated PSTAR.json ->", new_pstar["iteration_tag"], "macro=", new_pstar["oof_macro_f1"])
else:
    print("ℹ️ PSTAR.json unchanged (no candidate passed hard gate).")

# =====================
# 10) Append iteration_detail.jsonl (append-only)
# =====================
timestamp = datetime.utcnow().isoformat() + "Z"
hm_message = (
    "HM notes: real=0.7564 aligns with OOF=0.7565@thr~0.36; goal is reduce FP / improve F1(class0). "
    "Treat iter0002 as true P* (real 0.7768 / OOF 0.7629). Enforce hard gate: +0.002 macro, std not worse, pos_rate near prevalence. "
    "Also enforce scoped artifact discovery under artifacts/clai_ch3_v3 and checkpoints/clai_ch3_v3 only."
)
real_score = {"iter0002_real": 0.7768, "latest_real": 0.7564}

iteration_record = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": timestamp,
    "phase": "Modeling",
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "data_paths_used": {
        "stays_train": str(dp("stays_train.csv")),
        "stays_test": str(dp("stays_test.csv")),
        "patients": str(dp("patients.csv")),
        "vitals_timeseries": str(dp("vitals_timeseries.json")),
    },
    "join_keys": ["stays.patient_id -> patients.patient_id", "stays.stay_id -> vitals_timeseries.stay_id"],
    "leakage_checks_performed": [
        "Asserted vitals max(day) <= 10 (no day11 vitals).",
        "No use of target-derived columns or test labels.",
        "Scoped artifact discovery to v3 dirs only (prevents cross-version contamination).",
    ],
    "feature_summary": {
        "categorical": CAT_COLS,
        "text": {"col": TEXT_COL, "tfidf": sel_schema["tfidf"]},
        "numeric_count_noKW": len(schema_base["num_cols"]),
        "numeric_count_withKW": len(schema_kw["num_cols"]),
        "keywords_count": len(KEYWORDS),
        "vitals_cols": VITAL_COLS,
        "vitals_days_used": "1-10",
    },
    "models_attempted": [
        {k: cand0[k] for k in ["name","macro_f1","best_macro_overall","threshold","pos_rate","fold_mean","fold_std","f1_0","f1_1","auc","tn","fp","fn","tp","include_keywords","C"]},
        {k: cand1[k] for k in ["name","macro_f1","best_macro_overall","threshold","pos_rate","fold_mean","fold_std","f1_0","f1_1","auc","tn","fp","fn","tp","include_keywords","C"]},
        {k: cand2[k] for k in ["name","macro_f1","best_macro_overall","threshold","pos_rate","fold_mean","fold_std","f1_0","f1_1","auc","tn","fp","fn","tp","include_keywords","C"]},
    ],
    "selected_model": {
        "name": selected["name"],
        "C": float(sel_C),
        "penalty": "l1",
        "solver": "liblinear",
        "class_weight": "balanced",
        "include_keywords": bool(sel_include_kw),
        "threshold": float(thr),
        "decision": decision,
    },
    "validation_protocol": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "metrics": metrics,
    "artifacts_written": {
        "submission_csv": str(SUB_PATH),
        "candidate_snapshot": str(CAND_PATH),
        "oof_predictions": str(oof_path),
        "threshold_sweep": str(sweep_path),
        "ablation_summary": str(ABL_PATH),
        "checkpoint_dir": str(iter_ckpt_dir),
        "pstar_path": str(PSTAR_PATH),
    },
    "deltas_vs_prev": {
        "primary_goal": "Reduce FP / raise F1(class0) without collapsing F1(class1).",
        "changes": [
            "Added per-day vitals + deltas/trends (trajectory-aware) features.",
            "Added small, interpretable keyword-count features from notes (all + last2 days).",
            "Adjusted regularization C to control FP while keeping macro-F1 high.",
            "Threshold chosen with prevalence-aware tie-break to prevent runaway pos_rate."
        ],
    },
    "known_defects_limitations": [
        "Still a linear model; may miss non-linear interactions between vitals and text.",
        "Keyword list is hand-crafted and may be sensitive to synthetic note phrasing.",
        "TF-IDF vocab can drift; monitor feature count and coefficient sparsity.",
    ],
    "next_step_hypothesis": (
        "High-risk branch: controlled blend of (1) sparse LR (text+cats+vitals) and (2) "
        "dense GBDT/HGB on numeric vitals + keyword counts to capture non-linear vitals patterns; "
        "then tune threshold under the same hard gate."
    ),
    "hm_message": hm_message,
    "real_score": real_score,
}

with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
print("Appended iteration log ->", LOG_PATH)

# =====================
# 11) Consultant packet (v3: every iteration)
# =====================
consult_packet = {
    "tag": TAG,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": timestamp,
    "what_changed": iteration_record["deltas_vs_prev"]["changes"],
    "metric_changes": {
        "pstar_reference": pstar_metrics,
        "selected_metrics": metrics,
        "gate_decision": decision,
    },
    "suspected_leakage_risks": [
        "Vitals file includes only days 1-10; asserted max(day)<=10.",
        "No target leakage from future days; notes aligned to same 10-day window."
    ],
    "complexity_drift_risks": [
        "Keyword-count features add small dimensionality but could encode dataset-specific phrasing.",
        "TF-IDF + OneHot can still be high-dimensional; monitor coefficient sparsity and stability."
    ],
    "evaluation_alignment_risks": [
        "Threshold selected with prevalence-aware tie-break; verify it does not harm real-F1 alignment.",
        "Hard gate enforces macro-F1 uplift + stable std + reasonable pos_rate."
    ],
    "unknown_unknowns": [
        "Potential shift in note phrasing between train/test could reduce keyword usefulness.",
        "Zip3 one-hot could memorize rare regions; monitor fold variance."
    ],
    "recommendations_for_next_iter": [
        "If real-F1 improves but FP still high: increase pos_tol strictness or adjust class_weight ratio.",
        "Explore high-risk blend with dense GBDT on vitals/keywords to reduce FP via non-linear signals.",
        "Consider removing zip3 if it increases variance (ablation)."
    ]
}

consult_path = CONSULT_ROOT / f"{TAG}_iter{ITER_ID}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, ensure_ascii=False)
print("Wrote consultant packet ->", consult_path)

=== PREFLIGHT ===
CWD: d:\AgentDs\agent_ds_healthcare
PROJECT_ROOT: d:\AgentDs\agent_ds_healthcare
ART_ROOT: d:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT: d:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
Discovered iter tags (scoped): ['iter0000', 'iter0002', 'iter0003', 'iter0007']
ITER_ID=8  ITER_TAG=iter0008  BLOCK_ID=1  BLOCK_POS=3  BRANCH=W4


C:\Users\shend\AppData\Local\Temp\ipykernel_6244\1187089965.py:501: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",


=== P* (recomputed) ===
{
  "iteration_tag": "iter0002",
  "oof_macro_f1": 0.7629482620799912,
  "threshold": 0.45,
  "best_threshold": 0.45,
  "pos_rate": 0.649,
  "fold_mean": 0.7627652490625078,
  "fold_std": 0.010630182135354563,
  "f1_0": 0.6906474820143885,
  "f1_1": 0.8352490421455939,
  "confusion_matrix": [
    [
      240,
      104
    ],
    [
      111,
      545
    ]
  ]
}
=== Ablation summary (selected threshold per candidate) ===
              name  macro_f1  best_macro_overall  threshold  pos_rate  fold_mean  fold_std     f1_0     f1_1      auc  tn  fp  fn  tp  include_keywords    C
    A2_addKW_C0.12  0.768382            0.768382       0.45     0.683   0.768040  0.039753 0.689864 0.846901 0.850410 228 116  89 567              True 0.12
    A1_addKW_C0.15  0.761497            0.763091       0.46     0.671   0.761038  0.037154 0.683507 0.839488 0.846781 230 114  99 557              True 0.15
A0_base_noKW_C0.15  0.758823            0.760440       0.46     0.654   0.7587

C:\Users\shend\AppData\Local\Temp\ipykernel_6244\1187089965.py:753: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat() + "Z"


# NEW BRANCH

# Iteration 6
- 0.6770

In [36]:
import os, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
import joblib

# =====================
# 0) Config + paths
# =====================
TEAM, TASK, VERSION = "clai", "ch3", "v3"
TAG = f"{TEAM}_{TASK}_{VERSION}"
SEED = 42
N_SPLITS = 5

# W1 focus: FP control
POS_TOL = 0.01          # constrain predicted positive rate close to train prevalence
TFIDF_MAX_FEATURES = 1200
TFIDF_MIN_DF = 2
OHE_MIN_FREQ = 10
LR_C = 0.30             # stronger regularization to curb FP/overfit

def _find_project_root():
    cand = Path.cwd()
    if (cand / "stays_train.csv").exists():
        return cand
    cand2 = Path("/mnt/data")
    if (cand2 / "stays_train.csv").exists():
        return cand2
    return cand

PROJECT_ROOT = _find_project_root()
ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"
ART_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
CONSULT_ROOT.mkdir(parents=True, exist_ok=True)

LOG_PATH = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"
LOG_PATH.touch(exist_ok=True)

print("=== PREFLIGHT ===")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_ROOT:", ART_ROOT)
print("CKPT_ROOT:", CKPT_ROOT)
print("CONSULT_ROOT:", CONSULT_ROOT)
print("LOG_PATH:", LOG_PATH)

# =====================
# 1) Deterministic iteration id (append-only)
# =====================
def next_iteration_id(log_path: Path, default_start_id: int = 5) -> int:
    if (not log_path.exists()) or (log_path.stat().st_size == 0):
        # user requested "6th iteration" -> iter0005 (0-indexed)
        return default_start_id
    max_id = -1
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                max_id = max(max_id, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return max_id + 1

ITER_ID = next_iteration_id(LOG_PATH, default_start_id=5)
ITER_TAG = f"iter{ITER_ID:04d}"
BLOCK_ID = ITER_ID // 5
BLOCK_POS = ITER_ID % 5
BRANCH = {0:"W1", 1:"W2", 2:"W3", 3:"W4", 4:"BlockSummary"}.get(BLOCK_POS, "W1")
PHASE = "BlockSummary" if BRANCH == "BlockSummary" else "Modeling"

print(f"ITER_ID={ITER_ID} ITER_TAG={ITER_TAG} BLOCK_ID={BLOCK_ID} BLOCK_POS={BLOCK_POS} BRANCH={BRANCH} PHASE={PHASE}")

# =====================
# 2) Load data
# =====================
stays_train = pd.read_csv(PROJECT_ROOT / "stays_train.csv")
stays_test  = pd.read_csv(PROJECT_ROOT / "stays_test.csv")
patients    = pd.read_csv(PROJECT_ROOT / "patients.csv")

with open(PROJECT_ROOT / "vitals_timeseries.json", "r", encoding="utf-8") as f:
    vitals_list = json.load(f)

vitals_map = {int(x["stay_id"]): x["days"] for x in vitals_list}

# Join integrity checks
assert stays_train["patient_id"].isin(patients["patient_id"]).all(), "patient_id join mismatch in train"
assert stays_test["patient_id"].isin(patients["patient_id"]).all(), "patient_id join mismatch in test"
assert stays_train["stay_id"].isin(list(vitals_map.keys())).all(), "stay_id missing in vitals_map for train"
assert stays_test["stay_id"].isin(list(vitals_map.keys())).all(), "stay_id missing in vitals_map for test"

y_all = stays_train["discharge_ready_day11"].astype(int).values
train_prev = float(y_all.mean())
print("Train prevalence(y=1):", train_prev)

# =====================
# 3) Feature builder (W1: trajectory-only vitals + text + cats)
# =====================
VITALS = ["hr", "sbp", "dbp", "temp_c", "rr"]
DAYS = np.arange(1, 11, dtype=float)

def _safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def _slope(x: np.ndarray, y: np.ndarray) -> float:
    mask = ~np.isnan(y)
    if mask.sum() < 2:
        return np.nan
    xx = x[mask]
    yy = y[mask]
    x_mean = xx.mean()
    y_mean = yy.mean()
    cov = np.mean((xx - x_mean) * (yy - y_mean))
    var = np.mean((xx - x_mean) ** 2)
    return float(cov / var) if var > 0 else 0.0

def build_features(stays_df: pd.DataFrame) -> pd.DataFrame:
    df = stays_df.merge(patients, on="patient_id", how="left")

    # categorical casting
    for col in ["unit_type", "admission_reason", "sex", "insurance"]:
        df[col] = df[col].astype(str)
    df["zip3"] = df["zip3"].astype(int).astype(str)

    n = len(df)
    note_all = [""] * n

    feat = {}
    for v in VITALS:
        feat[f"{v}_mean"] = np.full(n, np.nan, dtype=float)
        feat[f"{v}_std"]  = np.full(n, np.nan, dtype=float)
        feat[f"{v}_min"]  = np.full(n, np.nan, dtype=float)
        feat[f"{v}_max"]  = np.full(n, np.nan, dtype=float)
        feat[f"{v}_first"] = np.full(n, np.nan, dtype=float)
        feat[f"{v}_last"]  = np.full(n, np.nan, dtype=float)
        feat[f"{v}_delta_last_first"] = np.full(n, np.nan, dtype=float)
        feat[f"{v}_delta_last_mean_first3"] = np.full(n, np.nan, dtype=float)
        feat[f"{v}_missing"] = np.zeros(n, dtype=float)
        feat[f"{v}_slope"] = np.full(n, np.nan, dtype=float)

    for i, sid in enumerate(df["stay_id"].astype(int).values):
        days = vitals_map.get(int(sid), [])
        max_day = max([int(r.get("day", 0)) for r in days], default=0)
        if max_day > 10:
            raise RuntimeError(f"[ABORT] Leakage risk: stay_id={sid} has max day={max_day} (>10)")

        rec_by_day = {int(r.get("day")): r for r in days if "day" in r}
        texts = []

        for v in VITALS:
            y = np.full(10, np.nan, dtype=float)
            for d in range(1, 11):
                r = rec_by_day.get(d, None)
                if r is None:
                    continue
                y[d-1] = _safe_float(r.get(v, np.nan))
                if v == VITALS[0]:
                    t = str(r.get("note", "") or "")
                    if t and t != "nan":
                        texts.append(t)

            feat[f"{v}_missing"][i] = float(np.isnan(y).sum())
            if np.all(np.isnan(y)):
                continue

            feat[f"{v}_mean"][i] = float(np.nanmean(y))
            feat[f"{v}_std"][i]  = float(np.nanstd(y))
            feat[f"{v}_min"][i]  = float(np.nanmin(y))
            feat[f"{v}_max"][i]  = float(np.nanmax(y))
            feat[f"{v}_first"][i] = float(y[0]) if not np.isnan(y[0]) else float(np.nan)
            feat[f"{v}_last"][i]  = float(y[-1]) if not np.isnan(y[-1]) else float(np.nan)

            if (not np.isnan(feat[f"{v}_last"][i])) and (not np.isnan(feat[f"{v}_first"][i])):
                feat[f"{v}_delta_last_first"][i] = float(feat[f"{v}_last"][i] - feat[f"{v}_first"][i])
            else:
                feat[f"{v}_delta_last_first"][i] = float(np.nan)

            first3 = y[:3]
            m_first3 = np.nanmean(first3) if not np.all(np.isnan(first3)) else np.nan
            if (not np.isnan(feat[f"{v}_last"][i])) and (not np.isnan(m_first3)):
                feat[f"{v}_delta_last_mean_first3"][i] = float(feat[f"{v}_last"][i] - m_first3)
            else:
                feat[f"{v}_delta_last_mean_first3"][i] = float(np.nan)

            feat[f"{v}_slope"][i] = _slope(DAYS, y)

        note_all[i] = " ".join(texts)

    for k, arr in feat.items():
        df[k] = arr
    df["note_text_all"] = note_all
    return df

train_df = build_features(stays_train)
test_df  = build_features(stays_test)

# =====================
# 4) Deterministic folds
# =====================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.empty(len(train_df), dtype=int)
for f, (_, va_idx) in enumerate(skf.split(np.zeros(len(train_df)), y_all)):
    folds[va_idx] = f

# =====================
# 5) Modeling (W1): LogReg(liblinear, L2) + prevalence-constrained threshold
# =====================
id_cols = ["stay_id", "patient_id"]
target_col = "discharge_ready_day11"
cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3"]
text_col = "note_text_all"

num_cols = sorted([c for c in train_df.columns if c not in id_cols + [target_col] + cat_cols + [text_col]])

X = train_df[id_cols + cat_cols + [text_col] + num_cols].copy()
y = y_all

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=OHE_MIN_FREQ)),
])
text_transformer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 2),
    min_df=TFIDF_MIN_DF
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
        ("txt", text_transformer, text_col),
    ],
    sparse_threshold=0.3
)

clf = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    C=LR_C,
    max_iter=1000,
    random_state=SEED
)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", clf),
])

# OOF predictions
oof_proba = np.zeros(len(train_df), dtype=float)
for f in range(N_SPLITS):
    tr_idx = np.where(folds != f)[0]
    va_idx = np.where(folds == f)[0]
    pipe.fit(X.iloc[tr_idx], y[tr_idx])
    oof_proba[va_idx] = pipe.predict_proba(X.iloc[va_idx])[:, 1]

auc = float(roc_auc_score(y, oof_proba))
print("OOF AUC:", auc)

# Threshold sweep (0.01 grid)
thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
rows = []
for t in thresholds:
    pred = (oof_proba >= t).astype(int)
    cm = confusion_matrix(y, pred)
    f1s = f1_score(y, pred, average=None)
    macro = f1_score(y, pred, average="macro")
    fold_scores = []
    for f in range(N_SPLITS):
        idx = folds == f
        fold_scores.append(f1_score(y[idx], pred[idx], average="macro"))
    rows.append({
        "threshold": float(t),
        "macro_f1": float(macro),
        "f1_0": float(f1s[0]),
        "f1_1": float(f1s[1]),
        "pos_rate": float(pred.mean()),
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1]),
    })

sweep_df = pd.DataFrame(rows)
sweep_df["pos_gap"] = (sweep_df["pos_rate"] - train_prev).abs()

def pick_best_row(sw: pd.DataFrame, pos_tol: float):
    sw = sw.copy()
    cand = sw[sw["pos_gap"] <= pos_tol].copy()
    if len(cand) == 0:
        cand = sw
    cand = cand.sort_values(["macro_f1", "pos_gap", "fold_std", "threshold"],
                            ascending=[False, True, True, True]).reset_index(drop=True)
    return cand.iloc[0].to_dict(), int((sw["pos_gap"] <= pos_tol).sum())

best, n_within = pick_best_row(sweep_df, POS_TOL)
best_threshold = float(best["threshold"])

best_pred = (oof_proba >= best_threshold).astype(int)
per_fold_macro = []
for f in range(N_SPLITS):
    idx = folds == f
    per_fold_macro.append(float(f1_score(y[idx], best_pred[idx], average="macro")))

print(
    f"Best threshold={best_threshold} | OOF macro_f1={best['macro_f1']:.6f} | "
    f"F1_0={best['f1_0']:.4f} F1_1={best['f1_1']:.4f} | "
    f"within_pos_tol={n_within}/{len(sweep_df)} | pos_rate={best['pos_rate']:.3f} | gap={abs(best['pos_rate']-train_prev):.3f}"
)
print("OOF confusion_matrix:", [[int(best["tn"]), int(best["fp"])], [int(best["fn"]), int(best["tp"])]])
print("Per-fold macro-F1 @best_thr:", per_fold_macro)

# =====================
# 6) Write artifacts (scoped)
# =====================
oof_df = pd.DataFrame({
    "stay_id": train_df["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "oof_proba": oof_proba.astype(float),
    "fold": folds.astype(int),
})

oof_path = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
sweep_path = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"
oof_df.to_csv(oof_path, index=False)
sweep_df.drop(columns=["pos_gap"]).to_csv(sweep_path, index=False)
print("✅ wrote:", oof_path.name, "|", sweep_path.name)

# Candidate + submission
X_test = test_df[id_cols + cat_cols + [text_col] + num_cols].copy()
pipe.fit(X, y)
test_proba = pipe.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

cand_df = pd.DataFrame({
    "stay_id": test_df["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred.astype(int),
})

cand_path1 = ART_ROOT / f"{TAG}_{ITER_TAG}_candidate.csv"
cand_path2 = ART_ROOT / f"{TAG}_{BRANCH}_block{BLOCK_ID}_candidate.csv"
cand_df.to_csv(cand_path1, index=False)
cand_df.to_csv(cand_path2, index=False)

SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"
cand_df.to_csv(SUB_PATH, index=False)

print("✅ wrote candidate:", cand_path1.name, "and", cand_path2.name, "| pos_rate:", float(cand_df["discharge_ready_day11"].mean()))
print("✅ wrote submission:", SUB_PATH)

# =====================
# 7) Checkpoint bundle + P*
# =====================
iter_ckpt_dir = CKPT_ROOT / ITER_TAG
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

PSTAR_PATH = CKPT_ROOT / "PSTAR.json"
prev_pstar = None
if PSTAR_PATH.exists():
    try:
        prev_pstar = json.loads(PSTAR_PATH.read_text(encoding="utf-8")).get("pstar", None)
    except Exception:
        prev_pstar = None

is_new_pstar = False
if prev_pstar is None:
    is_new_pstar = True
else:
    try:
        is_new_pstar = float(best["macro_f1"]) > float(prev_pstar.get("oof_macro_f1", -1.0)) + 1e-9
    except Exception:
        is_new_pstar = False

pstar_payload = {
    "iteration_tag": ITER_TAG,
    "oof_macro_f1": float(best["macro_f1"]),
    "threshold": float(best_threshold),
    "best_threshold": float(best_threshold),
    "pos_rate": float(best["pos_rate"]),
    "fold_mean": float(best["fold_mean"]),
    "fold_std": float(best["fold_std"]),
    "f1_0": float(best["f1_0"]),
    "f1_1": float(best["f1_1"]),
    "confusion_matrix": [[int(best["tn"]), int(best["fp"])], [int(best["fn"]), int(best["tp"])]],
}

timestamp = datetime.utcnow().isoformat() + "Z"
if is_new_pstar:
    with open(PSTAR_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "tag": TAG,
            "schema_version": 3,
            "timestamp": timestamp,
            "train_prevalence": train_prev,
            "pstar": pstar_payload,
            "notes": "P* updated automatically from deterministic OOF macro-F1."
        }, f, indent=2, ensure_ascii=False)
    print("🏁 P* UPDATED ->", PSTAR_PATH)
else:
    print("ℹ️ P* unchanged (current best does not exceed existing P*).")

joblib.dump(pipe, iter_ckpt_dir / "pipeline.joblib")

ckpt_config = {
    "team": TEAM, "task": TASK, "version": VERSION,
    "iteration_id": ITER_ID, "iteration_tag": ITER_TAG,
    "phase": PHASE, "branch": BRANCH,
    "validation": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "artifact_scoping": {"ART_ROOT": str(ART_ROOT), "CKPT_ROOT": str(CKPT_ROOT)},
    "feature_params": {
        "vitals": VITALS,
        "numeric_features": num_cols,
        "categorical_features": cat_cols,
        "text_feature": text_col,
        "tfidf_max_features": TFIDF_MAX_FEATURES,
        "tfidf_min_df": TFIDF_MIN_DF,
        "ohe_min_frequency": OHE_MIN_FREQ,
    },
    "model_params": {"family": "LogisticRegression", "solver": "liblinear", "penalty": "l2", "C": LR_C},
    "thresholding": {"pos_tol": POS_TOL, "selection_rule": "macro_f1 desc, pos_gap asc, fold_std asc, threshold asc; with pos_gap<=pos_tol filter"},
    "pstar_update": bool(is_new_pstar),
}

with open(iter_ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(ckpt_config, f, indent=2, ensure_ascii=False)

metrics_payload = {
    "oof": {
        "macro_f1": float(best["macro_f1"]),
        "threshold": float(best_threshold),
        "f1_0": float(best["f1_0"]),
        "f1_1": float(best["f1_1"]),
        "pos_rate": float(best["pos_rate"]),
        "confusion_matrix": [[int(best["tn"]), int(best["fp"])], [int(best["fn"]), int(best["tp"])]],
        "auc": float(auc),
        "fold_mean": float(best["fold_mean"]),
        "fold_std": float(best["fold_std"]),
        "per_fold_macro_f1": per_fold_macro,
    },
    "train_prevalence": float(train_prev),
    "artifact_paths": {"oof": str(oof_path), "sweep": str(sweep_path)},
}
with open(iter_ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)

print("✅ Saved checkpoint ->", iter_ckpt_dir)

# =====================
# 8) Append iteration_detail.jsonl (append-only)
# =====================
prev_record = None
try:
    with open(LOG_PATH, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    if lines:
        prev_record = json.loads(lines[-1])
except Exception:
    prev_record = None

delta = {}
if prev_record and isinstance(prev_record, dict):
    try:
        prev_m = prev_record.get("metrics", {}).get("oof_macro_f1_mean", None)
        if prev_m is not None:
            delta["oof_macro_f1_mean_delta"] = float(best["macro_f1"]) - float(prev_m)
    except Exception:
        pass

iteration_record = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": timestamp,
    "phase": PHASE,
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "data_paths_used": {
        "stays_train": str(PROJECT_ROOT / "stays_train.csv"),
        "stays_test": str(PROJECT_ROOT / "stays_test.csv"),
        "patients": str(PROJECT_ROOT / "patients.csv"),
        "vitals_timeseries": str(PROJECT_ROOT / "vitals_timeseries.json"),
        "art_root": str(ART_ROOT),
        "ckpt_root": str(CKPT_ROOT),
    },
    "join_keys": ["stays.patient_id -> patients.patient_id", "stays.stay_id -> vitals_timeseries.stay_id"],
    "leakage_checks_performed": [
        "Guardrail: vitals max(day) must be <=10 (asserted during feature build).",
        "No cross-version artifact discovery (only scoped ART_ROOT/CKPT_ROOT used).",
    ],
    "feature_summary": {
        "numeric": {
            "age": True,
            "vitals_trajectory_only": True,
            "vitals": VITALS,
            "per_vital_features": ["mean", "std", "min", "max", "first", "last", "delta_last_first", "delta_last_mean_first3", "missing", "slope"],
            "n_numeric_features": int(len(num_cols)),
        },
        "categorical": {"cols": cat_cols, "ohe_min_frequency": OHE_MIN_FREQ},
        "text": {"col": text_col, "tfidf_max_features": TFIDF_MAX_FEATURES, "tfidf_min_df": TFIDF_MIN_DF, "ngram_range": [1, 2]},
    },
    "models_attempted": [
        {
            "name": "LogisticRegression(liblinear, L2)",
            "params": {"C": LR_C, "max_iter": 1000, "random_state": SEED},
            "cv": {"n_splits": N_SPLITS, "macro_f1_oof_best_threshold": float(best["macro_f1"]), "auc_oof": float(auc)},
            "notes": "W1 focus: reduce FP/overfit via stronger regularization + prevalence-constrained thresholding."
        }
    ],
    "selected_model": "LogisticRegression(liblinear, L2)",
    "validation_protocol": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "metrics": {
        "train_prevalence": float(train_prev),
        "oof_macro_f1_mean": float(best["macro_f1"]),
        "oof_auc": float(auc),
        "threshold": float(best_threshold),
        "pos_rate": float(best["pos_rate"]),
        "per_class_f1": {"f1_0": float(best["f1_0"]), "f1_1": float(best["f1_1"])},
        "confusion_matrix": [[int(best["tn"]), int(best["fp"])], [int(best["fn"]), int(best["tp"])]],
        "per_fold_macro_f1": per_fold_macro,
        "fold_mean_macro_f1": float(best["fold_mean"]),
        "fold_std_macro_f1": float(best["fold_std"]),
    },
    "deltas_vs_previous": delta,
    "artifacts_written": {
        "oof_predictions": str(oof_path),
        "threshold_sweep": str(sweep_path),
        "candidate_csv_iter": str(cand_path1),
        "candidate_csv_branch": str(cand_path2),
        "submission_csv": str(SUB_PATH),
        "checkpoint_dir": str(iter_ckpt_dir),
        "pstar_json": str(PSTAR_PATH),
    },
    "known_defects_limitations": [
        "Local CV may not perfectly align with hidden leaderboard distribution.",
        "Text notes are only aggregated (note_text_all); last2-day specific branch deferred to W2.",
    ],
    "next_step_hypothesis": (
        "If FP inflation persists on LB, tighten pos_tol further (e.g., 0.005) or reduce C; "
        "if recall drops too much, add last2-day text branch (W2) or mild non-linearity (W3)."
    ),
    "hm_message": "User requested: start the 6th iteration; focus W1 FP control per handover.",
    "real_score": None,
}

with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
print("✅ Appended iteration log ->", LOG_PATH)

# =====================
# 9) Consultant packet (v3: every iteration)
# =====================
consult_packet = {
    "tag": TAG,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": timestamp,
    "what_changed": [
        "W1: switched numeric vitals to trajectory-only stats (no per-day raw flattening)",
        f"W1: stronger L2 regularization (C={LR_C}) to curb overfitting/FP",
        f"W1: prevalence-constrained threshold selection (pos_tol={POS_TOL})",
    ],
    "metric_changes": {
        "oof_macro_f1": float(best["macro_f1"]),
        "oof_auc": float(auc),
        "threshold": float(best_threshold),
        "pos_rate": float(best["pos_rate"]),
        "confusion_matrix": [[int(best["tn"]), int(best["fp"])], [int(best["fn"]), int(best["tp"])]],
        "per_fold_macro_f1": per_fold_macro,
        "note": "Compare against PSTAR.json for whether P* updated."
    },
    "suspected_leakage_risks": [
        "Primary risk: day>10 vitals or notes; guarded by max(day)<=10 assertion.",
        "No use of target leakage sources; only stays/patients/vitals_timeseries.",
    ],
    "complexity_drift_risks": [
        f"TF-IDF capped at {TFIDF_MAX_FEATURES} features; keep caps to avoid feature explosion.",
        "Categorical OHE uses min_frequency to group infrequent categories.",
    ],
    "evaluation_alignment_risks": [
        "Threshold overfitting risk if tuned too aggressively; pos_tol constraint reduces drift but may underfit macro-F1.",
        "CV↔LB mismatch possible; monitor real score when HM provides it.",
    ],
    "unknown_unknowns": [
        "If notes contain systematic day-index artifacts, aggregation may wash out signal; W2 can add last2-day view.",
    ],
    "recommendations_for_next_iter": [
        "W1 follow-up: sweep C in a small grid (0.2, 0.3, 0.4) with same pos_tol and log stability.",
        "W2: add note_text_last2 TF-IDF branch with small max_features (500–1000).",
    ],
}

consult_path = CONSULT_ROOT / f"{TAG}_iter{ITER_ID}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, ensure_ascii=False)
print("✅ Wrote consultant packet ->", consult_path)

print("\n=== DONE ===")

=== PREFLIGHT ===
PROJECT_ROOT: d:\AgentDs\agent_ds_healthcare
ART_ROOT: d:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT: d:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
CONSULT_ROOT: d:\AgentDs\agent_ds_healthcare\consultant_packets
LOG_PATH: d:\AgentDs\agent_ds_healthcare\clai_ch3_v3_iteration_detail.jsonl
ITER_ID=13 ITER_TAG=iter0013 BLOCK_ID=2 BLOCK_POS=3 BRANCH=W4 PHASE=Modeling
Train prevalence(y=1): 0.656
OOF AUC: 0.75350077992059
Best threshold=0.61 | OOF macro_f1=0.700098 | F1_0=0.6038 F1_1=0.7964 | within_pos_tol=2/91 | pos_rate=0.665 | gap=0.009
OOF confusion_matrix: [[205, 139], [130, 526]]
Per-fold macro-F1 @best_thr: [0.7739739243088288, 0.6677085973360367, 0.6633374480978567, 0.6680362439117302, 0.7264957264957265]
✅ wrote: iter0013_oof_predictions.csv | iter0013_threshold_sweep.csv
✅ wrote candidate: clai_ch3_v3_iter0013_candidate.csv and clai_ch3_v3_W4_block2_candidate.csv | pos_rate: 0.668
✅ wrote submission: d:\AgentDs\agent_ds_healthcare\clai_c

C:\Users\shend\AppData\Local\Temp\ipykernel_6244\2842973671.py:410: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat() + "Z"


# Iteration 7
- 0.7739

In [40]:
import os, json, random, re, shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.base import clone
import joblib

# =====================
# 0) Config + paths
# =====================
TEAM, TASK, VERSION = "clai", "ch3", "v3"
TAG = f"{TEAM}_{TASK}_{VERSION}"
SEED = 42
N_SPLITS = 5

np.random.seed(SEED)
random.seed(SEED)

def _find_project_root():
    cand = Path.cwd()
    if (cand / "stays_train.csv").exists():
        return cand
    cand2 = Path("/mnt/data")
    if (cand2 / "stays_train.csv").exists():
        return cand2
    return cand

PROJECT_ROOT = _find_project_root()
ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"

ART_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
CONSULT_ROOT.mkdir(parents=True, exist_ok=True)

print("=== PREFLIGHT ===")
print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_ROOT:", ART_ROOT)
print("CKPT_ROOT:", CKPT_ROOT)

# =====================
# 1) Iteration id (append-only)
# =====================
LOG_PATH = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"

# If log exists but not writable (e.g., sandbox upload), replace with a writable copy
if LOG_PATH.exists() and (not os.access(LOG_PATH, os.W_OK)):
    backup_path = ART_ROOT / f"{TAG}_iteration_detail_readonly_backup_{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}.jsonl"
    shutil.copy2(LOG_PATH, backup_path)
    old_text = LOG_PATH.read_text(encoding="utf-8", errors="ignore")
    LOG_PATH.unlink()
    LOG_PATH.write_text(old_text, encoding="utf-8")
    print(f"[log fix] Replaced non-writable log. Backup -> {backup_path}")

LOG_PATH.touch(exist_ok=True)

def next_iteration_id(log_path: Path) -> int:
    max_id = -1
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                max_id = max(max_id, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return max_id + 1

ITER_ID = next_iteration_id(LOG_PATH)
ITER_TAG = f"iter{ITER_ID:04d}"
BLOCK_ID = ITER_ID // 5
BLOCK_POS = ITER_ID % 5
BRANCH = "W4"  # FP-control / thresholding

print(f"ITER_ID={ITER_ID}  ITER_TAG={ITER_TAG}  BLOCK_ID={BLOCK_ID}  BLOCK_POS={BLOCK_POS}  BRANCH={BRANCH}")

# =====================
# 2) Load data
# =====================
def dp(name: str) -> Path:
    p = PROJECT_ROOT / name
    if p.exists():
        return p
    p2 = Path("/mnt/data") / name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {name} under PROJECT_ROOT or /mnt/data")

stays_train = pd.read_csv(dp("stays_train.csv"))
stays_test  = pd.read_csv(dp("stays_test.csv"))
patients    = pd.read_csv(dp("patients.csv"))
with open(dp("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vit_list = json.load(f)

# Leakage check: vitals day must be <=10
max_day = max(d.get("day", 0) for item in vit_list for d in item.get("days", []))
assert max_day <= 10, f"Leakage risk: found vitals day={max_day} (>10). Abort."

vit_map = {int(item["stay_id"]): item.get("days", []) for item in vit_list}

y_all = stays_train["discharge_ready_day11"].astype(int).values
train_prev = float(y_all.mean())

# Deterministic fold map (for backfilling old OOF artifacts)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.empty(len(y_all), dtype=int)
for f, (_, va_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
    folds[va_idx] = f
fold_map = pd.DataFrame({"stay_id": stays_train["stay_id"].values, "fold": folds})

# =====================
# 3) Scoped artifact discovery + auto-fix missing sweeps
# =====================
def ensure_oof_has_fold(oof_df: pd.DataFrame) -> pd.DataFrame:
    if "fold" in oof_df.columns:
        return oof_df
    out = oof_df.merge(fold_map, on="stay_id", how="left")
    if out["fold"].isna().any():
        miss = out.loc[out["fold"].isna(), "stay_id"].head(10).tolist()
        raise RuntimeError(f"[ABORT] fold backfill failed; e.g. missing stay_id={miss}")
    out["fold"] = out["fold"].astype(int)
    return out

def compute_threshold_sweep(oof_df: pd.DataFrame, thresholds=None) -> pd.DataFrame:
    if thresholds is None:
        thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
    y = oof_df["y_true"].astype(int).values
    proba = oof_df["oof_proba"].astype(float).values
    fold_ids = oof_df["fold"].astype(int).values

    rows = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        cm = confusion_matrix(y, pred, labels=[0, 1])
        f1s = f1_score(y, pred, labels=[0, 1], average=None, zero_division=0)
        macro = f1_score(y, pred, average="macro", zero_division=0)

        fold_scores = []
        for f in range(N_SPLITS):
            idx = fold_ids == f
            fold_scores.append(f1_score(y[idx], pred[idx], average="macro", zero_division=0))

        rows.append({
            "threshold": float(t),
            "macro_f1": float(macro),
            "f1_0": float(f1s[0]),
            "f1_1": float(f1s[1]),
            "pos_rate": float(pred.mean()),
            "fold_mean": float(np.mean(fold_scores)),
            "fold_std": float(np.std(fold_scores)),
            "tn": int(cm[0, 0]), "fp": int(cm[0, 1]), "fn": int(cm[1, 0]), "tp": int(cm[1, 1]),
        })
    return pd.DataFrame(rows)

def discover_iter_tags_scoped(art_root: Path):
    oof_files = sorted(art_root.glob("iter????_oof_predictions.csv"))
    sweep_files = sorted(art_root.glob("iter????_threshold_sweep.csv"))
    return sorted({p.name.split("_")[0] for p in (oof_files + sweep_files)})

tags = discover_iter_tags_scoped(ART_ROOT)

# Auto-generate missing sweeps (if oof exists)
for t in tags:
    oof_p = ART_ROOT / f"{t}_oof_predictions.csv"
    sw_p  = ART_ROOT / f"{t}_threshold_sweep.csv"
    if oof_p.exists() and (not sw_p.exists()):
        oof_df = pd.read_csv(oof_p)
        oof_df = ensure_oof_has_fold(oof_df)
        oof_df.to_csv(oof_p, index=False)  # upgrade schema
        sw = compute_threshold_sweep(oof_df)
        sw.to_csv(sw_p, index=False)
        print("[generated sweep]", sw_p.name)

# Hard checks: every discovered tag must have BOTH oof + sweep
tags = discover_iter_tags_scoped(ART_ROOT)
for t in tags:
    if not (ART_ROOT / f"{t}_oof_predictions.csv").exists():
        raise RuntimeError(f"[ABORT] Missing OOF for tag={t} under {ART_ROOT}")
    if not (ART_ROOT / f"{t}_threshold_sweep.csv").exists():
        raise RuntimeError(f"[ABORT] Missing sweep for tag={t} under {ART_ROOT}")

print("Discovered iter tags (scoped):", tags)

# =====================
# 4) Recompute P* (prefer iter0002)
# =====================
def pick_best_row(sw: pd.DataFrame, prevalence: float):
    sw = sw.copy()
    if "macro_f1" not in sw.columns and "macro_f1_oof" in sw.columns:
        sw = sw.rename(columns={"macro_f1_oof": "macro_f1"})
    for col in ["pos_rate", "fold_std"]:
        if col not in sw.columns:
            sw[col] = np.nan
    sw["pos_gap"] = (sw["pos_rate"] - prevalence).abs()
    sw = sw.sort_values(["macro_f1", "pos_gap", "fold_std", "threshold"],
                        ascending=[False, True, True, True]).reset_index(drop=True)
    return sw.iloc[0].to_dict()

def load_iter_metrics(iter_tag: str, prevalence: float):
    oof_p = ART_ROOT / f"{iter_tag}_oof_predictions.csv"
    sw_p  = ART_ROOT / f"{iter_tag}_threshold_sweep.csv"
    if (not oof_p.exists()) or (not sw_p.exists()):
        return None

    oof_df = pd.read_csv(oof_p)
    oof_df = ensure_oof_has_fold(oof_df)
    oof_df.to_csv(oof_p, index=False)

    sw = pd.read_csv(sw_p)
    if "macro_f1" not in sw.columns and "macro_f1_oof" in sw.columns:
        sw = sw.rename(columns={"macro_f1_oof": "macro_f1"})
    need = {"threshold","macro_f1","f1_0","f1_1","tn","fp","fn","tp","pos_rate","fold_mean","fold_std"}
    if not need.issubset(set(sw.columns)):
        sw = compute_threshold_sweep(oof_df)
        sw.to_csv(sw_p, index=False)

    best = pick_best_row(sw, prevalence=prevalence)
    thr = float(best["threshold"])

    y_true = oof_df["y_true"].astype(int).values
    proba  = oof_df["oof_proba"].astype(float).values
    pred   = (proba >= thr).astype(int)

    cm = confusion_matrix(y_true, pred, labels=[0,1]).tolist()
    f1s = f1_score(y_true, pred, labels=[0,1], average=None, zero_division=0)

    fold_ids = oof_df["fold"].astype(int).values
    fold_scores = []
    for f in range(N_SPLITS):
        idx = fold_ids == f
        fold_scores.append(f1_score(y_true[idx], pred[idx], average="macro", zero_division=0))

    return {
        "iteration_tag": iter_tag,
        "oof_macro_f1": float(best["macro_f1"]),
        "threshold": float(thr),
        "best_threshold": float(thr),
        "pos_rate": float(pred.mean()),
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "f1_0": float(f1s[0]),
        "f1_1": float(f1s[1]),
        "confusion_matrix": cm,
    }

pstar_metrics = None
if "iter0002" in tags:
    pstar_metrics = load_iter_metrics("iter0002", prevalence=train_prev)
else:
    best_score = -1.0
    for t in tags:
        m = load_iter_metrics(t, prevalence=train_prev)
        if m and m["oof_macro_f1"] > best_score:
            best_score = m["oof_macro_f1"]
            pstar_metrics = m

if pstar_metrics is None:
    pstar_metrics = {
        "iteration_tag": "NONE",
        "oof_macro_f1": -1.0,
        "threshold": 0.5,
        "best_threshold": 0.5,
        "pos_rate": None,
        "fold_mean": None,
        "fold_std": None,
        "f1_0": None,
        "f1_1": None,
        "confusion_matrix": None,
    }

PSTAR_PATH = CKPT_ROOT / "PSTAR.json"
with open(PSTAR_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "tag": TAG,
        "pstar": pstar_metrics,
        "train_prevalence": train_prev,
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "schema_version": 2,
        "notes": "P* recomputed from scoped artifacts; keys normalized."
    }, f, indent=2, ensure_ascii=False)

print("=== P* (recomputed) ===")
print(json.dumps(pstar_metrics, indent=2, ensure_ascii=False))

# =====================
# 5) Feature engineering (day-aligned vitals + keywords)
# =====================
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]

KEYWORDS = [
    "stable","improving","worsening","afebrile","fever","chills","hypotensive","tachy",
    "culture","antibiotic","abx","iv","oxygen","weaned","dyspnea","cough",
    "pain","nausea","vomit","n/v","ambulated","pt","ot","telemetry","fluids","room air","sats",
    "discharge","transfer"
]
KW_PATTERNS = [(kw, re.compile(re.escape(kw), re.IGNORECASE)) for kw in KEYWORDS]

def safe_nan(fn, arr, default=np.nan):
    try:
        arr = np.asarray(arr, dtype=float)
        if arr.size == 0 or np.all(np.isnan(arr)):
            return default
        return fn(arr)
    except Exception:
        return default

def build_stay_features(stays_df: pd.DataFrame, include_keywords: bool) -> pd.DataFrame:
    base = stays_df.merge(patients, on="patient_id", how="left")
    rows = []
    for sid in base["stay_id"].astype(int).tolist():
        days = sorted(vit_map.get(int(sid), []), key=lambda x: x.get("day", 0))

        notes_by_day = {int(d.get("day", -1)): (d.get("note") or "") for d in days if d.get("day") is not None}
        note_all = " ".join([notes_by_day.get(day, "") for day in range(1, 11)]).strip()
        note_last2 = " ".join([notes_by_day.get(day, "") for day in (9, 10)]).strip()

        feat = {
            "stay_id": int(sid),
            "note_text_all": note_all,
            "note_text_last2": note_last2,
            "note_len_all": len(note_all),
            "note_len_last2": len(note_last2),
        }

        for col in VITAL_COLS:
            val_by_day = {
                int(d.get("day", -1)): (np.nan if d.get(col, np.nan) is None else d.get(col, np.nan))
                for d in days if d.get("day") is not None
            }
            vals = np.array([val_by_day.get(day, np.nan) for day in range(1, 11)], dtype=float)
            day_idx = np.arange(1, 11, dtype=float)

            for day in range(1, 11):
                feat[f"{col}_day{day}"] = val_by_day.get(day, np.nan)

            diffs = []
            for j in range(1, 10):
                v1 = val_by_day.get(j, np.nan)
                v2 = val_by_day.get(j+1, np.nan)
                dv = (v2 - v1) if (not np.isnan(v1) and not np.isnan(v2)) else np.nan
                feat[f"{col}_d{j+1}_minus_d{j}"] = dv
                diffs.append(dv)
            diffs = np.array(diffs, dtype=float)
            feat[f"{col}_delta10_9"] = feat[f"{col}_d10_minus_d9"]

            feat[f"{col}_mean"]   = safe_nan(np.nanmean, vals)
            feat[f"{col}_std"]    = safe_nan(np.nanstd,  vals)
            feat[f"{col}_min"]    = safe_nan(np.nanmin,  vals)
            feat[f"{col}_max"]    = safe_nan(np.nanmax,  vals)
            feat[f"{col}_median"] = safe_nan(np.nanmedian, vals)
            feat[f"{col}_first"]  = vals[0]
            feat[f"{col}_last"]   = vals[-1]

            last2 = vals[8:10]
            early = vals[0:8]
            feat[f"{col}_last2_mean"] = safe_nan(np.nanmean, last2)
            feat[f"{col}_early_mean"] = safe_nan(np.nanmean, early)
            feat[f"{col}_delta_last2_early"] = feat[f"{col}_last2_mean"] - feat[f"{col}_early_mean"]
            feat[f"{col}_delta_last_first"]  = feat[f"{col}_last"] - feat[f"{col}_first"]

            mask = (~np.isnan(vals)) & (~np.isnan(day_idx))
            feat[f"{col}_slope"] = np.polyfit(day_idx[mask], vals[mask], 1)[0] if mask.sum() >= 2 else np.nan

            feat[f"{col}_diff_mean"] = safe_nan(np.nanmean, diffs)
            feat[f"{col}_diff_std"]  = safe_nan(np.nanstd,  diffs)
            feat[f"{col}_last3_std"] = safe_nan(np.nanstd,  vals[7:10])

            feat[f"{col}_missing"]  = int(np.isnan(vals).sum())
            feat[f"{col}_observed"] = int((~np.isnan(vals)).sum())

            if col == "hr":
                feat["hr_gt100"] = int(np.nansum(vals > 100))
                feat["hr_lt60"]  = int(np.nansum(vals < 60))
            if col == "sbp":
                feat["sbp_lt90"]  = int(np.nansum(vals < 90))
                feat["sbp_gt160"] = int(np.nansum(vals > 160))
            if col == "temp_c":
                feat["temp_gt38"] = int(np.nansum(vals > 38))
                feat["temp_lt36"] = int(np.nansum(vals < 36))
            if col == "rr":
                feat["rr_gt22"] = int(np.nansum(vals > 22))
            if col == "dbp":
                feat["dbp_lt60"] = int(np.nansum(vals < 60))

        if include_keywords:
            for kw, pat in KW_PATTERNS:
                key = kw.replace(" ", "_").replace("/", "_")
                feat[f"all_kw_{key}"]   = len(pat.findall(note_all))
                feat[f"last2_kw_{key}"] = len(pat.findall(note_last2))

        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    return base.merge(feat_df, on="stay_id", how="left")

train_full = build_stay_features(stays_train, include_keywords=True)
test_full  = build_stay_features(stays_test,  include_keywords=True)

y = train_full["discharge_ready_day11"].astype(int).values
train_prev = float(y.mean())

# =====================
# 6) Modeling utilities
# =====================
ID_COLS = ["stay_id", "patient_id"]
BASE_CAT_COLS = ["unit_type", "sex", "insurance", "zip3", "admission_reason"]
TEXT_COL = "note_text_all"
DROP_TEXT_HELPER_COLS = ["note_text_last2"]

def text_selector(x):
    s = x.iloc[:, 0] if isinstance(x, pd.DataFrame) else pd.Series(x)
    return s.fillna("").astype(str).values

def make_preprocessor(df_X: pd.DataFrame, include_keywords: bool, include_zip3: bool,
                      tfidf_max_features: int = 4000, ngram_range=(1, 2)):
    cat_cols = [c for c in BASE_CAT_COLS if include_zip3 or (c != "zip3")]
    num_cols = [c for c in df_X.columns if c not in (ID_COLS + cat_cols + [TEXT_COL] + DROP_TEXT_HELPER_COLS)]
    if not include_keywords:
        num_cols = [c for c in num_cols if (not c.startswith("all_kw_")) and (not c.startswith("last2_kw_"))]

    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler(with_mean=False))]), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("note", Pipeline([("selector", FunctionTransformer(text_selector, validate=False)),
                               ("tfidf", TfidfVectorizer(min_df=2, max_features=tfidf_max_features, ngram_range=ngram_range))]),
             [TEXT_COL]),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    schema = {
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "text_col": TEXT_COL,
        "tfidf": {"min_df": 2, "max_features": tfidf_max_features, "ngram_range": ngram_range},
        "include_keywords": bool(include_keywords),
        "include_zip3": bool(include_zip3),
    }
    return pre, schema

def build_fold_cache(df_X: pd.DataFrame, y: np.ndarray, preprocessor: ColumnTransformer):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_cache = []
    fold_ids = np.zeros(len(df_X), dtype=int)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(df_X, y)):
        pre = clone(preprocessor)
        X_tr = pre.fit_transform(df_X.iloc[tr_idx], y[tr_idx])
        X_va = pre.transform(df_X.iloc[va_idx])
        fold_cache.append({"fold": fold, "tr_idx": tr_idx, "va_idx": va_idx, "X_tr": X_tr, "y_tr": y[tr_idx], "X_va": X_va})
        fold_ids[va_idx] = fold
    return fold_cache, fold_ids

def oof_from_cache(fold_cache, C: float, class_weight):
    n = int(max(fd["va_idx"].max() for fd in fold_cache) + 1)
    oof = np.zeros(n, dtype=float)
    for fd in fold_cache:
        clf = LogisticRegression(
            solver="liblinear",
            penalty="l1",
            C=float(C),
            class_weight=class_weight,
            max_iter=5000,
            random_state=SEED
        )
        clf.fit(fd["X_tr"], fd["y_tr"])
        oof[fd["va_idx"]] = clf.predict_proba(fd["X_va"])[:, 1]
    return oof

def threshold_sweep(oof_proba: np.ndarray, y_true: np.ndarray, fold_ids: np.ndarray, thresholds: np.ndarray):
    rows = []
    for t in thresholds:
        pred = (oof_proba >= t).astype(int)
        cm = confusion_matrix(y_true, pred, labels=[0, 1])
        f1s = f1_score(y_true, pred, labels=[0, 1], average=None, zero_division=0)
        macro = f1_score(y_true, pred, average="macro", zero_division=0)

        fold_scores = []
        for f in range(N_SPLITS):
            idx = fold_ids == f
            fold_scores.append(f1_score(y_true[idx], pred[idx], average="macro", zero_division=0))

        rows.append({
            "threshold": float(t),
            "macro_f1": float(macro),
            "f1_0": float(f1s[0]),
            "f1_1": float(f1s[1]),
            "pos_rate": float(pred.mean()),
            "fold_mean": float(np.mean(fold_scores)),
            "fold_std": float(np.std(fold_scores)),
            "tn": int(cm[0, 0]), "fp": int(cm[0, 1]), "fn": int(cm[1, 0]), "tp": int(cm[1, 1]),
        })
    return pd.DataFrame(rows).sort_values("threshold", ascending=True).reset_index(drop=True)

def choose_threshold(df_sweep: pd.DataFrame, prevalence: float, pos_tol: float = 0.07, macro_drop_tol: float = 0.005):
    best_macro = float(df_sweep["macro_f1"].max())
    df_pos_ok = df_sweep.loc[(df_sweep["pos_rate"] - prevalence).abs() <= pos_tol].copy()
    pool = df_pos_ok if len(df_pos_ok) else df_sweep.copy()

    pool_best = float(pool["macro_f1"].max())
    df_near = pool.loc[pool["macro_f1"] >= (pool_best - macro_drop_tol)].copy()
    df_near["pos_gap"] = (df_near["pos_rate"] - prevalence).abs()

    df_near = df_near.sort_values(["pos_gap", "fold_std", "macro_f1", "threshold"],
                                  ascending=[True, True, False, True]).reset_index(drop=True)
    chosen = df_near.iloc[0].to_dict()
    chosen["best_macro_overall"] = best_macro
    return chosen

def evaluate_candidate(name, df_X, y, fold_cache, fold_ids, include_keywords, include_zip3,
                       C, class_weight, pos_tol=0.07, macro_drop_tol=0.005):
    oof = oof_from_cache(fold_cache, C=C, class_weight=class_weight)
    thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
    df_sweep = threshold_sweep(oof, y, fold_ids, thresholds)
    chosen = choose_threshold(df_sweep, prevalence=train_prev, pos_tol=pos_tol, macro_drop_tol=macro_drop_tol)
    chosen.update({
        "name": name,
        "C": float(C),
        "class_weight": str(class_weight),
        "include_keywords": bool(include_keywords),
        "include_zip3": bool(include_zip3),
        "pos_tol": float(pos_tol),
        "macro_drop_tol": float(macro_drop_tol),
        "auc": float(roc_auc_score(y, oof)),
    })
    return chosen, df_sweep, oof

# =====================
# 7) Ablations (one change at a time)
# =====================
train_X = train_full.drop(columns=["discharge_ready_day11"]).copy()
test_X  = test_full.copy()

for c in BASE_CAT_COLS:
    train_X[c] = train_X[c].astype(str)
    test_X[c]  = test_X[c].astype(str)

pre_zip, schema_zip = make_preprocessor(train_X, include_keywords=True, include_zip3=True)
cache_zip, fold_ids_zip = build_fold_cache(train_X, y, pre_zip)

pre_nozip, schema_nozip = make_preprocessor(train_X, include_keywords=True, include_zip3=False)
cache_nozip, fold_ids_nozip = build_fold_cache(train_X, y, pre_nozip)

# B0: legacy-ish (balanced)
cand0, sweep0, oof0 = evaluate_candidate(
    name="B0_kw_zip3_balanced_C0.12",
    df_X=train_X, y=y, fold_cache=cache_zip, fold_ids=fold_ids_zip,
    include_keywords=True, include_zip3=True, C=0.12, class_weight="balanced"
)

# B1: NEW (no class_weight), tuned C
cand1, sweep1, oof1 = evaluate_candidate(
    name="B1_kw_zip3_noCW_C0.11",
    df_X=train_X, y=y, fold_cache=cache_zip, fold_ids=fold_ids_zip,
    include_keywords=True, include_zip3=True, C=0.11, class_weight=None
)

# B2: NEW (no zip3) + no class_weight (robustness)
cand2, sweep2, oof2 = evaluate_candidate(
    name="B2_kw_nozip3_noCW_C0.11",
    df_X=train_X, y=y, fold_cache=cache_nozip, fold_ids=fold_ids_nozip,
    include_keywords=True, include_zip3=False, C=0.11, class_weight=None
)

def attach_cm_counts(cand, oof):
    thr = float(cand["threshold"])
    pred = (oof >= thr).astype(int)
    cm = confusion_matrix(y, pred, labels=[0,1])
    cand["tn"], cand["fp"], cand["fn"], cand["tp"] = int(cm[0,0]), int(cm[0,1]), int(cm[1,0]), int(cm[1,1])
    return cand

cand0 = attach_cm_counts(cand0, oof0)
cand1 = attach_cm_counts(cand1, oof1)
cand2 = attach_cm_counts(cand2, oof2)

ablations = [cand0, cand1, cand2]
abl_df = pd.DataFrame(ablations).sort_values("macro_f1", ascending=False)

print("\n=== Ablation summary (selected threshold per candidate) ===")
cols_show = ["name","macro_f1","best_macro_overall","threshold","pos_rate","fold_mean","fold_std",
             "f1_0","f1_1","auc","tn","fp","fn","tp","class_weight","C","include_zip3"]
print(abl_df[cols_show].to_string(index=False))

ABL_PATH = ART_ROOT / f"{ITER_TAG}_ablation_summary.csv"
abl_df.to_csv(ABL_PATH, index=False)

# =====================
# 8) Candidate selection with hard gate vs P*
# =====================
MIN_DELTA = 0.002
STD_TOL = 0.01
POS_TOL = 0.07

pstar_macro = float(pstar_metrics.get("oof_macro_f1", -1.0))
pstar_std   = pstar_metrics.get("fold_std", None)

def passes_gate(c):
    if float(c["macro_f1"]) < pstar_macro + MIN_DELTA:
        return False
    if (pstar_std is not None) and (c.get("fold_std") is not None):
        if float(c["fold_std"]) > float(pstar_std) + STD_TOL:
            return False
    if abs(float(c["pos_rate"]) - train_prev) > POS_TOL:
        return False
    return True

eligible = [c for c in ablations if passes_gate(c)]
if eligible:
    selected = sorted(eligible, key=lambda d: float(d["macro_f1"]), reverse=True)[0]
    decision = f"SELECT_CANDIDATE (beats P* by >= {MIN_DELTA})"
else:
    selected = max(ablations, key=lambda d: float(d["macro_f1"]))
    decision = "NO_CANDIDATE_PASSES_GATE -> ROLLBACK_TO_PSTAR_IF_AVAILABLE"

print("\n=== Selection decision ===")
print(decision)
print("Selected candidate:", selected["name"], "| macro_f1=", selected["macro_f1"], "thr=", selected["threshold"], "pos_rate=", selected["pos_rate"])

# Map selected candidate to its artifacts/settings
if selected["name"].startswith("B0"):
    sel_oof, sel_folds, sel_sweep, sel_schema, sel_C, sel_cw, sel_include_zip3 = oof0, fold_ids_zip, sweep0, schema_zip, 0.12, "balanced", True
elif selected["name"].startswith("B1"):
    sel_oof, sel_folds, sel_sweep, sel_schema, sel_C, sel_cw, sel_include_zip3 = oof1, fold_ids_zip, sweep1, schema_zip, 0.11, None, True
else:
    sel_oof, sel_folds, sel_sweep, sel_schema, sel_C, sel_cw, sel_include_zip3 = oof2, fold_ids_nozip, sweep2, schema_nozip, 0.11, None, False

# Rollback pipeline path (if exists)
pstar_tag = str(pstar_metrics.get("iteration_tag", ""))
pstar_pipeline_path = CKPT_ROOT / pstar_tag / "pipeline.joblib"
use_rollback_pipeline = (not eligible) and pstar_pipeline_path.exists()

# IMPORTANT: if rolling back, align saved artifacts/metrics with rollback pipeline (avoid candidate/pipeline mismatch)
if use_rollback_pipeline:
    pstar_oof_path = ART_ROOT / f"{pstar_tag}_oof_predictions.csv"
    pstar_sweep_path = ART_ROOT / f"{pstar_tag}_threshold_sweep.csv"

    if pstar_oof_path.exists():
        _p_oof_df = pd.read_csv(pstar_oof_path)
        _p_oof_df = ensure_oof_has_fold(_p_oof_df)
        sel_oof = _p_oof_df["oof_proba"].astype(float).values
        sel_folds = _p_oof_df["fold"].astype(int).values

    if pstar_sweep_path.exists():
        _p_sw = pd.read_csv(pstar_sweep_path)
        if "macro_f1" not in _p_sw.columns and "macro_f1_oof" in _p_sw.columns:
            _p_sw = _p_sw.rename(columns={"macro_f1_oof": "macro_f1"})
        sel_sweep = _p_sw

    pstar_schema_path = CKPT_ROOT / pstar_tag / "schema.json"
    if pstar_schema_path.exists():
        try:
            with open(pstar_schema_path, "r", encoding="utf-8") as f:
                sel_schema = json.load(f)
        except Exception:
            pass

# =====================
# 9) Fit final pipeline + write submission
# =====================
if use_rollback_pipeline:
    print("[ROLLBACK] Using P* pipeline from:", pstar_pipeline_path)
    pipe = joblib.load(pstar_pipeline_path)
    thr = float(pstar_metrics["threshold"])
else:
    pre, _ = make_preprocessor(
        train_X, include_keywords=True, include_zip3=sel_include_zip3,
        tfidf_max_features=sel_schema["tfidf"]["max_features"],
        ngram_range=tuple(sel_schema["tfidf"]["ngram_range"])
    )
    clf = LogisticRegression(
        solver="liblinear",
        penalty="l1",
        C=float(sel_C),
        class_weight=sel_cw,
        max_iter=5000,
        random_state=SEED
    )
    pipe = Pipeline([("preprocess", pre), ("clf", clf)])
    pipe.fit(train_X, y)
    thr = float(selected["threshold"])

test_proba = pipe.predict_proba(test_X)[:, 1]
test_pred  = (test_proba >= thr).astype(int)

sub = pd.DataFrame({
    "stay_id": test_full["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred.astype(int)
})
SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"
sub.to_csv(SUB_PATH, index=False)

print("\n=== Submission ===")
print("Path:", SUB_PATH)
print("Threshold used:", thr)
print("Submission positive-rate:", float(sub["discharge_ready_day11"].mean()))
print(sub.head())

# Snapshot under artifacts (FIX: avoid 'iteriter' filename)
CAND_PATH = ART_ROOT / f"{TAG}_{BRANCH}_block{BLOCK_ID}_{ITER_TAG}_candidate.csv"
sub.to_csv(CAND_PATH, index=False)

# =====================
# 10) Save artifacts + checkpoint bundle
# =====================
oof_df = pd.DataFrame({
    "stay_id": train_full["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "oof_proba": np.asarray(sel_oof, dtype=float),
    "fold": np.asarray(sel_folds, dtype=int)
})
oof_path = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
oof_df.to_csv(oof_path, index=False)

sweep_path = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"
sel_sweep.to_csv(sweep_path, index=False)

iter_ckpt_dir = CKPT_ROOT / ITER_TAG
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

# If rollback, try to read true hyperparams from loaded pipeline for logging
rb_C = None
rb_cw = None
if use_rollback_pipeline:
    try:
        _clf = pipe.named_steps.get("clf", None)
        rb_C = getattr(_clf, "C", None)
        rb_cw = getattr(_clf, "class_weight", None)
    except Exception:
        pass

selected_name_for_log = f"ROLLBACK_{pstar_tag}" if use_rollback_pipeline else selected["name"]
selected_C_for_log = float(rb_C) if use_rollback_pipeline and (rb_C is not None) else (float(sel_C) if not use_rollback_pipeline else None)
selected_cw_for_log = str(rb_cw) if use_rollback_pipeline else str(sel_cw)

config = {
    "team": TEAM,
    "task": TASK,
    "version": VERSION,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "seed": SEED,
    "validation": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "selected": {
        "name": selected_name_for_log,
        "include_keywords": True,
        "include_zip3": bool(sel_schema.get("include_zip3", True)),
        "C": selected_C_for_log,
        "class_weight": selected_cw_for_log,
        "threshold": float(thr),
        "threshold_selection": {
            "pos_tol": float(selected.get("pos_tol", 0.07)),
            "macro_drop_tol": float(selected.get("macro_drop_tol", 0.005)),
        },
        "rollback_used": bool(use_rollback_pipeline),
        "rollback_from": pstar_tag if use_rollback_pipeline else None,
    },
    "tfidf": sel_schema["tfidf"],
    "paths": {"artifacts": str(ART_ROOT), "checkpoints": str(CKPT_ROOT)},
    "notes": [
        "Scoped artifacts enforced under artifacts/clai_ch3_v3 + checkpoints/clai_ch3_v3 only.",
        "Vitals strictly limited to days 1-10 (asserted max(day)<=10).",
        "This iter: ablation focused on class_weight balanced vs None + zip3 robustness.",
    ]
}

pred_sel = (np.asarray(sel_oof, dtype=float) >= thr).astype(int)
cm_sel = confusion_matrix(y, pred_sel, labels=[0,1]).tolist()
f1s_sel = f1_score(y, pred_sel, labels=[0,1], average=None, zero_division=0)
macro_sel = f1_score(y, pred_sel, average="macro", zero_division=0)

fold_scores_sel = []
for f in range(N_SPLITS):
    idx = np.asarray(sel_folds, dtype=int) == f
    fold_scores_sel.append(f1_score(y[idx], pred_sel[idx], average="macro", zero_division=0))

metrics = {
    "oof_macro_f1": float(macro_sel),
    "per_class_f1": {"f1_0": float(f1s_sel[0]), "f1_1": float(f1s_sel[1])},
    "confusion_matrix": cm_sel,
    "threshold": float(thr),
    "pos_rate": float(pred_sel.mean()),
    "fold_scores": [float(x) for x in fold_scores_sel],
    "fold_mean": float(np.mean(fold_scores_sel)),
    "fold_std": float(np.std(fold_scores_sel)),
    "auc": float(roc_auc_score(y, np.asarray(sel_oof, dtype=float))),
    "train_prevalence": train_prev,
    "gate": {"min_delta": MIN_DELTA, "std_tol": STD_TOL, "pos_tol": POS_TOL, "decision": decision},
    "pstar_reference": pstar_metrics,
}

with open(iter_ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "schema.json", "w", encoding="utf-8") as f:
    json.dump(sel_schema, f, indent=2, ensure_ascii=False)
joblib.dump(pipe, iter_ckpt_dir / "pipeline.joblib")

print("\nSaved checkpoint:", iter_ckpt_dir)
print("Saved artifacts:", oof_path.name, sweep_path.name, ABL_PATH.name, CAND_PATH.name)

# Update PSTAR.json if candidate passes gate
if eligible:
    new_pstar = {
        "iteration_tag": ITER_TAG,
        "oof_macro_f1": float(macro_sel),
        "threshold": float(thr),
        "best_threshold": float(thr),
        "pos_rate": float(pred_sel.mean()),
        "fold_mean": float(np.mean(fold_scores_sel)),
        "fold_std": float(np.std(fold_scores_sel)),
        "f1_0": float(f1s_sel[0]),
        "f1_1": float(f1s_sel[1]),
        "confusion_matrix": cm_sel,
    }
    with open(PSTAR_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "tag": TAG,
            "pstar": new_pstar,
            "train_prevalence": train_prev,
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "schema_version": 2,
            "notes": "Updated because candidate passed hard gate vs prior P*."
        }, f, indent=2, ensure_ascii=False)
    print("✅ Updated PSTAR.json ->", new_pstar["iteration_tag"], "macro=", new_pstar["oof_macro_f1"])
else:
    print("ℹ️ PSTAR.json unchanged (no candidate passed hard gate).")

# =====================
# 11) Append iteration_detail.jsonl (append-only)
# =====================
timestamp = datetime.utcnow().isoformat() + "Z"
hm_message = (
    "HM update: latest real F1=0.6765 (catastrophic regression). "
    "Baseline best real=0.7746 code provided. "
    "This iter: ablation class_weight ('balanced' vs None) + retune C, plus zip3 robustness ablation."
)
real_score = {"latest_real_f1": 0.6765, "prev_best_real_f1": 0.7746}

iteration_record = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": timestamp,
    "phase": "Modeling",
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "data_paths_used": {
        "stays_train": str(dp("stays_train.csv")),
        "stays_test": str(dp("stays_test.csv")),
        "patients": str(dp("patients.csv")),
        "vitals_timeseries": str(dp("vitals_timeseries.json")),
        "art_root": str(ART_ROOT),
        "ckpt_root": str(CKPT_ROOT),
    },
    "join_keys": ["stays.patient_id -> patients.patient_id", "stays.stay_id -> vitals_timeseries.stay_id"],
    "leakage_checks_performed": [
        "Asserted vitals max(day) <= 10 (no day11 leakage).",
        "Scoped artifact discovery ONLY under v3 dirs; auto-generated missing sweeps if oof exists.",
    ],
    "feature_summary": {
        "categorical": BASE_CAT_COLS,
        "text": {"col": TEXT_COL, "tfidf": sel_schema["tfidf"]},
        "keywords_count": len(KEYWORDS),
        "vitals_cols": VITAL_COLS,
        "vitals_days_used": "1-10",
        "numeric_feature_count": len(sel_schema["num_cols"]),
    },
    "models_attempted": [
        {k: cand0.get(k) for k in cols_show},
        {k: cand1.get(k) for k in cols_show},
        {k: cand2.get(k) for k in cols_show},
    ],
    "selected_model": {
        "name": selected_name_for_log,
        "C": selected_C_for_log,
        "penalty": "l1",
        "solver": "liblinear",
        "class_weight": selected_cw_for_log,
        "include_keywords": True,
        "include_zip3": bool(sel_schema.get("include_zip3", True)),
        "threshold": float(thr),
        "decision": decision,
        "rollback_used": bool(use_rollback_pipeline),
        "rollback_from": pstar_tag if use_rollback_pipeline else None,
    },
    "validation_protocol": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "metrics": metrics,
    "artifacts_written": {
        "submission_csv": str(SUB_PATH),
        "candidate_snapshot": str(CAND_PATH),
        "oof_predictions": str(oof_path),
        "threshold_sweep": str(sweep_path),
        "ablation_summary": str(ABL_PATH),
        "checkpoint_dir": str(iter_ckpt_dir),
        "pstar_path": str(PSTAR_PATH),
    },
    "deltas_vs_prev": {
        "primary_goal": "Recover from real-F1 regression; improve OOF and reduce misalignment risk.",
        "changes": [
            "Ablated LogisticRegression class_weight: 'balanced' vs None (NEW).",
            "Retuned C to 0.11 under fixed CV; kept vitals trajectory + TF-IDF + keyword counts.",
            "Ran robustness ablation by removing zip3 one-hot to reduce overfit risk."
        ],
    },
    "known_defects_limitations": [
        "Still a linear model; may miss non-linear interactions.",
        "Threshold selected on OOF grid; may still misalign if test prevalence shifts.",
    ],
    "next_step_hypothesis": (
        "If real-F1 still unstable, try conservative thresholding (tighter pos_tol) and/or "
        "blend a numeric-only non-linear model (HGB) with sparse LR(text) via OOF stacking."
    ),
    "hm_message": hm_message,
    "real_score": real_score,
}

with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
print("Appended iteration log ->", LOG_PATH)

# =====================
# 12) Consultant packet (v3: every iteration)
# =====================
consult_packet = {
    "tag": TAG,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": timestamp,
    "what_changed": iteration_record["deltas_vs_prev"]["changes"],
    "metric_changes": {
        "pstar_reference": pstar_metrics,
        "selected_metrics": metrics,
        "gate_decision": decision,
    },
    "suspected_leakage_risks": [
        "Vitals file includes only days 1-10; asserted max(day)<=10.",
        "Notes confined to days 1-10."
    ],
    "complexity_drift_risks": [
        "TF-IDF + OneHot can be high-dimensional; keep max_features controlled.",
        "Keyword-count features are hand-crafted; monitor stability if note phrasing shifts."
    ],
    "evaluation_alignment_risks": [
        "Changing class_weight shifts calibration; thresholds re-swept with prevalence-aware tie-break.",
        "OOF-best threshold may still misalign if test prevalence shifts."
    ],
    "unknown_unknowns": [
        "Distribution shift in notes/vitals could still cause OOF-real mismatch."
    ],
    "recommendations_for_next_iter": [
        "If real still low: tighten pos_tol or penalize pos_gap in threshold selection.",
        "Try numeric-only non-linear model + blend with LR(text) using OOF stacking."
    ]
}

consult_path = CONSULT_ROOT / f"{TAG}_iter{ITER_ID}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, ensure_ascii=False)
print("Wrote consultant packet ->", consult_path)

=== PREFLIGHT ===
CWD: d:\AgentDs\agent_ds_healthcare
PROJECT_ROOT: d:\AgentDs\agent_ds_healthcare
ART_ROOT: d:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT: d:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
ITER_ID=15  ITER_TAG=iter0015  BLOCK_ID=3  BLOCK_POS=0  BRANCH=W4
Discovered iter tags (scoped): ['iter0000', 'iter0002', 'iter0003', 'iter0007', 'iter0008', 'iter0009', 'iter0010', 'iter0011', 'iter0012', 'iter0013']


C:\Users\shend\AppData\Local\Temp\ipykernel_6244\906261623.py:292: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",


=== P* (recomputed) ===
{
  "iteration_tag": "iter0002",
  "oof_macro_f1": 0.766406614124761,
  "threshold": 0.44,
  "best_threshold": 0.44,
  "pos_rate": 0.655,
  "fold_mean": 0.7661866671646843,
  "fold_std": 0.012079082886103704,
  "f1_0": 0.6937590711175616,
  "f1_1": 0.8390541571319603,
  "confusion_matrix": [
    [
      239,
      105
    ],
    [
      106,
      550
    ]
  ]
}

=== Ablation summary (selected threshold per candidate) ===
                     name  macro_f1  best_macro_overall  threshold  pos_rate  fold_mean  fold_std     f1_0     f1_1      auc  tn  fp  fn  tp class_weight    C  include_zip3
    B1_kw_zip3_noCW_C0.11  0.774359            0.774359       0.59     0.694   0.774086  0.033223 0.695385 0.853333 0.852147 226 118  80 576         None 0.11          True
  B2_kw_nozip3_noCW_C0.11  0.773858            0.773858       0.60     0.684   0.773705  0.032556 0.696970 0.850746 0.852152 230 114  86 570         None 0.11         False
B0_kw_zip3_balanced_C0.12  0.7

C:\Users\shend\AppData\Local\Temp\ipykernel_6244\906261623.py:851: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat() + "Z"


# Iteration 8
- 0.7903

In [42]:
import os, json, re, shutil, warnings, random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.base import clone
import joblib

# =====================
# 0) Config + paths
# =====================
TEAM, TASK, VERSION = "clai", "ch3", "v3"
TAG = f"{TEAM}_{TASK}_{VERSION}"
SEED = 42
N_SPLITS = 5

np.random.seed(SEED)
random.seed(SEED)

def _find_project_root():
    cand = Path.cwd()
    if (cand / "stays_train.csv").exists():
        return cand
    cand2 = Path("/mnt/data")
    if (cand2 / "stays_train.csv").exists():
        return cand2
    return cand

PROJECT_ROOT = _find_project_root()
ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"

ART_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
CONSULT_ROOT.mkdir(parents=True, exist_ok=True)

print("=== PREFLIGHT ===")
print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_ROOT:", ART_ROOT)
print("CKPT_ROOT:", CKPT_ROOT)

# =====================
# 1) Bootstrap: copy any loose artifacts into scoped dir (idempotent)
# =====================
def _copy_if_missing(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists() and (not dst.exists()):
        shutil.copy2(src, dst)
        print("[copied]", src.name, "->", dst)

def _bootstrap_from(root: Path):
    for src in sorted(root.glob("iter????_oof_predictions.csv")):
        _copy_if_missing(src, ART_ROOT / src.name)
    for src in sorted(root.glob("iter????_threshold_sweep.csv")):
        _copy_if_missing(src, ART_ROOT / src.name)
    for src in sorted(root.glob("iter????_*ablation*.csv")):
        _copy_if_missing(src, ART_ROOT / src.name)
    for src in sorted(root.glob(f"{TAG}_*candidate*.csv")):
        _copy_if_missing(src, ART_ROOT / src.name)
    if (root / "PSTAR.json").exists():
        _copy_if_missing(root / "PSTAR.json", CKPT_ROOT / "PSTAR.json")

_bootstrap_from(PROJECT_ROOT)
if PROJECT_ROOT != Path("/mnt/data"):
    _bootstrap_from(Path("/mnt/data"))

LOG_PATH = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"
# If the uploaded log is owned by root (common in this sandbox), it may be read-only for our user.
# Create a writable copy in-place while preserving content.
if LOG_PATH.exists() and (not os.access(LOG_PATH, os.W_OK)):
    print("[warn] iteration_detail log not writable; recreating writable copy:", LOG_PATH)
    tmp = LOG_PATH.with_suffix(".tmp")
    shutil.copyfile(LOG_PATH, tmp)
    try:
        LOG_PATH.unlink()
    except Exception as e:
        raise RuntimeError(f"Cannot replace non-writable log file at {LOG_PATH}: {e}")
    shutil.move(tmp, LOG_PATH)
if not LOG_PATH.exists():
    LOG_PATH.write_text("", encoding="utf-8")

# =====================
# 2) Iteration id (robust: log + existing iter tags)
# =====================
def _max_iter_from_name(name: str):
    m = re.match(r"iter(\d{4})", name)
    return int(m.group(1)) if m else None

def _max_iter_in_paths():
    max_id = -1
    for p in list(ART_ROOT.glob("iter????_*")) + list(CKPT_ROOT.glob("iter????")):
        v = _max_iter_from_name(p.name)
        if v is not None:
            max_id = max(max_id, v)
    return max_id

def next_iteration_id(log_path: Path) -> int:
    max_log = -1
    if log_path.exists():
        with open(log_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    max_log = max(max_log, int(obj.get("iteration_id", -1)))
                except Exception:
                    continue
    max_art = _max_iter_in_paths()
    return max(max_log, max_art) + 1

ITER_ID = next_iteration_id(LOG_PATH)
ITER_TAG = f"iter{ITER_ID:04d}"
BLOCK_ID = ITER_ID // 5
BLOCK_POS = ITER_ID % 5
BRANCH = "W1"  # low-risk, physiology-derived vitals

print(f"ITER_ID={ITER_ID}  ITER_TAG={ITER_TAG}  BLOCK_ID={BLOCK_ID}  BLOCK_POS={BLOCK_POS}  BRANCH={BRANCH}")

# =====================
# 3) Data loading + fixed folds (for P* recompute & artifact hygiene)
# =====================
def dp(name: str) -> Path:
    p = PROJECT_ROOT / name
    if p.exists():
        return p
    p2 = Path("/mnt/data") / name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {name} under PROJECT_ROOT or /mnt/data")

stays_train = pd.read_csv(dp("stays_train.csv"))
y_all = stays_train["discharge_ready_day11"].astype(int).values
train_prev = float(y_all.mean())

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.empty(len(y_all), dtype=int)
for f, (_, va_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
    folds[va_idx] = f
fold_map = pd.DataFrame({"stay_id": stays_train["stay_id"].values, "fold": folds})

# =====================
# 4) Scoped artifact discovery + auto-fix (missing sweeps / fold col)
# =====================
def discover_iter_tags_scoped(art_root: Path):
    oof_files = sorted(art_root.glob("iter????_oof_predictions.csv"))
    sweep_files = sorted(art_root.glob("iter????_threshold_sweep.csv"))
    tags = sorted({p.name.split("_")[0] for p in (oof_files + sweep_files)})
    return tags

def ensure_oof_has_fold(oof_df: pd.DataFrame) -> pd.DataFrame:
    if "fold" in oof_df.columns:
        return oof_df
    out = oof_df.merge(fold_map, on="stay_id", how="left")
    if out["fold"].isna().any():
        miss = out.loc[out["fold"].isna(), "stay_id"].head(10).tolist()
        raise RuntimeError(f"[ABORT] fold backfill failed; e.g. missing stay_id={miss}")
    out["fold"] = out["fold"].astype(int)
    return out

def compute_threshold_sweep(oof_df: pd.DataFrame, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
    y = oof_df["y_true"].astype(int).values
    proba = oof_df["oof_proba"].astype(float).values
    fold_ids = oof_df["fold"].astype(int).values
    rows = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        cm = confusion_matrix(y, pred)
        f1s = f1_score(y, pred, average=None)
        macro = f1_score(y, pred, average="macro")
        fold_scores = []
        for f in range(N_SPLITS):
            idx = fold_ids == f
            fold_scores.append(f1_score(y[idx], pred[idx], average="macro"))
        rows.append({
            "threshold": float(t),
            "macro_f1": float(macro),
            "f1_0": float(f1s[0]),
            "f1_1": float(f1s[1]),
            "pos_rate": float(pred.mean()),
            "fold_mean": float(np.mean(fold_scores)),
            "fold_std": float(np.std(fold_scores)),
            "tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        })
    return pd.DataFrame(rows)

tags = discover_iter_tags_scoped(ART_ROOT)
print("Discovered iter tags (scoped):", tags)

# Generate missing sweeps if needed
for t in tags:
    oof_p = ART_ROOT / f"{t}_oof_predictions.csv"
    sw_p  = ART_ROOT / f"{t}_threshold_sweep.csv"
    if oof_p.exists() and (not sw_p.exists()):
        oof_df = pd.read_csv(oof_p)
        oof_df = ensure_oof_has_fold(oof_df)
        oof_df.to_csv(oof_p, index=False)
        sw = compute_threshold_sweep(oof_df)
        sw.to_csv(sw_p, index=False)
        print("[generated sweep]", sw_p.name)

# Hard check: every tag must have both
tags = discover_iter_tags_scoped(ART_ROOT)
for t in tags:
    oof_p = ART_ROOT / f"{t}_oof_predictions.csv"
    sw_p  = ART_ROOT / f"{t}_threshold_sweep.csv"
    if (not oof_p.exists()) or (not sw_p.exists()):
        raise RuntimeError(f"[ABORT] Incomplete artifacts for tag={t} under {ART_ROOT}")

def pick_best_row(sw: pd.DataFrame, prevalence: float):
    sw = sw.copy()
    if "macro_f1" not in sw.columns and "macro_f1_oof" in sw.columns:
        sw = sw.rename(columns={"macro_f1_oof": "macro_f1"})
    if "pos_rate" not in sw.columns:
        sw["pos_rate"] = np.nan
    if "fold_std" not in sw.columns:
        sw["fold_std"] = np.nan
    sw["pos_gap"] = (sw["pos_rate"] - prevalence).abs()
    sw = sw.sort_values(["macro_f1", "pos_gap", "fold_std", "threshold"],
                        ascending=[False, True, True, True]).reset_index(drop=True)
    return sw.iloc[0].to_dict()

# Build summary table for existing iters (small; used for P* reference + audit)
rows_out = []
for t in tags:
    oof_df = pd.read_csv(ART_ROOT / f"{t}_oof_predictions.csv")
    oof_df = ensure_oof_has_fold(oof_df)
    sw = pd.read_csv(ART_ROOT / f"{t}_threshold_sweep.csv")
    if "macro_f1" not in sw.columns and "macro_f1_oof" in sw.columns:
        sw = sw.rename(columns={"macro_f1_oof": "macro_f1"})
    # If sweep schema is thin, recompute fully
    need_cols = {"f1_0","f1_1","tn","fp","fn","tp","pos_rate","fold_mean","fold_std"}
    if not need_cols.issubset(set(sw.columns)):
        sw = compute_threshold_sweep(oof_df)
        sw.to_csv(ART_ROOT / f"{t}_threshold_sweep.csv", index=False)
    best = pick_best_row(sw, prevalence=train_prev)
    rows_out.append({
        "iter_tag": t,
        "oof_macro_f1": float(best["macro_f1"]),
        "threshold": float(best["threshold"]),
        "f1_0": float(best["f1_0"]),
        "f1_1": float(best["f1_1"]),
        "pos_rate": float(best["pos_rate"]),
        "fold_mean": float(best["fold_mean"]),
        "fold_std": float(best["fold_std"]),
        "tn": int(best["tn"]), "fp": int(best["fp"]), "fn": int(best["fn"]), "tp": int(best["tp"]),
        "auc": float(roc_auc_score(oof_df["y_true"].astype(int).values, oof_df["oof_proba"].astype(float).values))
    })

summary_df = pd.DataFrame(rows_out).sort_values("iter_tag").reset_index(drop=True)
if len(summary_df):
    print("\n=== Existing OOF summary (scoped artifacts) ===")
    print(summary_df.to_string(index=False))
else:
    print("\n(No prior iter artifacts found under ART_ROOT; continuing.)")

# P* (validation) = best macro among existing iters (if any)
if len(summary_df):
    pstar_val = summary_df.sort_values("oof_macro_f1", ascending=False).iloc[0].to_dict()
else:
    pstar_val = {"iter_tag":"NONE","oof_macro_f1":-1.0,"threshold":0.5,"fold_std":None,"pos_rate":None}

# Real-best anchor (if iter0002 exists)
if len(summary_df) and (summary_df["iter_tag"]=="iter0002").any():
    pstar_real_anchor = summary_df.loc[summary_df["iter_tag"]=="iter0002"].iloc[0].to_dict()
else:
    pstar_real_anchor = pstar_val.copy()

PSTAR_PATH = CKPT_ROOT / "PSTAR.json"
with open(PSTAR_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "tag": TAG,
        "schema_version": 3,
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "train_prevalence": train_prev,
        "pstar": {  # backward compatible alias
            "iteration_tag": pstar_val.get("iter_tag", "NONE"),
            "oof_macro_f1": float(pstar_val.get("oof_macro_f1", -1.0)),
            "threshold": float(pstar_val.get("threshold", 0.5)),
        },
        "pstar_validation": pstar_val,
        "pstar_real_anchor": pstar_real_anchor,
        "real_best_known": {"iteration_tag": "iter0002", "real_f1": 0.7768},
        "notes": "Schema v3: pstar_validation (best OOF) + pstar_real_anchor (iter0002 if available)."
    }, f, indent=2, ensure_ascii=False)

print("\n=== P* reference ===")
print("P* (validation best):", pstar_val.get("iter_tag"), "macro=", pstar_val.get("oof_macro_f1"))
print("P* (real anchor):    ", pstar_real_anchor.get("iter_tag"), "macro=", pstar_real_anchor.get("oof_macro_f1"))

# =====================
# 5) Load full modeling data
# =====================
stays_test  = pd.read_csv(dp("stays_test.csv"))
patients    = pd.read_csv(dp("patients.csv"))
with open(dp("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vit_list = json.load(f)

# Leakage check: max day must be <=10
max_day = max(d.get("day", 0) for item in vit_list for d in item.get("days", []))
assert max_day <= 10, f"Leakage risk: found vitals day={max_day} (>10). Abort."

vit_map2 = {int(item["stay_id"]): item.get("days", []) for item in vit_list}

# =====================
# 6) Feature engineering: add derived hemodynamics (PP, MAP, Shock Index)
# =====================
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]
DERIVED_SERIES = ["pp", "map", "si"]

KEYWORDS = [
    "stable","improving","worsening","afebrile","fever","chills","hypotensive","tachy",
    "culture","antibiotic","abx","iv","oxygen","weaned","dyspnea","cough",
    "pain","nausea","vomit","n/v","ambulated","pt","ot","telemetry","fluids","room air","sats",
    "discharge","transfer"
]
KW_PATTERNS = [(kw, re.compile(re.escape(kw), re.IGNORECASE)) for kw in KEYWORDS]

def nan_stat(fn, arr):
    arr = np.asarray(arr, dtype=float)
    if arr.size == 0 or np.isfinite(arr).sum() == 0:
        return np.nan
    return fn(arr)

def build_stay_features(stays_df: pd.DataFrame, include_keywords: bool = True, include_derived: bool = True) -> pd.DataFrame:
    base = stays_df.merge(patients, on="patient_id", how="left")
    rows = []
    for sid in base["stay_id"].astype(int).tolist():
        days = sorted(vit_map2.get(int(sid), []), key=lambda x: x.get("day", 0))
        day_idx = np.array([d.get("day", np.nan) for d in days], dtype=float)

        notes = [(d.get("note") or "") for d in days]
        note_all = " ".join(notes).strip()
        note_last2 = " ".join(notes[-2:]).strip()

        feat = {
            "stay_id": int(sid),
            "note_text_all": note_all,
            "note_text_last2": note_last2,
            "note_len_all": len(note_all),
            "note_len_last2": len(note_last2),
        }

        raw = {}
        for col in VITAL_COLS:
            raw[col] = np.array([d.get(col, np.nan) for d in days], dtype=float)

        if include_derived:
            sbp = raw["sbp"]
            dbp = raw["dbp"]
            hr  = raw["hr"]
            raw["pp"]  = sbp - dbp
            raw["map"] = dbp + (sbp - dbp) / 3.0
            raw["si"]  = np.where(sbp > 0, hr / sbp, np.nan)

        series_cols = VITAL_COLS + (DERIVED_SERIES if include_derived else [])
        for col in series_cols:
            vals = raw[col]

            # per-day values (day1..day10)
            for i, day in enumerate(range(1, 11)):
                feat[f"{col}_day{day}"] = vals[i] if i < len(vals) else np.nan

            # day-to-day diffs (d2-d1 .. d10-d9)
            diffs = np.diff(vals) if len(vals) >= 2 else np.array([], dtype=float)
            for j in range(1, 10):
                feat[f"{col}_d{j+1}_minus_d{j}"] = diffs[j-1] if (j-1) < len(diffs) else np.nan
            feat[f"{col}_delta10_9"] = vals[-1] - vals[-2] if len(vals) >= 2 else np.nan

            # stats
            feat[f"{col}_mean"]   = nan_stat(np.nanmean, vals)
            feat[f"{col}_std"]    = nan_stat(np.nanstd, vals)
            feat[f"{col}_min"]    = nan_stat(np.nanmin, vals)
            feat[f"{col}_max"]    = nan_stat(np.nanmax, vals)
            feat[f"{col}_median"] = nan_stat(np.nanmedian, vals)
            feat[f"{col}_first"]  = vals[0] if len(vals) else np.nan
            feat[f"{col}_last"]   = vals[-1] if len(vals) else np.nan

            # recent vs early (last 2 days vs first 8 days)
            last2 = vals[-2:] if len(vals) >= 2 else vals
            early = vals[:8] if len(vals) >= 8 else (vals[:-2] if len(vals) > 2 else vals)
            feat[f"{col}_last2_mean"] = nan_stat(np.nanmean, last2)
            feat[f"{col}_early_mean"] = nan_stat(np.nanmean, early)
            feat[f"{col}_delta_last2_early"] = feat[f"{col}_last2_mean"] - feat[f"{col}_early_mean"]
            feat[f"{col}_delta_last_first"]  = feat[f"{col}_last"] - feat[f"{col}_first"]

            # slope
            mask = (~np.isnan(vals)) & (~np.isnan(day_idx))
            feat[f"{col}_slope"] = np.polyfit(day_idx[mask], vals[mask], 1)[0] if mask.sum() >= 2 else np.nan

            # volatility
            feat[f"{col}_diff_mean"] = nan_stat(np.nanmean, diffs) if len(diffs) else np.nan
            feat[f"{col}_diff_std"]  = nan_stat(np.nanstd, diffs) if len(diffs) else np.nan
            feat[f"{col}_last3_std"] = nan_stat(np.nanstd, vals[-3:]) if len(vals) >= 3 else nan_stat(np.nanstd, vals)

            # missingness
            feat[f"{col}_missing"] = int(np.isnan(vals).sum())

        # clinical thresholds on raw vitals
        feat["hr_gt100"]  = int(np.nansum(raw["hr"] > 100))
        feat["hr_lt60"]   = int(np.nansum(raw["hr"] < 60))
        feat["sbp_lt90"]  = int(np.nansum(raw["sbp"] < 90))
        feat["sbp_gt160"] = int(np.nansum(raw["sbp"] > 160))
        feat["temp_gt38"] = int(np.nansum(raw["temp_c"] > 38))
        feat["temp_lt36"] = int(np.nansum(raw["temp_c"] < 36))
        feat["rr_gt22"]   = int(np.nansum(raw["rr"] > 22))
        feat["dbp_lt60"]  = int(np.nansum(raw["dbp"] < 60))

        if include_derived:
            feat["si_gt0p9"] = int(np.nansum(raw["si"] > 0.9))

        if include_keywords:
            for kw, pat in KW_PATTERNS:
                key = kw.replace(" ", "_").replace("/", "_")
                feat[f"all_kw_{key}"]   = len(pat.findall(note_all))
                feat[f"last2_kw_{key}"] = len(pat.findall(note_last2))

        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    out = base.merge(feat_df, on="stay_id", how="left")
    return out

print("\n[FE] Building features ...")
train_full = build_stay_features(stays_train, include_keywords=True, include_derived=True)
test_full  = build_stay_features(stays_test,  include_keywords=True, include_derived=True)

y = train_full["discharge_ready_day11"].astype(int).values
train_prev2 = float(y.mean())
print("[FE] Train prevalence:", train_prev2)

# =====================
# 7) Modeling utilities (single preprocessor cache; tune C / class_weight)
# =====================
ID_COLS = ["stay_id", "patient_id"]
CAT_COLS = ["unit_type", "sex", "insurance", "zip3", "admission_reason"]
TEXT_COL = "note_text_all"
DROP_TEXT_HELPER_COLS = ["note_text_last2"]

KEYWORD_PREFIXES = ("all_kw_", "last2_kw_")

def text_selector(x):
    s = x.iloc[:, 0] if isinstance(x, pd.DataFrame) else pd.Series(x)
    return s.fillna("").astype(str).values

def make_preprocessor(df: pd.DataFrame, include_keywords: bool = True,
                      tfidf_max_features: int = 4000, tfidf_min_df: int = 2):
    num_cols = [c for c in df.columns if c not in (["discharge_ready_day11"] + ID_COLS + CAT_COLS + [TEXT_COL] + DROP_TEXT_HELPER_COLS)]
    if not include_keywords:
        num_cols = [c for c in num_cols if (not c.startswith(KEYWORD_PREFIXES))]
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler(with_mean=False))]), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
            ("note", Pipeline([("selector", FunctionTransformer(text_selector, validate=False)),
                               ("tfidf", TfidfVectorizer(min_df=tfidf_min_df, max_features=tfidf_max_features, ngram_range=(1,2)))]),
             [TEXT_COL]),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    schema = {"num_cols": num_cols, "cat_cols": CAT_COLS, "text_col": TEXT_COL,
              "tfidf": {"min_df": tfidf_min_df, "max_features": tfidf_max_features, "ngram_range": (1,2)},
              "include_keywords": include_keywords,
              "include_derived": True,
              "derived_series": DERIVED_SERIES}
    return pre, schema

def build_fold_cache(df_X: pd.DataFrame, y: np.ndarray, preprocessor: ColumnTransformer):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_cache = []
    fold_ids = np.zeros(len(df_X), dtype=int)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(df_X, y)):
        pre = clone(preprocessor)
        X_tr = pre.fit_transform(df_X.iloc[tr_idx], y[tr_idx])
        X_va = pre.transform(df_X.iloc[va_idx])
        fold_cache.append({"fold": fold, "tr_idx": tr_idx, "va_idx": va_idx,
                           "X_tr": X_tr, "y_tr": y[tr_idx], "X_va": X_va})
        fold_ids[va_idx] = fold
    return fold_cache, fold_ids

def oof_from_cache(fold_cache, C: float, class_weight):
    n = max(int(fd["va_idx"].max()) for fd in fold_cache) + 1
    oof = np.zeros(n, dtype=float)
    for fd in fold_cache:
        clf = LogisticRegression(
            solver="liblinear",
            penalty="l1",
            C=C,
            class_weight=class_weight,
            max_iter=5000,
            random_state=SEED
        )
        clf.fit(fd["X_tr"], fd["y_tr"])
        oof[fd["va_idx"]] = clf.predict_proba(fd["X_va"])[:, 1]
    return oof

def threshold_sweep(oof_proba: np.ndarray, y_true: np.ndarray, fold_ids: np.ndarray, thresholds: np.ndarray):
    rows = []
    for t in thresholds:
        pred = (oof_proba >= t).astype(int)
        cm = confusion_matrix(y_true, pred)
        f1s = f1_score(y_true, pred, average=None)
        macro = f1_score(y_true, pred, average="macro")
        fold_scores = []
        for f in range(N_SPLITS):
            idx = fold_ids == f
            fold_scores.append(f1_score(y_true[idx], pred[idx], average="macro"))
        rows.append({
            "threshold": float(t),
            "macro_f1": float(macro),
            "f1_0": float(f1s[0]),
            "f1_1": float(f1s[1]),
            "pos_rate": float(pred.mean()),
            "fold_mean": float(np.mean(fold_scores)),
            "fold_std": float(np.std(fold_scores)),
            "tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        })
    return pd.DataFrame(rows).sort_values("threshold", ascending=True).reset_index(drop=True)

def choose_threshold(df_sweep: pd.DataFrame, prevalence: float, pos_tol: float = 0.07, macro_drop_tol: float = 0.005):
    best_macro = float(df_sweep["macro_f1"].max())
    df_pos_ok = df_sweep.loc[(df_sweep["pos_rate"] - prevalence).abs() <= pos_tol].copy()
    candidate_pool = df_pos_ok if len(df_pos_ok) else df_sweep.copy()

    pool_macro = float(candidate_pool["macro_f1"].max())
    df_near = candidate_pool.loc[candidate_pool["macro_f1"] >= (pool_macro - macro_drop_tol)].copy()
    df_near["pos_gap"] = (df_near["pos_rate"] - prevalence).abs()
    df_near = df_near.sort_values(["pos_gap", "fold_std", "macro_f1", "threshold"],
                                  ascending=[True, True, False, True]).reset_index(drop=True)
    chosen = df_near.iloc[0].to_dict()
    chosen["best_macro_overall"] = best_macro
    return chosen

def evaluate_candidate(name, fold_cache, fold_ids, y, C, class_weight):
    oof = oof_from_cache(fold_cache, C=C, class_weight=class_weight)
    thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
    df_sweep = threshold_sweep(oof, y, fold_ids, thresholds)
    chosen = choose_threshold(df_sweep, prevalence=train_prev2, pos_tol=0.07, macro_drop_tol=0.005)
    chosen.update({
        "name": name,
        "C": float(C),
        "class_weight": "balanced" if class_weight == "balanced" else "none",
        "auc": float(roc_auc_score(y, oof)),
    })
    return chosen, df_sweep, oof

train_X = train_full.drop(columns=["discharge_ready_day11"]).copy()
test_X  = test_full.copy()
for c in CAT_COLS:
    train_X[c] = train_X[c].astype(str)
    test_X[c]  = test_X[c].astype(str)

pre, schema = make_preprocessor(train_X, include_keywords=True, tfidf_max_features=4000, tfidf_min_df=2)
fold_cache, fold_ids = build_fold_cache(train_X, y, pre)

candidate_specs = [
    ("D1_derived_kw_C0.08_noCW", 0.08, None),
    ("D2_derived_kw_C0.10_noCW", 0.10, None),
    ("D3_derived_kw_C0.11_noCW", 0.11, None),
    ("D4_derived_kw_C0.10_balanced", 0.10, "balanced"),
]

cands, sweeps, oofs = [], {}, {}
for name, C, cw in candidate_specs:
    cand, sw, oof = evaluate_candidate(name, fold_cache, fold_ids, y, C=C, class_weight=cw)
    cands.append(cand); sweeps[name] = sw; oofs[name] = oof

abl_df = pd.DataFrame(cands).sort_values("macro_f1", ascending=False).reset_index(drop=True)
print("\n=== Ablation summary (derived vitals) ===")
cols_show = ["name","macro_f1","best_macro_overall","threshold","pos_rate","fold_mean","fold_std","f1_0","f1_1","auc","class_weight","C"]
print(abl_df[cols_show].to_string(index=False))

ABL_PATH = ART_ROOT / f"{ITER_TAG}_ablation_summary.csv"
abl_df.to_csv(ABL_PATH, index=False)

# =====================
# 8) Candidate selection (gate vs P* validation)
# =====================
MIN_DELTA = 0.002
STD_TOL = 0.02
POS_TOL = 0.10

pstar_macro = float(pstar_val.get("oof_macro_f1", -1.0))
pstar_std   = pstar_val.get("fold_std", None)

def passes_gate(c):
    if pstar_macro > 0 and float(c["macro_f1"]) < pstar_macro + MIN_DELTA:
        return False
    if (pstar_std is not None) and (c.get("fold_std") is not None) and (not pd.isna(pstar_std)):
        if float(c["fold_std"]) > float(pstar_std) + STD_TOL:
            return False
    if abs(float(c["pos_rate"]) - train_prev2) > POS_TOL:
        return False
    return True

eligible = [c for c in cands if passes_gate(c)]
if eligible:
    selected = sorted(eligible, key=lambda d: float(d["macro_f1"]), reverse=True)[0]
    decision = f"SELECT_CANDIDATE (passes gate vs P*_val; +{MIN_DELTA} macro)"
else:
    selected = max(cands, key=lambda d: float(d["macro_f1"]))
    decision = "NO_CANDIDATE_PASSES_GATE -> KEEP_PSTAR_VALIDATION"

print("\n=== Selection decision ===")
print(decision)
print("Selected:", selected["name"], "| macro_f1=", selected["macro_f1"], "thr=", selected["threshold"], "pos_rate=", selected["pos_rate"])

sel_name = selected["name"]
sel_oof = oofs[sel_name]
sel_sweep = sweeps[sel_name]
thr = float(selected["threshold"])
sel_C = float(selected["C"])
sel_cw = ("balanced" if selected["class_weight"] == "balanced" else None)

# =====================
# 9) Fit final pipeline + write submission
# =====================
pre_final, schema_final = make_preprocessor(train_X, include_keywords=True, tfidf_max_features=4000, tfidf_min_df=2)
clf_final = LogisticRegression(
    solver="liblinear",
    penalty="l1",
    C=sel_C,
    class_weight=sel_cw,
    max_iter=5000,
    random_state=SEED
)
pipe = Pipeline([("preprocess", pre_final), ("clf", clf_final)])
pipe.fit(train_X, y)

test_proba = pipe.predict_proba(test_X)[:, 1]
test_pred  = (test_proba >= thr).astype(int)

sub = pd.DataFrame({
    "stay_id": test_full["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred.astype(int)
})
SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"
sub.to_csv(SUB_PATH, index=False)

print("\n=== Submission ===")
print("Path:", SUB_PATH)
print("Threshold used:", thr)
print("Submission positive-rate:", float(sub["discharge_ready_day11"].mean()))
print(sub.head())

# Candidate snapshot for traceability
CAND_PATH = ART_ROOT / f"{TAG}_{BRANCH}_block{BLOCK_ID}_iter{ITER_ID:04d}_candidate.csv"
sub.to_csv(CAND_PATH, index=False)

# =====================
# 10) Save artifacts + checkpoint bundle
# =====================
oof_df = pd.DataFrame({
    "stay_id": train_full["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "oof_proba": sel_oof.astype(float),
    "fold": fold_ids.astype(int)
})
oof_path = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
oof_df.to_csv(oof_path, index=False)

sweep_path = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"
sel_sweep.to_csv(sweep_path, index=False)

iter_ckpt_dir = CKPT_ROOT / ITER_TAG
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

pred_sel = (sel_oof >= thr).astype(int)
cm_sel = confusion_matrix(y, pred_sel).tolist()
f1s_sel = f1_score(y, pred_sel, average=None)
macro_sel = f1_score(y, pred_sel, average="macro")
fold_scores_sel = [float(f1_score(y[fold_ids==f], pred_sel[fold_ids==f], average="macro")) for f in range(N_SPLITS)]

config = {
    "team": TEAM,
    "task": TASK,
    "version": VERSION,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "seed": SEED,
    "validation": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "selected": {
        "name": selected["name"],
        "C": float(sel_C),
        "class_weight": (sel_cw if sel_cw is not None else "none"),
        "threshold": float(thr),
        "threshold_selection": {"pos_tol": 0.07, "macro_drop_tol": 0.005},
    },
    "schema": schema_final,
    "paths": {"artifacts": str(ART_ROOT), "checkpoints": str(CKPT_ROOT)},
    "notes": [
        "Key change: add derived hemodynamics series (PP, MAP, Shock Index) with same trajectory features as raw vitals.",
        "Leakage guardrail: asserted vitals max(day) <= 10 (no day11 vitals).",
        "Scoped artifact discovery enforced (only artifacts/clai_ch3_v3).",
    ]
}

metrics = {
    "oof_macro_f1": float(macro_sel),
    "per_class_f1": {"f1_0": float(f1s_sel[0]), "f1_1": float(f1s_sel[1])},
    "confusion_matrix": cm_sel,
    "threshold": float(thr),
    "pos_rate": float(pred_sel.mean()),
    "fold_scores": fold_scores_sel,
    "fold_mean": float(np.mean(fold_scores_sel)),
    "fold_std": float(np.std(fold_scores_sel)),
    "auc": float(roc_auc_score(y, sel_oof)),
    "train_prevalence": train_prev2,
    "gate": {"min_delta": 0.002, "std_tol": 0.02, "pos_tol": 0.10, "decision": decision},
    "pstar_validation_reference": pstar_val,
    "pstar_real_anchor_reference": pstar_real_anchor,
}

with open(iter_ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema_final, f, indent=2, ensure_ascii=False)
joblib.dump(pipe, iter_ckpt_dir / "pipeline.joblib")

print("\nSaved checkpoint:", iter_ckpt_dir)
print("Saved artifacts:", oof_path.name, sweep_path.name, ABL_PATH.name, CAND_PATH.name)

# =====================
# 11) Update PSTAR.json if candidate passes gate
# =====================
if eligible:
    new_pstar_val = {
        "iter_tag": ITER_TAG,
        "oof_macro_f1": float(macro_sel),
        "threshold": float(thr),
        "pos_rate": float(pred_sel.mean()),
        "fold_mean": float(np.mean(fold_scores_sel)),
        "fold_std": float(np.std(fold_scores_sel)),
        "f1_0": float(f1s_sel[0]),
        "f1_1": float(f1s_sel[1]),
        "confusion_matrix": cm_sel,
        "auc": float(roc_auc_score(y, sel_oof)),
    }
    with open(PSTAR_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "tag": TAG,
            "schema_version": 3,
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "train_prevalence": train_prev2,
            "pstar": {
                "iteration_tag": ITER_TAG,
                "oof_macro_f1": float(macro_sel),
                "threshold": float(thr),
            },
            "pstar_validation": new_pstar_val,
            "pstar_real_anchor": pstar_real_anchor,
            "real_best_known": {"iteration_tag": "iter0002", "real_f1": 0.7768},
            "notes": "Updated pstar_validation because current iteration passed hard gate."
        }, f, indent=2, ensure_ascii=False)
    print("✅ Updated PSTAR.json ->", ITER_TAG, "macro=", float(macro_sel))
else:
    print("ℹ️ PSTAR.json unchanged (no candidate passed hard gate).")

# =====================
# 12) Append iteration_detail.jsonl (append-only)
# =====================
timestamp = datetime.utcnow().isoformat() + "Z"
hm_message = (
    "HM update: previous real F1 = 0.7739 (back to 0.77x). "
    "Goal: push beyond 0.7768 (best known real at iter0002). "
    "This iteration adds derived hemodynamics (PP/MAP/ShockIndex) to help linear model capture interactions."
)
real_score = {"best_real_iter0002": 0.7768, "prev_iter_real": 0.7739}

iteration_record = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": timestamp,
    "phase": "Modeling",
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "data_paths_used": {
        "stays_train": str(dp("stays_train.csv")),
        "stays_test": str(dp("stays_test.csv")),
        "patients": str(dp("patients.csv")),
        "vitals_timeseries": str(dp("vitals_timeseries.json")),
        "art_root": str(ART_ROOT),
        "ckpt_root": str(CKPT_ROOT),
    },
    "join_keys": ["stays.patient_id -> patients.patient_id", "stays.stay_id -> vitals_timeseries.stay_id"],
    "leakage_checks_performed": [
        "Asserted vitals max(day) <= 10 (no day11 vitals).",
        "No use of target-derived columns or test labels.",
        "Scoped artifact discovery to v3 dirs only.",
    ],
    "feature_summary": {
        "categorical": CAT_COLS,
        "text": {"col": TEXT_COL, "tfidf": schema_final["tfidf"]},
        "vitals_cols": VITAL_COLS,
        "derived_series": DERIVED_SERIES,
        "keywords_count": len(KEYWORDS),
        "vitals_days_used": "1-10",
        "numeric_feature_count": len(schema_final["num_cols"]),
    },
    "models_attempted": [
        {k: row[k] for k in ["name","macro_f1","best_macro_overall","threshold","pos_rate","fold_mean","fold_std","f1_0","f1_1","auc","class_weight","C"]}
        for row in cands
    ],
    "selected_model": {
        "name": selected["name"],
        "C": float(sel_C),
        "penalty": "l1",
        "solver": "liblinear",
        "class_weight": (sel_cw if sel_cw is not None else "none"),
        "threshold": float(thr),
        "decision": decision,
    },
    "validation_protocol": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "metrics": metrics,
    "artifacts_written": {
        "submission_csv": str(SUB_PATH),
        "candidate_snapshot": str(CAND_PATH),
        "oof_predictions": str(oof_path),
        "threshold_sweep": str(sweep_path),
        "ablation_summary": str(ABL_PATH),
        "checkpoint_dir": str(iter_ckpt_dir),
        "pstar_path": str(PSTAR_PATH),
    },
    "deltas_vs_prev": {
        "primary_goal": "Add stable, physiology-derived vitals interactions to improve ranking; keep FP controlled via prevalence-aware thresholding.",
        "changes": [
            "Added derived hemodynamics series: PP=SBP-DBP, MAP=DBP+(SBP-DBP)/3, ShockIndex=HR/SBP, with same trajectory features.",
            "Kept TF-IDF (note_text_all) + keywords + OHE cats; tuned C on small grid; class_weight ablated."
        ],
    },
    "known_defects_limitations": [
        "ShockIndex may be unstable when SBP is extremely low; code sets SI to NaN when SBP<=0.",
        "Still a linear model; may miss non-linear interactions beyond these derived ratios.",
        "High-dimensional sparse TF-IDF remains a potential overfit vector; keep max_features controlled."
    ],
    "next_step_hypothesis": (
        "If real-F1 still plateaus: try a lightweight 2-model ensemble (derived LR + baseline LR) and re-sweep threshold "
        "under the same prevalence constraint, or add a dense-only GBDT on numeric vitals then blend with LR(text)."
    ),
    "hm_message": hm_message,
    "real_score": real_score,
}

with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
print("\nAppended iteration log ->", LOG_PATH)

# =====================
# 13) Consultant packet (v3: every iteration)
# =====================
consult_packet = {
    "tag": TAG,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": timestamp,
    "what_changed": iteration_record["deltas_vs_prev"]["changes"],
    "metric_changes": {
        "pstar_validation_reference": pstar_val,
        "selected_metrics": metrics,
        "gate_decision": decision,
    },
    "suspected_leakage_risks": [
        "Vitals file includes only days 1-10; asserted max(day)<=10.",
        "Derived metrics (PP/MAP/SI) are computed only from day<=10 vitals.",
        "Text comes from the same 10-day window notes."
    ],
    "complexity_drift_risks": [
        "Derived features increase numeric dimensionality; should remain stable and interpretable.",
        "TF-IDF + OHE remains high-dimensional; keep max_features capped and monitor fold_std."
    ],
    "evaluation_alignment_risks": [
        "Threshold selected with prevalence-aware tie-break; verify it stays aligned to real-F1.",
        "Large OOF gains could still fail if leaderboard distribution shifts; watch pos_rate and FP counts."
    ],
    "unknown_unknowns": [
        "Synthetic note phrasing drift between train/test could reduce TF-IDF generalization.",
        "If any vitals values are systematically missing in test, derived ratios may behave differently."
    ],
    "recommendations_for_next_iter": [
        "If real-F1 improves: continue small, reversible changes (ensemble or tighter pos_tol).",
        "If real-F1 regresses: rollback to pstar_validation or pstar_real_anchor and ablate derived SI threshold count.",
        "Consider adding a dense-only tree model on numeric vitals and blending with LR(text) via OOF stacking."
    ]
}

consult_path = CONSULT_ROOT / f"{TAG}_iter{ITER_ID}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, ensure_ascii=False)
print("Wrote consultant packet ->", consult_path)

=== PREFLIGHT ===
CWD: d:\AgentDs\agent_ds_healthcare
PROJECT_ROOT: d:\AgentDs\agent_ds_healthcare
ART_ROOT: d:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT: d:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
ITER_ID=16  ITER_TAG=iter0016  BLOCK_ID=3  BLOCK_POS=1  BRANCH=W1
Discovered iter tags (scoped): ['iter0000', 'iter0002', 'iter0003', 'iter0007', 'iter0008', 'iter0009', 'iter0010', 'iter0011', 'iter0012', 'iter0013', 'iter0015']

=== Existing OOF summary (scoped artifacts) ===
iter_tag  oof_macro_f1  threshold     f1_0     f1_1  pos_rate  fold_mean  fold_std  tn  fp  fn  tp      auc
iter0000      0.717792       0.37 0.602369 0.833215     0.753   0.716883  0.039636 178 166  69 587 0.765824
iter0002      0.766407       0.44 0.693759 0.839054     0.655   0.766187  0.012079 239 105 106 550 0.846294
iter0003      0.727955       0.40 0.628931 0.826979     0.708   0.727570  0.030966 200 144  92 564 0.794717
iter0007      0.756537       0.36 0.662316 0.850757     0.731

C:\Users\shend\AppData\Local\Temp\ipykernel_6244\2835644561.py:289: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",


[FE] Train prevalence: 0.656

=== Ablation summary (derived vitals) ===
                        name  macro_f1  best_macro_overall  threshold  pos_rate  fold_mean  fold_std     f1_0     f1_1      auc class_weight    C
    D2_derived_kw_C0.10_noCW  0.795454            0.797793       0.60     0.676   0.795281  0.029345 0.727545 0.863363 0.882750         none 0.10
    D1_derived_kw_C0.08_noCW  0.795089            0.797447       0.61     0.671   0.794999  0.024230 0.728083 0.862095 0.879662         none 0.08
    D3_derived_kw_C0.11_noCW  0.794483            0.797151       0.60     0.675   0.794266  0.028951 0.726457 0.862509 0.883468         none 0.11
D4_derived_kw_C0.10_balanced  0.792184            0.797133       0.47     0.668   0.791813  0.025866 0.724852 0.859517 0.881257     balanced 0.10

=== Selection decision ===
SELECT_CANDIDATE (passes gate vs P*_val; +0.002 macro)
Selected: D2_derived_kw_C0.10_noCW | macro_f1= 0.795454136771502 thr= 0.6 pos_rate= 0.676

=== Submission ===
Path:

C:\Users\shend\AppData\Local\Temp\ipykernel_6244\2835644561.py:765: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",
C:\Users\shend\AppData\Local\Temp\ipykernel_6244\2835644561.py:784: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat() + "Z"


# Iteration 9
-  0.7906

In [44]:
import os, json, random, re, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.base import clone
import joblib

# =========================================================
# Iteration #17 (auto-increment) - W1: derived vitals + interactions
# Goal: push beyond real 0.7903 using MAP / pulse pressure / shock index features + C sweep
# =========================================================

# ---------------------
# 0) Config + paths
# ---------------------
TEAM, TASK, VERSION = "clai", "ch3", "v3"
TAG = f"{TEAM}_{TASK}_{VERSION}"
SEED = 42
N_SPLITS = 5

np.random.seed(SEED)
random.seed(SEED)

def utc_now_z():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def _find_project_root():
    cand = Path.cwd()
    if (cand / "stays_train.csv").exists():
        return cand
    cand2 = Path("/mnt/data")
    if (cand2 / "stays_train.csv").exists():
        return cand2
    return cand

PROJECT_ROOT = _find_project_root()
ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"

ART_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
CONSULT_ROOT.mkdir(parents=True, exist_ok=True)

print("=== PREFLIGHT ===")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_ROOT:", ART_ROOT)
print("CKPT_ROOT:", CKPT_ROOT)

def discover_iter_tags_scoped(art_root: Path):
    oof_files = sorted(art_root.glob("iter????_oof_predictions.csv"))
    sweep_files = sorted(art_root.glob("iter????_threshold_sweep.csv"))
    tags = sorted({p.name.split("_")[0] for p in (oof_files + sweep_files)})

    # Hard check: prevent incomplete bundles inside this TAG folder.
    for t in tags:
        oof_p = art_root / f"{t}_oof_predictions.csv"
        sw_p  = art_root / f"{t}_threshold_sweep.csv"
        if (not oof_p.exists()) or (not sw_p.exists()):
            raise RuntimeError(
                f"[ABORT] Incomplete artifacts for tag={t} under {art_root}. "
                f"Expected BOTH oof + sweep."
            )
    return tags

discovered_tags = discover_iter_tags_scoped(ART_ROOT)
print("Discovered iter tags (scoped):", discovered_tags)

# ---------------------
# 1) Iteration id
# ---------------------
LOG_PATH = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"

def next_iteration_id(log_path: Path) -> int:
    if not log_path.exists():
        return 0
    max_id = -1
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                max_id = max(max_id, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return max_id + 1

ITER_ID  = next_iteration_id(LOG_PATH)
ITER_TAG = f"iter{ITER_ID:04d}"
BLOCK_ID = ITER_ID // 5
BLOCK_POS = ITER_ID % 5
BRANCH = "W1"  # derived vitals direction

print(f"ITER_ID={ITER_ID}  ITER_TAG={ITER_TAG}  BLOCK_ID={BLOCK_ID}  BLOCK_POS={BLOCK_POS}  BRANCH={BRANCH}")

# ---------------------
# 2) Load data
# ---------------------
def dp(name: str) -> Path:
    p = PROJECT_ROOT / name
    if p.exists():
        return p
    p2 = Path("/mnt/data") / name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {name} under PROJECT_ROOT or /mnt/data")

stays_train = pd.read_csv(dp("stays_train.csv"))
stays_test  = pd.read_csv(dp("stays_test.csv"))
patients    = pd.read_csv(dp("patients.csv"))
with open(dp("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vit_list = json.load(f)

# Leakage check: vitals only days 1..10
max_day = max(d.get("day", 0) for item in vit_list for d in item.get("days", []))
assert max_day <= 10, f"Leakage risk: found vitals day={max_day} (>10). Abort."

vit_map = {int(item["stay_id"]): item.get("days", []) for item in vit_list}

# ---------------------
# 3) Feature engineering (derived vitals + interactions + keywords)
# ---------------------
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice")

VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]

# Interactions: low-cost, explainable, often strong
# - MAP ≈ (2*DBP + SBP) / 3
# - Pulse Pressure = SBP - DBP
# - Shock Index = HR / SBP
INTERACTION_COLS = ["map", "pulse_pressure", "shock_index"]

KEYWORDS = [
    "stable","improving","worsening","afebrile","fever","chills","hypotensive","tachy",
    "culture","antibiotic","abx","iv","oxygen","weaned","dyspnea","cough",
    "pain","nausea","vomit","n/v","ambulated","pt","ot","telemetry","fluids","room air","sats",
    "discharge","transfer",
    # small additions (often appear in discharge-ish phrasing)
    "home","follow up","f/u","ready"
]
KW_PATTERNS = [(kw, re.compile(re.escape(kw), re.IGNORECASE)) for kw in KEYWORDS]

def safe_nan(fn, arr, default=np.nan):
    try:
        return fn(arr)
    except Exception:
        return default

def build_stay_features_derived_interactions(stays_df: pd.DataFrame, include_keywords: bool = True) -> pd.DataFrame:
    base = stays_df.merge(patients, on="patient_id", how="left")
    rows = []

    for sid in base["stay_id"].astype(int).tolist():
        days = sorted(vit_map.get(int(sid), []), key=lambda x: x.get("day", 0))
        day_map = {int(d.get("day")): d for d in days if d.get("day") is not None}

        # notes (days 1..10)
        notes = [(day_map.get(day, {}).get("note") or "") for day in range(1, 11)]
        note_all = " ".join(notes).strip()
        note_last2 = " ".join(notes[-2:]).strip()

        feat = {
            "stay_id": int(sid),
            "note_text_all": note_all,
            "note_text_last2": note_last2,
            "note_len_all": len(note_all),
            "note_len_last2": len(note_last2),
        }

        # Build day-aligned vitals arrays (robust even if days missing)
        arrs = {}
        for col in VITAL_COLS:
            arrs[col] = np.array([day_map.get(day, {}).get(col, np.nan) for day in range(1, 11)], dtype=float)

        # interactions
        hr  = arrs["hr"]
        sbp = arrs["sbp"]
        dbp = arrs["dbp"]
        arrs["map"] = (2.0 * dbp + sbp) / 3.0
        arrs["pulse_pressure"] = sbp - dbp
        arrs["shock_index"] = hr / np.where(sbp == 0, np.nan, sbp)

        day_idx = np.arange(1, 11, dtype=float)

        for col, vals in arrs.items():
            diffs = np.diff(vals) if len(vals) >= 2 else np.array([], dtype=float)

            # derived stats (NO raw per-day features to limit dimensionality)
            feat[f"{col}_mean"]   = safe_nan(np.nanmean, vals)
            feat[f"{col}_std"]    = safe_nan(np.nanstd, vals)
            feat[f"{col}_min"]    = safe_nan(np.nanmin, vals)
            feat[f"{col}_max"]    = safe_nan(np.nanmax, vals)
            feat[f"{col}_median"] = safe_nan(np.nanmedian, vals)
            feat[f"{col}_first"]  = vals[0] if len(vals) else np.nan
            feat[f"{col}_last"]   = vals[-1] if len(vals) else np.nan

            last2 = vals[-2:] if len(vals) >= 2 else vals
            early = vals[:8] if len(vals) >= 8 else (vals[:-2] if len(vals) > 2 else vals)

            feat[f"{col}_last2_mean"] = safe_nan(np.nanmean, last2)
            feat[f"{col}_early_mean"] = safe_nan(np.nanmean, early)
            feat[f"{col}_delta_last2_early"] = feat[f"{col}_last2_mean"] - feat[f"{col}_early_mean"]
            feat[f"{col}_delta_last_first"]  = feat[f"{col}_last"] - feat[f"{col}_first"]

            mask = ~np.isnan(vals)
            feat[f"{col}_slope"] = np.polyfit(day_idx[mask], vals[mask], 1)[0] if mask.sum() >= 2 else np.nan

            feat[f"{col}_diff_mean"] = safe_nan(np.nanmean, diffs) if len(diffs) else np.nan
            feat[f"{col}_diff_std"]  = safe_nan(np.nanstd, diffs)  if len(diffs) else np.nan
            feat[f"{col}_last3_std"] = safe_nan(np.nanstd, vals[-3:]) if len(vals) >= 3 else safe_nan(np.nanstd, vals)

            # missingness
            feat[f"{col}_missing"] = int(np.isnan(vals).sum())

            # simple clinically-motivated thresholds
            if col == "hr":
                feat["hr_gt100"] = int(np.nansum(vals > 100))
                feat["hr_lt60"]  = int(np.nansum(vals < 60))
            if col == "sbp":
                feat["sbp_lt90"]  = int(np.nansum(vals < 90))
                feat["sbp_gt160"] = int(np.nansum(vals > 160))
            if col == "temp_c":
                feat["temp_gt38"] = int(np.nansum(vals > 38))
                feat["temp_lt36"] = int(np.nansum(vals < 36))
            if col == "rr":
                feat["rr_gt22"] = int(np.nansum(vals > 22))
            if col == "dbp":
                feat["dbp_lt60"] = int(np.nansum(vals < 60))
            if col == "map":
                feat["map_lt65"] = int(np.nansum(vals < 65))
            if col == "shock_index":
                feat["si_gt0_9"] = int(np.nansum(vals > 0.9))

        if include_keywords:
            for kw, pat in KW_PATTERNS:
                key = kw.replace(" ", "_").replace("/", "_")
                feat[f"all_kw_{key}"]   = len(pat.findall(note_all))
                feat[f"last2_kw_{key}"] = len(pat.findall(note_last2))

        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    return base.merge(feat_df, on="stay_id", how="left")

print("=== [FE] Building features ===")
train_full = build_stay_features_derived_interactions(stays_train, include_keywords=True)
test_full  = build_stay_features_derived_interactions(stays_test,  include_keywords=True)

y = train_full["discharge_ready_day11"].astype(int).values
train_prev = float(y.mean())
print(f"[FE] Train prevalence: {train_prev:.3f}")

# ---------------------
# 4) Modeling utilities
# ---------------------
ID_COLS = ["stay_id", "patient_id"]
CAT_COLS = ["unit_type", "sex", "insurance", "zip3", "admission_reason"]
TEXT_COL = "note_text_all"
DROP_TEXT_HELPER_COLS = ["note_text_last2"]  # keep for keyword counts, not for TFIDF

train_X = train_full.drop(columns=["discharge_ready_day11"]).copy()
test_X  = test_full.copy()

for c in CAT_COLS:
    train_X[c] = train_X[c].astype(str)
    test_X[c]  = test_X[c].astype(str)

def text_selector(x):
    s = x.iloc[:, 0] if isinstance(x, pd.DataFrame) else pd.Series(x)
    return s.fillna("").astype(str).values

def make_preprocessor(df_X: pd.DataFrame):
    num_cols = [c for c in df_X.columns if c not in (ID_COLS + CAT_COLS + [TEXT_COL] + DROP_TEXT_HELPER_COLS)]
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler(with_mean=False))]), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
            ("note", Pipeline([("selector", FunctionTransformer(text_selector, validate=False)),
                               ("tfidf", TfidfVectorizer(min_df=2, max_features=4000, ngram_range=(1, 2)))]),
             [TEXT_COL]),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    schema = {
        "num_cols": num_cols,
        "cat_cols": CAT_COLS,
        "text_col": TEXT_COL,
        "tfidf": {"min_df": 2, "max_features": 4000, "ngram_range": (1, 2)},
        "vitals_cols": VITAL_COLS,
        "interaction_cols": INTERACTION_COLS,
        "keywords_count": len(KEYWORDS),
    }
    return pre, schema

def build_fold_cache(df_X: pd.DataFrame, y: np.ndarray, preprocessor: ColumnTransformer):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_cache = []
    fold_ids = np.zeros(len(df_X), dtype=int)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(df_X, y)):
        pre = clone(preprocessor)
        X_tr = pre.fit_transform(df_X.iloc[tr_idx], y[tr_idx])
        X_va = pre.transform(df_X.iloc[va_idx])
        fold_cache.append({"fold": fold, "tr_idx": tr_idx, "va_idx": va_idx, "X_tr": X_tr, "y_tr": y[tr_idx], "X_va": X_va})
        fold_ids[va_idx] = fold
    return fold_cache, fold_ids

def threshold_sweep(oof_proba: np.ndarray, y_true: np.ndarray, fold_ids: np.ndarray, thresholds: np.ndarray):
    rows = []
    for t in thresholds:
        pred = (oof_proba >= t).astype(int)
        cm = confusion_matrix(y_true, pred)
        f1s = f1_score(y_true, pred, average=None)
        macro = f1_score(y_true, pred, average="macro")
        pos_rate = float(pred.mean())

        fold_scores = []
        for f in range(N_SPLITS):
            idx = fold_ids == f
            fold_scores.append(f1_score(y_true[idx], pred[idx], average="macro"))

        rows.append({
            "threshold": float(t),
            "macro_f1": float(macro),
            "f1_0": float(f1s[0]),
            "f1_1": float(f1s[1]),
            "pos_rate": pos_rate,
            "fold_mean": float(np.mean(fold_scores)),
            "fold_std": float(np.std(fold_scores)),
            "tn": int(cm[0, 0]), "fp": int(cm[0, 1]), "fn": int(cm[1, 0]), "tp": int(cm[1, 1]),
        })
    return pd.DataFrame(rows).sort_values("threshold", ascending=True).reset_index(drop=True)

def choose_threshold(df_sweep: pd.DataFrame, prevalence: float, pos_tol: float = 0.07, macro_drop_tol: float = 0.005):
    best_macro = float(df_sweep["macro_f1"].max())
    df_pos_ok = df_sweep.loc[(df_sweep["pos_rate"] - prevalence).abs() <= pos_tol].copy()
    candidate_pool = df_pos_ok if len(df_pos_ok) else df_sweep.copy()

    pool_macro = float(candidate_pool["macro_f1"].max())
    df_near = candidate_pool.loc[candidate_pool["macro_f1"] >= (pool_macro - macro_drop_tol)].copy()
    df_near["pos_gap"] = (df_near["pos_rate"] - prevalence).abs()

    # Tie-break: pos_gap -> fold_std -> macro -> threshold
    df_near = df_near.sort_values(["pos_gap", "fold_std", "macro_f1", "threshold"],
                                  ascending=[True, True, False, True]).reset_index(drop=True)

    chosen = df_near.iloc[0].to_dict()
    chosen["best_macro_overall"] = best_macro
    chosen["pos_gap"] = float(chosen["pos_gap"])
    return chosen

def oof_from_cache(fold_cache, C: float):
    n = max(int(fd["va_idx"].max() + 1) for fd in fold_cache)
    oof = np.zeros(n, dtype=float)
    for fd in fold_cache:
        clf = LogisticRegression(
            solver="liblinear",
            penalty="l1",
            C=float(C),
            class_weight=None,
            max_iter=5000,
            random_state=SEED
        )
        clf.fit(fd["X_tr"], fd["y_tr"])
        oof[fd["va_idx"]] = clf.predict_proba(fd["X_va"])[:, 1]
    return oof

def evaluate_candidate(name, y, fold_cache, fold_ids, C: float, prevalence: float):
    oof = oof_from_cache(fold_cache, C=C)
    thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
    df_sweep = threshold_sweep(oof, y, fold_ids, thresholds)
    chosen = choose_threshold(df_sweep, prevalence=prevalence, pos_tol=0.07, macro_drop_tol=0.005)
    chosen.update({
        "name": name,
        "C": float(C),
        "class_weight": "none",
        "auc": float(roc_auc_score(y, oof)),
    })
    return chosen, df_sweep, oof

# ---------------------
# 5) Load P* (from PSTAR.json if exists)
# ---------------------
PSTAR_PATH = CKPT_ROOT / "PSTAR.json"
if PSTAR_PATH.exists():
    try:
        with open(PSTAR_PATH, "r", encoding="utf-8") as f:
            pstar_obj = json.load(f)
        pstar_metrics = pstar_obj.get("pstar", {})
    except Exception:
        pstar_metrics = {}
else:
    pstar_metrics = {}

if not pstar_metrics:
    pstar_metrics = {
        "iteration_tag": "NONE",
        "oof_macro_f1": -1.0,
        "threshold": 0.5,
        "pos_rate": None,
        "fold_std": None,
    }

pstar_macro = float(pstar_metrics.get("oof_macro_f1", -1.0))
pstar_std   = pstar_metrics.get("fold_std", None)

print("=== P* (loaded) ===")
print(json.dumps(pstar_metrics, indent=2, ensure_ascii=False))

# ---------------------
# 6) Build fold cache once + C sweep
# ---------------------
pre, schema = make_preprocessor(train_X)
fold_cache, fold_ids = build_fold_cache(train_X, y, pre)

C_GRID = [0.12, 0.14, 0.16, 0.17, 0.18, 0.20, 0.22, 0.24]

candidates = []
store = {}  # name -> {oof, sweep}
for C in C_GRID:
    name = f"E_intDerived_kw_C{C:.2f}_noCW"
    cand, sweep, oof = evaluate_candidate(name, y, fold_cache, fold_ids, C=C, prevalence=train_prev)
    candidates.append(cand)
    store[name] = {"oof": oof, "sweep": sweep}

abl_df = pd.DataFrame(candidates).sort_values(
    ["macro_f1", "pos_gap", "fold_std", "threshold"],
    ascending=[False, True, True, True]
).reset_index(drop=True)

print("\n=== Ablation summary (derived vitals + interactions) ===")
cols_show = ["name","macro_f1","best_macro_overall","threshold","pos_rate","pos_gap","fold_mean","fold_std","f1_0","f1_1","auc","class_weight","C","tn","fp","fn","tp"]
print(abl_df[cols_show].to_string(index=False))

ABL_PATH = ART_ROOT / f"{ITER_TAG}_ablation_summary.csv"
abl_df.to_csv(ABL_PATH, index=False)

# ---------------------
# 7) Selection + gate vs P*
# ---------------------
MIN_DELTA = 0.002
STD_TOL   = 0.01
POS_TOL   = 0.07

selected = abl_df.iloc[0].to_dict()
sel_name = selected["name"]
sel_C    = float(selected["C"])
thr      = float(selected["threshold"])

sel_oof   = store[sel_name]["oof"]
sel_sweep = store[sel_name]["sweep"]

def passes_gate(c):
    if float(c["macro_f1"]) < pstar_macro + MIN_DELTA:
        return False
    if (pstar_std is not None) and (c.get("fold_std") is not None):
        if float(c["fold_std"]) > float(pstar_std) + STD_TOL:
            return False
    if abs(float(c["pos_rate"]) - train_prev) > POS_TOL:
        return False
    return True

eligible = passes_gate(selected)
decision = "SELECT_CANDIDATE (passes gate vs P*)" if eligible else "SELECT_CANDIDATE (candidate; P* unchanged)"

print("\n=== Selection decision ===")
print(decision)
print(f"Selected: {sel_name} | macro_f1={selected['macro_f1']:.6f} | thr={thr:.2f} | pos_rate={selected['pos_rate']:.3f} | C={sel_C:.2f}")

# ---------------------
# 8) Fit final pipeline + write submission
# ---------------------
clf = LogisticRegression(
    solver="liblinear",
    penalty="l1",
    C=float(sel_C),
    class_weight=None,
    max_iter=5000,
    random_state=SEED
)
pipe = Pipeline([("preprocess", pre), ("clf", clf)])
pipe.fit(train_X, y)

test_proba = pipe.predict_proba(test_X)[:, 1]
test_pred  = (test_proba >= thr).astype(int)

sub = pd.DataFrame({
    "stay_id": test_full["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred.astype(int)
})
SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"
sub.to_csv(SUB_PATH, index=False)

print("\n=== Submission ===")
print("Path:", SUB_PATH)
print("Threshold used:", thr)
print("Submission positive-rate:", float(sub["discharge_ready_day11"].mean()))
print(sub.head())

CAND_PATH = ART_ROOT / f"{TAG}_{BRANCH}_block{BLOCK_ID}_{ITER_TAG}_candidate.csv"
sub.to_csv(CAND_PATH, index=False)

# ---------------------
# 9) Save artifacts + checkpoint bundle
# ---------------------
oof_df = pd.DataFrame({
    "stay_id": train_full["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "oof_proba": sel_oof.astype(float),
    "fold": fold_ids.astype(int),
})
oof_path = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
oof_df.to_csv(oof_path, index=False)

sweep_path = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"
sel_sweep.to_csv(sweep_path, index=False)

iter_ckpt_dir = CKPT_ROOT / ITER_TAG
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

pred_sel = (sel_oof >= thr).astype(int)
cm_sel = confusion_matrix(y, pred_sel).tolist()
f1s_sel = f1_score(y, pred_sel, average=None)
macro_sel = float(f1_score(y, pred_sel, average="macro"))

fold_scores_sel = []
for f in range(N_SPLITS):
    idx = fold_ids == f
    fold_scores_sel.append(float(f1_score(y[idx], pred_sel[idx], average="macro")))

metrics = {
    "oof_macro_f1": macro_sel,
    "per_class_f1": {"f1_0": float(f1s_sel[0]), "f1_1": float(f1s_sel[1])},
    "confusion_matrix": cm_sel,
    "threshold": float(thr),
    "pos_rate": float(pred_sel.mean()),
    "fold_scores": fold_scores_sel,
    "fold_mean": float(np.mean(fold_scores_sel)),
    "fold_std": float(np.std(fold_scores_sel)),
    "auc": float(roc_auc_score(y, sel_oof)),
    "train_prevalence": train_prev,
    "gate": {"min_delta": MIN_DELTA, "std_tol": STD_TOL, "pos_tol": POS_TOL, "decision": decision},
    "pstar_reference": pstar_metrics,
}

config = {
    "team": TEAM,
    "task": TASK,
    "version": VERSION,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "seed": SEED,
    "validation": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "feature_engineering": {
        "type": "derived_vitals_plus_interactions",
        "vitals_cols": VITAL_COLS,
        "interaction_cols": INTERACTION_COLS,
        "keywords_count": len(KEYWORDS),
        "vitals_days_used": "1-10",
    },
    "selected": {
        "name": sel_name,
        "C": float(sel_C),
        "penalty": "l1",
        "solver": "liblinear",
        "class_weight": None,
        "threshold": float(thr),
        "threshold_selection": {"pos_tol": 0.07, "macro_drop_tol": 0.005},
    },
    "tfidf": schema["tfidf"],
    "paths": {"artifacts": str(ART_ROOT), "checkpoints": str(CKPT_ROOT)},
    "notes": [
        "Scoped artifacts/checkpoints under v3 tag only.",
        "Leakage check: asserted vitals max(day) <= 10.",
        "Derived interactions added: MAP, pulse_pressure, shock_index.",
    ],
}

with open(iter_ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)

joblib.dump(pipe, iter_ckpt_dir / "pipeline.joblib")

print("\nSaved checkpoint:", iter_ckpt_dir)
print("Saved artifacts:", oof_path, sweep_path, ABL_PATH, CAND_PATH)

# Update PSTAR.json if passes gate
if eligible:
    new_pstar = {
        "iteration_tag": ITER_TAG,
        "oof_macro_f1": float(metrics["oof_macro_f1"]),
        "threshold": float(thr),
        "best_threshold": float(thr),
        "pos_rate": float(metrics["pos_rate"]),
        "fold_mean": float(metrics["fold_mean"]),
        "fold_std": float(metrics["fold_std"]),
        "f1_0": float(metrics["per_class_f1"]["f1_0"]),
        "f1_1": float(metrics["per_class_f1"]["f1_1"]),
        "confusion_matrix": cm_sel,
    }
    with open(PSTAR_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "tag": TAG,
            "pstar": new_pstar,
            "train_prevalence": train_prev,
            "timestamp": utc_now_z(),
            "schema_version": 3,
            "notes": "Updated because candidate passed hard gate vs prior P*."
        }, f, indent=2, ensure_ascii=False)
    print("✅ Updated PSTAR.json ->", new_pstar["iteration_tag"], "macro=", new_pstar["oof_macro_f1"])
else:
    print("ℹ️ PSTAR.json unchanged (candidate did not pass hard gate).")

# ---------------------
# 10) Append iteration_detail.jsonl (append-only)
# ---------------------
hm_message = (
    "HM notes: last real F1=0.7903. Continue derived-vitals direction; add interaction features (MAP/PP/SI) "
    "and widen C sweep to find a higher-performing sparse LR. Watch pos_rate (risk of FP inflation)."
)
real_score = {"latest_real": 0.7903}

iteration_record = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": utc_now_z(),
    "phase": "Modeling",
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "data_paths_used": {
        "stays_train": str(dp("stays_train.csv")),
        "stays_test": str(dp("stays_test.csv")),
        "patients": str(dp("patients.csv")),
        "vitals_timeseries": str(dp("vitals_timeseries.json")),
    },
    "join_keys": ["stays.patient_id -> patients.patient_id", "stays.stay_id -> vitals_timeseries.stay_id"],
    "leakage_checks_performed": [
        "Asserted vitals max(day) <= 10 (no day11 leakage).",
        "No target-derived columns used.",
        "Artifacts discovery scoped under artifacts/clai_ch3_v3 only.",
    ],
    "feature_summary": {
        "categorical": CAT_COLS,
        "text": {"col": TEXT_COL, "tfidf": schema["tfidf"]},
        "numeric_count": len(schema["num_cols"]),
        "keywords_count": len(KEYWORDS),
        "vitals_cols": VITAL_COLS,
        "interaction_cols": INTERACTION_COLS,
        "vitals_days_used": "1-10",
    },
    "models_attempted": [
        {k: row[k] for k in ["name","macro_f1","best_macro_overall","threshold","pos_rate","pos_gap","fold_mean","fold_std","f1_0","f1_1","auc","tn","fp","fn","tp","class_weight","C"]}
        for row in candidates
    ],
    "selected_model": {
        "name": sel_name,
        "C": float(sel_C),
        "penalty": "l1",
        "solver": "liblinear",
        "class_weight": None,
        "threshold": float(thr),
        "decision": decision,
    },
    "validation_protocol": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "metrics": metrics,
    "artifacts_written": {
        "submission_csv": str(SUB_PATH),
        "candidate_snapshot": str(CAND_PATH),
        "oof_predictions": str(oof_path),
        "threshold_sweep": str(sweep_path),
        "ablation_summary": str(ABL_PATH),
        "checkpoint_dir": str(iter_ckpt_dir),
        "pstar_path": str(PSTAR_PATH),
    },
    "deltas_vs_prev": {
        "primary_goal": "Push real F1 beyond 0.7903 while keeping FP under control.",
        "changes": [
            "Added interaction-derived vitals features: MAP, pulse_pressure, shock_index.",
            "Expanded C sweep range to explore less-regularized sparse LR solutions.",
            "Kept prevalence-aware threshold selection to avoid runaway pos_rate."
        ],
    },
    "known_defects_limitations": [
        "If selected C pushes pos_rate too high, real F1 may drop due to FP inflation.",
        "Still a linear model; may miss non-linear vitals interactions beyond these handcrafted ones.",
    ],
    "next_step_hypothesis": (
        "If real improves: keep interactions and refine thresholding to trim FP. "
        "If real regresses (FP heavy): keep model but tighten pos_tol / macro_drop_tol, "
        "or try dropping zip3 / mild class_weight to stabilize."
    ),
    "hm_message": hm_message,
    "real_score": real_score,
}

with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
print("Appended iteration log ->", LOG_PATH)

# ---------------------
# 11) Consultant packet (every iteration)
# ---------------------
consult_packet = {
    "tag": TAG,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": iteration_record["timestamp"],
    "what_changed": iteration_record["deltas_vs_prev"]["changes"],
    "metric_changes": {
        "pstar_reference": pstar_metrics,
        "selected_metrics": metrics,
        "gate_decision": decision,
    },
    "suspected_leakage_risks": [
        "Vitals restricted to days 1-10; asserted max(day)<=10.",
        "Notes aggregated from same 10-day window."
    ],
    "complexity_drift_risks": [
        "Interactions add small numeric dimensionality (low risk).",
        "TF-IDF remains high-dimensional; rely on L1 sparsity + monitor fold_std."
    ],
    "evaluation_alignment_risks": [
        "Threshold selected for macro-F1 with prevalence-aware tie-break; "
        "pos_rate may still rise (monitor FP / confusion matrix)."
    ],
    "recommendations_for_next_iter": [
        "If FP rises on real: keep same model, but adjust threshold selection to prioritize lower pos_gap / FP.",
        "Try ablation dropping zip3 if variance grows.",
        "Optionally test mild class_weight dict to penalize FP more without collapsing recall."
    ],
}

consult_path = CONSULT_ROOT / f"{TAG}_iter{ITER_ID}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, ensure_ascii=False)
print("Wrote consultant packet ->", consult_path)

=== PREFLIGHT ===
PROJECT_ROOT: d:\AgentDs\agent_ds_healthcare
ART_ROOT: d:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT: d:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
Discovered iter tags (scoped): ['iter0000', 'iter0002', 'iter0003', 'iter0007', 'iter0008', 'iter0009', 'iter0010', 'iter0011', 'iter0012', 'iter0013', 'iter0015', 'iter0016']
ITER_ID=17  ITER_TAG=iter0017  BLOCK_ID=3  BLOCK_POS=2  BRANCH=W1
=== [FE] Building features ===
[FE] Train prevalence: 0.656
=== P* (loaded) ===
{
  "iteration_tag": "iter0016",
  "oof_macro_f1": 0.795454136771502,
  "threshold": 0.6
}

=== Ablation summary (derived vitals + interactions) ===
                      name  macro_f1  best_macro_overall  threshold  pos_rate  pos_gap  fold_mean  fold_std     f1_0     f1_1      auc class_weight    C    tn    fp   fn    tp
E_intDerived_kw_C0.14_noCW  0.793554            0.795733       0.58     0.681    0.025   0.793353  0.028690 0.723982 0.863126 0.887957         none 0.14 240.0 10

# Iteration 10
- 0.7965

In [46]:
import os, json, re, random, shutil, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.base import clone
import joblib

# =====================
# 0) Warnings (keep logs readable)
# =====================
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice")
warnings.filterwarnings("ignore", category=RuntimeWarning)

# =====================
# 1) Config + paths
# =====================
TEAM, TASK, VERSION = "clai", "ch3", "v3"
TAG = f"{TEAM}_{TASK}_{VERSION}"
SEED = 42
N_SPLITS = 5

np.random.seed(SEED)
random.seed(SEED)

def utc_ts() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def _find_project_root() -> Path:
    cand = Path.cwd()
    if (cand / "stays_train.csv").exists():
        return cand
    cand2 = Path("/mnt/data")
    if (cand2 / "stays_train.csv").exists():
        return cand2
    return cand

PROJECT_ROOT = _find_project_root()
ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"

ART_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
CONSULT_ROOT.mkdir(parents=True, exist_ok=True)

print("=== PREFLIGHT ===")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_ROOT:", ART_ROOT)
print("CKPT_ROOT:", CKPT_ROOT)

# =====================
# 2) Bootstrap: copy loose artifacts into scoped dir (idempotent)
# =====================
def _copy_if_missing(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists() and (not dst.exists()):
        shutil.copy2(src, dst)

for pat in ["iter????_oof_predictions.csv", "iter????_threshold_sweep.csv", "iter????_ablation_summary.csv"]:
    for src in sorted(PROJECT_ROOT.glob(pat)):
        _copy_if_missing(src, ART_ROOT / src.name)
for src in sorted(PROJECT_ROOT.glob(f"{TAG}_*candidate*.csv")):
    _copy_if_missing(src, ART_ROOT / src.name)
for src in sorted(PROJECT_ROOT.glob(f"{TAG}_W*_block*_iter*_candidate.csv")):
    _copy_if_missing(src, ART_ROOT / src.name)

def _extract_iter_num(name: str):
    m = re.search(r"iter(\d{4})", name)
    return int(m.group(1)) if m else None

def discover_complete_iter_tags(art_root: Path):
    oof_tags = {p.name.split("_")[0] for p in art_root.glob("iter????_oof_predictions.csv")}
    sw_tags  = {p.name.split("_")[0] for p in art_root.glob("iter????_threshold_sweep.csv")}
    complete = sorted(oof_tags & sw_tags)
    incomplete = sorted((oof_tags | sw_tags) - set(complete))
    if incomplete:
        print("⚠️ Incomplete iter artifacts detected (ignored for P*/history):", incomplete)
    return complete, incomplete

def max_iter_num_from_artifacts(art_root: Path) -> int:
    mx = -1
    for p in art_root.glob("iter????_*"):
        n = _extract_iter_num(p.name)
        if n is not None:
            mx = max(mx, n)
    return mx

def max_iter_num_from_ckpt(ckpt_root: Path) -> int:
    mx = -1
    for d in ckpt_root.glob("iter????"):
        if d.is_dir():
            n = _extract_iter_num(d.name)
            if n is not None:
                mx = max(mx, n)
    return mx

tags_complete, tags_incomplete = discover_complete_iter_tags(ART_ROOT)
print("Discovered COMPLETE iter tags (scoped):", tags_complete)

# =====================
# 3) Iteration id (robust): max(log, artifacts, ckpts)+1
# =====================
LOG_PATH = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"

def max_iteration_id_from_log(log_path: Path) -> int:
    if not log_path.exists():
        return -1
    mx = -1
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                mx = max(mx, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return mx

max_log = max_iteration_id_from_log(LOG_PATH)
max_art = max_iter_num_from_artifacts(ART_ROOT)
max_ckpt = max_iter_num_from_ckpt(CKPT_ROOT)

ITER_ID = max(max_log, max_art, max_ckpt) + 1
ITER_TAG = f"iter{ITER_ID:04d}"
BLOCK_ID = ITER_ID // 5
BLOCK_POS = ITER_ID % 5
BRANCH = "W2"  # new angle: clinically-derived indices + macro-first threshold policy (still logs prev-first)

print(f"ITER_ID={ITER_ID}  ITER_TAG={ITER_TAG}  BLOCK_ID={BLOCK_ID}  BLOCK_POS={BLOCK_POS}  BRANCH={BRANCH}")

# =====================
# 4) Load data + leakage check
# =====================
def dp(name: str) -> Path:
    p = PROJECT_ROOT / name
    if p.exists():
        return p
    p2 = Path("/mnt/data") / name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {name}")

stays_train = pd.read_csv(dp("stays_train.csv"))
stays_test  = pd.read_csv(dp("stays_test.csv"))
patients    = pd.read_csv(dp("patients.csv"))
with open(dp("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vit_list = json.load(f)

max_day = max(d.get("day", 0) for item in vit_list for d in item.get("days", []))
assert max_day <= 10, f"Leakage risk: found vitals day={max_day} (>10). Abort."

vit_map = {int(item["stay_id"]): item.get("days", []) for item in vit_list}

# =====================
# 5) Feature engineering (derived indices: MAP, shock index, pulse pressure, temp deviation)
# =====================
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]

KEYWORDS = [
    "stable","improving","worsening","afebrile","fever","chills","hypotensive","tachy",
    "culture","antibiotic","abx","iv","oxygen","weaned","dyspnea","cough",
    "pain","nausea","vomit","n/v","ambulated","pt","ot","telemetry","fluids","room air","sats",
    "discharge","transfer"
]
KW_PATTERNS = [(kw, re.compile(re.escape(kw), re.IGNORECASE)) for kw in KEYWORDS]

def safe_nan(fn, arr, default=np.nan):
    try:
        with np.errstate(all="ignore"):
            return fn(arr)
    except Exception:
        return default

def build_features(stays_df: pd.DataFrame, include_keywords: bool = True) -> pd.DataFrame:
    base = stays_df.merge(patients, on="patient_id", how="left")
    rows = []
    for sid in base["stay_id"].astype(int).tolist():
        days = sorted(vit_map.get(int(sid), []), key=lambda x: x.get("day", 0))
        day_idx = np.array([d.get("day", np.nan) for d in days], dtype=float)

        vit = {c: np.array([d.get(c, np.nan) for d in days], dtype=float) for c in VITAL_COLS}

        # derived per-day indices
        pulse_pressure = vit["sbp"] - vit["dbp"]
        map_est = (vit["sbp"] + 2.0 * vit["dbp"]) / 3.0
        shock_index = vit["hr"] / vit["sbp"]
        shock_index = np.where(np.isfinite(shock_index), shock_index, np.nan)
        temp_dev = vit["temp_c"] - 37.0

        series = {
            "hr": vit["hr"],
            "sbp": vit["sbp"],
            "dbp": vit["dbp"],
            "rr": vit["rr"],
            "temp_c": vit["temp_c"],
            "pulse_pressure": pulse_pressure,
            "map": map_est,
            "shock_index": shock_index,
            "temp_dev": temp_dev,
        }

        notes = [(d.get("note") or "") for d in days]
        note_all = " ".join(notes).strip()
        note_last2 = " ".join(notes[-2:]).strip()

        feat = {
            "stay_id": int(sid),
            "note_len_all": len(note_all),
            "note_len_last2": len(note_last2),
            "note_days": int(sum(1 for n in notes if str(n).strip() != "")),
            "days_recorded": int(len(days)),
            "day_first": float(day_idx[0]) if len(day_idx) else np.nan,
            "day_last": float(day_idx[-1]) if len(day_idx) else np.nan,
        }

        for name, vals in series.items():
            vals = np.array(vals, dtype=float)
            diffs = np.diff(vals) if len(vals) >= 2 else np.array([], dtype=float)

            feat[f"{name}_mean"] = safe_nan(np.nanmean, vals)
            feat[f"{name}_std"]  = safe_nan(np.nanstd,  vals)
            feat[f"{name}_min"]  = safe_nan(np.nanmin,  vals)
            feat[f"{name}_max"]  = safe_nan(np.nanmax,  vals)
            feat[f"{name}_median"] = safe_nan(np.nanmedian, vals)
            feat[f"{name}_first"] = vals[0] if len(vals) else np.nan
            feat[f"{name}_last"]  = vals[-1] if len(vals) else np.nan

            last2 = vals[-2:] if len(vals) >= 2 else vals
            early = vals[:8] if len(vals) >= 8 else (vals[:-2] if len(vals) > 2 else vals)
            feat[f"{name}_last2_mean"] = safe_nan(np.nanmean, last2)
            feat[f"{name}_early_mean"] = safe_nan(np.nanmean, early)
            feat[f"{name}_delta_last2_early"] = feat[f"{name}_last2_mean"] - feat[f"{name}_early_mean"]
            feat[f"{name}_delta_last_first"] = feat[f"{name}_last"] - feat[f"{name}_first"]

            mask = (~np.isnan(vals)) & (~np.isnan(day_idx))
            feat[f"{name}_slope"] = np.polyfit(day_idx[mask], vals[mask], 1)[0] if mask.sum() >= 2 else np.nan

            feat[f"{name}_diff_mean"] = safe_nan(np.nanmean, diffs) if len(diffs) else np.nan
            feat[f"{name}_diff_std"]  = safe_nan(np.nanstd,  diffs) if len(diffs) else np.nan
            feat[f"{name}_last3_std"] = safe_nan(np.nanstd, vals[-3:]) if len(vals) >= 3 else safe_nan(np.nanstd, vals)

            feat[f"{name}_missing"] = int(np.isnan(vals).sum())
            feat[f"{name}_n_obs"]   = int(np.sum(~np.isnan(vals)))

        # simple clinical threshold counts
        hr, sbp, dbp, temp, rr = vit["hr"], vit["sbp"], vit["dbp"], vit["temp_c"], vit["rr"]
        feat["hr_gt100"] = int(np.nansum(hr > 100))
        feat["hr_lt60"]  = int(np.nansum(hr < 60))
        feat["sbp_lt90"] = int(np.nansum(sbp < 90))
        feat["sbp_gt160"] = int(np.nansum(sbp > 160))
        feat["dbp_lt60"] = int(np.nansum(dbp < 60))
        feat["temp_gt38"] = int(np.nansum(temp > 38))
        feat["temp_lt36"] = int(np.nansum(temp < 36))
        feat["rr_gt22"] = int(np.nansum(rr > 22))
        feat["map_lt65"] = int(np.nansum(map_est < 65))
        feat["shock_gt1"] = int(np.nansum(shock_index > 1.0))
        feat["pp_lt30"] = int(np.nansum(pulse_pressure < 30))

        if include_keywords:
            for kw, pat in KW_PATTERNS:
                key = kw.replace(" ", "_").replace("/", "_")
                feat[f"all_kw_{key}"] = len(pat.findall(note_all))
                feat[f"last2_kw_{key}"] = len(pat.findall(note_last2))

        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    return base.merge(feat_df, on="stay_id", how="left")

print("=== [FE] Building features ===")
train_full = build_features(stays_train, include_keywords=True)
test_full  = build_features(stays_test,  include_keywords=True)

y = train_full["discharge_ready_day11"].astype(int).values
train_prev = float(y.mean())
print("[FE] Train prevalence:", round(train_prev, 3))

# =====================
# 6) Modeling (LR L1; no TF-IDF; dual threshold policy)
# =====================
ID_COLS = ["stay_id", "patient_id"]
CAT_COLS = ["unit_type", "sex", "insurance", "zip3", "admission_reason"]

train_X = train_full.drop(columns=["discharge_ready_day11"]).copy()
test_X  = test_full.copy()
for c in CAT_COLS:
    train_X[c] = train_X[c].astype(str)
    test_X[c]  = test_X[c].astype(str)

def make_preprocessor_no_tfidf(df: pd.DataFrame):
    # numeric = everything except ids/cats and any raw note_text columns (we don't use TF-IDF here)
    drop_text_cols = [c for c in df.columns if c.startswith("note_text_")]
    num_cols = [c for c in df.columns if c not in (ID_COLS + CAT_COLS + drop_text_cols)]
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler(with_mean=False))]), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    schema = {
        "num_cols": num_cols,
        "cat_cols": CAT_COLS,
        "text": "keyword_counts_only (no TF-IDF)",
        "derived_indices": {
            "pulse_pressure": "sbp - dbp",
            "map": "(sbp + 2*dbp) / 3",
            "shock_index": "hr / sbp",
            "temp_dev": "temp_c - 37",
        },
        "keywords_count": len(KEYWORDS),
        "vitals_cols": VITAL_COLS,
        "vitals_days_used": "1-10",
    }
    return pre, schema

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

def build_fold_cache(df_X: pd.DataFrame, y: np.ndarray, preprocessor: ColumnTransformer):
    fold_cache = []
    fold_ids = np.zeros(len(df_X), dtype=int)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(df_X, y)):
        pre = clone(preprocessor)
        X_tr = pre.fit_transform(df_X.iloc[tr_idx], y[tr_idx])
        X_va = pre.transform(df_X.iloc[va_idx])
        fold_cache.append({"fold": fold, "tr_idx": tr_idx, "va_idx": va_idx,
                           "X_tr": X_tr, "y_tr": y[tr_idx], "X_va": X_va})
        fold_ids[va_idx] = fold
    return fold_cache, fold_ids

def oof_from_cache(fold_cache, C: float):
    n = max(fd["va_idx"].max() for fd in fold_cache) + 1
    oof = np.zeros(n, dtype=float)
    for fd in fold_cache:
        clf = LogisticRegression(
            solver="liblinear",
            penalty="l1",
            C=float(C),
            class_weight=None,
            max_iter=5000,
            random_state=SEED
        )
        clf.fit(fd["X_tr"], fd["y_tr"])
        oof[fd["va_idx"]] = clf.predict_proba(fd["X_va"])[:, 1]
    return oof

def threshold_sweep(oof_proba: np.ndarray, y_true: np.ndarray, fold_ids: np.ndarray):
    thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
    rows = []
    for t in thresholds:
        pred = (oof_proba >= t).astype(int)
        cm = confusion_matrix(y_true, pred)
        f1s = f1_score(y_true, pred, average=None)
        macro = f1_score(y_true, pred, average="macro")
        fold_scores = []
        for f in range(N_SPLITS):
            idx = fold_ids == f
            fold_scores.append(f1_score(y_true[idx], pred[idx], average="macro"))
        rows.append({
            "threshold": float(t),
            "macro_f1": float(macro),
            "f1_0": float(f1s[0]),
            "f1_1": float(f1s[1]),
            "pos_rate": float(pred.mean()),
            "fold_mean": float(np.mean(fold_scores)),
            "fold_std": float(np.std(fold_scores)),
            "tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        })
    return pd.DataFrame(rows)

def choose_threshold_prev_first(df_sweep: pd.DataFrame, prevalence: float, pos_tol: float = 0.07, macro_drop_tol: float = 0.005):
    best_macro = float(df_sweep["macro_f1"].max())
    df_pos_ok = df_sweep.loc[(df_sweep["pos_rate"] - prevalence).abs() <= pos_tol].copy()
    pool = df_pos_ok if len(df_pos_ok) else df_sweep.copy()
    pool_best = float(pool["macro_f1"].max())
    df_near = pool.loc[pool["macro_f1"] >= (pool_best - macro_drop_tol)].copy()
    df_near["pos_gap"] = (df_near["pos_rate"] - prevalence).abs()
    df_near = df_near.sort_values(["pos_gap", "fold_std", "macro_f1", "threshold"],
                                  ascending=[True, True, False, True]).reset_index(drop=True)
    chosen = df_near.iloc[0].to_dict()
    chosen["best_macro_overall"] = best_macro
    chosen["policy"] = "prev_first"
    return chosen

def choose_threshold_macro_first(df_sweep: pd.DataFrame, prevalence: float):
    df = df_sweep.copy()
    df["pos_gap"] = (df["pos_rate"] - prevalence).abs()
    df = df.sort_values(["macro_f1", "fold_std", "pos_gap", "threshold"],
                        ascending=[False, True, True, True]).reset_index(drop=True)
    chosen = df.iloc[0].to_dict()
    chosen["best_macro_overall"] = float(df_sweep["macro_f1"].max())
    chosen["policy"] = "macro_first"
    return chosen

# Build fold cache once
pre, schema = make_preprocessor_no_tfidf(train_X)
fold_cache, fold_ids = build_fold_cache(train_X, y, pre)

# =====================
# 7) Load P* (prefer PSTAR.json)
# =====================
PSTAR_PATH = CKPT_ROOT / "PSTAR.json"
pstar = None
if PSTAR_PATH.exists():
    try:
        pstar = json.loads(PSTAR_PATH.read_text(encoding="utf-8")).get("pstar")
    except Exception:
        pstar = None

# fallback: use iter0016 threshold sweep if available; else best among complete tags
def pstar_fallback_from_sweeps(tags):
    best = None
    for t in tags:
        sw_p = ART_ROOT / f"{t}_threshold_sweep.csv"
        if not sw_p.exists():
            continue
        sw = pd.read_csv(sw_p)
        row = sw.sort_values("macro_f1", ascending=False).iloc[0].to_dict()
        cand = {"iteration_tag": t, "oof_macro_f1": float(row["macro_f1"]), "threshold": float(row["threshold"]), "fold_std": float(row.get("fold_std", np.nan))}
        if (best is None) or (cand["oof_macro_f1"] > best["oof_macro_f1"]):
            best = cand
    return best

if pstar is None:
    sw16 = ART_ROOT / "iter0016_threshold_sweep.csv"
    if sw16.exists():
        sw = pd.read_csv(sw16)
        if (sw["threshold"] == 0.60).any():
            row = sw.loc[sw["threshold"] == 0.60].iloc[0].to_dict()
            pstar = {"iteration_tag": "iter0016", "oof_macro_f1": float(row["macro_f1"]), "threshold": float(row["threshold"]), "fold_std": float(row.get("fold_std", np.nan))}
        else:
            row = sw.sort_values("macro_f1", ascending=False).iloc[0].to_dict()
            pstar = {"iteration_tag": "iter0016", "oof_macro_f1": float(row["macro_f1"]), "threshold": float(row["threshold"]), "fold_std": float(row.get("fold_std", np.nan))}
    else:
        pstar = pstar_fallback_from_sweeps(tags_complete) or {"iteration_tag": "NONE", "oof_macro_f1": -1.0, "threshold": 0.5, "fold_std": None}

print("=== P* (loaded) ===")
print(json.dumps(pstar, indent=2, ensure_ascii=False))

# =====================
# 8) Ablation grid (C x threshold policy)
# =====================
C_GRID = [0.06, 0.08, 0.10, 0.11, 0.12, 0.14]
oof_store, sweep_store = {}, {}

rows = []
for C in C_GRID:
    oof = oof_from_cache(fold_cache, C=C)
    oof_store[C] = oof
    sw = threshold_sweep(oof, y, fold_ids)
    sweep_store[C] = sw

    auc = float(roc_auc_score(y, oof))
    ch_prev = choose_threshold_prev_first(sw, prevalence=train_prev)
    ch_macro = choose_threshold_macro_first(sw, prevalence=train_prev)

    for ch in [ch_prev, ch_macro]:
        rows.append({
            "name": f"DV2_derivedIdx_kw_C{C:.2f}_{ch['policy']}",
            "macro_f1": float(ch["macro_f1"]),
            "best_macro_overall": float(ch["best_macro_overall"]),
            "threshold": float(ch["threshold"]),
            "pos_rate": float(ch["pos_rate"]),
            "pos_gap": float(abs(float(ch["pos_rate"]) - train_prev)),
            "fold_mean": float(ch["fold_mean"]),
            "fold_std": float(ch["fold_std"]),
            "f1_0": float(ch["f1_0"]),
            "f1_1": float(ch["f1_1"]),
            "auc": auc,
            "C": float(C),
            "class_weight": "none",
            "policy": ch["policy"],
            "tn": int(ch["tn"]), "fp": int(ch["fp"]), "fn": int(ch["fn"]), "tp": int(ch["tp"]),
        })

abl_df = pd.DataFrame(rows)
abl_df["delta_vs_pstar_macro"] = abl_df["macro_f1"] - float(pstar.get("oof_macro_f1", -1.0))
pstar_std = pstar.get("fold_std", None)
pstar_std = float(pstar_std) if pstar_std is not None else np.nan
abl_df["delta_vs_pstar_std"] = abl_df["fold_std"] - pstar_std

abl_df = abl_df.sort_values(["macro_f1", "fold_std"], ascending=[False, True]).reset_index(drop=True)

print("\n=== Ablation summary (derived indices + keyword text; dual threshold policy) ===")
cols_show = [
    "name","macro_f1","delta_vs_pstar_macro","best_macro_overall","threshold",
    "pos_rate","pos_gap","fold_mean","fold_std","delta_vs_pstar_std",
    "f1_0","f1_1","auc","class_weight","C","policy","tn","fp","fn","tp"
]
print(abl_df[cols_show].to_string(index=False))

ABL_PATH = ART_ROOT / f"{ITER_TAG}_ablation_summary.csv"
abl_df.to_csv(ABL_PATH, index=False)

# =====================
# 9) Selection gate + fallback
# =====================
MIN_DELTA, STD_TOL, POS_TOL = 0.001, 0.01, 0.07
pstar_macro = float(pstar.get("oof_macro_f1", -1.0))

def passes_gate(r):
    if float(r["macro_f1"]) < pstar_macro + MIN_DELTA:
        return False
    if not np.isnan(pstar_std):
        if float(r["fold_std"]) > float(pstar_std) + STD_TOL:
            return False
    if abs(float(r["pos_rate"]) - train_prev) > POS_TOL:
        return False
    return True

eligible = [r for r in abl_df.to_dict("records") if passes_gate(r)]
if eligible:
    selected = sorted(eligible, key=lambda d: float(d["macro_f1"]), reverse=True)[0]
    decision = "SELECT_CANDIDATE (passes gate vs P*)"
else:
    selected = abl_df.iloc[0].to_dict()
    decision = "NO_CANDIDATE_PASSES_GATE -> produce candidate only (P* unchanged)"

print("\n=== Selection decision ===")
print(decision)
print("Selected:", selected["name"], "| macro_f1=", round(float(selected["macro_f1"]),6),
      "thr=", selected["threshold"], "pos_rate=", round(float(selected["pos_rate"]),3),
      "C=", selected["C"], "policy=", selected["policy"])

sel_C = float(selected["C"])
thr = float(selected["threshold"])
sel_oof = oof_store[sel_C]
sel_sweep = sweep_store[sel_C]

# Optional rollback if gate fails and P* pipeline exists
pstar_iter_tag = str(pstar.get("iteration_tag", "NONE"))
pstar_pipe_path = CKPT_ROOT / pstar_iter_tag / "pipeline.joblib"
use_rollback = (not eligible) and pstar_pipe_path.exists()

# =====================
# 10) Fit final model + write submission
# =====================
SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"

if use_rollback:
    print("[ROLLBACK] Using P* pipeline:", pstar_pipe_path)
    pipe = joblib.load(pstar_pipe_path)
    thr = float(pstar.get("threshold", 0.5))
else:
    clf = LogisticRegression(
        solver="liblinear",
        penalty="l1",
        C=float(sel_C),
        class_weight=None,
        max_iter=5000,
        random_state=SEED
    )
    pipe = Pipeline([("preprocess", pre), ("clf", clf)])
    pipe.fit(train_X, y)

test_proba = pipe.predict_proba(test_X)[:, 1]
test_pred = (test_proba >= thr).astype(int)

sub = pd.DataFrame({
    "stay_id": test_full["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred.astype(int),
})
sub.to_csv(SUB_PATH, index=False)

print("\n=== Submission ===")
print("Path:", SUB_PATH)
print("Threshold used:", thr)
print("Submission positive-rate:", float(sub["discharge_ready_day11"].mean()))
print(sub.head())

CAND_PATH = ART_ROOT / f"{TAG}_{BRANCH}_block{BLOCK_ID}_iter{ITER_ID:04d}_candidate.csv"
sub.to_csv(CAND_PATH, index=False)

# =====================
# 11) Save artifacts + checkpoint
# =====================
oof_df = pd.DataFrame({
    "stay_id": train_full["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "oof_proba": sel_oof.astype(float),
    "fold": fold_ids.astype(int),
})
oof_path = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
oof_df.to_csv(oof_path, index=False)

sweep_path = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"
sel_sweep.to_csv(sweep_path, index=False)

iter_ckpt_dir = CKPT_ROOT / ITER_TAG
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(pipe, iter_ckpt_dir / "pipeline.joblib")

pred_oof = (sel_oof >= thr).astype(int)
cm = confusion_matrix(y, pred_oof).tolist()
f1s = f1_score(y, pred_oof, average=None)
macro = f1_score(y, pred_oof, average="macro")

fold_scores = []
for f in range(N_SPLITS):
    idx = fold_ids == f
    fold_scores.append(float(f1_score(y[idx], pred_oof[idx], average="macro")))

metrics = {
    "oof_macro_f1": float(macro),
    "per_class_f1": {"f1_0": float(f1s[0]), "f1_1": float(f1s[1])},
    "confusion_matrix": cm,
    "threshold": float(thr),
    "pos_rate": float(pred_oof.mean()),
    "fold_scores": fold_scores,
    "fold_mean": float(np.mean(fold_scores)),
    "fold_std": float(np.std(fold_scores)),
    "auc": float(roc_auc_score(y, sel_oof)),
    "train_prevalence": train_prev,
    "gate": {"min_delta": MIN_DELTA, "std_tol": STD_TOL, "pos_tol": POS_TOL, "decision": decision},
    "pstar_reference": pstar,
}

config = {
    "team": TEAM, "task": TASK, "version": VERSION,
    "iteration_id": ITER_ID, "iteration_tag": ITER_TAG,
    "branch": BRANCH, "block_id": BLOCK_ID, "block_pos": BLOCK_POS,
    "seed": SEED,
    "validation": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "selected": {
        "name": selected["name"],
        "C": float(sel_C),
        "class_weight": "none",
        "threshold": float(thr),
        "threshold_policy": selected["policy"],
        "threshold_policies_compared": ["prev_first", "macro_first"],
    },
    "schema": schema,
    "notes": [
        "Reversible change: add clinically-derived indices (MAP, shock_index, pulse_pressure, temp_dev) computed only from days 1-10.",
        "Auditable: two threshold policies compared and logged; selection is gate-controlled vs P*.",
        "No TF-IDF this iter: keyword counts only (dimensionality control).",
    ],
}

with open(iter_ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)

print("\nSaved checkpoint:", iter_ckpt_dir)
print("Saved artifacts:", oof_path.name, sweep_path.name, ABL_PATH.name, CAND_PATH.name)

# =====================
# 12) Diagnostics: group-wise FP/FN (failure explanation)
# =====================
diag = train_full[["stay_id"] + CAT_COLS].copy()
diag["y"] = y
diag["pred"] = pred_oof
diag["proba"] = sel_oof
diag["is_fp"] = (diag["y"] == 0) & (diag["pred"] == 1)
diag["is_fn"] = (diag["y"] == 1) & (diag["pred"] == 0)

def group_err(df: pd.DataFrame, col: str) -> pd.DataFrame:
    g = df.groupby(col).agg(
        n=("stay_id", "count"),
        y_prev=("y", "mean"),
        pred_rate=("pred", "mean"),
        fp=("is_fp", "sum"),
        fn=("is_fn", "sum"),
    ).reset_index()
    # denominators (avoid div by 0)
    neg = np.maximum(1.0, g["n"] - g["y_prev"] * g["n"])
    pos = np.maximum(1.0, g["y_prev"] * g["n"])
    g["fp_rate_in_neg"] = g["fp"] / neg
    g["fn_rate_in_pos"] = g["fn"] / pos
    return g.sort_values(["fp_rate_in_neg", "fn_rate_in_pos"], ascending=False)

for col in ["unit_type", "insurance", "admission_reason"]:
    g = group_err(diag, col)
    print(f"\n[DIAG] Top groups by FP-rate (col={col})")
    print(g.head(8).to_string(index=False))

DIAG_PATH = ART_ROOT / f"{ITER_TAG}_group_error_summary.csv"
g_unit = group_err(diag, "unit_type"); g_unit["group_col"] = "unit_type"
g_ins  = group_err(diag, "insurance"); g_ins["group_col"] = "insurance"
g_adm  = group_err(diag, "admission_reason"); g_adm["group_col"] = "admission_reason"
pd.concat([g_unit, g_ins, g_adm], ignore_index=True).to_csv(DIAG_PATH, index=False)

# =====================
# 13) Update PSTAR.json (only if passes gate)
# =====================
if eligible:
    new_pstar = {
        "iteration_tag": ITER_TAG,
        "oof_macro_f1": float(macro),
        "threshold": float(thr),
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "pos_rate": float(pred_oof.mean()),
        "f1_0": float(f1s[0]),
        "f1_1": float(f1s[1]),
        "confusion_matrix": cm,
    }
    try:
        with open(PSTAR_PATH, "w", encoding="utf-8") as f:
            json.dump({
                "tag": TAG,
                "pstar": new_pstar,
                "train_prevalence": train_prev,
                "timestamp": utc_ts(),
                "schema_version": 3,
                "notes": "Updated because candidate passed hard gate vs prior P*."
            }, f, indent=2, ensure_ascii=False)
        print("✅ Updated PSTAR.json ->", new_pstar["iteration_tag"], "macro=", round(new_pstar["oof_macro_f1"], 6))
    except PermissionError as e:
        print("⚠️ Could not write PSTAR.json:", e)
else:
    print("ℹ️ PSTAR.json unchanged (no candidate passed hard gate).")

# =====================
# 14) Append iteration log (append-only; fallback if permission issue)
# =====================
hm_message = "HM: iter0017 real F1=0.7906, improvement limited; request new angle: reversible/auditable changes, clear failure explanations, risk + fallback each iter."
real_score = {"iter0017_real_f1": 0.7906}

iteration_record = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": utc_ts(),
    "phase": "Modeling",
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "data_paths_used": {
        "stays_train": str(dp("stays_train.csv")),
        "stays_test": str(dp("stays_test.csv")),
        "patients": str(dp("patients.csv")),
        "vitals_timeseries": str(dp("vitals_timeseries.json")),
    },
    "join_keys": ["stays.patient_id -> patients.patient_id", "stays.stay_id -> vitals_timeseries.stay_id"],
    "leakage_checks_performed": [
        "Asserted vitals max(day) <= 10 (no day11 leakage).",
        "Threshold tuned on OOF only (no test leakage).",
        "Artifact discovery scoped under artifacts/clai_ch3_v3.",
    ],
    "feature_summary": schema,
    "models_attempted": abl_df[cols_show].to_dict("records"),
    "selected_model": {
        "name": selected["name"],
        "C": float(sel_C),
        "penalty": "l1",
        "solver": "liblinear",
        "class_weight": "none",
        "threshold": float(thr),
        "threshold_policy": selected["policy"],
        "decision": decision,
    },
    "validation_protocol": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "metrics": metrics,
    "artifacts_written": {
        "submission_csv": str(SUB_PATH),
        "candidate_snapshot": str(CAND_PATH),
        "oof_predictions": str(oof_path),
        "threshold_sweep": str(sweep_path),
        "ablation_summary": str(ABL_PATH),
        "group_error_summary": str(DIAG_PATH),
        "checkpoint_dir": str(iter_ckpt_dir),
        "pstar_path": str(PSTAR_PATH),
    },
    "deltas_vs_prev": {
        "changes": [
            "Added clinically-derived indices: MAP=(SBP+2*DBP)/3, shock_index=HR/SBP, pulse_pressure=SBP-DBP, temp_dev=temp-37.",
            "Compared threshold policies: prev_first vs macro_first; selection is gate-controlled.",
            "Added group-wise FP/FN diagnostics as an artifact to explain failures.",
        ],
        "risk": [
            "Derived indices can amplify noise when SBP/DBP is missing; L1 regularization mitigates overfit.",
            "Macro-first threshold may increase pos_rate; monitor FP via confusion matrix + group diagnostics.",
        ],
        "fallback": "If candidate does not pass gate, do not update P*; optionally rollback to P* pipeline if available.",
    },
    "known_defects_limitations": [
        "Keyword counts are coarse and may miss synonyms; intentionally kept to control dimensionality.",
    ],
    "next_step_hypothesis": (
        "If real-F1 plateaus: try a 2-model OOF blend (LR on text/cats + HistGradientBoosting on dense numeric vitals/indices), "
        "with strict leakage checks and the same gate discipline."
    ),
    "hm_message": hm_message,
    "real_score": real_score,
}

log_written_to = str(LOG_PATH)
try:
    if not LOG_PATH.exists():
        LOG_PATH.write_text("", encoding="utf-8")
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
except PermissionError:
    alt_log = CKPT_ROOT / f"{TAG}_iteration_detail.jsonl"
    log_written_to = str(alt_log)
    with open(alt_log, "a", encoding="utf-8") as f:
        f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")

print("Appended iteration log ->", log_written_to)

# =====================
# 15) Consultant packet (every iter)
# =====================
consult_packet = {
    "tag": TAG,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": iteration_record["timestamp"],
    "what_changed": iteration_record["deltas_vs_prev"]["changes"],
    "metric_changes": {
        "pstar_reference": pstar,
        "selected_metrics": metrics,
        "gate_decision": decision,
    },
    "suspected_leakage_risks": iteration_record["leakage_checks_performed"],
    "complexity_drift_risks": [
        "Added a few derived indices; still linear + sparse (no TF-IDF) to limit complexity drift."
    ],
    "evaluation_alignment_risks": [
        "Threshold tuned on OOF; macro-first can overfit, but we track fold_std + pos_gap and enforce a gate."
    ],
    "unknown_unknowns": [
        "Possible distribution shift in SBP/DBP measurement could affect derived indices on test."
    ],
    "recommendations_for_next_iter": [
        "If regresses: keep features but revert threshold policy to prev_first; compare FP/FN shifts.",
        "If improves: consider a single ablation removing zip3 to reduce variance.",
        "High-risk next: OOF blend LR(text) + HGB(numeric) with strict audit trail."
    ],
}
consult_path = CONSULT_ROOT / f"{TAG}_iter{ITER_ID}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, ensure_ascii=False)
print("Wrote consultant packet ->", consult_path)

=== PREFLIGHT ===
PROJECT_ROOT: d:\AgentDs\agent_ds_healthcare
ART_ROOT: d:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT: d:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
Discovered COMPLETE iter tags (scoped): ['iter0000', 'iter0002', 'iter0003', 'iter0007', 'iter0008', 'iter0009', 'iter0010', 'iter0011', 'iter0012', 'iter0013', 'iter0015', 'iter0016', 'iter0017']
ITER_ID=18  ITER_TAG=iter0018  BLOCK_ID=3  BLOCK_POS=3  BRANCH=W2
=== [FE] Building features ===
[FE] Train prevalence: 0.656
=== P* (loaded) ===
{
  "iteration_tag": "iter0016",
  "oof_macro_f1": 0.795454136771502,
  "threshold": 0.6
}

=== Ablation summary (derived indices + keyword text; dual threshold policy) ===
                               name  macro_f1  delta_vs_pstar_macro  best_macro_overall  threshold  pos_rate  pos_gap  fold_mean  fold_std  delta_vs_pstar_std     f1_0     f1_1      auc class_weight    C      policy  tn  fp  fn  tp
 DV2_derivedIdx_kw_C0.08_prev_first  0.801044              0

# NEW BRANCH

# Iteration 11
- 0.7965

In [50]:
import os, json, re, random, shutil, warnings
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# =========================================================
# clai_ch3_v3 — Successor iteration (single cell)
# Behavior-first: reversible + auditable + attributable
# Outputs per iter:
#   - ablation_summary.csv (candidates + deltas vs P*)
#   - oof_predictions.csv + threshold_sweep.csv
#   - group_error_summary.csv (FP/FN hotspots)
#   - decision_md (human readable)
#   - checkpoint/config.json + metrics.json + model joblib(s)
#   - consultant packet json
#   - append-only iteration_detail.jsonl
# Guardrails:
#   - Strictly forbid vitals day>10 (no day11 leakage)
#   - Fixed CV: StratifiedKFold(5, shuffle=True, random_state=42)
#   - Scoped artifacts: only under artifacts/checkpoints for this TAG
# =========================================================

# ---------------------
# 0) Config (small-step)
# ---------------------
TEAM, TASK, VERSION = "clai", "ch3", "v3"
TAG = f"{TEAM}_{TASK}_{VERSION}"

SEED = 42
N_SPLITS = 5

# keep OFF by default: TF-IDF adds dimensionality/overfit risk
ENABLE_TFIDF_VARIANT = False

# submission strategy
# - "safe": if gate fails and P* pipeline exists, rollback for submission
# - "explore": always submit selected candidate
SUBMISSION_MODE = "safe"

# tiny, controlled candidate search (incremental)
BASE_C_GRID = [0.08, 0.10]  # iter0018 uses ~0.08; add 0.10 as mild perturbation
EVAL_VARIANTS = [
    {"variant": "zip3_kw",   "include_zip3": True,  "use_tfidf": False},
    {"variant": "nozip3_kw", "include_zip3": False, "use_tfidf": False},
]
if ENABLE_TFIDF_VARIANT:
    EVAL_VARIANTS.append({"variant": "zip3_kw_tfidf1k", "include_zip3": True, "use_tfidf": True, "tfidf_max_features": 1000})

# hard gate vs P* (only update PSTAR.json when passing)
MIN_DELTA = 0.001
STD_TOL   = 0.01
POS_TOL   = 0.07

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

np.random.seed(SEED)
random.seed(SEED)

def utc_ts() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def _find_project_root() -> Path:
    cand = Path.cwd()
    if (cand / "stays_train.csv").exists():
        return cand
    cand2 = Path("/mnt/data")
    if (cand2 / "stays_train.csv").exists():
        return cand2
    return cand

PROJECT_ROOT = _find_project_root()
ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"

ART_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
CONSULT_ROOT.mkdir(parents=True, exist_ok=True)

print("=== PREFLIGHT ===")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_ROOT:", ART_ROOT)
print("CKPT_ROOT:", CKPT_ROOT)
print("CONSULT_ROOT:", CONSULT_ROOT)

# ---------------------
# 1) Bootstrap: copy any loose artifacts into scoped ART_ROOT (idempotent)
# ---------------------
def _copy_if_missing(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists() and (not dst.exists()):
        shutil.copy2(src, dst)

for pat in [
    "iter????_oof_predictions.csv",
    "iter????_threshold_sweep.csv",
    "iter????_ablation_summary.csv",
    "iter????_group_error_summary.csv",
]:
    for src in sorted(PROJECT_ROOT.glob(pat)):
        _copy_if_missing(src, ART_ROOT / src.name)

for src in sorted(PROJECT_ROOT.glob(f"{TAG}_*candidate*.csv")):
    _copy_if_missing(src, ART_ROOT / src.name)

# ---------------------
# 2) Append-only log path (fallback if unwritable)
# ---------------------
RAW_LOG_PATH = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"

def ensure_appendable(path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if os.access(path, os.W_OK):
            return path
        alt = path.with_name(path.stem + "_appended" + path.suffix)
        if not alt.exists():
            try:
                shutil.copy2(path, alt)
            except Exception:
                pass
        return alt
    else:
        try:
            with open(path, "a", encoding="utf-8") as _:
                pass
            return path
        except Exception:
            alt = path.with_name(path.stem + "_appended" + path.suffix)
            with open(alt, "a", encoding="utf-8") as _:
                pass
            return alt

LOG_PATH = ensure_appendable(RAW_LOG_PATH)
print("LOG_PATH:", LOG_PATH)

# ---------------------
# 3) Robust iteration id: max(log, artifacts, ckpts)+1
# ---------------------
def _extract_iter_num(name: str):
    m = re.search(r"iter(\d{4})", name)
    return int(m.group(1)) if m else None

def max_iteration_id_from_log(log_path: Path) -> int:
    if not log_path.exists():
        return -1
    mx = -1
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                mx = max(mx, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return mx

def max_iter_num_from_artifacts(art_root: Path) -> int:
    mx = -1
    for p in art_root.glob("iter????_*"):
        n = _extract_iter_num(p.name)
        if n is not None:
            mx = max(mx, n)
    return mx

def max_iter_num_from_ckpt(ckpt_root: Path) -> int:
    mx = -1
    for d in ckpt_root.glob("iter????"):
        if d.is_dir():
            n = _extract_iter_num(d.name)
            if n is not None:
                mx = max(mx, n)
    return mx

ITER_ID = max(max_iteration_id_from_log(LOG_PATH), max_iter_num_from_artifacts(ART_ROOT), max_iter_num_from_ckpt(CKPT_ROOT)) + 1
ITER_TAG = f"iter{ITER_ID:04d}"
BLOCK_ID = ITER_ID // 5
BLOCK_POS = ITER_ID % 5
BRANCH = "W3_ensembleLR"

print(f"ITER_ID={ITER_ID}  ITER_TAG={ITER_TAG}  BLOCK_ID={BLOCK_ID}  BLOCK_POS={BLOCK_POS}  BRANCH={BRANCH}")

# ---------------------
# 4) Load data + hard leakage check
# ---------------------
def dp(name: str) -> Path:
    p = PROJECT_ROOT / name
    if p.exists():
        return p
    p2 = Path("/mnt/data") / name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {name}")

stays_train = pd.read_csv(dp("stays_train.csv"))
stays_test  = pd.read_csv(dp("stays_test.csv"))
patients    = pd.read_csv(dp("patients.csv"))
with open(dp("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vit_list = json.load(f)

max_day = max(d.get("day", 0) for item in vit_list for d in item.get("days", []))
assert max_day <= 10, f"Leakage risk: found vitals day={max_day} (>10). Abort."

vit_map = {int(item["stay_id"]): item.get("days", []) for item in vit_list}

y = stays_train["discharge_ready_day11"].astype(int).values
train_prev = float(y.mean())
print(f"[DATA] Train prevalence(y=1): {train_prev:.3f}")

# Fixed CV (must-follow)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Global fold ids (for OOF bookkeeping + ensemble eval)
fold_ids_global = np.zeros(len(stays_train), dtype=int)
for f, (_, va_idx) in enumerate(skf.split(np.zeros(len(stays_train)), y)):
    fold_ids_global[va_idx] = f

# ---------------------
# 5) Read predecessor jsonl -> successor received summary (auditable handover)
# ---------------------
def read_jsonl(path: Path):
    recs = []
    if not path.exists():
        return recs
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                recs.append(json.loads(line))
            except Exception:
                continue
    return recs

prev_recs = read_jsonl(RAW_LOG_PATH)
print(f"[LOG] Historical records found: {len(prev_recs)}")

def _safe_get(d, *keys, default=np.nan):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

timeline_rows = []
for r in prev_recs:
    it = r.get("iteration_id", None)
    timeline_rows.append({
        "iter": f"iter{int(it):04d}" if it is not None else "NA",
        "timestamp": r.get("timestamp", ""),
        "phase": r.get("phase", ""),
        "branch": r.get("branch", ""),
        "selected": _safe_get(r, "selected_model", "name", default=str(r.get("selected_model",""))),
        "oof_macro_f1": _safe_get(r, "metrics", "oof_macro_f1", default=np.nan),
        "threshold": _safe_get(r, "metrics", "threshold", default=np.nan),
        "pos_rate": _safe_get(r, "metrics", "pos_rate", default=np.nan),
        "fold_std": _safe_get(r, "metrics", "fold_std", default=np.nan),
    })

timeline_df = pd.DataFrame(timeline_rows)
if len(timeline_df):
    tmp = timeline_df.copy()
    tmp["oof_macro_f1_num"] = pd.to_numeric(tmp["oof_macro_f1"], errors="coerce")
    best_log = tmp.dropna(subset=["oof_macro_f1_num"]).sort_values("oof_macro_f1_num", ascending=False).head(1)
    best_log = best_log.iloc[0].to_dict() if len(best_log) else None
else:
    best_log = None

print("[LOG] Best OOF macro_f1 (from log):", best_log)

REAL_TIMELINE = [
    {"iter": "FAIL_CASE", "real_f1": 0.6765, "note": "artifact contamination / wrong run context"},
    {"iter": "iter0015", "real_f1": 0.7746, "note": "recovered baseline"},
    {"iter": "iter0016", "real_f1": 0.7903, "note": "derived indices added"},
    {"iter": "iter0017", "real_f1": 0.7906, "note": "interaction-derived vitals; C sweep"},
    {"iter": "iter0018", "real_f1": 0.7965, "note": "best so far; OOF macro≈0.8010 @ thr≈0.58"},
]
print("[REAL] Timeline (provided by HM/user):")
print(pd.DataFrame(REAL_TIMELINE).to_string(index=False))

SUCCESSOR_SUMMARY_MD  = ART_ROOT / "SUCCESSOR_RECEIVED_SUMMARY.md"
SUCCESSOR_SUMMARY_CSV = ART_ROOT / "SUCCESSOR_RECEIVED_SUMMARY.csv"

summary_lines = []
summary_lines.append(f"# {TAG} — Successor received summary\nGenerated: {utc_ts()}\n")
summary_lines.append("## Mission\nPredict **discharge_ready_day11** (binary). Metric: **Macro-F1**.\n")
summary_lines.append("## Guardrails (must-follow)\n"
                     "- Fixed CV: StratifiedKFold(5, shuffle=True, random_state=42)\n"
                     "- Leakage: vitals day must be <=10 (no day11 vitals)\n"
                     "- Scoped artifacts only under artifacts/checkpoints for this TAG\n"
                     "- Each iter: reversible + auditable + attributable; failures explainable\n")
summary_lines.append("## Real-score timeline (from HM/user)\n")
summary_lines.append(pd.DataFrame(REAL_TIMELINE).to_markdown(index=False))
summary_lines.append("\n## Historical iteration timeline (from jsonl log)\n")
summary_lines.append(timeline_df.to_markdown(index=False) if len(timeline_df) else "_No historical records found._")
summary_lines.append("\n## What worked (validated)\n"
                     "- Derived/trajectory vitals beats naive per-day flattening\n"
                     "- Clinically-derived indices (MAP, shock_index, pulse_pressure, temp_dev)\n"
                     "- No class_weight='balanced' + tuned C reduced FP inflation\n"
                     "- Prevalence-aware thresholding improved CV↔real alignment\n"
                     "- Scoped artifact hygiene prevented cross-version contamination\n")
summary_lines.append("\n## What hurt (avoid)\n"
                     "- Cross-version artifact discovery (rglob '.') → contamination → huge real drop\n"
                     "- Feature explosion (too-large TF-IDF / too-many sparse cats) on ~1k train\n"
                     "- pos_rate too high → FP inflation → class0 F1 collapses\n"
                     "- Per-day alignment bug risk: must use explicit 'day'\n")
SUCCESSOR_SUMMARY_MD.write_text("\n".join(summary_lines), encoding="utf-8")
timeline_df.to_csv(SUCCESSOR_SUMMARY_CSV, index=False)
print("✅ Wrote successor summary ->", SUCCESSOR_SUMMARY_MD)

# ---------------------
# 6) Load P* (prefer PSTAR.json; fallback to iter0018 from log); ensure PSTAR exists
# ---------------------
PSTAR_PATH = CKPT_ROOT / "PSTAR.json"

def load_pstar():
    if PSTAR_PATH.exists():
        try:
            obj = json.loads(PSTAR_PATH.read_text(encoding="utf-8"))
            if isinstance(obj, dict) and isinstance(obj.get("pstar"), dict):
                return obj["pstar"], obj
        except Exception:
            pass
    # fallback: iter0018 metrics in log
    rec18 = next((r for r in prev_recs if r.get("iteration_id") == 18), None)
    if rec18 and isinstance(rec18.get("metrics"), dict):
        m = rec18["metrics"]
        return {
            "iteration_tag": "iter0018",
            "oof_macro_f1": float(m.get("oof_macro_f1")),
            "threshold": float(m.get("threshold", 0.5)),
            "fold_std": float(m.get("fold_std", np.nan)),
            "pos_rate": float(m.get("pos_rate", np.nan)),
            "confusion_matrix": m.get("confusion_matrix", None),
        }, {"tag": TAG, "timestamp": utc_ts(), "schema_version": "fallback_from_log"}
    # ultimate fallback
    return {"iteration_tag": "NONE", "oof_macro_f1": -1.0, "threshold": 0.5, "fold_std": np.nan, "pos_rate": np.nan}, {"tag": TAG, "timestamp": utc_ts(), "schema_version": "empty"}

pstar, pstar_container = load_pstar()
pstar_iter_tag = str(pstar.get("iteration_tag", "NONE"))
pstar_macro = float(pstar.get("oof_macro_f1", -1.0))
pstar_std = pstar.get("fold_std", np.nan)
pstar_std = float(pstar_std) if pstar_std is not None else np.nan
pstar_thr = float(pstar.get("threshold", 0.5))

print("=== P* anchor ===")
print(json.dumps(pstar, indent=2, ensure_ascii=False))

# Initialize PSTAR.json if missing (so downstream code can rely on it)
if not PSTAR_PATH.exists():
    with open(PSTAR_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "tag": TAG,
            "schema_version": 4,
            "timestamp": utc_ts(),
            "train_prevalence": train_prev,
            "pstar": pstar,
            "notes": "Initialized from historical log fallback; no training executed."
        }, f, indent=2, ensure_ascii=False)
    print("✅ Initialized PSTAR.json from log fallback ->", PSTAR_PATH)

# ---------------------
# 7) Feature engineering (iter0018-style: derived indices + keywords; keep raw text cols for optional TF-IDF)
# ---------------------
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]
CAT_COLS_BASE = ["unit_type", "sex", "insurance", "zip3", "admission_reason"]
ID_COLS = ["stay_id", "patient_id"]
TEXT_COLS = ["note_text_all", "note_text_last2"]

KEYWORDS = [
    "stable","improving","worsening","afebrile","fever","chills","hypotensive","tachy",
    "culture","antibiotic","abx","iv","oxygen","weaned","dyspnea","cough",
    "pain","nausea","vomit","n/v","ambulated","pt","ot","telemetry","fluids","room air","sats",
    "discharge","transfer"
]
KW_PATTERNS = [(kw, re.compile(re.escape(kw), re.IGNORECASE)) for kw in KEYWORDS]

def safe_nan(fn, arr, default=np.nan):
    try:
        with np.errstate(all="ignore"):
            return fn(arr)
    except Exception:
        return default

def build_features(stays_df: pd.DataFrame) -> pd.DataFrame:
    base = stays_df.merge(patients, on="patient_id", how="left")
    rows = []
    for sid in base["stay_id"].astype(int).tolist():
        days = sorted(vit_map.get(int(sid), []), key=lambda x: x.get("day", 0))
        # defensive per-stay leakage check
        if any((d.get("day", 0) > 10) for d in days):
            raise RuntimeError(f"Leakage risk: stay_id={sid} contains day>10.")
        day_idx = np.array([d.get("day", np.nan) for d in days], dtype=float)

        vit = {c: np.array([d.get(c, np.nan) for d in days], dtype=float) for c in VITAL_COLS}

        # derived indices
        pulse_pressure = vit["sbp"] - vit["dbp"]
        map_est = (vit["sbp"] + 2.0 * vit["dbp"]) / 3.0
        shock_index = vit["hr"] / vit["sbp"]
        shock_index = np.where(np.isfinite(shock_index), shock_index, np.nan)
        temp_dev = vit["temp_c"] - 37.0

        series = {
            "hr": vit["hr"],
            "sbp": vit["sbp"],
            "dbp": vit["dbp"],
            "rr": vit["rr"],
            "temp_c": vit["temp_c"],
            "pulse_pressure": pulse_pressure,
            "map": map_est,
            "shock_index": shock_index,
            "temp_dev": temp_dev,
        }

        notes = [(d.get("note") or "") for d in days]
        note_all = " ".join(notes).strip()
        note_last2 = " ".join(notes[-2:]).strip()

        feat = {
            "stay_id": int(sid),
            "note_text_all": note_all,
            "note_text_last2": note_last2,
            "note_len_all": len(note_all),
            "note_len_last2": len(note_last2),
            "note_days": int(sum(1 for n in notes if str(n).strip() != "")),
            "days_recorded": int(len(days)),
            "day_first": float(day_idx[0]) if len(day_idx) else np.nan,
            "day_last": float(day_idx[-1]) if len(day_idx) else np.nan,
        }

        # trajectory summaries
        for name, vals in series.items():
            vals = np.array(vals, dtype=float)
            diffs = np.diff(vals) if len(vals) >= 2 else np.array([], dtype=float)

            feat[f"{name}_mean"] = safe_nan(np.nanmean, vals)
            feat[f"{name}_std"]  = safe_nan(np.nanstd,  vals)
            feat[f"{name}_min"]  = safe_nan(np.nanmin,  vals)
            feat[f"{name}_max"]  = safe_nan(np.nanmax,  vals)
            feat[f"{name}_median"] = safe_nan(np.nanmedian, vals)
            feat[f"{name}_first"] = vals[0] if len(vals) else np.nan
            feat[f"{name}_last"]  = vals[-1] if len(vals) else np.nan

            last2 = vals[-2:] if len(vals) >= 2 else vals
            early = vals[:8] if len(vals) >= 8 else (vals[:-2] if len(vals) > 2 else vals)

            feat[f"{name}_last2_mean"] = safe_nan(np.nanmean, last2)
            feat[f"{name}_early_mean"] = safe_nan(np.nanmean, early)
            feat[f"{name}_delta_last2_early"] = feat[f"{name}_last2_mean"] - feat[f"{name}_early_mean"]
            feat[f"{name}_delta_last_first"] = feat[f"{name}_last"] - feat[f"{name}_first"]

            mask = (~np.isnan(vals)) & (~np.isnan(day_idx))
            feat[f"{name}_slope"] = np.polyfit(day_idx[mask], vals[mask], 1)[0] if mask.sum() >= 2 else np.nan

            feat[f"{name}_diff_mean"] = safe_nan(np.nanmean, diffs) if len(diffs) else np.nan
            feat[f"{name}_diff_std"]  = safe_nan(np.nanstd,  diffs) if len(diffs) else np.nan
            feat[f"{name}_last3_std"] = safe_nan(np.nanstd, vals[-3:]) if len(vals) >= 3 else safe_nan(np.nanstd, vals)

            feat[f"{name}_missing"] = int(np.isnan(vals).sum())
            feat[f"{name}_n_obs"]   = int(np.sum(~np.isnan(vals)))

        # clinical threshold counts
        hr, sbp, dbp, temp, rr = vit["hr"], vit["sbp"], vit["dbp"], vit["temp_c"], vit["rr"]
        feat["hr_gt100"] = int(np.nansum(hr > 100))
        feat["hr_lt60"]  = int(np.nansum(hr < 60))
        feat["sbp_lt90"] = int(np.nansum(sbp < 90))
        feat["sbp_gt160"] = int(np.nansum(sbp > 160))
        feat["dbp_lt60"] = int(np.nansum(dbp < 60))
        feat["temp_gt38"] = int(np.nansum(temp > 38))
        feat["temp_lt36"] = int(np.nansum(temp < 36))
        feat["rr_gt22"] = int(np.nansum(rr > 22))
        feat["map_lt65"] = int(np.nansum(map_est < 65))
        feat["shock_gt1"] = int(np.nansum(shock_index > 1.0))
        feat["pp_lt30"] = int(np.nansum(pulse_pressure < 30))

        # keyword counts (cheap, interpretable)
        for kw, pat in KW_PATTERNS:
            key = kw.replace(" ", "_").replace("/", "_")
            feat[f"all_kw_{key}"] = len(pat.findall(note_all))
            feat[f"last2_kw_{key}"] = len(pat.findall(note_last2))

        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    return base.merge(feat_df, on="stay_id", how="left")

print("=== [FE] Building features ===")
train_full = build_features(stays_train)
test_full  = build_features(stays_test)

train_X = train_full.drop(columns=["discharge_ready_day11"]).copy()
test_X  = test_full.copy()

# categorical casting
for c in CAT_COLS_BASE:
    train_X[c] = train_X[c].astype(str)
    test_X[c]  = test_X[c].astype(str)

# ---------------------
# 8) Ensure P* pipeline exists (optional reconstruction for iter0018 only)
# ---------------------
pstar_pipe_path = CKPT_ROOT / pstar_iter_tag / "pipeline.joblib"

def reconstruct_pstar_pipeline_if_needed():
    if pstar_iter_tag != "iter0018":
        return
    if pstar_pipe_path.exists():
        return
    # reconstruct from log if possible (default C=0.08)
    rec = next((r for r in prev_recs if r.get("iteration_id") == 18), None)
    C = 0.08
    if rec and isinstance(rec.get("selected_model"), dict) and rec["selected_model"].get("C") is not None:
        try:
            C = float(rec["selected_model"]["C"])
        except Exception:
            C = 0.08

    cat_cols = CAT_COLS_BASE  # iter0018 baseline used zip3
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler(with_mean=False)),
            ]), [c for c in train_X.columns if c not in (ID_COLS + list(cat_cols) + TEXT_COLS)]),
            ("cat", OneHotEncoder(handle_unknown="ignore"), list(cat_cols)),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    clf = LogisticRegression(
        solver="liblinear",
        penalty="l1",
        C=float(C),
        class_weight=None,
        max_iter=4000,
        random_state=SEED
    )
    pipe = Pipeline([("preprocess", pre), ("clf", clf)])
    pipe.fit(train_X, y)

    d = CKPT_ROOT / pstar_iter_tag
    d.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipe, pstar_pipe_path)

    note = {
        "timestamp": utc_ts(),
        "reason": "P* pipeline missing; reconstructed from log for safe rollback",
        "pstar_iter": pstar_iter_tag,
        "C": float(C),
        "threshold": float(pstar_thr),
        "schema": "iter0018-style derived indices + keyword counts; no TF-IDF"
    }
    with open(d / "RECONSTRUCTED_PIPELINE.json", "w", encoding="utf-8") as f:
        json.dump(note, f, indent=2, ensure_ascii=False)
    print("✅ Reconstructed missing P* pipeline ->", pstar_pipe_path)

reconstruct_pstar_pipeline_if_needed()

# ---------------------
# 9) Modeling helpers (preprocessor, fold-cache, OOF, threshold policy)
# ---------------------
def make_preprocessor(df_X: pd.DataFrame, cat_cols, use_tfidf: bool, tfidf_max_features: int = 1000):
    num_cols = [c for c in df_X.columns if c not in (ID_COLS + list(cat_cols) + TEXT_COLS)]
    transformers = [
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler(with_mean=False))]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), list(cat_cols)),
    ]
    if use_tfidf:
        selector = FunctionTransformer(lambda x: x.astype(str).to_numpy().ravel(), validate=False)
        tfidf = TfidfVectorizer(
            max_features=int(tfidf_max_features),
            min_df=2,
            ngram_range=(1, 2),
            lowercase=True,
        )
        transformers.append(("tfidf", Pipeline([("sel", selector), ("tfidf", tfidf)]), "note_text_last2"))

    pre = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3)

    schema = {
        "num_cols": num_cols,
        "cat_cols": list(cat_cols),
        "use_tfidf": bool(use_tfidf),
        "tfidf_max_features": int(tfidf_max_features) if use_tfidf else 0,
        "keywords_count": len(KEYWORDS),
        "vitals_days_used": "1-10",
        "derived_indices": ["map", "shock_index", "pulse_pressure", "temp_dev"],
    }
    return pre, schema

def build_fold_cache(df_X: pd.DataFrame, y: np.ndarray, pre: ColumnTransformer):
    cache = []
    fold_ids = np.zeros(len(df_X), dtype=int)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(df_X, y)):
        pre_f = clone(pre)
        X_tr = pre_f.fit_transform(df_X.iloc[tr_idx], y[tr_idx])
        X_va = pre_f.transform(df_X.iloc[va_idx])
        cache.append({"fold": fold, "tr_idx": tr_idx, "va_idx": va_idx, "X_tr": X_tr, "y_tr": y[tr_idx], "X_va": X_va})
        fold_ids[va_idx] = fold
    return cache, fold_ids

def oof_from_cache(cache, C: float):
    oof = np.zeros(len(train_X), dtype=float)
    for fd in cache:
        clf = LogisticRegression(
            solver="liblinear",
            penalty="l1",
            C=float(C),
            class_weight=None,
            max_iter=4000,
            random_state=SEED
        )
        clf.fit(fd["X_tr"], fd["y_tr"])
        oof[fd["va_idx"]] = clf.predict_proba(fd["X_va"])[:, 1]
    return oof

def threshold_sweep(oof_proba: np.ndarray, y_true: np.ndarray, fold_ids: np.ndarray):
    thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
    rows = []
    for t in thresholds:
        pred = (oof_proba >= t).astype(int)
        cm = confusion_matrix(y_true, pred)
        f1s = f1_score(y_true, pred, average=None)
        macro = f1_score(y_true, pred, average="macro")
        fold_scores = [f1_score(y_true[fold_ids == f], pred[fold_ids == f], average="macro") for f in range(N_SPLITS)]
        rows.append({
            "threshold": float(t),
            "macro_f1": float(macro),
            "f1_0": float(f1s[0]),
            "f1_1": float(f1s[1]),
            "pos_rate": float(pred.mean()),
            "fold_mean": float(np.mean(fold_scores)),
            "fold_std": float(np.std(fold_scores)),
            "tn": int(cm[0, 0]), "fp": int(cm[0, 1]), "fn": int(cm[1, 0]), "tp": int(cm[1, 1]),
        })
    return pd.DataFrame(rows)

def choose_threshold_prev_first(sw: pd.DataFrame, prevalence: float, pos_tol: float = 0.07, macro_drop_tol: float = 0.005):
    best_macro = float(sw["macro_f1"].max())
    sw_ok = sw.loc[(sw["pos_rate"] - prevalence).abs() <= pos_tol].copy()
    pool = sw_ok if len(sw_ok) else sw.copy()
    pool_best = float(pool["macro_f1"].max())
    near = pool.loc[pool["macro_f1"] >= (pool_best - macro_drop_tol)].copy()
    near["pos_gap"] = (near["pos_rate"] - prevalence).abs()
    near = near.sort_values(["pos_gap", "fold_std", "macro_f1", "threshold"],
                            ascending=[True, True, False, True]).reset_index(drop=True)
    out = near.iloc[0].to_dict()
    out["best_macro_overall"] = best_macro
    out["policy"] = "prev_first"
    return out

def choose_threshold_macro_first(sw: pd.DataFrame, prevalence: float):
    df = sw.copy()
    df["pos_gap"] = (df["pos_rate"] - prevalence).abs()
    df = df.sort_values(["macro_f1", "fold_std", "pos_gap", "threshold"],
                        ascending=[False, True, True, True]).reset_index(drop=True)
    out = df.iloc[0].to_dict()
    out["best_macro_overall"] = float(sw["macro_f1"].max())
    out["policy"] = "macro_first"
    return out

# ---------------------
# 10) Evaluate base candidates (variant x C) + policy variants; then small OOF ensemble (avg)
# ---------------------
cand_rows = []
oof_by_key = {}     # base_key -> oof_proba
spec_by_key = {}    # base_key -> spec dict (variant, cat_cols, C, schema)

for v in EVAL_VARIANTS:
    cat_cols = [c for c in CAT_COLS_BASE if (v["include_zip3"] or c != "zip3")]
    pre, schema = make_preprocessor(train_X, cat_cols=cat_cols, use_tfidf=bool(v.get("use_tfidf", False)),
                                    tfidf_max_features=int(v.get("tfidf_max_features", 1000)))
    cache, fold_ids = build_fold_cache(train_X, y, pre)

    for C in BASE_C_GRID:
        base_key = f"{v['variant']}_C{float(C):.2f}"
        oof = oof_from_cache(cache, C=float(C))
        oof_by_key[base_key] = oof
        spec_by_key[base_key] = {"variant": v, "cat_cols": cat_cols, "C": float(C), "schema": schema}

        auc = float(roc_auc_score(y, oof))
        sw = threshold_sweep(oof, y, fold_ids)

        for ch in [choose_threshold_prev_first(sw, train_prev), choose_threshold_macro_first(sw, train_prev)]:
            name = f"{base_key}_{ch['policy']}"
            cand_rows.append({
                "name": name,
                "type": "single",
                "base_key": base_key,
                "policy": ch["policy"],
                "threshold": float(ch["threshold"]),
                "macro_f1": float(ch["macro_f1"]),
                "best_macro_overall": float(ch["best_macro_overall"]),
                "pos_rate": float(ch["pos_rate"]),
                "pos_gap": float(abs(float(ch["pos_rate"]) - train_prev)),
                "fold_mean": float(ch["fold_mean"]),
                "fold_std": float(ch["fold_std"]),
                "f1_0": float(ch["f1_0"]),
                "f1_1": float(ch["f1_1"]),
                "auc": auc,
                "tn": int(ch["tn"]), "fp": int(ch["fp"]), "fn": int(ch["fn"]), "tp": int(ch["tp"]),
                "variant": v["variant"],
                "include_zip3": bool(v["include_zip3"]),
                "use_tfidf": bool(v.get("use_tfidf", False)),
                "tfidf_max_features": int(v.get("tfidf_max_features", 0)),
                "C": float(C),
            })

cand_df = pd.DataFrame(cand_rows)
cand_df["delta_vs_pstar_macro"] = cand_df["macro_f1"] - pstar_macro
cand_df["delta_vs_pstar_std"] = cand_df["fold_std"] - (pstar_std if not np.isnan(pstar_std) else np.nan)

# Ensemble pool: top K base_keys by their best macro_f1 across policies (dedup)
base_best = cand_df.groupby("base_key")["macro_f1"].max().reset_index().sort_values("macro_f1", ascending=False)
TOPK_FOR_ENSEMBLE = min(3, len(base_best))
pool_base = base_best.head(TOPK_FOR_ENSEMBLE)["base_key"].tolist()

ens_rows = []
if len(pool_base) >= 2:
    for comb in combinations(pool_base, 2):
        oof_avg = np.mean([oof_by_key[bk] for bk in comb], axis=0)
        auc = float(roc_auc_score(y, oof_avg))
        sw = threshold_sweep(oof_avg, y, fold_ids_global)
        for ch in [choose_threshold_prev_first(sw, train_prev), choose_threshold_macro_first(sw, train_prev)]:
            name = f"ENS2[{'+'.join(comb)}]_{ch['policy']}"
            ens_rows.append({
                "name": name,
                "type": "ensemble",
                "base_key": None,
                "members": list(comb),
                "policy": ch["policy"],
                "threshold": float(ch["threshold"]),
                "macro_f1": float(ch["macro_f1"]),
                "best_macro_overall": float(ch["best_macro_overall"]),
                "pos_rate": float(ch["pos_rate"]),
                "pos_gap": float(abs(float(ch["pos_rate"]) - train_prev)),
                "fold_mean": float(ch["fold_mean"]),
                "fold_std": float(ch["fold_std"]),
                "f1_0": float(ch["f1_0"]),
                "f1_1": float(ch["f1_1"]),
                "auc": auc,
                "tn": int(ch["tn"]), "fp": int(ch["fp"]), "fn": int(ch["fn"]), "tp": int(ch["tp"]),
                "variant": "ensemble2",
                "include_zip3": None,
                "use_tfidf": any(bool(spec_by_key[b]["variant"].get("use_tfidf", False)) for b in comb),
                "tfidf_max_features": None,
                "C": None,
            })
            # store OOF under name for easy access later
            oof_by_key[name] = oof_avg
            spec_by_key[name] = {"ensemble_members": list(comb)}

if ens_rows:
    ens_df = pd.DataFrame(ens_rows)
    ens_df["delta_vs_pstar_macro"] = ens_df["macro_f1"] - pstar_macro
    ens_df["delta_vs_pstar_std"] = ens_df["fold_std"] - (pstar_std if not np.isnan(pstar_std) else np.nan)
    cand_df = pd.concat([cand_df, ens_df], ignore_index=True)

cand_df = cand_df.sort_values(["macro_f1", "fold_std", "pos_gap"], ascending=[False, True, True]).reset_index(drop=True)

print("\n=== Candidate leaderboard (top 12) ===")
SHOW_COLS = [
    "name","type","macro_f1","delta_vs_pstar_macro","threshold","pos_rate","pos_gap",
    "fold_std","delta_vs_pstar_std","f1_0","f1_1","auc","policy"
]
print(cand_df[SHOW_COLS].head(12).to_string(index=False))

ABL_PATH = ART_ROOT / f"{ITER_TAG}_ablation_summary.csv"
cand_df.to_csv(ABL_PATH, index=False)
print("✅ Saved ablation summary ->", ABL_PATH)

# ---------------------
# 11) Gate + selection (clear decision; rollback path)
# ---------------------
def passes_gate(row) -> bool:
    if float(row["macro_f1"]) < pstar_macro + MIN_DELTA:
        return False
    if not np.isnan(pstar_std) and float(row["fold_std"]) > pstar_std + STD_TOL:
        return False
    if abs(float(row["pos_rate"]) - train_prev) > POS_TOL:
        return False
    return True

eligible_df = cand_df[cand_df.apply(passes_gate, axis=1)].copy()

if len(eligible_df):
    eligible_df = eligible_df.sort_values(["macro_f1", "fold_std", "pos_gap"], ascending=[False, True, True])
    selected = eligible_df.iloc[0].to_dict()
    decision = "SELECT_CANDIDATE (passes gate vs P*)"
else:
    selected = cand_df.iloc[0].to_dict()
    decision = "NO_CANDIDATE_PASSES_GATE"

selected_name = selected["name"]
selected_type = selected["type"]
selected_thr  = float(selected["threshold"])
selected_policy = selected.get("policy", "prev_first")

print("\n=== Selection ===")
print("Decision:", decision)
print("Selected:", selected_name, "| type=", selected_type, "| macro_f1=", round(float(selected["macro_f1"]),6),
      "| thr=", selected_thr, "| pos_rate=", round(float(selected["pos_rate"]),3))

# decide whether to rollback submission
use_rollback_for_submission = (decision == "NO_CANDIDATE_PASSES_GATE") and (SUBMISSION_MODE == "safe") and pstar_pipe_path.exists()
if use_rollback_for_submission:
    print("[SUBMISSION] SAFE MODE -> rollback to P* pipeline:", pstar_pipe_path)
else:
    if decision == "NO_CANDIDATE_PASSES_GATE" and SUBMISSION_MODE == "safe":
        print("[SUBMISSION] SAFE MODE but no P* pipeline found; will submit selected candidate to avoid empty.")
    else:
        print("[SUBMISSION] Explore mode or candidate passes gate -> submit selected candidate.")

# ---------------------
# 12) Fit final model(s) + write submission/candidate
# ---------------------
SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"
CAND_PATH = ART_ROOT / f"{TAG}_{BRANCH}_block{BLOCK_ID}_iter{ITER_ID:04d}_candidate.csv"

def fit_single_pipeline(spec_key: str):
    spec = spec_by_key[spec_key]
    v = spec["variant"]
    cat_cols = spec["cat_cols"]
    pre, schema = make_preprocessor(
        train_X, cat_cols=cat_cols,
        use_tfidf=bool(v.get("use_tfidf", False)),
        tfidf_max_features=int(v.get("tfidf_max_features", 1000))
    )
    clf = LogisticRegression(
        solver="liblinear",
        penalty="l1",
        C=float(spec["C"]),
        class_weight=None,
        max_iter=4000,
        random_state=SEED
    )
    pipe = Pipeline([("preprocess", pre), ("clf", clf)])
    pipe.fit(train_X, y)
    return pipe, schema

def predict_with_threshold(proba: np.ndarray, thr: float) -> np.ndarray:
    return (proba >= float(thr)).astype(int)

thr_used = selected_thr
final_model_bundle = {}
final_schema = None

if use_rollback_for_submission:
    pipe = joblib.load(pstar_pipe_path)
    thr_used = pstar_thr
    test_proba = pipe.predict_proba(test_X)[:, 1]
    test_pred  = predict_with_threshold(test_proba, thr_used)
    final_model_bundle = {"type": "rollback", "pstar_pipeline": str(pstar_pipe_path)}
else:
    if selected_type == "ensemble":
        members = spec_by_key[selected_name]["ensemble_members"]
        member_pipes = {}
        member_schemas = {}
        member_probas = []
        for bk in members:
            pipe_m, schema_m = fit_single_pipeline(bk)
            member_pipes[bk] = pipe_m
            member_schemas[bk] = schema_m
            member_probas.append(pipe_m.predict_proba(test_X)[:, 1])
        test_proba = np.mean(member_probas, axis=0)
        test_pred  = predict_with_threshold(test_proba, thr_used)
        final_model_bundle = {"type": "ensemble2", "members": list(members)}
        final_schema = {"members": list(members), "member_schemas": member_schemas}
    else:
        base_key = selected["base_key"]
        pipe, schema = fit_single_pipeline(base_key)
        test_proba = pipe.predict_proba(test_X)[:, 1]
        test_pred  = predict_with_threshold(test_proba, thr_used)
        final_model_bundle = {"type": "single", "base_key": base_key, "C": spec_by_key[base_key]["C"]}
        final_schema = schema

sub = pd.DataFrame({
    "stay_id": test_full["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred.astype(int),
})
sub.to_csv(SUB_PATH, index=False)
sub.to_csv(CAND_PATH, index=False)

print("\n=== Submission written ===")
print("SUB_PATH:", SUB_PATH)
print("CAND_PATH:", CAND_PATH)
print("Threshold used:", thr_used)
print("Submission pos_rate:", round(float(sub["discharge_ready_day11"].mean()), 3))

# ---------------------
# 13) Save OOF/sweep/diagnostics (for failure explanations)
# ---------------------
# selected OOF proba (single uses base_key; ensemble uses selected_name)
if selected_type == "ensemble":
    selected_oof = oof_by_key[selected_name]
else:
    selected_oof = oof_by_key[selected["base_key"]]

# store OOF
oof_path = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
oof_df = pd.DataFrame({
    "stay_id": train_full["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "oof_proba": selected_oof.astype(float),
    "fold": fold_ids_global.astype(int),
})
oof_df.to_csv(oof_path, index=False)

# store sweep
sweep_path = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"
sw_sel = threshold_sweep(selected_oof, y, fold_ids_global)
sw_sel.to_csv(sweep_path, index=False)

# metrics on selected oof @ selected_thr (even if rollback submission used)
pred_oof = predict_with_threshold(selected_oof, selected_thr)
cm = confusion_matrix(y, pred_oof).tolist()
f1s = f1_score(y, pred_oof, average=None)
macro = f1_score(y, pred_oof, average="macro")
fold_scores = [float(f1_score(y[fold_ids_global == f], pred_oof[fold_ids_global == f], average="macro")) for f in range(N_SPLITS)]

# group-wise FP/FN
diag = train_full[["stay_id"] + CAT_COLS_BASE].copy()
for c in CAT_COLS_BASE:
    diag[c] = diag[c].astype(str)
diag["y"] = y
diag["pred"] = pred_oof
diag["proba"] = selected_oof
diag["is_fp"] = (diag["y"] == 0) & (diag["pred"] == 1)
diag["is_fn"] = (diag["y"] == 1) & (diag["pred"] == 0)

def group_err(df: pd.DataFrame, col: str) -> pd.DataFrame:
    g = df.groupby(col).agg(
        n=("stay_id", "count"),
        y_prev=("y", "mean"),
        pred_rate=("pred", "mean"),
        fp=("is_fp", "sum"),
        fn=("is_fn", "sum"),
    ).reset_index()
    neg = np.maximum(1.0, g["n"] - g["y_prev"] * g["n"])
    pos = np.maximum(1.0, g["y_prev"] * g["n"])
    g["fp_rate_in_neg"] = g["fp"] / neg
    g["fn_rate_in_pos"] = g["fn"] / pos
    return g.sort_values(["fp_rate_in_neg", "fn_rate_in_pos"], ascending=False)

g_unit = group_err(diag, "unit_type"); g_unit["group_col"] = "unit_type"
g_ins  = group_err(diag, "insurance"); g_ins["group_col"] = "insurance"
g_adm  = group_err(diag, "admission_reason"); g_adm["group_col"] = "admission_reason"

DIAG_PATH = ART_ROOT / f"{ITER_TAG}_group_error_summary.csv"
pd.concat([g_unit, g_ins, g_adm], ignore_index=True).to_csv(DIAG_PATH, index=False)

print("✅ Saved OOF ->", oof_path.name)
print("✅ Saved sweep ->", sweep_path.name)
print("✅ Saved group diagnostics ->", DIAG_PATH.name)

# ---------------------
# 14) Checkpoint bundle (config/metrics + model artifacts)
# ---------------------
iter_ckpt_dir = CKPT_ROOT / ITER_TAG
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

# Save model(s)
if use_rollback_for_submission:
    with open(iter_ckpt_dir / "rollback_used.txt", "w", encoding="utf-8") as f:
        f.write(str(pstar_pipe_path))
else:
    if selected_type == "ensemble":
        # save each member pipeline
        for bk, pipe_m in member_pipes.items():
            joblib.dump(pipe_m, iter_ckpt_dir / f"pipeline_{bk}.joblib")
        with open(iter_ckpt_dir / "ensemble_spec.json", "w", encoding="utf-8") as f:
            json.dump(final_model_bundle, f, indent=2, ensure_ascii=False)
    else:
        joblib.dump(pipe, iter_ckpt_dir / "pipeline.joblib")

metrics = {
    "oof_macro_f1": float(macro),
    "per_class_f1": {"f1_0": float(f1s[0]), "f1_1": float(f1s[1])},
    "confusion_matrix": cm,
    "threshold": float(selected_thr),
    "pos_rate": float(pred_oof.mean()),
    "fold_scores": fold_scores,
    "fold_mean": float(np.mean(fold_scores)),
    "fold_std": float(np.std(fold_scores)),
    "auc": float(roc_auc_score(y, selected_oof)),
    "train_prevalence": train_prev,
    "gate": {"min_delta": MIN_DELTA, "std_tol": STD_TOL, "pos_tol": POS_TOL, "decision": decision},
    "pstar_reference": pstar,
    "submission": {"mode": SUBMISSION_MODE, "rollback_used": bool(use_rollback_for_submission), "threshold_used": float(thr_used)},
}

config = {
    "team": TEAM, "task": TASK, "version": VERSION,
    "iteration_id": ITER_ID, "iteration_tag": ITER_TAG,
    "branch": BRANCH, "block_id": BLOCK_ID, "block_pos": BLOCK_POS,
    "seed": SEED,
    "validation": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "selected": {
        "name": selected_name,
        "type": selected_type,
        "decision": decision,
        "threshold": float(selected_thr),
        "threshold_policy": selected_policy,
        "base_key": selected.get("base_key", None),
        "ensemble_members": selected.get("members", None),
    },
    "schema": final_schema,
    "deltas_vs_prev": {
        "changes": [
            "W3 incremental: evaluate tiny OOF-level ensemble of top LR variants (zip3 on/off, small C perturbation).",
            "Kept iter0018 feature set (derived indices + keyword counts); TF-IDF optional and OFF by default.",
            "Compared threshold policies (prev_first vs macro_first) with prevalence-aware selection.",
        ],
        "risk": [
            "Ensembling or C perturbation can shift pos_rate and inflate FP; we guard with POS_TOL and log confusion matrix + group FP hotspots.",
            "Threshold selection can overfit; we log fold_std and require gate vs P* to update PSTAR.json.",
        ],
        "fallback": [
            "If gate fails and P* pipeline exists: SAFE submission rolls back to P* pipeline.",
            "Even if rollback submission is used, we still save candidate OOF + diagnostics for learning.",
        ],
    },
}

with open(iter_ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("✅ Saved checkpoint ->", iter_ckpt_dir)

# Human-readable decision note
DECISION_MD = ART_ROOT / f"{ITER_TAG}_DECISION.md"
dec = []
dec.append(f"# {TAG} — {ITER_TAG} decision\nGenerated: {utc_ts()}\n")
dec.append("## P* reference\n```json\n" + json.dumps(pstar, indent=2, ensure_ascii=False) + "\n```\n")
dec.append("## Candidate leaderboard (top 12)\n")
dec.append(cand_df[SHOW_COLS].head(12).to_markdown(index=False))
dec.append("\n## Selected\n```json\n" + json.dumps({k: selected.get(k) for k in ["name","type","macro_f1","threshold","pos_rate","fold_std","policy","delta_vs_pstar_macro","delta_vs_pstar_std"]}, indent=2, ensure_ascii=False) + "\n```\n")
dec.append(f"\n## Gate decision\n- {decision}\n")
dec.append("\n## Risks\n- " + "\n- ".join(config["deltas_vs_prev"]["risk"]) + "\n")
dec.append("\n## Fallback\n- " + "\n- ".join(config["deltas_vs_prev"]["fallback"]) + "\n")
DECISION_MD.write_text("\n".join(dec), encoding="utf-8")
print("✅ Wrote decision note ->", DECISION_MD)

# ---------------------
# 15) Update PSTAR.json only if passes gate
# ---------------------
if decision.startswith("SELECT_CANDIDATE"):
    new_pstar = {
        "iteration_tag": ITER_TAG,
        "oof_macro_f1": float(macro),
        "threshold": float(selected_thr),
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "pos_rate": float(pred_oof.mean()),
        "f1_0": float(f1s[0]),
        "f1_1": float(f1s[1]),
        "confusion_matrix": cm,
    }
    with open(PSTAR_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "tag": TAG,
            "schema_version": 4,
            "timestamp": utc_ts(),
            "train_prevalence": train_prev,
            "pstar": new_pstar,
            "notes": "Updated because candidate passed hard gate vs prior P*."
        }, f, indent=2, ensure_ascii=False)
    print("✅ Updated PSTAR.json ->", new_pstar["iteration_tag"], "macro=", round(new_pstar["oof_macro_f1"], 6))
else:
    print("ℹ️ PSTAR.json unchanged (no candidate passed hard gate).")

# ---------------------
# 16) Append iteration log (append-only)
# ---------------------
iteration_record = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": utc_ts(),
    "phase": "Modeling",
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "data_paths_used": {
        "stays_train": str(dp("stays_train.csv")),
        "stays_test": str(dp("stays_test.csv")),
        "patients": str(dp("patients.csv")),
        "vitals_timeseries": str(dp("vitals_timeseries.json")),
        "art_root": str(ART_ROOT),
        "ckpt_root": str(CKPT_ROOT),
    },
    "join_keys": ["stays.patient_id -> patients.patient_id", "stays.stay_id -> vitals_timeseries.stay_id"],
    "leakage_checks_performed": [
        "Asserted vitals max(day) <= 10 (no day11 leakage).",
        "Defensive per-stay check: no day>10 present.",
        "Threshold tuned on OOF only.",
        "Artifact discovery scoped under artifacts/clai_ch3_v3.",
    ],
    "feature_summary": {
        "baseline": "iter0018-style derived indices + keyword counts",
        "tfidf_enabled": ENABLE_TFIDF_VARIANT,
    },
    "models_attempted": cand_df[SHOW_COLS].head(30).to_dict("records"),
    "selected_model": config["selected"],
    "validation_protocol": config["validation"],
    "metrics": metrics,
    "artifacts_written": {
        "successor_summary_md": str(SUCCESSOR_SUMMARY_MD),
        "successor_summary_csv": str(SUCCESSOR_SUMMARY_CSV),
        "decision_md": str(DECISION_MD),
        "submission_csv": str(SUB_PATH),
        "candidate_snapshot": str(CAND_PATH),
        "oof_predictions": str(oof_path),
        "threshold_sweep": str(sweep_path),
        "ablation_summary": str(ABL_PATH),
        "group_error_summary": str(DIAG_PATH),
        "checkpoint_dir": str(iter_ckpt_dir),
        "pstar_path": str(PSTAR_PATH),
    },
    "deltas_vs_prev": config["deltas_vs_prev"],
    "known_defects_limitations": [
        "Ensemble selection is still OOF-based; mitigated by small candidate set + stability metrics.",
        "TF-IDF remains OFF by default to prevent complexity drift; enable only as a gated experiment.",
    ],
    "next_step_hypothesis": (
        "If real-F1 still plateaus: consider enabling small TF-IDF last2 (<=1k feats) as a strictly-gated experiment, "
        "or a 2-model blend: LR(sparse) + tree(dense vitals) with the same audit/rollback discipline."
    ),
    "hm_message": (
        "Successor run: prioritize reversible/auditable/attributable changes. "
        "Keep CV fixed and prevent leakage; include risk+fallback each iteration."
    ),
}

try:
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
    print("✅ Appended iteration log ->", LOG_PATH)
except Exception as e:
    alt = CKPT_ROOT / f"{TAG}_iteration_detail_fallback.jsonl"
    with open(alt, "a", encoding="utf-8") as f:
        f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
    print("⚠️ Could not append to LOG_PATH; wrote to:", alt, "err=", str(e))

# ---------------------
# 17) Consultant packet (every iter)
# ---------------------
consult_packet = {
    "tag": TAG,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": iteration_record["timestamp"],
    "what_changed": config["deltas_vs_prev"]["changes"],
    "metric_changes": {
        "pstar_reference": pstar,
        "selected_summary": {k: selected.get(k) for k in ["name","type","macro_f1","threshold","pos_rate","fold_std","policy"]},
        "gate_decision": decision,
        "oof_metrics": {"macro_f1": float(macro), "f1_0": float(f1s[0]), "f1_1": float(f1s[1]), "auc": float(metrics["auc"])},
    },
    "suspected_leakage_risks": iteration_record["leakage_checks_performed"],
    "complexity_drift_risks": [
        "Candidate search kept tiny; TF-IDF disabled by default."
    ],
    "evaluation_alignment_risks": [
        "Threshold tuned on OOF; mitigated by prevalence-aware policy + fold_std logging + hard gate vs P*."
    ],
    "unknown_unknowns": [
        "Group-wise error hotspots may shift on test; see group_error_summary artifact."
    ],
    "recommendations_for_next_iter": [
        "If regress: rollback submission to P*; compare FP/FN shifts by group and pos_rate drift.",
        "If stable but no gain: enable TF-IDF last2 as gated experiment (max_features<=1k), ablate and monitor fold_std/pos_rate.",
    ],
}

consult_path = CONSULT_ROOT / f"{TAG}_iter{ITER_ID}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, ensure_ascii=False)
print("✅ Wrote consultant packet ->", consult_path)

print("\nDONE ✅")
print("Key outputs:")
print(" - Successor summary:", SUCCESSOR_SUMMARY_MD)
print(" - Submission       :", SUB_PATH)
print(" - Candidate        :", CAND_PATH)
print(" - Ablation         :", ABL_PATH)
print(" - OOF              :", oof_path)
print(" - Sweep            :", sweep_path)
print(" - Group diag       :", DIAG_PATH)
print(" - Decision note    :", DECISION_MD)
print(" - PSTAR            :", PSTAR_PATH)
print(" - Checkpoint dir   :", iter_ckpt_dir)

=== PREFLIGHT ===
PROJECT_ROOT: d:\AgentDs\agent_ds_healthcare
ART_ROOT: d:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT: d:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
CONSULT_ROOT: d:\AgentDs\agent_ds_healthcare\consultant_packets
LOG_PATH: d:\AgentDs\agent_ds_healthcare\clai_ch3_v3_iteration_detail.jsonl
ITER_ID=19  ITER_TAG=iter0019  BLOCK_ID=3  BLOCK_POS=4  BRANCH=W3_ensembleLR
[DATA] Train prevalence(y=1): 0.656
[LOG] Historical records found: 17
[LOG] Best OOF macro_f1 (from log): {'iter': 'iter0018', 'timestamp': '2026-03-02T06:01:17.915093Z', 'phase': 'Modeling', 'branch': 'W2', 'selected': 'DV2_derivedIdx_kw_C0.08_prev_first', 'oof_macro_f1': 0.8010438942644934, 'threshold': 0.58, 'pos_rate': 0.691, 'fold_std': 0.017551713766308295, 'oof_macro_f1_num': 0.8010438942644934}
[REAL] Timeline (provided by HM/user):
     iter  real_f1                                       note
FAIL_CASE   0.6765 artifact contamination / wrong run context
 iter0015   0.7746   

# Iteration 12
- 0.8168

In [52]:
# =========================================================
# clai_ch3_v3 — Successor Iteration Cell (W4: text + TE-HGB blend)
# Goal: jump out of LR-variants local optimum WITHOUT breaking the proven behaviors:
#   ✅ reversible changes  ✅ auditable artifacts  ✅ attributable deltas
#   ✅ failure is explainable  ✅ strict leakage guardrail (vitals day<=10)
#   ✅ fixed CV: StratifiedKFold(5, shuffle=True, random_state=42)
#
# What this cell does (single paste):
# 1) Preflight + scoped artifact hygiene (no cross-version contamination)
# 2) Load data + hard leakage assert (max day <= 10)
# 3) Build features:
#    - DV2 summary/trajectory vitals + derived indices (MAP, shock index, pulse pressure, temp_dev)
#    - keyword counts (validated low-dim text proxy)
#    - NEW (controlled): explicit per-day vitals features aligned by day (small, not "feature explosion")
#    - keep raw note text columns for TF-IDF model
# 4) Train diverse candidate models with fixed CV:
#    - BASE: LR (L1) on structured features (P* anchor family)
#    - TEXT: LR (L2) on small TF-IDF(note_last2) + a few numeric note stats (capped features)
#    - HGB: HistGradientBoosting on dense numeric + TargetEncoded categories (non-linear interactions)
#    - search simple blends (OOF) to reduce correlation → attempt to break plateau
# 5) Threshold sweep + two policies (prev_first / macro_first) + prevalence-aware tie-break
# 6) Gate vs P* (hard safety): if candidate fails, PSTAR not updated; submission can rollback to P*
# 7) Write *every* required artifact: ablation CSV, OOF, sweep, group diagnostics, decision MD,
#    checkpoint (pipelines + config + metrics), consultant packet, append-only jsonl log
# 8) Write successor context summary (pitfalls + why iter0019 stalled + what we try now)
# =========================================================

import os, json, re, random, shutil, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
import joblib

# =====================
# 0) Runtime toggles
# =====================
FAST_MODE = False  # True = faster debug (smaller HGB iters / TF-IDF cap). Default False.
SAFE_SUBMISSION = True  # If gate fails and P* pipeline exists -> rollback for SUBMISSION csv.
FIT_AND_SAVE_CANDIDATES_EVEN_IF_ROLLBACK = True  # keep audit reproducibility.

# =====================
# 1) Determinism + warnings
# =====================
TEAM, TASK, VERSION = "clai", "ch3", "v3"
TAG = f"{TEAM}_{TASK}_{VERSION}"
SEED = 42
N_SPLITS = 5

np.random.seed(SEED)
random.seed(SEED)

warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice")
warnings.filterwarnings("ignore", category=RuntimeWarning)

def utc_ts() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def _find_project_root() -> Path:
    cand = Path.cwd()
    if (cand / "stays_train.csv").exists():
        return cand
    cand2 = Path("/mnt/data")
    if (cand2 / "stays_train.csv").exists():
        return cand2
    return cand

PROJECT_ROOT = _find_project_root()
ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"

for p in [ART_ROOT, CKPT_ROOT, CONSULT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=== PREFLIGHT ===")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_ROOT:", ART_ROOT)
print("CKPT_ROOT:", CKPT_ROOT)
print("CONSULT_ROOT:", CONSULT_ROOT)

# =====================
# 2) Scoped artifact bootstrap (idempotent)
#    Guard against the historic disaster: cross-version artifact contamination.
# =====================
def _copy_if_missing(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists() and (not dst.exists()):
        shutil.copy2(src, dst)

for pat in [
    "iter????_oof_predictions.csv",
    "iter????_threshold_sweep.csv",
    "iter????_ablation_summary.csv",
    "iter????_*group_error_summary*.csv",
    f"{TAG}_*candidate*.csv",
]:
    for src in sorted(PROJECT_ROOT.glob(pat)):
        _copy_if_missing(src, ART_ROOT / src.name)

# =====================
# 3) Iteration id (max(log, artifacts, ckpts)+1) — deterministic
# =====================
LOG_PATH = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"

def _extract_iter_num(name: str):
    m = re.search(r"iter(\d{4})", name)
    return int(m.group(1)) if m else None

def max_iteration_id_from_log(path: Path) -> int:
    if not path.exists():
        return -1
    mx = -1
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    if "iteration_id" in obj:
                        mx = max(mx, int(obj["iteration_id"]))
                except Exception:
                    continue
    except Exception:
        pass
    return mx

def max_iter_num_from_artifacts(art_root: Path) -> int:
    mx = -1
    for p in art_root.glob("iter????_*"):
        n = _extract_iter_num(p.name)
        if n is not None:
            mx = max(mx, n)
    return mx

def max_iter_num_from_ckpt(ckpt_root: Path) -> int:
    mx = -1
    for d in ckpt_root.glob("iter????"):
        if d.is_dir():
            n = _extract_iter_num(d.name)
            if n is not None:
                mx = max(mx, n)
    return mx

ITER_ID = max(max_iteration_id_from_log(LOG_PATH), max_iter_num_from_artifacts(ART_ROOT), max_iter_num_from_ckpt(CKPT_ROOT)) + 1
ITER_TAG = f"iter{ITER_ID:04d}"
BLOCK_ID = ITER_ID // 5
BLOCK_POS = ITER_ID % 5
BRANCH = "W4_text+TEHGB_blend"

print(f"ITER_ID={ITER_ID}  ITER_TAG={ITER_TAG}  BLOCK_ID={BLOCK_ID}  BLOCK_POS={BLOCK_POS}  BRANCH={BRANCH}")

# =====================
# 4) Load data + leakage guardrail
# =====================
def dp(name: str) -> Path:
    p = PROJECT_ROOT / name
    if p.exists():
        return p
    p2 = Path("/mnt/data") / name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {name}")

stays_train = pd.read_csv(dp("stays_train.csv"))
stays_test = pd.read_csv(dp("stays_test.csv"))
patients = pd.read_csv(dp("patients.csv"))
with open(dp("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vit_list = json.load(f)

max_day = max((d.get("day", 0) for item in vit_list for d in item.get("days", [])), default=0)
assert max_day <= 10, f"Leakage risk: vitals max(day)={max_day} (>10). ABORT."

vit_map = {int(item["stay_id"]): item.get("days", []) for item in vit_list}

# =====================
# 5) Read history + load P*
# =====================
def load_log_records(log_path: Path):
    out = []
    if not log_path.exists():
        return out
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                out.append(json.loads(line))
            except Exception:
                continue
    return out

def load_pstar(ckpt_root: Path, log_path: Path):
    pstar_path = ckpt_root / "PSTAR.json"
    if pstar_path.exists():
        try:
            obj = json.loads(pstar_path.read_text(encoding="utf-8"))
            p = obj.get("pstar")
            if p:
                return p
        except Exception:
            pass

    # fallback: best oof_macro_f1 in log
    best = None
    recs = load_log_records(log_path)
    for obj in recs:
        it_tag = obj.get("iteration_tag") or (f"iter{int(obj.get('iteration_id')):04d}" if obj.get("iteration_id") is not None else None)
        metrics = obj.get("metrics", {})
        if not isinstance(metrics, dict) or it_tag is None:
            continue
        macro = metrics.get("oof_macro_f1") or metrics.get("oof_macro_f1_num")
        if macro is None:
            continue
        cand = {
            "iteration_tag": it_tag,
            "oof_macro_f1": float(macro),
            "threshold": float(metrics.get("threshold", 0.5)),
            "fold_std": float(metrics.get("fold_std", np.nan)) if metrics.get("fold_std") is not None else np.nan,
            "pos_rate": float(metrics.get("pos_rate", metrics.get("oof_pos_rate", np.nan))) if metrics.get("pos_rate") is not None else np.nan,
            "f1_0": float((metrics.get("per_class_f1", {}) or {}).get("f1_0", np.nan)),
            "f1_1": float((metrics.get("per_class_f1", {}) or {}).get("f1_1", np.nan)),
            "confusion_matrix": metrics.get("confusion_matrix", None),
        }
        if (best is None) or (cand["oof_macro_f1"] > best["oof_macro_f1"]):
            best = cand
    return best or {"iteration_tag": "NONE", "oof_macro_f1": -1.0, "threshold": 0.5, "fold_std": np.nan}

recs = load_log_records(LOG_PATH)
pstar = load_pstar(CKPT_ROOT, LOG_PATH)

print("[DATA] Train prevalence(y=1):", round(float(stays_train["discharge_ready_day11"].mean()), 3))
print("[LOG] Historical records found:", len(recs))
print("=== P* (loaded) ===")
print(json.dumps(pstar, indent=2, ensure_ascii=False))

# =====================
# 6) Successor context summary (detailed, concrete, explainable)
# =====================
REAL_TIMELINE = [
    {"iter": "FAIL_CASE", "real_f1": 0.6765, "note": "artifact contamination / wrong run context"},
    {"iter": "iter0015", "real_f1": 0.7746, "note": "recovered baseline"},
    {"iter": "iter0016", "real_f1": 0.7903, "note": "derived indices added"},
    {"iter": "iter0017", "real_f1": 0.7906, "note": "interaction-derived vitals; C sweep"},
    {"iter": "iter0018", "real_f1": 0.7965, "note": "best so far; OOF macro≈0.8010 @ thr≈0.58"},
]

def _to_markdown_safe(df: pd.DataFrame, n=15):
    try:
        return df.head(n).to_markdown(index=False)
    except Exception:
        return df.head(n).to_string(index=False)

hist_rows = []
for obj in recs:
    it_id = obj.get("iteration_id", None)
    it_tag = obj.get("iteration_tag") or (f"iter{int(it_id):04d}" if it_id is not None else None)
    metrics = obj.get("metrics", {})
    if it_tag is None or not isinstance(metrics, dict):
        continue
    macro = metrics.get("oof_macro_f1") or metrics.get("oof_macro_f1_num")
    thr = metrics.get("threshold")
    pos = metrics.get("pos_rate")
    std = metrics.get("fold_std")
    branch = obj.get("branch", "")
    phase = obj.get("phase", "")
    ts = obj.get("timestamp", "")
    if macro is None:
        continue
    hist_rows.append({
        "iter": it_tag,
        "timestamp": ts,
        "phase": phase,
        "branch": branch,
        "oof_macro_f1": float(macro),
        "threshold": float(thr) if thr is not None else np.nan,
        "pos_rate": float(pos) if pos is not None else np.nan,
        "fold_std": float(std) if std is not None else np.nan,
    })

hist_df = pd.DataFrame(hist_rows).sort_values(["oof_macro_f1", "fold_std"], ascending=[False, True]).reset_index(drop=True)
SUCCESSOR_MD = ART_ROOT / "SUCCESSOR_CONTEXT_AND_PLAN.md"
succ = []
succ.append(f"# {TAG} — Successor context + plan\nGenerated: {utc_ts()}\n")
succ.append("## Mission\nPredict **discharge_ready_day11** (binary). Primary metric: **Macro-F1**.\n")
succ.append("## Non-negotiable guardrails (validated as the only way we stay alive)\n"
            "- Fixed CV: StratifiedKFold(5, shuffle=True, random_state=42)\n"
            "- Leakage: **NO day11 vitals**. Enforce `max(day) <= 10` before any feature build.\n"
            "- Artifact hygiene: discovery is scoped under artifacts/checkpoints for this TAG.\n"
            "- Every iter must output: changes, metric deltas, risks, fallback.\n")
succ.append("## Current best anchor (P*)\n```json\n" + json.dumps(pstar, indent=2, ensure_ascii=False) + "\n```\n")
succ.append("## Real-score timeline (provided by HM/user)\n" + _to_markdown_safe(pd.DataFrame(REAL_TIMELINE), 50) + "\n")
if len(hist_df):
    succ.append("## Best OOF history from jsonl (top 15)\n" + _to_markdown_safe(hist_df, 15) + "\n")
succ.append("## What we learned (high-signal)\n"
            "### ✅ Behaviors that worked\n"
            "- Reversible and auditable iterations (small deltas, always keep rollback path)\n"
            "- Scoped artifacts (avoids the 0.6765 catastrophe)\n"
            "- Prevalence-aware threshold policy (prevents FP inflation)\n"
            "- Derived indices (MAP, shock index, pulse pressure, temp deviation) helped generalization\n\n"
            "### ❌ Pitfalls we must not repeat\n"
            "- Cross-version artifact discovery (rglob '.') → wrong P*/wrong sweep → real collapse\n"
            "- TF-IDF / sparse explosion on ~1k train → overfit and unstable\n"
            "- pos_rate drift too high → FP inflation → class0 F1 collapses\n"
            "- Per-day feature alignment by position instead of explicit day\n\n"
            "### Why iter0019 stalled\n"
            "- LR variants (zip3 on/off, nearby C) produced almost identical OOF → ensembles were highly correlated → no gain.\n"
            "- Conclusion: we need **model diversity**, not another LR micro-variant.\n")
succ.append("## This iteration plan (W4)\n"
            "- Add **TEXT model**: small capped TF-IDF on `note_last2` (orthogonal signal)\n"
            "- Add **Non-linear model**: HGB on dense features + TargetEncoded categories (captures interactions)\n"
            "- Search simple OOF blends; keep the same threshold policies + hard gate vs P*.\n"
            "- If gate fails: do NOT update PSTAR; optionally rollback for submission.\n")
SUCCESSOR_MD.write_text("\n".join(succ), encoding="utf-8")
print("✅ Wrote successor context summary ->", SUCCESSOR_MD)

# =====================
# 7) Feature engineering (DV2 + controlled day-aligned features + keep text columns)
# =====================
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]

KEYWORDS = [
    "stable","improving","worsening","afebrile","fever","chills","hypotensive","tachy",
    "culture","antibiotic","abx","iv","oxygen","weaned","dyspnea","cough",
    "pain","nausea","vomit","n/v","ambulated","pt","ot","telemetry","fluids","room air","sats",
    "discharge","transfer",
]
KW_PATTERNS = [(kw, re.compile(re.escape(kw), re.IGNORECASE)) for kw in KEYWORDS]

def safe_nan(fn, arr, default=np.nan):
    try:
        with np.errstate(all="ignore"):
            return fn(arr)
    except Exception:
        return default

def build_features(stays_df: pd.DataFrame, include_keywords: bool = True, include_day_features: bool = True) -> pd.DataFrame:
    base = stays_df.merge(patients, on="patient_id", how="left")
    rows = []
    for sid in base["stay_id"].astype(int).tolist():
        days = sorted(vit_map.get(int(sid), []), key=lambda x: x.get("day", 0))
        day_idx = np.array([d.get("day", np.nan) for d in days], dtype=float)
        vit = {c: np.array([d.get(c, np.nan) for d in days], dtype=float) for c in VITAL_COLS}

        # derived indices per day
        pulse_pressure = vit["sbp"] - vit["dbp"]
        map_est = (vit["sbp"] + 2.0 * vit["dbp"]) / 3.0
        shock_index = vit["hr"] / vit["sbp"]
        shock_index = np.where(np.isfinite(shock_index), shock_index, np.nan)
        temp_dev = vit["temp_c"] - 37.0

        series = {
            "hr": vit["hr"],
            "sbp": vit["sbp"],
            "dbp": vit["dbp"],
            "rr": vit["rr"],
            "temp_c": vit["temp_c"],
            "pulse_pressure": pulse_pressure,
            "map": map_est,
            "shock_index": shock_index,
            "temp_dev": temp_dev,
        }

        notes = [(d.get("note") or "") for d in days]
        note_all = " ".join(notes).strip()
        note_last2 = " ".join(notes[-2:]).strip()
        note_last1 = (notes[-1] if notes else "").strip()

        feat = {
            "stay_id": int(sid),
            "note_all": note_all,
            "note_last2": note_last2,
            "note_last1": note_last1,
            "note_len_all": len(note_all),
            "note_len_last2": len(note_last2),
            "note_len_last1": len(note_last1),
            "note_days": int(sum(1 for n in notes if str(n).strip() != "")),
            "days_recorded": int(len(days)),
            "day_first": float(day_idx[0]) if len(day_idx) else np.nan,
            "day_last": float(day_idx[-1]) if len(day_idx) else np.nan,
        }

        # DV2 summary + trajectory stats
        for name, vals in series.items():
            vals = np.array(vals, dtype=float)
            diffs = np.diff(vals) if len(vals) >= 2 else np.array([], dtype=float)

            feat[f"{name}_mean"] = safe_nan(np.nanmean, vals)
            feat[f"{name}_std"]  = safe_nan(np.nanstd,  vals)
            feat[f"{name}_min"]  = safe_nan(np.nanmin,  vals)
            feat[f"{name}_max"]  = safe_nan(np.nanmax,  vals)
            feat[f"{name}_median"] = safe_nan(np.nanmedian, vals)

            feat[f"{name}_first"] = float(vals[0]) if len(vals) else np.nan
            feat[f"{name}_last"]  = float(vals[-1]) if len(vals) else np.nan

            last2 = vals[-2:] if len(vals) >= 2 else vals
            early = vals[:8] if len(vals) >= 8 else (vals[:-2] if len(vals) > 2 else vals)
            feat[f"{name}_last2_mean"] = safe_nan(np.nanmean, last2)
            feat[f"{name}_early_mean"] = safe_nan(np.nanmean, early)
            feat[f"{name}_delta_last2_early"] = feat[f"{name}_last2_mean"] - feat[f"{name}_early_mean"]
            feat[f"{name}_delta_last_first"] = feat[f"{name}_last"] - feat[f"{name}_first"]

            mask = (~np.isnan(vals)) & (~np.isnan(day_idx))
            feat[f"{name}_slope"] = np.polyfit(day_idx[mask], vals[mask], 1)[0] if mask.sum() >= 2 else np.nan

            feat[f"{name}_diff_mean"] = safe_nan(np.nanmean, diffs) if len(diffs) else np.nan
            feat[f"{name}_diff_std"]  = safe_nan(np.nanstd,  diffs) if len(diffs) else np.nan
            feat[f"{name}_last3_std"] = safe_nan(np.nanstd, vals[-3:]) if len(vals) >= 3 else safe_nan(np.nanstd, vals)

            feat[f"{name}_missing"] = int(np.isnan(vals).sum())
            feat[f"{name}_n_obs"]   = int(np.sum(~np.isnan(vals)))

        # simple clinical threshold counts
        hr, sbp, dbp, temp, rr = vit["hr"], vit["sbp"], vit["dbp"], vit["temp_c"], vit["rr"]
        feat["hr_gt100"] = int(np.nansum(hr > 100))
        feat["hr_lt60"]  = int(np.nansum(hr < 60))
        feat["sbp_lt90"] = int(np.nansum(sbp < 90))
        feat["sbp_gt160"] = int(np.nansum(sbp > 160))
        feat["dbp_lt60"] = int(np.nansum(dbp < 60))
        feat["temp_gt38"] = int(np.nansum(temp > 38))
        feat["temp_lt36"] = int(np.nansum(temp < 36))
        feat["rr_gt22"] = int(np.nansum(rr > 22))
        feat["map_lt65"] = int(np.nansum(map_est < 65))
        feat["shock_gt1"] = int(np.nansum(shock_index > 1.0))
        feat["pp_lt30"] = int(np.nansum(pulse_pressure < 30))

        # NEW but controlled: per-day features, explicit day alignment
        if include_day_features:
            dmap = {int(d.get("day", -1)): d for d in days if "day" in d}
            for day in range(1, 11):
                rec = dmap.get(day, {})
                for c in VITAL_COLS:
                    feat[f"{c}_d{day:02d}"] = float(rec.get(c, np.nan)) if rec.get(c, None) is not None else np.nan

                sbp_d = feat[f"sbp_d{day:02d}"]
                dbp_d = feat[f"dbp_d{day:02d}"]
                hr_d  = feat[f"hr_d{day:02d}"]
                temp_d= feat[f"temp_c_d{day:02d}"]

                pp = (sbp_d - dbp_d) if (np.isfinite(sbp_d) and np.isfinite(dbp_d)) else np.nan
                mp = ((sbp_d + 2.0*dbp_d)/3.0) if (np.isfinite(sbp_d) and np.isfinite(dbp_d)) else np.nan
                si = (hr_d / sbp_d) if (np.isfinite(hr_d) and np.isfinite(sbp_d) and sbp_d != 0) else np.nan
                td = (temp_d - 37.0) if np.isfinite(temp_d) else np.nan

                feat[f"pp_d{day:02d}"] = float(pp) if np.isfinite(pp) else np.nan
                feat[f"map_d{day:02d}"] = float(mp) if np.isfinite(mp) else np.nan
                feat[f"si_d{day:02d}"] = float(si) if np.isfinite(si) else np.nan
                feat[f"tempdev_d{day:02d}"] = float(td) if np.isfinite(td) else np.nan

        # keyword counts (validated low-dimensional text proxy)
        if include_keywords:
            for kw, pat in KW_PATTERNS:
                key = kw.replace(" ", "_").replace("/", "_")
                feat[f"all_kw_{key}"] = int(len(pat.findall(note_all)))
                feat[f"last2_kw_{key}"] = int(len(pat.findall(note_last2)))
                feat[f"last1_kw_{key}"] = int(len(pat.findall(note_last1)))

        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    return base.merge(feat_df, on="stay_id", how="left")

print("=== [FE] Building features ===")
train_full = build_features(stays_train, include_keywords=True, include_day_features=True)
test_full  = build_features(stays_test,  include_keywords=True, include_day_features=True)

y = train_full["discharge_ready_day11"].astype(int).values
train_prev = float(y.mean())
print("[FE] Train prevalence:", round(train_prev, 3), "| n=", len(y))

# =====================
# 8) Feature sets + preprocessors
# =====================
ID_COLS = ["stay_id", "patient_id"]
CAT_COLS = ["unit_type", "sex", "insurance", "zip3", "admission_reason"]
TEXT_COLS = ["note_all", "note_last2", "note_last1"]

train_X = train_full.drop(columns=["discharge_ready_day11"]).copy()
test_X  = test_full.copy()

for c in CAT_COLS:
    train_X[c] = train_X[c].astype(str)
    test_X[c]  = test_X[c].astype(str)
for c in TEXT_COLS:
    train_X[c] = train_X[c].fillna("").astype(str)
    test_X[c]  = test_X[c].fillna("").astype(str)

def make_preprocessor_lr(df: pd.DataFrame, include_day: bool, include_zip3: bool = True):
    cats = CAT_COLS.copy()
    if not include_zip3 and "zip3" in cats:
        cats = [c for c in cats if c != "zip3"]
    drop = set(ID_COLS + cats + TEXT_COLS)
    num_cols = [c for c in df.columns if c not in drop]
    if not include_day:
        num_cols = [c for c in num_cols if not re.search(r"_d\d{2}$", c)]
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler(with_mean=False)),
            ]), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cats),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    schema = {"num_cols": num_cols, "cat_cols": cats, "include_day": include_day, "include_zip3": include_zip3}
    return pre, schema

def make_preprocessor_text(df: pd.DataFrame, max_features: int = 1800, use_cats: bool = False):
    # keep tiny numeric anchors to help calibration; avoid adding structured vitals here for diversity
    text_col = "note_last2"
    num_feats = [c for c in ["age", "note_len_last2", "note_days", "days_recorded"] if c in df.columns]
    cats = ["unit_type", "admission_reason"] if use_cats else []
    pre = ColumnTransformer(
        transformers=[
            ("tfidf", TfidfVectorizer(
                max_features=max_features,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
                strip_accents="unicode",
            ), text_col),
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler(with_mean=False)),
            ]), num_feats),
        ] + ([("cat", OneHotEncoder(handle_unknown="ignore"), cats)] if cats else []),
        remainder="drop",
        sparse_threshold=0.3
    )
    schema = {"text_col": text_col, "max_features": max_features, "ngram_range": (1, 2), "min_df": 2, "num_feats": num_feats, "cats": cats}
    return pre, schema

class TargetEncoder(BaseEstimator, TransformerMixin):
    """
    Target mean encoding with smoothing.
    IMPORTANT: Must be inside CV folds / Pipeline to avoid leakage.
    Robust to ColumnTransformer passing either DataFrame or ndarray.
    """
    def __init__(self, smoothing: float = 20.0):
        self.smoothing = float(smoothing)
        self.global_mean_ = None
        self.maps_ = {}
        self.cols_ = None

    def fit(self, X, y):
        y_arr = np.asarray(y).astype(float)
        self.global_mean_ = float(np.mean(y_arr))
        self.maps_ = {}

        if isinstance(X, pd.DataFrame):
            self.cols_ = list(X.columns)
            X_df = X.copy()
        else:
            X_arr = np.asarray(X)
            self.cols_ = [f"col{i}" for i in range(X_arr.shape[1])]
            X_df = pd.DataFrame(X_arr, columns=self.cols_)

        for col in self.cols_:
            s = X_df[col].astype(str).fillna("__MISSING__")
            stats = pd.DataFrame({"cat": s, "y": y_arr}).groupby("cat")["y"].agg(["mean", "count"])
            smooth = (stats["count"] * stats["mean"] + self.smoothing * self.global_mean_) / (stats["count"] + self.smoothing)
            self.maps_[col] = smooth.to_dict()
        return self

    def transform(self, X):
        if self.global_mean_ is None:
            raise RuntimeError("TargetEncoder not fitted")
        if isinstance(X, pd.DataFrame):
            X_df = X.copy()
            # if columns mismatch for any reason, align by position
            if self.cols_ is not None and list(X_df.columns) != list(self.cols_):
                X_df = X_df.iloc[:, :len(self.cols_)].copy()
                X_df.columns = list(self.cols_)
        else:
            X_arr = np.asarray(X)
            X_df = pd.DataFrame(X_arr, columns=list(self.cols_))

        out = np.zeros((len(X_df), len(self.cols_)), dtype=float)
        for j, col in enumerate(self.cols_):
            mp = self.maps_.get(col, {})
            s = X_df[col].astype(str).fillna("__MISSING__")
            out[:, j] = s.map(mp).fillna(self.global_mean_).astype(float).values
        return out

def make_preprocessor_hgb_te(df: pd.DataFrame, include_day: bool = True, smoothing: float = 25.0):
    cats = CAT_COLS.copy()
    drop = set(ID_COLS + cats + TEXT_COLS)
    num_cols = [c for c in df.columns if c not in drop]
    if not include_day:
        num_cols = [c for c in num_cols if not re.search(r"_d\d{2}$", c)]
    pre = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num_cols),
            ("te", TargetEncoder(smoothing=smoothing), cats),
        ],
        remainder="drop",
        sparse_threshold=0.0
    )
    schema = {"num_cols": num_cols, "cat_cols": cats, "include_day": include_day, "smoothing": smoothing}
    return pre, schema

# =====================
# 9) CV, sweeps, threshold policies
# =====================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = list(skf.split(train_X, y))  # deterministic splits
fold_ids = np.zeros(len(y), dtype=int)
for f, (_, va_idx) in enumerate(splits):
    fold_ids[va_idx] = f

def threshold_sweep(oof_proba: np.ndarray, y_true: np.ndarray, fold_ids: np.ndarray):
    thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
    rows = []
    for t in thresholds:
        pred = (oof_proba >= t).astype(int)
        cm = confusion_matrix(y_true, pred)
        f1s = f1_score(y_true, pred, average=None)
        macro = f1_score(y_true, pred, average="macro")
        fold_scores = []
        for f in range(N_SPLITS):
            idx = fold_ids == f
            fold_scores.append(f1_score(y_true[idx], pred[idx], average="macro"))
        rows.append({
            "threshold": float(t),
            "macro_f1": float(macro),
            "f1_0": float(f1s[0]),
            "f1_1": float(f1s[1]),
            "pos_rate": float(pred.mean()),
            "fold_mean": float(np.mean(fold_scores)),
            "fold_std": float(np.std(fold_scores)),
            "tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        })
    return pd.DataFrame(rows)

def choose_threshold_prev_first(df_sweep: pd.DataFrame, prevalence: float, pos_tol: float = 0.07, macro_drop_tol: float = 0.005):
    best_macro = float(df_sweep["macro_f1"].max())
    df_pos_ok = df_sweep.loc[(df_sweep["pos_rate"] - prevalence).abs() <= pos_tol].copy()
    pool = df_pos_ok if len(df_pos_ok) else df_sweep.copy()
    pool_best = float(pool["macro_f1"].max())
    df_near = pool.loc[pool["macro_f1"] >= (pool_best - macro_drop_tol)].copy()
    df_near["pos_gap"] = (df_near["pos_rate"] - prevalence).abs()
    df_near = df_near.sort_values(["pos_gap", "fold_std", "macro_f1", "threshold"],
                                  ascending=[True, True, False, True]).reset_index(drop=True)
    chosen = df_near.iloc[0].to_dict()
    chosen["best_macro_overall"] = best_macro
    chosen["policy"] = "prev_first"
    return chosen

def choose_threshold_macro_first(df_sweep: pd.DataFrame, prevalence: float):
    df = df_sweep.copy()
    df["pos_gap"] = (df["pos_rate"] - prevalence).abs()
    df = df.sort_values(["macro_f1", "fold_std", "pos_gap", "threshold"],
                        ascending=[False, True, True, True]).reset_index(drop=True)
    chosen = df.iloc[0].to_dict()
    chosen["best_macro_overall"] = float(df_sweep["macro_f1"].max())
    chosen["policy"] = "macro_first"
    return chosen

def oof_predict_proba(pipe: Pipeline, X: pd.DataFrame, y: np.ndarray, splits):
    oof = np.zeros(len(y), dtype=float)
    for tr_idx, va_idx in splits:
        model = clone(pipe)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    return oof

# =====================
# 10) Candidate models (diverse) + blends
# =====================
# BASE (anchor)
pre_base, schema_base = make_preprocessor_lr(train_X, include_day=False, include_zip3=True)
pipe_base = Pipeline([
    ("preprocess", pre_base),
    ("clf", LogisticRegression(
        solver="liblinear",
        penalty="l1",
        C=0.08,
        class_weight=None,
        max_iter=5000,
        random_state=SEED
    ))
])

# TEXT (small TF-IDF) — keep capped to avoid the known overfit pitfall
tfidf_cap = 1200 if FAST_MODE else 2000
pre_text, schema_text = make_preprocessor_text(train_X, max_features=tfidf_cap, use_cats=False)
pipe_text = Pipeline([
    ("preprocess", pre_text),
    ("clf", LogisticRegression(
        solver="saga",
        penalty="l2",
        C=2.0,
        class_weight=None,
        max_iter=(3500 if FAST_MODE else 7000),
        random_state=SEED
    ))
])

# HGB + TargetEncoding (dense, non-linear interactions)
pre_hgb, schema_hgb = make_preprocessor_hgb_te(train_X, include_day=True, smoothing=25.0)
pipe_hgb = Pipeline([
    ("preprocess", pre_hgb),
    ("clf", HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=3,
        max_iter=(120 if FAST_MODE else 350),
        min_samples_leaf=20,
        l2_regularization=0.0,
        random_state=SEED
    ))
])

print("=== [OOF] Training diverse base models (fixed CV) ===")
oof_base = oof_predict_proba(pipe_base, train_X, y, splits)
oof_text = oof_predict_proba(pipe_text, train_X, y, splits)
oof_hgb  = oof_predict_proba(pipe_hgb,  train_X, y, splits)

auc_base = float(roc_auc_score(y, oof_base))
auc_text = float(roc_auc_score(y, oof_text))
auc_hgb  = float(roc_auc_score(y, oof_hgb))

def score_candidate(name: str, oof_proba: np.ndarray, kind: str, policy: str, w_base: float, w_text: float, w_hgb: float):
    sw = threshold_sweep(oof_proba, y, fold_ids)
    ch = choose_threshold_prev_first(sw, train_prev) if policy == "prev_first" else choose_threshold_macro_first(sw, train_prev)
    auc = {"base": auc_base, "text": auc_text, "hgb": auc_hgb}.get(kind, np.nan)
    row = {
        "name": name,
        "type": kind,
        "policy": policy,
        "macro_f1": float(ch["macro_f1"]),
        "best_macro_overall": float(ch["best_macro_overall"]),
        "threshold": float(ch["threshold"]),
        "pos_rate": float(ch["pos_rate"]),
        "pos_gap": float(abs(float(ch["pos_rate"]) - train_prev)),
        "fold_mean": float(ch["fold_mean"]),
        "fold_std": float(ch["fold_std"]),
        "f1_0": float(ch["f1_0"]),
        "f1_1": float(ch["f1_1"]),
        "tn": int(ch["tn"]), "fp": int(ch["fp"]), "fn": int(ch["fn"]), "tp": int(ch["tp"]),
        "auc": float(auc),
        "w_base": float(w_base),
        "w_text": float(w_text),
        "w_hgb": float(w_hgb),
    }
    return row, sw

candidates = []
sweep_store = {}

for pol in ["prev_first", "macro_first"]:
    r, sw = score_candidate("BASE_LR_sum", oof_base, "base", pol, 1.0, 0.0, 0.0)
    candidates.append(r); sweep_store[(r["name"], pol)] = sw
    r, sw = score_candidate("TEXT_LR_tfidf_last2", oof_text, "text", pol, 0.0, 1.0, 0.0)
    candidates.append(r); sweep_store[(r["name"], pol)] = sw
    r, sw = score_candidate("HGB_TE_dense", oof_hgb, "hgb", pol, 0.0, 0.0, 1.0)
    candidates.append(r); sweep_store[(r["name"], pol)] = sw

# Blends (simple, auditable). The point is to reduce correlation and break LR-local-optimum.
W_GRID = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
for w in W_GRID:
    oof_bt = w * oof_base + (1 - w) * oof_text
    for pol in ["prev_first", "macro_first"]:
        r, sw = score_candidate(f"ENS_base{w:.1f}_text{1-w:.1f}", oof_bt, "ens2", pol, w, 1-w, 0.0)
        candidates.append(r); sweep_store[(r["name"], pol)] = sw

    oof_bh = w * oof_base + (1 - w) * oof_hgb
    for pol in ["prev_first", "macro_first"]:
        r, sw = score_candidate(f"ENS_base{w:.1f}_hgb{1-w:.1f}", oof_bh, "ens2", pol, w, 0.0, 1-w)
        candidates.append(r); sweep_store[(r["name"], pol)] = sw

# 3-way equal blend
oof_bth = (oof_base + oof_text + oof_hgb) / 3.0
for pol in ["prev_first", "macro_first"]:
    r, sw = score_candidate("ENS3_equal", oof_bth, "ens3", pol, 1/3, 1/3, 1/3)
    candidates.append(r); sweep_store[(r["name"], pol)] = sw

abl_df = pd.DataFrame(candidates)
abl_df["delta_vs_pstar_macro"] = abl_df["macro_f1"] - float(pstar.get("oof_macro_f1", -1.0))
pstar_std = float(pstar.get("fold_std", np.nan)) if pstar.get("fold_std") is not None else np.nan
abl_df["delta_vs_pstar_std"] = abl_df["fold_std"] - pstar_std

abl_df = abl_df.sort_values(["macro_f1", "fold_std", "pos_gap"], ascending=[False, True, True]).reset_index(drop=True)

cols_show = [
    "name","type","macro_f1","delta_vs_pstar_macro","threshold","pos_rate","pos_gap",
    "fold_mean","fold_std","delta_vs_pstar_std","f1_0","f1_1","auc","policy","w_base","w_text","w_hgb"
]
print("\n=== Candidate leaderboard (top 12) ===")
print(abl_df[cols_show].head(12).to_string(index=False))

ABL_PATH = ART_ROOT / f"{ITER_TAG}_ablation_summary.csv"
abl_df.to_csv(ABL_PATH, index=False)
print("✅ Saved ablation summary ->", ABL_PATH)

# =====================
# 11) Gate vs P* + selection
# =====================
MIN_DELTA, STD_TOL, POS_TOL = 0.001, 0.01, 0.07
pstar_macro = float(pstar.get("oof_macro_f1", -1.0))

def passes_gate(r):
    if float(r["macro_f1"]) < pstar_macro + MIN_DELTA:
        return False
    if not np.isnan(pstar_std) and float(r["fold_std"]) > float(pstar_std) + STD_TOL:
        return False
    if abs(float(r["pos_rate"]) - train_prev) > POS_TOL:
        return False
    return True

eligible = [r for r in abl_df.to_dict("records") if passes_gate(r)]
if eligible:
    selected = sorted(eligible, key=lambda d: float(d["macro_f1"]), reverse=True)[0]
    decision = "SELECT_CANDIDATE (passes gate vs P*)"
else:
    selected = abl_df.iloc[0].to_dict()
    decision = "NO_CANDIDATE_PASSES_GATE"

print("\n=== Selection decision ===")
print("Decision:", decision)
print("Selected:", selected["name"], "| type=", selected["type"], "| macro_f1=", round(float(selected["macro_f1"]), 6),
      "| thr=", selected["threshold"], "| pos_rate=", round(float(selected["pos_rate"]), 3),
      "| weights=", (selected.get("w_base", 0), selected.get("w_text", 0), selected.get("w_hgb", 0)),
      "| policy=", selected["policy"])

# Selected threshold for candidate evaluation
thr_sel = float(selected["threshold"])

# =====================
# 12) Submission strategy (safe rollback if gate fails)
# =====================
SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"
CAND_PATH = ART_ROOT / f"{TAG}_{BRANCH}_block{BLOCK_ID}_iter{ITER_ID:04d}_candidate.csv"

pstar_iter_tag = str(pstar.get("iteration_tag", "NONE"))
pstar_pipe_path = CKPT_ROOT / pstar_iter_tag / "pipeline.joblib"

use_rollback_for_submission = SAFE_SUBMISSION and (not eligible) and pstar_pipe_path.exists()
submission_mode = "rollback_to_pstar" if use_rollback_for_submission else "use_candidate"

# =====================
# 13) Fit full models (for audit & for candidate submission when allowed)
# =====================
class BlendedModel(BaseEstimator):
    """
    A picklable, auditable blended predictor.
    Fits each component pipeline only if its weight > 0.
    """
    def __init__(self, base_pipe=None, text_pipe=None, hgb_pipe=None, w_base=1.0, w_text=0.0, w_hgb=0.0):
        self.base_pipe = base_pipe
        self.text_pipe = text_pipe
        self.hgb_pipe = hgb_pipe
        self.w_base = float(w_base)
        self.w_text = float(w_text)
        self.w_hgb = float(w_hgb)

    def fit(self, X, y):
        if self.w_base > 0 and self.base_pipe is not None:
            self.base_pipe.fit(X, y)
        if self.w_text > 0 and self.text_pipe is not None:
            self.text_pipe.fit(X, y)
        if self.w_hgb > 0 and self.hgb_pipe is not None:
            self.hgb_pipe.fit(X, y)
        return self

    def predict_proba(self, X):
        # Always compute in a stable way; if weight is 0, skip component
        denom = self.w_base + self.w_text + self.w_hgb
        if denom <= 0:
            raise ValueError("Invalid blend: sum of weights must be > 0")
        proba = np.zeros(len(X), dtype=float)

        if self.w_base > 0:
            proba += self.w_base * self.base_pipe.predict_proba(X)[:, 1]
        if self.w_text > 0:
            proba += self.w_text * self.text_pipe.predict_proba(X)[:, 1]
        if self.w_hgb > 0:
            proba += self.w_hgb * self.hgb_pipe.predict_proba(X)[:, 1]

        proba = proba / denom
        # return 2-col probability matrix
        return np.vstack([1 - proba, proba]).T

# Fit candidate blended model (even if submission rolls back — for audit)
blend_model = BlendedModel(
    base_pipe=clone(pipe_base),
    text_pipe=clone(pipe_text),
    hgb_pipe=clone(pipe_hgb),
    w_base=float(selected.get("w_base", 1.0)),
    w_text=float(selected.get("w_text", 0.0)),
    w_hgb=float(selected.get("w_hgb", 0.0)),
)

if (not use_rollback_for_submission) or FIT_AND_SAVE_CANDIDATES_EVEN_IF_ROLLBACK:
    print("\n=== [FIT] Fitting candidate blended model on full train (audit) ===")
    blend_model.fit(train_X, y)

# Candidate test proba (for potential use)
test_proba_candidate = None
if (not use_rollback_for_submission) or FIT_AND_SAVE_CANDIDATES_EVEN_IF_ROLLBACK:
    test_proba_candidate = blend_model.predict_proba(test_X)[:, 1]

# Submission proba/threshold
if use_rollback_for_submission:
    print("[SUBMISSION] SAFE MODE -> rollback to P* pipeline:", pstar_pipe_path)
    pipe_pstar = joblib.load(pstar_pipe_path)
    thr_submit = float(pstar.get("threshold", 0.5))
    test_proba_submit = pipe_pstar.predict_proba(test_X)[:, 1]
else:
    thr_submit = thr_sel
    test_proba_submit = test_proba_candidate

test_pred = (test_proba_submit >= thr_submit).astype(int)
sub = pd.DataFrame({"stay_id": test_full["stay_id"].astype(int).values,
                    "discharge_ready_day11": test_pred.astype(int)})

sub.to_csv(SUB_PATH, index=False)
sub.to_csv(CAND_PATH, index=False)

print("\n=== Submission written ===")
print("SUB_PATH:", SUB_PATH)
print("CAND_PATH:", CAND_PATH)
print("submission_mode:", submission_mode)
print("Threshold used (submit):", thr_submit)
print("Submission pos_rate:", float(sub["discharge_ready_day11"].mean()))

# =====================
# 14) Selected candidate OOF artifacts (always candidate, NOT rollback)
# =====================
# Candidate selected OOF proba is the same weights applied to base OOFs.
sel_oof = (float(selected.get("w_base", 0.0)) * oof_base +
           float(selected.get("w_text", 0.0)) * oof_text +
           float(selected.get("w_hgb", 0.0))  * oof_hgb)
den = float(selected.get("w_base", 0.0)) + float(selected.get("w_text", 0.0)) + float(selected.get("w_hgb", 0.0))
sel_oof = sel_oof / den if den > 0 else oof_base

pred_oof = (sel_oof >= thr_sel).astype(int)

OOF_PATH = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
SWEEP_PATH = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"

pd.DataFrame({
    "stay_id": train_full["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "oof_proba": sel_oof.astype(float),
    "fold": fold_ids.astype(int),
}).to_csv(OOF_PATH, index=False)

sel_sw = sweep_store.get((selected["name"], selected["policy"]))
if sel_sw is None:
    sel_sw = threshold_sweep(sel_oof, y, fold_ids)
sel_sw.to_csv(SWEEP_PATH, index=False)

print("✅ Saved OOF ->", OOF_PATH.name)
print("✅ Saved sweep ->", SWEEP_PATH.name)

# =====================
# 15) Diagnostics: group-wise FP/FN (failure explanation)
# =====================
diag = train_full[["stay_id"] + CAT_COLS].copy()
diag["y"] = y
diag["pred"] = pred_oof
diag["proba"] = sel_oof
diag["is_fp"] = (diag["y"] == 0) & (diag["pred"] == 1)
diag["is_fn"] = (diag["y"] == 1) & (diag["pred"] == 0)

def group_err(df: pd.DataFrame, col: str) -> pd.DataFrame:
    g = df.groupby(col).agg(
        n=("stay_id", "count"),
        y_prev=("y", "mean"),
        pred_rate=("pred", "mean"),
        fp=("is_fp", "sum"),
        fn=("is_fn", "sum"),
    ).reset_index()
    neg = np.maximum(1.0, g["n"] - g["y_prev"] * g["n"])
    pos = np.maximum(1.0, g["y_prev"] * g["n"])
    g["fp_rate_in_neg"] = g["fp"] / neg
    g["fn_rate_in_pos"] = g["fn"] / pos
    return g.sort_values(["fp_rate_in_neg", "fn_rate_in_pos"], ascending=False)

g_unit = group_err(diag, "unit_type"); g_unit["group_col"] = "unit_type"
g_ins  = group_err(diag, "insurance"); g_ins["group_col"] = "insurance"
g_adm  = group_err(diag, "admission_reason"); g_adm["group_col"] = "admission_reason"

DIAG_PATH = ART_ROOT / f"{ITER_TAG}_group_error_summary.csv"
pd.concat([g_unit, g_ins, g_adm], ignore_index=True).to_csv(DIAG_PATH, index=False)
print("✅ Saved group diagnostics ->", DIAG_PATH.name)

# =====================
# 16) Save checkpoint bundle (models + config + metrics + decision note)
# =====================
iter_ckpt_dir = CKPT_ROOT / ITER_TAG
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

# Save fitted candidate model (even if submission rolled back)
joblib.dump(blend_model, iter_ckpt_dir / "pipeline.joblib")
# Save components too (easy ablation/replay)
joblib.dump(pipe_base, iter_ckpt_dir / "pipeline_base_template.joblib")
joblib.dump(pipe_text, iter_ckpt_dir / "pipeline_text_template.joblib")
joblib.dump(pipe_hgb,  iter_ckpt_dir / "pipeline_hgb_template.joblib")

cm = confusion_matrix(y, pred_oof).tolist()
f1s = f1_score(y, pred_oof, average=None)
macro = f1_score(y, pred_oof, average="macro")

fold_scores = []
for f in range(N_SPLITS):
    idx = fold_ids == f
    fold_scores.append(float(f1_score(y[idx], pred_oof[idx], average="macro")))

metrics = {
    "oof_macro_f1": float(macro),
    "per_class_f1": {"f1_0": float(f1s[0]), "f1_1": float(f1s[1])},
    "confusion_matrix": cm,
    "threshold_selected": float(thr_sel),
    "pos_rate_selected": float(pred_oof.mean()),
    "fold_scores": fold_scores,
    "fold_mean": float(np.mean(fold_scores)),
    "fold_std": float(np.std(fold_scores)),
    "auc": float(roc_auc_score(y, sel_oof)),
    "train_prevalence": train_prev,
    "gate": {"min_delta": MIN_DELTA, "std_tol": STD_TOL, "pos_tol": POS_TOL, "decision": decision},
    "pstar_reference": pstar,
    "submission": {
        "mode": submission_mode,
        "threshold_submit": float(thr_submit),
        "safe_submission": bool(SAFE_SUBMISSION),
        "rollback_used": bool(use_rollback_for_submission),
        "pstar_pipeline_path": str(pstar_pipe_path) if pstar_pipe_path.exists() else None
    }
}

config = {
    "team": TEAM, "task": TASK, "version": VERSION,
    "iteration_id": ITER_ID, "iteration_tag": ITER_TAG,
    "branch": BRANCH, "block_id": BLOCK_ID, "block_pos": BLOCK_POS,
    "seed": SEED,
    "validation": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "selected": selected,
    "schemas": {"base": schema_base, "text": schema_text, "hgb_te": schema_hgb},
    "notes": [
        "New angle: model diversity beyond LR micro-variants (iter0019 stall diagnosis).",
        "TF-IDF is capped (avoid feature explosion pitfall).",
        "Target encoding is inside Pipeline+CV folds to avoid leakage.",
        "Hard gate vs P*; rollback option keeps submissions safe.",
    ],
}

with open(iter_ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("✅ Saved checkpoint ->", iter_ckpt_dir)

# =====================
# 17) Decision note (human readable)
# =====================
def md_table(df: pd.DataFrame, n=12):
    try:
        return df.head(n).to_markdown(index=False)
    except Exception:
        return df.head(n).to_string(index=False)

DECISION_MD = ART_ROOT / f"{ITER_TAG}_DECISION.md"
ts = utc_ts()
md = []
md.append(f"# {TAG} — {ITER_TAG} decision\nGenerated: {ts}\n")
md.append("## P* reference\n```json\n" + json.dumps(pstar, indent=2, ensure_ascii=False) + "\n```\n")
md.append("## Candidate leaderboard (top 12)\n" + md_table(abl_df[cols_show], 12) + "\n")
md.append("## Selected\n```json\n" + json.dumps(selected, indent=2, ensure_ascii=False) + "\n```\n")
md.append("## Gate decision\n- " + decision + "\n")
md.append("## Submission mode\n- " + submission_mode + "\n")
if submission_mode == "rollback_to_pstar":
    md.append(f"- Rolled back to P* pipeline: `{pstar_pipe_path}`\n")
md.append("## What changed (this iter)\n"
          "- Added small TF-IDF LR on note_last2 (capped features)\n"
          "- Added TargetEncoded + HistGradientBoosting on dense vitals/indices (non-linear interactions)\n"
          "- Searched OOF blends (reduce correlation; attempt to break local optimum)\n")
md.append("## Why iter0019 plateaued\n"
          "- LR variants/ensembles were too correlated → no measurable OOF gain.\n"
          "- Therefore we add orthogonal signals (text) + non-linear interactions (HGB).\n")
md.append("## Risks\n"
          "- TF-IDF overfit risk: cap vocab + min_df>=2\n"
          "- Target encoding leakage risk: only safe inside CV pipeline\n"
          "- Blend may drift pos_rate: guard with POS_TOL + confusion matrix tracking\n")
md.append("## Fallback\n"
          "- If candidate fails gate: PSTAR unchanged\n"
          "- If SAFE_SUBMISSION and P* pipeline exists: submission rolls back to P*\n")

DECISION_MD.write_text("\n".join(md), encoding="utf-8")
print("✅ Wrote decision note ->", DECISION_MD)

# =====================
# 18) Update PSTAR.json (only if passes gate)
# =====================
PSTAR_PATH = CKPT_ROOT / "PSTAR.json"
if eligible:
    new_pstar = {
        "iteration_tag": ITER_TAG,
        "oof_macro_f1": float(macro),
        "threshold": float(thr_sel),
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "pos_rate": float(pred_oof.mean()),
        "f1_0": float(f1s[0]),
        "f1_1": float(f1s[1]),
        "confusion_matrix": cm,
    }
    with open(PSTAR_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "tag": TAG,
            "pstar": new_pstar,
            "train_prevalence": train_prev,
            "timestamp": ts,
            "schema_version": 5,
            "notes": "Updated because candidate passed hard gate vs prior P*."
        }, f, indent=2, ensure_ascii=False)
    print("✅ Updated PSTAR.json ->", new_pstar["iteration_tag"], "macro=", round(new_pstar["oof_macro_f1"], 6))
else:
    print("ℹ️ PSTAR.json unchanged (no candidate passed hard gate).")

# =====================
# 19) Append iteration log (append-only with fallback)
# =====================
def ensure_appendable(path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and os.access(path, os.W_OK):
        return path
    if not path.exists():
        try:
            with open(path, "a", encoding="utf-8"):
                pass
            return path
        except Exception:
            pass
    alt = path.with_name(path.stem + "_appended" + path.suffix)
    if path.exists() and (not alt.exists()):
        try:
            shutil.copy2(path, alt)
        except Exception:
            pass
    with open(alt, "a", encoding="utf-8"):
        pass
    return alt

LOG_PATH2 = ensure_appendable(LOG_PATH)

iteration_record = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": ts,
    "phase": "Modeling",
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "data_paths_used": {
        "stays_train": str(dp("stays_train.csv")),
        "stays_test": str(dp("stays_test.csv")),
        "patients": str(dp("patients.csv")),
        "vitals_timeseries": str(dp("vitals_timeseries.json")),
    },
    "join_keys": ["stays.patient_id -> patients.patient_id", "stays.stay_id -> vitals_timeseries.stay_id"],
    "leakage_checks_performed": [
        "Asserted vitals max(day) <= 10 (no day11 leakage).",
        "Target encoding fit occurs inside CV folds via Pipeline (no leakage).",
        "Threshold tuned on OOF only (no test leakage).",
        "Artifacts scoped under artifacts/checkpoints for this TAG (no contamination).",
    ],
    "feature_summary": {
        "base": schema_base,
        "text": schema_text,
        "hgb_te": schema_hgb,
        "train_prevalence": train_prev,
        "note": "This iter targets diversity to break LR-variant plateau.",
    },
    "models_attempted": abl_df[cols_show].head(30).to_dict("records"),
    "selected_model": {
        "name": selected["name"],
        "type": selected["type"],
        "weights": {"w_base": float(selected.get("w_base", 0.0)), "w_text": float(selected.get("w_text", 0.0)), "w_hgb": float(selected.get("w_hgb", 0.0))},
        "threshold_selected": float(thr_sel),
        "policy": selected["policy"],
        "decision": decision,
        "submission_mode": submission_mode,
    },
    "validation_protocol": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "metrics": metrics,
    "artifacts_written": {
        "successor_context_md": str(SUCCESSOR_MD),
        "submission_csv": str(SUB_PATH),
        "candidate_snapshot": str(CAND_PATH),
        "oof_predictions": str(OOF_PATH),
        "threshold_sweep": str(SWEEP_PATH),
        "ablation_summary": str(ABL_PATH),
        "group_error_summary": str(DIAG_PATH),
        "checkpoint_dir": str(iter_ckpt_dir),
        "pstar_path": str(PSTAR_PATH),
        "decision_md": str(DECISION_MD),
    },
    "deltas_vs_prev": {
        "changes": [
            "Added TF-IDF LR (note_last2) with strict feature cap.",
            "Added TargetEncoder + HistGradientBoostingClassifier on dense features.",
            "Added OOF blend search for model diversity.",
            "Added controlled per-day features aligned by explicit day.",
        ],
        "risk": [
            "TF-IDF can overfit; mitigate by max_features cap + min_df.",
            "Target encoding leakage risk; mitigated by fitting inside CV pipeline only.",
            "Blends can drift pos_rate; mitigated by POS_TOL + fold_std gate checks.",
        ],
        "fallback": "If candidate fails gate, do not update P*; if SAFE_SUBMISSION and P* pipeline exists, rollback for submission.",
    },
    "known_defects_limitations": [
        "Blend is simple averaging; stacking could be explored later with stricter audit.",
        "No per-group thresholding in this iter (possible future controlled experiment).",
    ],
    "next_step_hypothesis": (
        "If this still plateaus: try calibrated LinearSVC on TF-IDF (more margin), "
        "or learn per-group thresholds *inside folds* to reduce FP hotspots without leaking."
    ),
    "hm_message": "Successor: escape LR-variant local optimum by adding orthogonal text signal + non-linear interactions, keeping reversible/auditable discipline.",
    "real_score": {},
}

with open(LOG_PATH2, "a", encoding="utf-8") as f:
    f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
print("✅ Appended iteration log ->", LOG_PATH2)

# =====================
# 20) Consultant packet (every iter)
# =====================
consult_packet = {
    "tag": TAG,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": ts,
    "what_changed": iteration_record["deltas_vs_prev"]["changes"],
    "metric_changes": {
        "pstar_reference": pstar,
        "selected_metrics": metrics,
        "gate_decision": decision,
        "submission_mode": submission_mode,
    },
    "suspected_leakage_risks": [
        "Vitals day>10 leakage (guarded by assert).",
        "Target encoding leakage if fit on full data (guarded by CV Pipeline).",
        "Threshold overfit risk (tracked by fold_std + prevalence-aware policy).",
    ],
    "complexity_drift_risks": [
        "TF-IDF vocab explosion (mitigated by max_features cap).",
        "HGB overfit on small data (mitigated by conservative depth/leaf).",
    ],
    "evaluation_alignment_risks": [
        "OOF↔real drift; monitor pos_rate + class0 F1 (FP inflation) explicitly.",
    ],
    "unknown_unknowns": [
        "Distribution shift in note phrasing or measurement patterns across train/test.",
    ],
    "recommendations_for_next_iter": [
        "If TEXT helps: try adding (unit_type, admission_reason) into text model as a controlled delta.",
        "If HGB helps: tune min_samples_leaf and l2_regularization; consider include_day=False ablation for robustness.",
        "If neither helps: try calibrated LinearSVC text model (margin-based) and blend with BASE.",
    ],
}

consult_path = CONSULT_ROOT / f"{TAG}_iter{ITER_ID}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, ensure_ascii=False)
print("✅ Wrote consultant packet ->", consult_path)

print("\nDONE ✅")
print("Key outputs:")
print(" - Successor context:", SUCCESSOR_MD)
print(" - Submission       :", SUB_PATH, "| mode:", submission_mode)
print(" - Candidate        :", CAND_PATH)
print(" - Ablation         :", ABL_PATH)
print(" - OOF              :", OOF_PATH)
print(" - Sweep            :", SWEEP_PATH)
print(" - Group diag       :", DIAG_PATH)
print(" - Decision note    :", DECISION_MD)
print(" - PSTAR            :", PSTAR_PATH)
print(" - Checkpoint dir   :", iter_ckpt_dir)

=== PREFLIGHT ===
PROJECT_ROOT: d:\AgentDs\agent_ds_healthcare
ART_ROOT: d:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT: d:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
CONSULT_ROOT: d:\AgentDs\agent_ds_healthcare\consultant_packets
ITER_ID=20  ITER_TAG=iter0020  BLOCK_ID=4  BLOCK_POS=0  BRANCH=W4_text+TEHGB_blend
[DATA] Train prevalence(y=1): 0.656
[LOG] Historical records found: 18
=== P* (loaded) ===
{
  "iteration_tag": "iter0018",
  "oof_macro_f1": 0.8010438942644934,
  "threshold": 0.58,
  "fold_mean": 0.8008722978857806,
  "fold_std": 0.017551713766308295,
  "pos_rate": 0.691,
  "f1_0": 0.7320061255742726,
  "f1_1": 0.8700816629547142,
  "confusion_matrix": [
    [
      239,
      105
    ],
    [
      70,
      586
    ]
  ]
}
✅ Wrote successor context summary -> d:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3\SUCCESSOR_CONTEXT_AND_PLAN.md
=== [FE] Building features ===
[FE] Train prevalence: 0.656 | n= 1000
=== [OOF] Training diverse base models (

# Iteration 13
- 0.8168

In [54]:
import os, json, re, random, shutil, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.base import clone
import joblib

# =========================================================
# clai_ch3_v3 — Iteration Runner (single cell)
# Branch: W5_stack3_teHGB_tune
#
# Goals (behavior-first, NOT “one trick”):
# - Reversible change: introduce stronger orthogonal learners + meta-stacking (can roll back)
# - Auditable: fixed CV, deterministic folds, scoped artifacts, append-only logs (+ fallback)
# - Attributable: per-candidate ablation table + decision note
# - Failure explainable: group-wise FP/FN diagnostics + pos_rate/CM tracking
#
# Inputs:
# - stays_train.csv, stays_test.csv, patients.csv, vitals_timeseries.json
#
# Outputs every iter:
# - submission CSV (safe: candidate if passes gate else P* rollback)
# - artifacts: oof_predictions.csv, threshold_sweep.csv, ablation_summary.csv,
#              group_error_summary.csv, DECISION.md, candidate snapshot
# - checkpoints: pipeline/joblib bundle + config/metrics/schema
# - logs: append iteration_detail.jsonl (+ fallback if unwritable)
# - consultant packet json (every iter)
# =========================================================

# ---------------------
# 0) Warnings & seeds
# ---------------------
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice")
warnings.filterwarnings("ignore", category=RuntimeWarning)

TEAM, TASK, VERSION = "clai", "ch3", "v3"
TAG = f"{TEAM}_{TASK}_{VERSION}"
SEED = 42
N_SPLITS = 5

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

def utc_ts() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

# ---------------------
# 1) Paths (robust)
# ---------------------
def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / "stays_train.csv").exists():
            return p
    # common fallbacks
    for cand in [Path("d:/AgentDs/agent_ds_healthcare"), Path("/mnt/data")]:
        if (cand / "stays_train.csv").exists():
            return cand.resolve()
    return here

PROJECT_ROOT = find_project_root()

def dp(name: str) -> Path:
    cands = [
        PROJECT_ROOT / name,
        PROJECT_ROOT / "data" / name,
        Path("/mnt/data") / name,
    ]
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {name}. Tried: {[str(p) for p in cands]}")

ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"
ART_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
CONSULT_ROOT.mkdir(parents=True, exist_ok=True)

print("=== PREFLIGHT ===")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_ROOT:", ART_ROOT)
print("CKPT_ROOT:", CKPT_ROOT)
print("CONSULT_ROOT:", CONSULT_ROOT)

# ---------------------
# 2) Bootstrap: copy loose artifacts into scoped dir (idempotent)
# ---------------------
def _copy_if_missing(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists() and (not dst.exists()):
        shutil.copy2(src, dst)

for pat in [
    "iter????_oof_predictions.csv",
    "iter????_threshold_sweep.csv",
    "iter????_ablation_summary.csv",
    "iter????_*group_error_summary*.csv",
    f"{TAG}_*candidate*.csv",
]:
    for src in sorted(PROJECT_ROOT.glob(pat)):
        _copy_if_missing(src, ART_ROOT / src.name)

# ---------------------
# 3) Iteration id (robust): max(log, artifacts, ckpts)+1; avoid collisions
# ---------------------
LOG_PATH_RAW = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"

def ensure_appendable(path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and os.access(path, os.W_OK):
        return path
    if not path.exists():
        try:
            with open(path, "a", encoding="utf-8") as f:
                pass
            return path
        except Exception:
            pass
    alt = path.with_name(path.stem + "_appended" + path.suffix)
    if not alt.exists() and path.exists():
        try:
            shutil.copy2(path, alt)
        except Exception:
            pass
    with open(alt, "a", encoding="utf-8") as f:
        pass
    return alt

LOG_PATH = ensure_appendable(LOG_PATH_RAW)

def _extract_iter_num(s: str):
    m = re.search(r"iter(\d{4})", s)
    return int(m.group(1)) if m else None

def max_iter_from_artifacts(art_root: Path) -> int:
    mx = -1
    for p in art_root.glob("iter????_*"):
        n = _extract_iter_num(p.name)
        if n is not None:
            mx = max(mx, n)
    return mx

def max_iter_from_ckpts(ckpt_root: Path) -> int:
    mx = -1
    for d in ckpt_root.glob("iter????"):
        if d.is_dir():
            n = _extract_iter_num(d.name)
            if n is not None:
                mx = max(mx, n)
    return mx

def max_iter_from_log(log_path: Path) -> int:
    if not log_path.exists():
        return -1
    mx = -1
    try:
        with open(log_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    mx = max(mx, int(obj.get("iteration_id", -1)))
                except Exception:
                    continue
    except Exception:
        pass
    return mx

ITER_ID = max(max_iter_from_log(LOG_PATH), max_iter_from_artifacts(ART_ROOT), max_iter_from_ckpts(CKPT_ROOT)) + 1

# avoid collision if rerun
while True:
    ITER_TAG = f"iter{ITER_ID:04d}"
    if not (CKPT_ROOT / ITER_TAG).exists() and not list(ART_ROOT.glob(f"{ITER_TAG}_*")):
        break
    ITER_ID += 1

BLOCK_ID = ITER_ID // 5
BLOCK_POS = ITER_ID % 5
BRANCH = "W5_stack3_teHGB_tune"

print("LOG_PATH:", LOG_PATH)
print(f"ITER_ID={ITER_ID}  ITER_TAG={ITER_TAG}  BLOCK_ID={BLOCK_ID}  BLOCK_POS={BLOCK_POS}  BRANCH={BRANCH}")

# ---------------------
# 4) Load data + leakage guardrail
# ---------------------
stays_train = pd.read_csv(dp("stays_train.csv"))
stays_test  = pd.read_csv(dp("stays_test.csv"))
patients    = pd.read_csv(dp("patients.csv"))

with open(dp("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vit_list = json.load(f)

max_day = max(d.get("day", 0) for item in vit_list for d in item.get("days", []))
assert max_day <= 10, f"Leakage risk: found vitals day={max_day} (>10). Abort."

vit_map = {int(item["stay_id"]): item.get("days", []) for item in vit_list}

y_all = stays_train["discharge_ready_day11"].astype(int).values
train_prev = float(np.mean(y_all))
print(f"[DATA] Train prevalence(y=1): {train_prev:.3f} | n={len(y_all)}")

# deterministic folds
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = list(skf.split(np.zeros(len(y_all)), y_all))
fold_ids = np.empty(len(y_all), dtype=int)
for f, (_, va_idx) in enumerate(splits):
    fold_ids[va_idx] = f

# ---------------------
# 5) Feature engineering (keep iter0018+ discipline; add raw note text columns for optional text models)
# ---------------------
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]

KEYWORDS = [
    "stable","improving","worsening","afebrile","fever","chills","hypotensive","tachy",
    "culture","antibiotic","abx","iv","oxygen","weaned","dyspnea","cough",
    "pain","nausea","vomit","n/v","ambulated","pt","ot","telemetry","fluids","room air","sats",
    "discharge","transfer"
]
KW_PATTERNS = [(kw, re.compile(re.escape(kw), re.IGNORECASE)) for kw in KEYWORDS]

def safe_nan(fn, arr, default=np.nan):
    try:
        with np.errstate(all="ignore"):
            return fn(arr)
    except Exception:
        return default

def build_features(stays_df: pd.DataFrame, include_keywords: bool = True, include_text: bool = True) -> pd.DataFrame:
    base = stays_df.merge(patients, on="patient_id", how="left")
    rows = []
    for sid in base["stay_id"].astype(int).tolist():
        days = sorted(vit_map.get(int(sid), []), key=lambda x: x.get("day", 0))
        day_idx = np.array([d.get("day", np.nan) for d in days], dtype=float)

        vit = {c: np.array([d.get(c, np.nan) for d in days], dtype=float) for c in VITAL_COLS}

        pulse_pressure = vit["sbp"] - vit["dbp"]
        map_est = (vit["sbp"] + 2.0 * vit["dbp"]) / 3.0
        shock_index = vit["hr"] / vit["sbp"]
        shock_index = np.where(np.isfinite(shock_index), shock_index, np.nan)
        temp_dev = vit["temp_c"] - 37.0

        series = {
            "hr": vit["hr"],
            "sbp": vit["sbp"],
            "dbp": vit["dbp"],
            "rr": vit["rr"],
            "temp_c": vit["temp_c"],
            "pulse_pressure": pulse_pressure,
            "map": map_est,
            "shock_index": shock_index,
            "temp_dev": temp_dev,
        }

        notes = [(d.get("note") or "") for d in days]
        note_all = " ".join(notes).strip()
        note_last2 = " ".join(notes[-2:]).strip()

        feat = {
            "stay_id": int(sid),
            "note_len_all": len(note_all),
            "note_len_last2": len(note_last2),
            "note_days": int(sum(1 for n in notes if str(n).strip() != "")),
            "days_recorded": int(len(days)),
            "day_first": float(day_idx[0]) if len(day_idx) else np.nan,
            "day_last": float(day_idx[-1]) if len(day_idx) else np.nan,
        }
        if include_text:
            feat["note_text_all"] = note_all
            feat["note_text_last2"] = note_last2

        for name, vals in series.items():
            vals = np.array(vals, dtype=float)
            diffs = np.diff(vals) if len(vals) >= 2 else np.array([], dtype=float)

            feat[f"{name}_mean"] = safe_nan(np.nanmean, vals)
            feat[f"{name}_std"]  = safe_nan(np.nanstd,  vals)
            feat[f"{name}_min"]  = safe_nan(np.nanmin,  vals)
            feat[f"{name}_max"]  = safe_nan(np.nanmax,  vals)
            feat[f"{name}_median"] = safe_nan(np.nanmedian, vals)
            feat[f"{name}_first"] = vals[0] if len(vals) else np.nan
            feat[f"{name}_last"]  = vals[-1] if len(vals) else np.nan

            last2 = vals[-2:] if len(vals) >= 2 else vals
            early = vals[:8] if len(vals) >= 8 else (vals[:-2] if len(vals) > 2 else vals)

            feat[f"{name}_last2_mean"] = safe_nan(np.nanmean, last2)
            feat[f"{name}_early_mean"] = safe_nan(np.nanmean, early)
            feat[f"{name}_delta_last2_early"] = feat[f"{name}_last2_mean"] - feat[f"{name}_early_mean"]
            feat[f"{name}_delta_last_first"] = feat[f"{name}_last"] - feat[f"{name}_first"]

            mask = (~np.isnan(vals)) & (~np.isnan(day_idx))
            feat[f"{name}_slope"] = np.polyfit(day_idx[mask], vals[mask], 1)[0] if mask.sum() >= 2 else np.nan

            feat[f"{name}_diff_mean"] = safe_nan(np.nanmean, diffs) if len(diffs) else np.nan
            feat[f"{name}_diff_std"]  = safe_nan(np.nanstd,  diffs) if len(diffs) else np.nan
            feat[f"{name}_last3_std"] = safe_nan(np.nanstd, vals[-3:]) if len(vals) >= 3 else safe_nan(np.nanstd, vals)

            feat[f"{name}_missing"] = int(np.isnan(vals).sum())
            feat[f"{name}_n_obs"]   = int(np.sum(~np.isnan(vals)))

        # simple clinical threshold counts
        hr, sbp, dbp, temp, rr = vit["hr"], vit["sbp"], vit["dbp"], vit["temp_c"], vit["rr"]
        feat["hr_gt100"] = int(np.nansum(hr > 100))
        feat["hr_lt60"]  = int(np.nansum(hr < 60))
        feat["sbp_lt90"] = int(np.nansum(sbp < 90))
        feat["sbp_gt160"] = int(np.nansum(sbp > 160))
        feat["dbp_lt60"] = int(np.nansum(dbp < 60))
        feat["temp_gt38"] = int(np.nansum(temp > 38))
        feat["temp_lt36"] = int(np.nansum(temp < 36))
        feat["rr_gt22"] = int(np.nansum(rr > 22))
        feat["map_lt65"] = int(np.nansum(map_est < 65))
        feat["shock_gt1"] = int(np.nansum(shock_index > 1.0))
        feat["pp_lt30"] = int(np.nansum(pulse_pressure < 30))

        if include_keywords:
            for kw, pat in KW_PATTERNS:
                key = kw.replace(" ", "_").replace("/", "_")
                feat[f"all_kw_{key}"] = len(pat.findall(note_all))
                feat[f"last2_kw_{key}"] = len(pat.findall(note_last2))

        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    return base.merge(feat_df, on="stay_id", how="left")

print("=== [FE] Building features ===")
train_full = build_features(stays_train, include_keywords=True, include_text=True)
test_full  = build_features(stays_test,  include_keywords=True, include_text=True)

y = train_full["discharge_ready_day11"].astype(int).values
assert len(y) == len(y_all)

# ---------------------
# 6) Columns setup
# ---------------------
ID_COLS = [c for c in ["stay_id", "patient_id"] if c in train_full.columns]
CAT_COLS_ALL = [c for c in ["unit_type", "sex", "insurance", "zip3", "admission_reason"] if c in train_full.columns]
TEXT_COLS = [c for c in ["note_text_all", "note_text_last2"] if c in train_full.columns]

train_X_full = train_full.drop(columns=["discharge_ready_day11"]).copy()
test_X_full  = test_full.copy()

# enforce cat types
for c in CAT_COLS_ALL:
    train_X_full[c] = train_X_full[c].astype(str)
    test_X_full[c]  = test_X_full[c].astype(str)

# text series (safe: only from vitals day<=10)
text_train_last2 = train_X_full.get("note_text_last2", pd.Series([""] * len(train_X_full))).fillna("").astype(str)
text_test_last2  = test_X_full.get("note_text_last2", pd.Series([""] * len(test_X_full))).fillna("").astype(str)

text_nonempty = float((text_train_last2.str.len() > 0).mean())
print(f"[FE] text_nonempty_rate(note_last2): {text_nonempty:.3f}")

# ---------------------
# 7) Load P* (prefer PSTAR.json)
# ---------------------
PSTAR_PATH = CKPT_ROOT / "PSTAR.json"

def load_pstar(pstar_path: Path):
    if not pstar_path.exists():
        return None
    try:
        obj = json.loads(pstar_path.read_text(encoding="utf-8"))
        p = obj.get("pstar") or obj.get("PSTAR") or obj.get("best") or None
        return p
    except Exception:
        return None

pstar = load_pstar(PSTAR_PATH)
if pstar is None:
    # fallback: use best sweep among complete tags in ART_ROOT
    best = None
    for sw_p in sorted(ART_ROOT.glob("iter????_threshold_sweep.csv")):
        try:
            sw = pd.read_csv(sw_p)
            if "macro_f1" not in sw.columns:
                continue
            row = sw.sort_values("macro_f1", ascending=False).iloc[0].to_dict()
            cand = {"iteration_tag": sw_p.name.split("_")[0],
                    "oof_macro_f1": float(row["macro_f1"]),
                    "threshold": float(row.get("threshold", 0.5)),
                    "fold_std": float(row.get("fold_std", np.nan))}
            if (best is None) or (cand["oof_macro_f1"] > best["oof_macro_f1"]):
                best = cand
        except Exception:
            continue
    pstar = best or {"iteration_tag": "NONE", "oof_macro_f1": -1.0, "threshold": 0.5, "fold_std": np.nan}

print("=== P* (loaded) ===")
print(json.dumps(pstar, indent=2, ensure_ascii=False))

pstar_iter_tag = str(pstar.get("iteration_tag", "NONE"))
pstar_macro = float(pstar.get("oof_macro_f1", -1.0))
pstar_std = pstar.get("fold_std", np.nan)
try:
    pstar_std = float(pstar_std)
except Exception:
    pstar_std = np.nan

# ---------------------
# 8) Threshold utilities (OOF-only)
# ---------------------
def threshold_sweep(oof_proba: np.ndarray, y_true: np.ndarray, fold_ids_: np.ndarray, thresholds=None) -> pd.DataFrame:
    if thresholds is None:
        thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
    rows = []
    for t in thresholds:
        pred = (oof_proba >= t).astype(int)
        cm = confusion_matrix(y_true, pred)
        f1s = f1_score(y_true, pred, average=None)
        macro = f1_score(y_true, pred, average="macro")
        fold_scores = []
        for f in range(N_SPLITS):
            idx = fold_ids_ == f
            fold_scores.append(float(f1_score(y_true[idx], pred[idx], average="macro")))
        rows.append({
            "threshold": float(t),
            "macro_f1": float(macro),
            "f1_0": float(f1s[0]),
            "f1_1": float(f1s[1]),
            "pos_rate": float(pred.mean()),
            "fold_mean": float(np.mean(fold_scores)),
            "fold_std": float(np.std(fold_scores)),
            "tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        })
    return pd.DataFrame(rows)

def choose_threshold_prev_first(df_sweep: pd.DataFrame, prevalence: float, pos_tol: float = 0.07, macro_drop_tol: float = 0.005):
    best_macro = float(df_sweep["macro_f1"].max())
    df = df_sweep.copy()
    df_pos_ok = df.loc[(df["pos_rate"] - prevalence).abs() <= pos_tol].copy()
    pool = df_pos_ok if len(df_pos_ok) else df
    pool_best = float(pool["macro_f1"].max())
    df_near = pool.loc[pool["macro_f1"] >= (pool_best - macro_drop_tol)].copy()
    df_near["pos_gap"] = (df_near["pos_rate"] - prevalence).abs()
    df_near = df_near.sort_values(["pos_gap","fold_std","macro_f1","threshold"],
                                  ascending=[True, True, False, True]).reset_index(drop=True)
    chosen = df_near.iloc[0].to_dict()
    chosen["best_macro_overall"] = best_macro
    chosen["policy"] = "prev_first"
    return chosen

def choose_threshold_macro_first(df_sweep: pd.DataFrame, prevalence: float):
    df = df_sweep.copy()
    df["pos_gap"] = (df["pos_rate"] - prevalence).abs()
    df = df.sort_values(["macro_f1","fold_std","pos_gap","threshold"],
                        ascending=[False, True, True, True]).reset_index(drop=True)
    chosen = df.iloc[0].to_dict()
    chosen["best_macro_overall"] = float(df_sweep["macro_f1"].max())
    chosen["policy"] = "macro_first"
    return chosen

# ---------------------
# 9) Base model: LR(L1) on engineered vitals+derived+keywords + OHE(cats)
# ---------------------
def make_base_preprocessor(df: pd.DataFrame, cat_cols: list, drop_extra_cols: list):
    # numeric = everything except IDs/cats/text + explicit drop
    drop_cols = set(ID_COLS + cat_cols + TEXT_COLS + drop_extra_cols)
    num_cols = [c for c in df.columns if c not in drop_cols]
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler(with_mean=False))]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    schema = {"num_cols": num_cols, "cat_cols": cat_cols, "drop_cols": sorted(list(drop_cols))}
    return pre, schema

def oof_lr_base(df_X: pd.DataFrame, y_true: np.ndarray, cat_cols: list, C: float, drop_extra_cols: list):
    pre, schema = make_base_preprocessor(df_X, cat_cols=cat_cols, drop_extra_cols=drop_extra_cols)
    oof = np.zeros(len(y_true), dtype=float)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        pre_f = clone(pre)
        X_tr = pre_f.fit_transform(df_X.iloc[tr_idx], y_true[tr_idx])
        X_va = pre_f.transform(df_X.iloc[va_idx])
        clf = LogisticRegression(
            solver="liblinear",
            penalty="l1",
            C=float(C),
            class_weight=None,
            max_iter=5000,
            random_state=SEED
        )
        clf.fit(X_tr, y_true[tr_idx])
        oof[va_idx] = clf.predict_proba(X_va)[:, 1]
    return oof, schema

# ---------------------
# 10) Text model: small TF-IDF LR (kept capped to control overfit)
# ---------------------
def oof_text_lr(text_train: pd.Series, y_true: np.ndarray, max_features: int = 2000, C: float = 2.0):
    oof = np.zeros(len(y_true), dtype=float)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        vec = TfidfVectorizer(
            ngram_range=(1,2),
            max_features=int(max_features),
            min_df=2,
            max_df=0.95,
        )
        clf = LogisticRegression(
            solver="liblinear",
            penalty="l2",
            C=float(C),
            class_weight=None,
            max_iter=5000,
            random_state=SEED
        )
        pipe = Pipeline([("tfidf", vec), ("clf", clf)])
        pipe.fit(text_train.iloc[tr_idx], y_true[tr_idx])
        oof[va_idx] = pipe.predict_proba(text_train.iloc[va_idx])[:, 1]
    return oof

# ---------------------
# 11) HGB model: dense numeric + Target Encoding(cats) + small hyperparam grid (orthogonal signal)
# ---------------------
def fit_te_maps(X_cat_tr: pd.DataFrame, y_tr: np.ndarray, alpha: float = 10.0):
    global_mean = float(np.mean(y_tr))
    maps = {}
    for col in X_cat_tr.columns:
        s = pd.DataFrame({"cat": X_cat_tr[col].astype(str).values, "y": y_tr})
        agg = s.groupby("cat")["y"].agg(["sum", "count"])
        enc = (agg["sum"] + alpha * global_mean) / (agg["count"] + alpha)
        maps[col] = {
            "global_mean": global_mean,
            "alpha": float(alpha),
            "enc": enc.to_dict(),
            "count": agg["count"].to_dict(),
        }
    return maps

def transform_te(X_cat_df: pd.DataFrame, te_maps: dict):
    out = {}
    for col, mp in te_maps.items():
        g = float(mp["global_mean"])
        enc_map = mp["enc"]
        cnt_map = mp["count"]
        vals = X_cat_df[col].astype(str).values
        out[f"te_{col}"] = np.array([enc_map.get(v, g) for v in vals], dtype=float)
        out[f"cnt_{col}"] = np.array([cnt_map.get(v, 0) for v in vals], dtype=float)
    return pd.DataFrame(out, index=X_cat_df.index)

def oof_hgb_te(df_X: pd.DataFrame,
               y_true: np.ndarray,
               cat_cols: list,
               num_cols: list,
               hgb_params: dict,
               te_alpha: float = 10.0,
               add_text_svd: bool = False,
               text_series: pd.Series = None,
               svd_dim: int = 32,
               tfidf_max_features: int = 1200):
    oof = np.zeros(len(y_true), dtype=float)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        X_tr_num = df_X.iloc[tr_idx][num_cols].apply(pd.to_numeric, errors="coerce")
        X_va_num = df_X.iloc[va_idx][num_cols].apply(pd.to_numeric, errors="coerce")

        X_tr_cat = df_X.iloc[tr_idx][cat_cols].astype(str)
        X_va_cat = df_X.iloc[va_idx][cat_cols].astype(str)

        te_maps = fit_te_maps(X_tr_cat, y_true[tr_idx], alpha=te_alpha)
        X_tr_te = transform_te(X_tr_cat, te_maps)
        X_va_te = transform_te(X_va_cat, te_maps)

        Xtr = pd.concat([X_tr_num.reset_index(drop=True), X_tr_te.reset_index(drop=True)], axis=1)
        Xva = pd.concat([X_va_num.reset_index(drop=True), X_va_te.reset_index(drop=True)], axis=1)

        # Optional: add a tiny text SVD block (still leakage-safe: fit on train fold only; unsupervised)
        if add_text_svd and (text_series is not None):
            t_tr = text_series.iloc[tr_idx].fillna("").astype(str)
            t_va = text_series.iloc[va_idx].fillna("").astype(str)

            vec = TfidfVectorizer(
                ngram_range=(1,2),
                max_features=int(tfidf_max_features),
                min_df=2,
                max_df=0.95,
            )
            Xtr_tfidf = vec.fit_transform(t_tr)
            Xva_tfidf = vec.transform(t_va)

            # If vocabulary collapses (rare), skip text block
            if Xtr_tfidf.shape[1] >= 2:
                n_comp = int(min(svd_dim, Xtr_tfidf.shape[1] - 1))
                svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
                Xtr_svd = svd.fit_transform(Xtr_tfidf)
                Xva_svd = svd.transform(Xva_tfidf)

                svd_cols = [f"svd_{i}" for i in range(Xtr_svd.shape[1])]
                Xtr = pd.concat([Xtr.reset_index(drop=True), pd.DataFrame(Xtr_svd, columns=svd_cols)], axis=1)
                Xva = pd.concat([Xva.reset_index(drop=True), pd.DataFrame(Xva_svd, columns=svd_cols)], axis=1)

        model = HistGradientBoostingClassifier(random_state=SEED, **hgb_params)
        model.fit(Xtr, y_true[tr_idx])
        oof[va_idx] = model.predict_proba(Xva)[:, 1]
    return oof

# ---------------------
# 12) Meta stacking on OOF preds (2nd-level LR) — replaces hand-tuned weights
# ---------------------
def oof_meta_lr(pred_matrix: np.ndarray, y_true: np.ndarray, C: float = 1.0):
    """
    Proper meta OOF: for each fold, train meta on other folds using their OOF preds -> predict on fold.
    This avoids "meta overfit" on the same rows it predicts.
    """
    oof = np.zeros(len(y_true), dtype=float)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        meta = LogisticRegression(
            solver="lbfgs",
            penalty="l2",
            C=float(C),
            class_weight=None,
            max_iter=5000,
            random_state=SEED
        )
        meta.fit(pred_matrix[tr_idx], y_true[tr_idx])
        oof[va_idx] = meta.predict_proba(pred_matrix[va_idx])[:, 1]
    return oof

# ---------------------
# 13) Define model configs (small, auditable grids)
# ---------------------
# Base LR variants (zip3 on/off). Keep C anchored near proven 0.08 for stability.
BASE_CFGS = [
    {"name": "BASE_zip3_C0.08", "include_zip3": True,  "C": 0.08},
    {"name": "BASE_nozip3_C0.08", "include_zip3": False, "C": 0.08},
]

# Text model (small TF-IDF). If text is weak, it will be downweighted by stacker.
TEXT_CFG = {"name": "TEXT_tfidf_last2_C2", "max_features": 2000, "C": 2.0}

# HGB configs: include a default-ish + two tuned variants, plus optional tiny SVD-on-text variant.
HGB_CFGS = [
    {
        "name": "HGB_TE_v0_default",
        "include_zip3": True,
        "te_alpha": 10.0,
        "params": dict(
            learning_rate=0.1,
            max_iter=250,
            max_depth=3,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=0.0,
            early_stopping=False,
        ),
        "add_text_svd": False,
    },
    {
        "name": "HGB_TE_v1_tuned",
        "include_zip3": True,
        "te_alpha": 10.0,
        "params": dict(
            learning_rate=0.05,
            max_iter=500,
            max_depth=3,
            max_leaf_nodes=63,
            min_samples_leaf=10,
            l2_regularization=0.1,
            early_stopping=False,
        ),
        "add_text_svd": False,
    },
    {
        "name": "HGB_TE_v2_tuned_stronger_reg",
        "include_zip3": True,
        "te_alpha": 20.0,
        "params": dict(
            learning_rate=0.08,
            max_iter=350,
            max_depth=3,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=1.0,
            early_stopping=False,
        ),
        "add_text_svd": False,
    },
]

# Optional: enable ONE text-SVD HGB variant only if text coverage is decent (keeps runtime reasonable)
if text_nonempty >= 0.20:
    HGB_CFGS.append({
        "name": "HGB_TE_v1_plusSVD32",
        "include_zip3": True,
        "te_alpha": 10.0,
        "params": dict(
            learning_rate=0.05,
            max_iter=500,
            max_depth=3,
            max_leaf_nodes=63,
            min_samples_leaf=10,
            l2_regularization=0.1,
            early_stopping=False,
        ),
        "add_text_svd": True,
        "svd_dim": 32,
        "tfidf_max_features": 1200,
    })

# ---------------------
# 14) Prepare shared numeric columns for HGB
# ---------------------
# Use "all columns except IDs/cats/text" as dense numeric; zip3 never treated as numeric.
drop_for_num = set(ID_COLS + CAT_COLS_ALL + TEXT_COLS)
NUM_COLS_ALL = [c for c in train_X_full.columns if c not in drop_for_num]
print(f"[SCHEMA] CAT_COLS_ALL={CAT_COLS_ALL} | NUM_COLS_ALL={len(NUM_COLS_ALL)} | TEXT_COLS={TEXT_COLS}")

# ---------------------
# 15) Train OOF predictions for all base learners (fixed CV)
# ---------------------
print("\n=== [OOF] Training base learners (fixed CV) ===")

oof_preds = {}   # model_name -> oof_proba
model_meta = {}  # model_name -> config/schema info for audit

# Base LR OOF
for cfg in BASE_CFGS:
    cat_cols = CAT_COLS_ALL.copy()
    drop_extra = []
    if not cfg["include_zip3"] and ("zip3" in cat_cols):
        cat_cols = [c for c in cat_cols if c != "zip3"]
        drop_extra = ["zip3"]  # ensure zip3 excluded entirely (not numeric)
    oof, schema = oof_lr_base(train_X_full, y, cat_cols=cat_cols, C=cfg["C"], drop_extra_cols=drop_extra)
    oof_preds[cfg["name"]] = oof
    model_meta[cfg["name"]] = {"type": "lr_base", "cfg": cfg, "schema": schema}

# Text OOF (optional but cheap)
try:
    oof_text = oof_text_lr(text_train_last2, y, max_features=TEXT_CFG["max_features"], C=TEXT_CFG["C"])
    oof_preds[TEXT_CFG["name"]] = oof_text
    model_meta[TEXT_CFG["name"]] = {"type": "lr_text", "cfg": TEXT_CFG, "text_col": "note_text_last2"}
except Exception as e:
    print("⚠️ Text model skipped due to error:", repr(e))

# HGB OOF
for cfg in HGB_CFGS:
    cat_cols = CAT_COLS_ALL.copy()
    drop_extra = []
    if not cfg.get("include_zip3", True) and ("zip3" in cat_cols):
        cat_cols = [c for c in cat_cols if c != "zip3"]
        drop_extra = ["zip3"]
    # numeric cols remain same; zip3 never numeric anyway
    add_svd = bool(cfg.get("add_text_svd", False))
    svd_dim = int(cfg.get("svd_dim", 32))
    tfidf_max = int(cfg.get("tfidf_max_features", 1200))
    text_series_for_hgb = text_train_last2 if add_svd else None

    oof_hgb = oof_hgb_te(
        df_X=train_X_full,
        y_true=y,
        cat_cols=cat_cols,
        num_cols=NUM_COLS_ALL,
        hgb_params=cfg["params"],
        te_alpha=float(cfg.get("te_alpha", 10.0)),
        add_text_svd=add_svd,
        text_series=text_series_for_hgb,
        svd_dim=svd_dim,
        tfidf_max_features=tfidf_max,
    )
    oof_preds[cfg["name"]] = oof_hgb
    model_meta[cfg["name"]] = {
        "type": "hgb_te",
        "cfg": cfg,
        "cat_cols": cat_cols,
        "num_cols": NUM_COLS_ALL,
        "text_svd": add_svd,
        "svd_dim": svd_dim if add_svd else None,
        "tfidf_max_features": tfidf_max if add_svd else None,
        "text_col": "note_text_last2" if add_svd else None,
    }

# ---------------------
# 16) Candidate generation (single / blends / stacking) + evaluation
# ---------------------
print("\n=== [EVAL] Candidate search (single / blends / stacking) ===")

cand_registry = {}  # cand_name -> spec
rows = []

def eval_candidate(cand_name: str, cand_type: str, proba: np.ndarray, extra: dict):
    sw = threshold_sweep(proba, y, fold_ids)
    auc = float(roc_auc_score(y, proba))
    ch_prev = choose_threshold_prev_first(sw, prevalence=train_prev)
    ch_macro = choose_threshold_macro_first(sw, prevalence=train_prev)
    for ch in [ch_prev, ch_macro]:
        rows.append({
            "name": cand_name,
            "type": cand_type,
            "macro_f1": float(ch["macro_f1"]),
            "threshold": float(ch["threshold"]),
            "pos_rate": float(ch["pos_rate"]),
            "pos_gap": float(abs(float(ch["pos_rate"]) - train_prev)),
            "fold_mean": float(ch["fold_mean"]),
            "fold_std": float(ch["fold_std"]),
            "f1_0": float(ch["f1_0"]),
            "f1_1": float(ch["f1_1"]),
            "tn": int(ch["tn"]), "fp": int(ch["fp"]), "fn": int(ch["fn"]), "tp": int(ch["tp"]),
            "auc": auc,
            "policy": ch["policy"],
            **extra
        })
    return sw  # for potential reuse

# 16.1 singles
for mname, oof in oof_preds.items():
    cand_registry[mname] = {"type": "single", "models": [mname], "weights": [1.0]}
    eval_candidate(mname, "single", oof, extra={"models": "+".join([mname]), "w": "1.0"})

# 16.2 blends: BASE x HGB (like iter0020 but expanded)
base_names = [c["name"] for c in BASE_CFGS if c["name"] in oof_preds]
hgb_names  = [c["name"] for c in HGB_CFGS if c["name"] in oof_preds]

WEIGHTS = [0.2,0.3,0.4,0.5,0.6,0.7,0.8]
for b in base_names:
    for h in hgb_names:
        for wb in WEIGHTS:
            wh = 1.0 - wb
            cname = f"ENS_{b[:10]}{wb:.1f}_{h[:10]}{wh:.1f}"
            proba = wb * oof_preds[b] + wh * oof_preds[h]
            cand_registry[cname] = {"type": "blend", "models": [b, h], "weights": [wb, wh]}
            eval_candidate(cname, "blend2", proba, extra={"models": f"{b}+{h}", "w": f"{wb:.2f},{wh:.2f}"})

# 16.3 stacking: meta-LR on (base + best2 HGB + optional text)
def best_model_by_auc(prefix_list):
    best = None
    for m in prefix_list:
        if m not in oof_preds:
            continue
        auc = roc_auc_score(y, oof_preds[m])
        if (best is None) or (auc > best[1]):
            best = (m, auc)
    return best[0] if best else None

# pick one strong base + two strong HGBs by AUC (orthogonality proxy)
base_best = best_model_by_auc(base_names) or (base_names[0] if base_names else None)
hgb_sorted = sorted([(m, roc_auc_score(y, oof_preds[m])) for m in hgb_names], key=lambda x: x[1], reverse=True)
hgb_top = [m for m,_ in hgb_sorted[:2]]

text_name = TEXT_CFG["name"] if TEXT_CFG["name"] in oof_preds else None

stack_sets = []
if base_best and len(hgb_top) >= 1:
    stack_sets.append(("STACK2_base+hgb", [base_best, hgb_top[0]]))
if base_best and len(hgb_top) >= 2:
    stack_sets.append(("STACK3_base+hgb1+hgb2", [base_best, hgb_top[0], hgb_top[1]]))
if base_best and len(hgb_top) >= 1 and text_name:
    stack_sets.append(("STACK3_base+hgb+text", [base_best, hgb_top[0], text_name]))

for sname, comps in stack_sets:
    Xmeta = np.vstack([oof_preds[m] for m in comps]).T
    for metaC in [0.5, 1.0, 2.0]:
        cname = f"{sname}_C{metaC}"
        proba = oof_meta_lr(Xmeta, y, C=metaC)
        cand_registry[cname] = {"type": "stack", "models": comps, "meta_C": metaC}
        eval_candidate(cname, "stack", proba, extra={"models": "+".join(comps), "w": f"metaC={metaC}"})

abl_df = pd.DataFrame(rows)
abl_df["delta_vs_pstar_macro"] = abl_df["macro_f1"] - pstar_macro
abl_df["delta_vs_pstar_std"] = abl_df["fold_std"] - (pstar_std if np.isfinite(pstar_std) else 0.0)

abl_df = abl_df.sort_values(["macro_f1", "fold_std", "pos_gap"], ascending=[False, True, True]).reset_index(drop=True)

print("\n=== Candidate leaderboard (top 15) ===")
show_cols = ["name","type","macro_f1","delta_vs_pstar_macro","threshold","pos_rate","pos_gap","fold_std","delta_vs_pstar_std","f1_0","f1_1","auc","policy","models","w"]
print(abl_df[show_cols].head(15).to_string(index=False))

ABL_PATH = ART_ROOT / f"{ITER_TAG}_ablation_summary.csv"
abl_df.to_csv(ABL_PATH, index=False)
print("✅ Saved ablation summary ->", ABL_PATH)

# ---------------------
# 17) Gate vs P* (hard gate, reversible)
# ---------------------
MIN_DELTA, STD_TOL, POS_TOL = 0.001, 0.01, 0.07

def passes_gate(row):
    if float(row["macro_f1"]) < pstar_macro + MIN_DELTA:
        return False
    if np.isfinite(pstar_std):
        if float(row["fold_std"]) > pstar_std + STD_TOL:
            return False
    if abs(float(row["pos_rate"]) - train_prev) > POS_TOL:
        return False
    return True

eligible = abl_df.loc[abl_df.apply(passes_gate, axis=1)].copy()
if len(eligible):
    selected_row = eligible.iloc[0].to_dict()
    decision = "SELECT_CANDIDATE (passes gate vs P*)"
else:
    selected_row = abl_df.iloc[0].to_dict()
    decision = "NO_CANDIDATE_PASSES_GATE -> SAFE SUBMISSION uses P* (if available)"

sel_name = str(selected_row["name"])
sel_policy = str(selected_row["policy"])
thr = float(selected_row["threshold"])
sel_spec = cand_registry.get(sel_name, None)

print("\n=== Selection decision ===")
print("Decision:", decision)
print("Selected:", sel_name, "| type=", selected_row["type"], "| macro_f1=", round(float(selected_row["macro_f1"]), 6),
      "| thr=", thr, "| pos_rate=", round(float(selected_row["pos_rate"]), 3), "| policy=", sel_policy)
print("Selected spec:", json.dumps(sel_spec, ensure_ascii=False, indent=2) if sel_spec else None)

# ---------------------
# 18) Fit selected candidate on full train + write submission (with SAFE fallback)
# ---------------------
SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"
CAND_PATH = ART_ROOT / f"{TAG}_{BRANCH}_block{BLOCK_ID}_iter{ITER_ID:04d}_candidate.csv"

def fit_full_base_lr(cfg):
    cat_cols = CAT_COLS_ALL.copy()
    drop_extra = []
    if not cfg["include_zip3"] and ("zip3" in cat_cols):
        cat_cols = [c for c in cat_cols if c != "zip3"]
        drop_extra = ["zip3"]
    pre, schema = make_base_preprocessor(train_X_full, cat_cols=cat_cols, drop_extra_cols=drop_extra)
    clf = LogisticRegression(
        solver="liblinear", penalty="l1", C=float(cfg["C"]),
        class_weight=None, max_iter=5000, random_state=SEED
    )
    pipe = Pipeline([("preprocess", pre), ("clf", clf)])
    pipe.fit(train_X_full, y)
    test_proba = pipe.predict_proba(test_X_full)[:, 1]
    return pipe, test_proba, {"schema": schema, "cat_cols": cat_cols, "drop_extra": drop_extra}

def fit_full_text_lr(cfg):
    vec = TfidfVectorizer(
        ngram_range=(1,2),
        max_features=int(cfg["max_features"]),
        min_df=2,
        max_df=0.95,
    )
    clf = LogisticRegression(
        solver="liblinear", penalty="l2", C=float(cfg["C"]),
        class_weight=None, max_iter=5000, random_state=SEED
    )
    pipe = Pipeline([("tfidf", vec), ("clf", clf)])
    pipe.fit(text_train_last2, y)
    test_proba = pipe.predict_proba(text_test_last2)[:, 1]
    return pipe, test_proba

def fit_full_hgb_te(cfg):
    cat_cols = CAT_COLS_ALL.copy()
    if not cfg.get("include_zip3", True) and ("zip3" in cat_cols):
        cat_cols = [c for c in cat_cols if c != "zip3"]

    X_num_tr = train_X_full[NUM_COLS_ALL].apply(pd.to_numeric, errors="coerce")
    X_num_te = test_X_full[NUM_COLS_ALL].apply(pd.to_numeric, errors="coerce")
    X_cat_tr = train_X_full[cat_cols].astype(str)
    X_cat_te = test_X_full[cat_cols].astype(str)

    te_maps = fit_te_maps(X_cat_tr, y, alpha=float(cfg.get("te_alpha", 10.0)))
    X_tr_te = transform_te(X_cat_tr, te_maps)
    X_te_te = transform_te(X_cat_te, te_maps)

    Xtr = pd.concat([X_num_tr.reset_index(drop=True), X_tr_te.reset_index(drop=True)], axis=1)
    Xte = pd.concat([X_num_te.reset_index(drop=True), X_te_te.reset_index(drop=True)], axis=1)

    add_svd = bool(cfg.get("add_text_svd", False))
    vec, svd = None, None
    if add_svd:
        svd_dim = int(cfg.get("svd_dim", 32))
        tfidf_max = int(cfg.get("tfidf_max_features", 1200))
        vec = TfidfVectorizer(
            ngram_range=(1,2),
            max_features=int(tfidf_max),
            min_df=2,
            max_df=0.95,
        )
        Xtr_tfidf = vec.fit_transform(text_train_last2)
        Xte_tfidf = vec.transform(text_test_last2)
        if Xtr_tfidf.shape[1] >= 2:
            n_comp = int(min(svd_dim, Xtr_tfidf.shape[1] - 1))
            svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
            Xtr_svd = svd.fit_transform(Xtr_tfidf)
            Xte_svd = svd.transform(Xte_tfidf)
            svd_cols = [f"svd_{i}" for i in range(Xtr_svd.shape[1])]
            Xtr = pd.concat([Xtr.reset_index(drop=True), pd.DataFrame(Xtr_svd, columns=svd_cols)], axis=1)
            Xte = pd.concat([Xte.reset_index(drop=True), pd.DataFrame(Xte_svd, columns=svd_cols)], axis=1)

    model = HistGradientBoostingClassifier(random_state=SEED, **cfg["params"])
    model.fit(Xtr, y)
    test_proba = model.predict_proba(Xte)[:, 1]

    pack = {
        "model": model,
        "te_maps": te_maps,
        "cat_cols": cat_cols,
        "num_cols": NUM_COLS_ALL,
        "cfg": cfg,
        "text_svd": add_svd,
        "vectorizer": vec,
        "svd": svd,
    }
    return pack, test_proba

def compute_test_proba_for_candidate(sel_spec: dict):
    """
    Fit only required components; then combine into final test proba for selected candidate.
    Returns: (test_proba_final, fitted_bundle_dict, per_component_test_proba_dict)
    """
    fitted = {"base_pipes": {}, "text_pipe": None, "hgb_packs": {}, "meta_model": None}
    test_pred = {}

    def fit_component(model_name: str):
        if model_name in test_pred:
            return
        # base?
        if model_name.startswith("BASE_"):
            cfg = next(c for c in BASE_CFGS if c["name"] == model_name)
            pipe, proba, meta = fit_full_base_lr(cfg)
            fitted["base_pipes"][model_name] = {"pipe": pipe, "meta": meta}
            test_pred[model_name] = proba
            return
        # text?
        if model_name == TEXT_CFG["name"]:
            pipe, proba = fit_full_text_lr(TEXT_CFG)
            fitted["text_pipe"] = {"name": model_name, "pipe": pipe, "cfg": TEXT_CFG}
            test_pred[model_name] = proba
            return
        # hgb?
        if model_name.startswith("HGB_"):
            cfg = next(c for c in HGB_CFGS if c["name"] == model_name)
            pack, proba = fit_full_hgb_te(cfg)
            fitted["hgb_packs"][model_name] = pack
            test_pred[model_name] = proba
            return
        raise KeyError(f"Unknown model component: {model_name}")

    # Fit needed components
    comps = sel_spec.get("models", [])
    for m in comps:
        fit_component(m)

    # Combine
    if sel_spec["type"] == "single":
        final = test_pred[comps[0]]
        return final, fitted, test_pred

    if sel_spec["type"] == "blend":
        ws = sel_spec["weights"]
        final = np.zeros(len(test_X_full), dtype=float)
        for m, w in zip(comps, ws):
            final += float(w) * test_pred[m]
        return final, fitted, test_pred

    if sel_spec["type"] == "stack":
        metaC = float(sel_spec.get("meta_C", 1.0))
        # fit meta on FULL OOF preds (training-time, already computed & leakage-safe)
        Xmeta_tr = np.vstack([oof_preds[m] for m in comps]).T
        meta = LogisticRegression(
            solver="lbfgs",
            penalty="l2",
            C=float(metaC),
            class_weight=None,
            max_iter=5000,
            random_state=SEED
        )
        meta.fit(Xmeta_tr, y)
        fitted["meta_model"] = {"model": meta, "meta_C": metaC, "models": comps}

        Xmeta_te = np.vstack([test_pred[m] for m in comps]).T
        final = meta.predict_proba(Xmeta_te)[:, 1]
        return final, fitted, test_pred

    raise ValueError(f"Unknown candidate type: {sel_spec['type']}")

# always compute candidate snapshot (even if not passing gate)
if sel_spec is None:
    raise RuntimeError(f"Selected candidate '{sel_name}' not found in registry; abort for audit safety.")

print("\n=== [FIT] Fitting selected candidate on full train ===")
test_proba_candidate, fitted_bundle, test_pred_components = compute_test_proba_for_candidate(sel_spec)
test_pred_candidate = (test_proba_candidate >= thr).astype(int)

candidate_df = pd.DataFrame({
    "stay_id": test_full["stay_id"].astype(int).values,
    "discharge_ready_day11": test_pred_candidate.astype(int),
})
candidate_df.to_csv(CAND_PATH, index=False)
print("✅ Wrote candidate snapshot ->", CAND_PATH, "| pos_rate:", float(candidate_df["discharge_ready_day11"].mean()))

# SAFE SUBMISSION: if no candidate passes gate, submit P* snapshot if available
def find_candidate_csv_for_iter(iter_tag: str):
    # Search scoped ART_ROOT first
    pats = [
        f"{TAG}_*{iter_tag}*candidate*.csv",
        f"{TAG}_*iter{_extract_iter_num(iter_tag) if _extract_iter_num(iter_tag) else ''}*candidate*.csv",
    ]
    cands = []
    for pat in pats:
        cands.extend(sorted(ART_ROOT.glob(pat)))
    # fallback: any candidate containing iter_tag
    if not cands:
        cands = sorted([p for p in ART_ROOT.glob(f"{TAG}_*candidate*.csv") if iter_tag in p.name])
    return cands[-1] if cands else None

submission_mode = "use_candidate"
if decision.startswith("NO_CANDIDATE_PASSES_GATE"):
    pstar_cand = find_candidate_csv_for_iter(pstar_iter_tag)
    if pstar_cand and pstar_cand.exists():
        shutil.copy2(pstar_cand, SUB_PATH)
        submission_mode = f"SAFE_MODE_copy_PSTAR_candidate({pstar_iter_tag})"
        print("[SUBMISSION] SAFE MODE -> copied P* candidate:", pstar_cand.name)
    else:
        # fallback to candidate anyway (still auditable)
        candidate_df.to_csv(SUB_PATH, index=False)
        submission_mode = "SAFE_MODE_no_pstar_candidate_found_use_current_candidate"
        print("[SUBMISSION] SAFE MODE -> no P* candidate found; using current candidate.")
else:
    candidate_df.to_csv(SUB_PATH, index=False)

print("\n=== Submission written ===")
print("SUB_PATH:", SUB_PATH)
print("submission_mode:", submission_mode)
print("Threshold used:", thr)
print("Submission pos_rate:", float(pd.read_csv(SUB_PATH)["discharge_ready_day11"].mean()))

# ---------------------
# 19) Persist OOF for selected candidate + sweep + diagnostics
# ---------------------
# Reconstruct selected candidate OOF proba for audit:
def get_candidate_oof_proba(sel_spec: dict):
    comps = sel_spec.get("models", [])
    if sel_spec["type"] == "single":
        return oof_preds[comps[0]]
    if sel_spec["type"] == "blend":
        ws = sel_spec["weights"]
        out = np.zeros(len(y), dtype=float)
        for m, w in zip(comps, ws):
            out += float(w) * oof_preds[m]
        return out
    if sel_spec["type"] == "stack":
        metaC = float(sel_spec.get("meta_C", 1.0))
        Xmeta = np.vstack([oof_preds[m] for m in comps]).T
        return oof_meta_lr(Xmeta, y, C=metaC)  # proper meta OOF
    raise ValueError("Unknown sel_spec type")

sel_oof = get_candidate_oof_proba(sel_spec)

# Save OOF with component columns for attribution (extra cols are OK for downstream scripts)
oof_df = pd.DataFrame({
    "stay_id": train_full["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "fold": fold_ids.astype(int),
    "oof_proba": sel_oof.astype(float),
})
# attach components (optional)
for m in sel_spec.get("models", []):
    oof_df[f"oof_{m}"] = oof_preds[m].astype(float)

OOF_PATH = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
oof_df.to_csv(OOF_PATH, index=False)
print("✅ Saved OOF ->", OOF_PATH.name)

SWEEP_DF = threshold_sweep(sel_oof, y, fold_ids)
SWEEP_PATH = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"
SWEEP_DF.to_csv(SWEEP_PATH, index=False)
print("✅ Saved sweep ->", SWEEP_PATH.name)

# Group diagnostics (failure explanation)
diag_cols = [c for c in ["unit_type","insurance","admission_reason","sex","zip3"] if c in train_full.columns]
diag = train_full[["stay_id"] + diag_cols].copy()
diag["y"] = y
diag["proba"] = sel_oof
diag["pred"] = (sel_oof >= thr).astype(int)
diag["is_fp"] = (diag["y"] == 0) & (diag["pred"] == 1)
diag["is_fn"] = (diag["y"] == 1) & (diag["pred"] == 0)

# optional: age bin
if "age" in train_full.columns:
    diag["age_bin"] = pd.cut(train_full["age"], bins=[0,30,45,60,75,200], include_lowest=True).astype(str)
    diag_cols = diag_cols + ["age_bin"]

def group_err(df: pd.DataFrame, col: str) -> pd.DataFrame:
    g = df.groupby(col).agg(
        n=("stay_id", "count"),
        y_prev=("y", "mean"),
        pred_rate=("pred", "mean"),
        fp=("is_fp", "sum"),
        fn=("is_fn", "sum"),
    ).reset_index()
    neg = np.maximum(1.0, g["n"] - g["y_prev"] * g["n"])
    pos = np.maximum(1.0, g["y_prev"] * g["n"])
    g["fp_rate_in_neg"] = g["fp"] / neg
    g["fn_rate_in_pos"] = g["fn"] / pos
    return g.sort_values(["fp_rate_in_neg","fn_rate_in_pos","n"], ascending=[False, False, False])

g_all = []
for col in diag_cols:
    try:
        gg = group_err(diag, col)
        gg["group_col"] = col
        g_all.append(gg)
    except Exception:
        pass

DIAG_PATH = ART_ROOT / f"{ITER_TAG}_group_error_summary.csv"
if g_all:
    pd.concat(g_all, ignore_index=True).to_csv(DIAG_PATH, index=False)
    print("✅ Saved group diagnostics ->", DIAG_PATH.name)
else:
    print("⚠️ No group diagnostics written (no group cols).")

# ---------------------
# 20) Checkpoint bundle (audit)
# ---------------------
iter_ckpt_dir = CKPT_ROOT / ITER_TAG
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

pred_oof = (sel_oof >= thr).astype(int)
cm = confusion_matrix(y, pred_oof).tolist()
f1s = f1_score(y, pred_oof, average=None)
macro = f1_score(y, pred_oof, average="macro")
fold_scores = []
for f in range(N_SPLITS):
    idx = fold_ids == f
    fold_scores.append(float(f1_score(y[idx], pred_oof[idx], average="macro")))

metrics = {
    "oof_macro_f1": float(macro),
    "per_class_f1": {"f1_0": float(f1s[0]), "f1_1": float(f1s[1])},
    "confusion_matrix": cm,
    "threshold": float(thr),
    "pos_rate": float(pred_oof.mean()),
    "fold_scores": fold_scores,
    "fold_mean": float(np.mean(fold_scores)),
    "fold_std": float(np.std(fold_scores)),
    "auc": float(roc_auc_score(y, sel_oof)),
    "train_prevalence": train_prev,
    "gate": {"min_delta": MIN_DELTA, "std_tol": STD_TOL, "pos_tol": POS_TOL, "decision": decision},
    "pstar_reference": pstar,
}

schema = {
    "id_cols": ID_COLS,
    "cat_cols_all": CAT_COLS_ALL,
    "num_cols_all": NUM_COLS_ALL,
    "text_cols": TEXT_COLS,
    "keywords_count": len(KEYWORDS),
    "vitals_cols": VITAL_COLS,
    "vitals_days_used": "1-10",
    "base_cfgs": BASE_CFGS,
    "hgb_cfgs": HGB_CFGS,
    "text_cfg": TEXT_CFG,
    "stack_sets": stack_sets,
}

config = {
    "team": TEAM, "task": TASK, "version": VERSION, "tag": TAG,
    "iteration_id": ITER_ID, "iteration_tag": ITER_TAG,
    "branch": BRANCH, "block_id": BLOCK_ID, "block_pos": BLOCK_POS,
    "seed": SEED,
    "validation": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "selected": {
        "name": sel_name,
        "type": str(selected_row["type"]),
        "policy": sel_policy,
        "threshold": float(thr),
        "spec": sel_spec,
    },
    "submission_mode": submission_mode,
    "notes": [
        "Behavior-first iteration: reversible, auditable, attributable; failure explainable.",
        "New in W5: small HGB hyperparam grid + meta stacking on OOF preds (avoids hand-tuned blend weights).",
        "Leakage guardrail enforced: vitals max(day)<=10.",
        "Threshold selected on OOF only; policies compared: prev_first vs macro_first.",
    ],
}

# Save bundle as joblib dict (portable)
bundle = {
    "selected_spec": sel_spec,
    "threshold": float(thr),
    "components_fitted": fitted_bundle,  # contains fitted base/text/hgb/meta needed for candidate
    "schema": schema,
    "config": config,
    "metrics": metrics,
}
joblib.dump(bundle, iter_ckpt_dir / "pipeline.joblib")

with open(iter_ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)

print("✅ Saved checkpoint ->", iter_ckpt_dir)

# Decision note (must-have: changes, metrics, risks, fallback)
DECISION_PATH = ART_ROOT / f"{ITER_TAG}_DECISION.md"
topN = abl_df[show_cols].head(12).copy()

def df_to_md(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except Exception:
        return "```\n" + df.to_string(index=False) + "\n```"

decision_lines = []
decision_lines.append(f"# {TAG} — {ITER_TAG} decision note\nGenerated: {utc_ts()}\n")
decision_lines.append("## What changed (reversible)\n"
                      "- Added orthogonal learners: **HGB (Target Encoding) small hyperparam grid**\n"
                      "- Added **meta stacking (2nd-level LR)** over OOF preds (reduces reliance on hand-tuned blend weights)\n"
                      "- Kept guardrails: fixed CV, Macro-F1, scoped artifacts, day<=10 vitals, threshold on OOF\n")
decision_lines.append("## P* reference\n" + "```json\n" + json.dumps(pstar, indent=2, ensure_ascii=False) + "\n```")
decision_lines.append("## Candidate leaderboard (top 12)\n" + df_to_md(topN))
decision_lines.append("\n## Selection\n"
                      f"- Decision: **{decision}**\n"
                      f"- Selected: **{sel_name}** (type={selected_row['type']}, policy={sel_policy})\n"
                      f"- OOF Macro-F1: **{metrics['oof_macro_f1']:.6f}** | thr={thr:.2f} | pos_rate={metrics['pos_rate']:.3f} | fold_std={metrics['fold_std']:.6f}\n"
                      f"- Confusion matrix: {metrics['confusion_matrix']}\n")
decision_lines.append("\n## Risks\n"
                      "- HGB may increase variance / fold_std; monitored by gate + diagnostics.\n"
                      "- Stacking can overfit at meta-level; mitigated by **proper meta OOF** (train meta on other folds only).\n"
                      "- Text features are capped; if they add noise, stacker should downweight.\n")
decision_lines.append("\n## Fallback\n"
                      "- If gate fails: **do not update P***; submission uses SAFE_MODE copy of P* candidate if available.\n"
                      "- All artifacts are scoped under artifacts/checkpoints for TAG to prevent contamination.\n")
DECISION_PATH.write_text("\n".join(decision_lines), encoding="utf-8")
print("✅ Wrote decision note ->", DECISION_PATH)

# ---------------------
# 21) Update PSTAR.json (only if passes gate)
# ---------------------
if decision.startswith("SELECT_CANDIDATE"):
    new_pstar = {
        "iteration_tag": ITER_TAG,
        "oof_macro_f1": float(metrics["oof_macro_f1"]),
        "threshold": float(thr),
        "fold_mean": float(metrics["fold_mean"]),
        "fold_std": float(metrics["fold_std"]),
        "pos_rate": float(metrics["pos_rate"]),
        "f1_0": float(metrics["per_class_f1"]["f1_0"]),
        "f1_1": float(metrics["per_class_f1"]["f1_1"]),
        "confusion_matrix": metrics["confusion_matrix"],
    }
    history = []
    if PSTAR_PATH.exists():
        try:
            old = json.loads(PSTAR_PATH.read_text(encoding="utf-8"))
            history = old.get("history", []) or []
        except Exception:
            history = []
    history.append({"timestamp": utc_ts(), "iteration_tag": ITER_TAG, "oof_macro_f1": float(metrics["oof_macro_f1"]), "threshold": float(thr)})
    with open(PSTAR_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "schema_version": 3,
            "tag": TAG,
            "timestamp": utc_ts(),
            "train_prevalence": train_prev,
            "pstar": new_pstar,
            "history": history[-50:],  # keep last 50
            "notes": "Updated because candidate passed hard gate vs prior P*.",
        }, f, indent=2, ensure_ascii=False)
    print("✅ Updated PSTAR.json ->", new_pstar["iteration_tag"], "macro=", round(new_pstar["oof_macro_f1"], 6))
else:
    print("ℹ️ PSTAR.json unchanged (no candidate passed hard gate).")

# ---------------------
# 22) Append iteration log (append-only; fallback if permission)
# ---------------------
iteration_record = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": utc_ts(),
    "phase": "Modeling",
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "data_paths_used": {
        "stays_train": str(dp("stays_train.csv")),
        "stays_test": str(dp("stays_test.csv")),
        "patients": str(dp("patients.csv")),
        "vitals_timeseries": str(dp("vitals_timeseries.json")),
        "art_root": str(ART_ROOT),
        "ckpt_root": str(CKPT_ROOT),
    },
    "join_keys": ["stays.patient_id -> patients.patient_id", "stays.stay_id -> vitals_timeseries.stay_id"],
    "leakage_checks_performed": [
        "Asserted vitals max(day) <= 10 (no day11 leakage).",
        "Threshold tuned on OOF only (no test leakage).",
        "Artifact discovery scoped under artifacts/clai_ch3_v3.",
        "Target encoding fit within fold only (no label leakage into val).",
        "Meta stacking uses proper meta OOF (train meta on other folds).",
    ],
    "feature_summary": schema,
    "models_attempted": topN.to_dict("records"),
    "selected_model": {
        "name": sel_name,
        "type": sel_spec["type"],
        "models": sel_spec.get("models"),
        "weights": sel_spec.get("weights"),
        "meta_C": sel_spec.get("meta_C"),
        "threshold": float(thr),
        "threshold_policy": sel_policy,
        "decision": decision,
    },
    "validation_protocol": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "metrics": metrics,
    "artifacts_written": {
        "submission_csv": str(SUB_PATH),
        "candidate_snapshot": str(CAND_PATH),
        "oof_predictions": str(OOF_PATH),
        "threshold_sweep": str(SWEEP_PATH),
        "ablation_summary": str(ABL_PATH),
        "group_error_summary": str(DIAG_PATH),
        "decision_note": str(DECISION_PATH),
        "checkpoint_dir": str(iter_ckpt_dir),
        "pstar_path": str(PSTAR_PATH),
    },
    "deltas_vs_prev": {
        "changes": [
            "Added HGB(TargetEncoding) hyperparam micro-grid for orthogonal signal.",
            "Added meta stacking (2nd-level LR) trained with proper meta OOF.",
            "Kept base feature set stable (derived indices + keyword counts) to ensure reversibility.",
        ],
        "risk": [
            "HGB variance may rise; controlled by fold_std gate and pos_rate tolerance.",
            "Stacking may overfit; mitigated by fold-wise meta OOF procedure.",
        ],
        "fallback": "If gate fails: keep P* unchanged; safe submission copies P* candidate (if found).",
    },
    "next_step_hypothesis": (
        "If real F1 still plateaus: consider adding a 2nd orthogonal nonlinear model (ExtraTrees on dense+TE) "
        "or richer time-series summary features (per-day quantiles), but keep changes small and gated."
    ),
}

log_written_to = str(LOG_PATH)
try:
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
except PermissionError:
    alt_log = CKPT_ROOT / f"{TAG}_iteration_detail.jsonl"
    log_written_to = str(alt_log)
    with open(alt_log, "a", encoding="utf-8") as f:
        f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")

print("✅ Appended iteration log ->", log_written_to)

# ---------------------
# 23) Consultant packet (every iter)
# ---------------------
consult_packet = {
    "tag": TAG,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": iteration_record["timestamp"],
    "what_changed": iteration_record["deltas_vs_prev"]["changes"],
    "metric_changes": {
        "pstar_reference": pstar,
        "selected_metrics": metrics,
        "gate_decision": decision,
    },
    "suspected_leakage_risks": [
        "Vitals leakage if any day>10 appears; enforced by assert max(day)<=10.",
        "Target encoding leakage if fit on full data; enforced fold-wise fit.",
        "Meta overfit if trained/evaluated on same rows; mitigated by meta OOF.",
    ],
    "complexity_drift_risks": [
        "HGB grid adds complexity; kept micro-grid only.",
        "Text TF-IDF is capped; optional only.",
    ],
    "evaluation_alignment_risks": [
        "Threshold can overfit; use simple sweep + prevalence-aware policy + fold_std tracking.",
        "Monitor pos_rate drift; enforce POS_TOL gate.",
    ],
    "unknown_unknowns": [
        "Distribution shift across hospitals/units could change FP hotspots.",
        "Text note style may differ in test; stacker should downweight if noisy.",
    ],
    "recommendations_for_next_iter": [
        "If this improves OOF and real: keep winner; then explore 1 extra orthogonal model (ExtraTrees dense+TE) for stacking.",
        "If this regresses: rollback to iter0020 P* immediately and only change ONE axis next time.",
    ],
}
consult_path = CONSULT_ROOT / f"{TAG}_iter{ITER_ID}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, ensure_ascii=False)
print("✅ Wrote consultant packet ->", consult_path)

print("\nDONE ✅")
print("Key outputs:")
print(" - Submission       :", SUB_PATH, "| mode:", submission_mode)
print(" - Candidate        :", CAND_PATH)
print(" - Ablation         :", ABL_PATH)
print(" - OOF              :", OOF_PATH)
print(" - Sweep            :", SWEEP_PATH)
print(" - Group diag       :", DIAG_PATH)
print(" - Decision note    :", DECISION_PATH)
print(" - PSTAR            :", PSTAR_PATH)
print(" - Checkpoint dir   :", iter_ckpt_dir)

=== PREFLIGHT ===
PROJECT_ROOT: D:\AgentDs\agent_ds_healthcare
ART_ROOT: D:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT: D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
CONSULT_ROOT: D:\AgentDs\agent_ds_healthcare\consultant_packets
LOG_PATH: D:\AgentDs\agent_ds_healthcare\clai_ch3_v3_iteration_detail.jsonl
ITER_ID=21  ITER_TAG=iter0021  BLOCK_ID=4  BLOCK_POS=1  BRANCH=W5_stack3_teHGB_tune
[DATA] Train prevalence(y=1): 0.656 | n=1000
=== [FE] Building features ===
[FE] text_nonempty_rate(note_last2): 1.000
=== P* (loaded) ===
{
  "iteration_tag": "iter0020",
  "oof_macro_f1": 0.8249641287951847,
  "threshold": 0.63,
  "fold_mean": 0.8248897805164553,
  "fold_std": 0.024420082144040903,
  "pos_rate": 0.665,
  "f1_0": 0.7687776141384389,
  "f1_1": 0.8811506434519304,
  "confusion_matrix": [
    [
      261,
      83
    ],
    [
      74,
      582
    ]
  ]
}
[SCHEMA] CAT_COLS_ALL=['unit_type', 'sex', 'insurance', 'zip3', 'admission_reason'] | NUM_COLS_ALL=229 | T

# Iteration 14
- 0.7321

In [56]:
# ============================================================
# clai_ch3_v3 - iter0022 (Plateau breaker): patient-history + real text model + tri-blend
# - Hard gate vs current P* (iter0020) + SAFE_MODE rollback (never worse than current best submission)
# - Strict CV hygiene (no leakage): encoding/TE/vectorizers are fit inside CV folds
# ============================================================

import os, re, json, math, random, shutil, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.base import clone

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse

import joblib

warnings.filterwarnings("ignore")

# -----------------------------
# 0) Config
# -----------------------------
TAG = "clai_ch3_v3"
BRANCH = "W6_patientHistory+TextTriBlend"
BLOCK_ID = 5
BLOCK_POS = 0

SEED = 42
N_SPLITS = 5

# Gate config (protect real score; only accept clear OOF improvement)
MIN_DELTA_MACRO = 0.0010
MAX_STD_INCREASE = 0.0100
MAX_POS_GAP = 0.08   # |pred_pos_rate - train_prevalence| must be <= this

# Feature toggles (new info sources)
USE_ED_COST = True
USE_ADMISSIONS_HISTORY = True

# Text toggles (plateau breaker)
ENABLE_TEXT_MODEL = True
TEXT_USE_LAST2 = True           # use stay note last2
TEXT_USE_ALL = False            # optional; keep False to reduce noise
TEXT_WORD_MAX_FEATS = 2500
TEXT_CHAR_MAX_FEATS = 2500
TEXT_LR_C_GRID = [0.7, 1.0, 1.5, 2.0]

# Base LR grid (keep near proven C=0.08, but allow small exploration)
BASE_LR_C_GRID = [0.06, 0.08, 0.10, 0.12]

# HGB grid (small, targeted)
HGB_PARAM_GRID = [
    dict(max_depth=3, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0, max_iter=500),
    dict(max_depth=4, learning_rate=0.03, max_leaf_nodes=63, min_samples_leaf=20, l2_regularization=2.0, max_iter=800),
]

# Target encoding smoothing (prevents rare-category overfit)
TE_ALPHA = 10.0

np.random.seed(SEED)
random.seed(SEED)

# -----------------------------
# 1) Paths / Scoping / Iter id
# -----------------------------
def utc_ts() -> str:
    return datetime.now(timezone.utc).isoformat()

def find_project_root() -> Path:
    # Works for both Windows project and this sandbox layout
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / "stays_train.csv").exists() and (p / "stays_test.csv").exists():
            return p
    # fallback for some notebook setups
    if Path("/mnt/data/stays_train.csv").exists():
        return Path("/mnt/data")
    raise FileNotFoundError("Cannot find project root containing stays_train.csv / stays_test.csv")

PROJECT_ROOT = find_project_root()
ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"
LOG_PATH = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"
SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"

for d in [ART_ROOT, CKPT_ROOT, CONSULT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

def infer_next_iter_id() -> int:
    max_id = 0
    # checkpoints
    if CKPT_ROOT.exists():
        for p in CKPT_ROOT.glob("iter*"):
            m = re.match(r"iter(\d+)", p.name)
            if m:
                max_id = max(max_id, int(m.group(1)))
    # log
    if LOG_PATH.exists():
        try:
            with open(LOG_PATH, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        rec = json.loads(line)
                        it = rec.get("iter_id", None)
                        tag = rec.get("iteration_tag", None)
                        if isinstance(it, int):
                            max_id = max(max_id, it)
                        elif isinstance(tag, str) and re.match(r"iter\d{4}$", tag):
                            max_id = max(max_id, int(tag.replace("iter", "")))
                    except Exception:
                        continue
        except Exception:
            pass
    return max_id + 1

ITER_ID = infer_next_iter_id()
ITER_TAG = f"iter{ITER_ID:04d}"

ITER_ART_DIR = ART_ROOT  # keep all artifacts in one scoped root for this tag
ITER_CKPT_DIR = CKPT_ROOT / ITER_TAG
ITER_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print("=== PREFLIGHT ===")
print("PROJECT_ROOT:", str(PROJECT_ROOT))
print("ART_ROOT    :", str(ART_ROOT))
print("CKPT_ROOT   :", str(CKPT_ROOT))
print("CONSULT_ROOT:", str(CONSULT_ROOT))
print("LOG_PATH    :", str(LOG_PATH))
print(f"ITER_ID={ITER_ID}  ITER_TAG={ITER_TAG}  BLOCK_ID={BLOCK_ID}  BLOCK_POS={BLOCK_POS}  BRANCH={BRANCH}")

# -----------------------------
# 2) Load data
# -----------------------------
def dp(name: str) -> Path:
    p = PROJECT_ROOT / name
    if p.exists():
        return p
    # common alternative layout
    p2 = PROJECT_ROOT / "data" / name
    if p2.exists():
        return p2
    p3 = Path("/mnt/data") / name
    if p3.exists():
        return p3
    raise FileNotFoundError(f"Missing required file: {name}")

stays_train = pd.read_csv(dp("stays_train.csv"))
stays_test  = pd.read_csv(dp("stays_test.csv"))
patients    = pd.read_csv(dp("patients.csv"))

y = stays_train["discharge_ready_day11"].astype(int).values
train_prev = float(np.mean(y))
print(f"[DATA] Train prevalence(y=1): {train_prev:.3f} | n={len(y)}")

# vitals
with open(dp("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vit_list = json.load(f)
vit_map = {int(item["stay_id"]): item.get("days", []) for item in vit_list}
max_day = max((d.get("day", 0) for item in vit_list for d in item.get("days", [])), default=0)
if max_day > 10:
    raise RuntimeError(f"[LEAKAGE] vitals_timeseries contains day={max_day} (>10). Refuse to proceed.")
print(f"[LEAKAGE_CHECK] Max day in vitals_timeseries: {max_day} (OK <= 10)")

# Optional extra data (patient history)
adm_train = adm_test = ed_train = ed_test = None
if USE_ADMISSIONS_HISTORY:
    adm_train = pd.read_csv(dp("admissions_train.csv"))
    adm_test  = pd.read_csv(dp("admissions_test.csv"))
if USE_ED_COST:
    ed_train = pd.read_csv(dp("ed_cost_train.csv"))
    ed_test  = pd.read_csv(dp("ed_cost_test.csv"))

# -----------------------------
# 3) Patient history features (SAFE)
# -----------------------------
def build_patient_history_features() -> pd.DataFrame:
    # Start from patients to guarantee full coverage
    hist = patients[["patient_id"]].copy()

    # ED cost: only prior features (drop target)
    if USE_ED_COST and ed_train is not None and ed_test is not None:
        ed_train_use = ed_train.drop(columns=[c for c in ["ed_cost_next3y_usd"] if c in ed_train.columns])
        ed_all = pd.concat([ed_train_use, ed_test], axis=0, ignore_index=True)
        # ensure unique per patient
        ed_all = ed_all.sort_values("patient_id").drop_duplicates("patient_id", keep="last")
        hist = hist.merge(ed_all, on="patient_id", how="left")

    # Admissions history: aggregate only non-label, non-future-ish fields (no readmit label, no los_days, no discharge_weekday)
    if USE_ADMISSIONS_HISTORY and adm_train is not None and adm_test is not None:
        drop_cols = [c for c in ["readmit_30d", "los_days", "discharge_weekday"] if c in adm_train.columns]
        adm_train_use = adm_train.drop(columns=drop_cols, errors="ignore")
        adm_test_use  = adm_test.copy()

        adm_all = pd.concat([adm_train_use, adm_test_use], axis=0, ignore_index=True)

        # coerce numeric
        for c in ["acuity_emergent", "charlson_band", "ed_visits_6m"]:
            if c in adm_all.columns:
                adm_all[c] = pd.to_numeric(adm_all[c], errors="coerce")

        g = adm_all.groupby("patient_id", dropna=False)

        out = pd.DataFrame({"patient_id": g.size().index})
        out["adm_count"] = g.size().values

        def gmean(col):
            return g[col].mean().values if col in adm_all.columns else np.nan

        def gmax(col):
            return g[col].max().values if col in adm_all.columns else np.nan

        out["adm_acuity_mean"]   = gmean("acuity_emergent")
        out["adm_charlson_mean"] = gmean("charlson_band")
        out["adm_charlson_max"]  = gmax("charlson_band")
        out["adm_edvis6m_mean"]  = gmean("ed_visits_6m")
        out["adm_edvis6m_max"]   = gmax("ed_visits_6m")

        # dx counts (only those present)
        if "primary_dx" in adm_all.columns:
            for dx in ["HF", "Pneumonia", "DiabetesComp"]:
                out[f"adm_dx_{dx}_count"] = g["primary_dx"].apply(lambda s: int((s == dx).sum())).values

        hist = hist.merge(out, on="patient_id", how="left")

    return hist

patient_hist = build_patient_history_features()

# -----------------------------
# 4) Vitals + note features (core)
# -----------------------------
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]
KEYWORDS = [
    "stable","improving","worsening","afebrile","fever","chills","hypotensive","tachy",
    "culture","antibiotic","abx","iv","oxygen","weaned","dyspnea","cough",
    "pain","nausea","vomit","n/v","ambulated","pt","ot","telemetry","fluids","room air","sats",
    "discharge","transfer"
]
KW_PATTERNS = [(kw, re.compile(re.escape(kw), re.IGNORECASE)) for kw in KEYWORDS]

def safe_nan(fn, arr, default=np.nan):
    try:
        with np.errstate(all="ignore"):
            return fn(arr)
    except Exception:
        return default

def build_features(stays_df: pd.DataFrame) -> pd.DataFrame:
    base = stays_df.merge(patients, on="patient_id", how="left").merge(patient_hist, on="patient_id", how="left")

    rows = []
    for sid in base["stay_id"].astype(int).tolist():
        days = sorted(vit_map.get(int(sid), []), key=lambda x: x.get("day", 0))
        day_idx = np.array([d.get("day", np.nan) for d in days], dtype=float)

        vit = {c: np.array([d.get(c, np.nan) for d in days], dtype=float) for c in VITAL_COLS}

        pulse_pressure = vit["sbp"] - vit["dbp"]
        map_est = (vit["sbp"] + 2.0 * vit["dbp"]) / 3.0
        shock_index = vit["hr"] / vit["sbp"]
        shock_index = np.where(np.isfinite(shock_index), shock_index, np.nan)
        temp_dev = vit["temp_c"] - 37.0

        series = {
            "hr": vit["hr"], "sbp": vit["sbp"], "dbp": vit["dbp"], "rr": vit["rr"], "temp_c": vit["temp_c"],
            "pulse_pressure": pulse_pressure, "map": map_est, "shock_index": shock_index, "temp_dev": temp_dev
        }

        notes = [(d.get("note") or "") for d in days]
        note_all = " ".join(notes).strip()
        note_last2 = " ".join(notes[-2:]).strip()

        feat = {
            "stay_id": int(sid),
            "note_text_all": note_all,
            "note_text_last2": note_last2,
            "note_len_all": len(note_all),
            "note_len_last2": len(note_last2),
            "note_days": int(sum(1 for n in notes if str(n).strip() != "")),
            "days_recorded": int(len(days)),
        }

        # richer but still stable stats: include p25/p75/iqr
        for name, vals in series.items():
            vals = np.array(vals, dtype=float)
            diffs = np.diff(vals) if len(vals) >= 2 else np.array([], dtype=float)

            feat[f"{name}_mean"] = safe_nan(np.nanmean, vals)
            feat[f"{name}_std"]  = safe_nan(np.nanstd, vals)
            feat[f"{name}_min"]  = safe_nan(np.nanmin, vals)
            feat[f"{name}_max"]  = safe_nan(np.nanmax, vals)
            feat[f"{name}_median"] = safe_nan(np.nanmedian, vals)

            p25 = safe_nan(lambda a: np.nanpercentile(a, 25), vals)
            p75 = safe_nan(lambda a: np.nanpercentile(a, 75), vals)
            feat[f"{name}_p25"] = p25
            feat[f"{name}_p75"] = p75
            feat[f"{name}_iqr"] = (p75 - p25) if (np.isfinite(p75) and np.isfinite(p25)) else np.nan

            feat[f"{name}_first"] = vals[0] if len(vals) else np.nan
            feat[f"{name}_last"]  = vals[-1] if len(vals) else np.nan

            last2 = vals[-2:] if len(vals) >= 2 else vals
            early = vals[:8] if len(vals) >= 8 else (vals[:-2] if len(vals) > 2 else vals)

            feat[f"{name}_last2_mean"] = safe_nan(np.nanmean, last2)
            feat[f"{name}_early_mean"] = safe_nan(np.nanmean, early)
            feat[f"{name}_delta_last2_early"] = feat[f"{name}_last2_mean"] - feat[f"{name}_early_mean"]
            feat[f"{name}_delta_last_first"]  = feat[f"{name}_last"] - feat[f"{name}_first"]

            mask = (~np.isnan(vals)) & (~np.isnan(day_idx))
            feat[f"{name}_slope"] = np.polyfit(day_idx[mask], vals[mask], 1)[0] if mask.sum() >= 2 else np.nan

            feat[f"{name}_diff_mean"] = safe_nan(np.nanmean, diffs) if len(diffs) else np.nan
            feat[f"{name}_diff_std"]  = safe_nan(np.nanstd, diffs) if len(diffs) else np.nan

            feat[f"{name}_missing"] = int(np.isnan(vals).sum())
            feat[f"{name}_n_obs"]   = int(np.sum(~np.isnan(vals)))

        # threshold counts
        hr, sbp, dbp, temp, rr = vit["hr"], vit["sbp"], vit["dbp"], vit["temp_c"], vit["rr"]
        feat["hr_gt100"]   = int(np.nansum(hr > 100))
        feat["hr_lt60"]    = int(np.nansum(hr < 60))
        feat["sbp_lt90"]   = int(np.nansum(sbp < 90))
        feat["sbp_gt160"]  = int(np.nansum(sbp > 160))
        feat["temp_gt38"]  = int(np.nansum(temp > 38))
        feat["temp_lt36"]  = int(np.nansum(temp < 36))
        feat["rr_gt22"]    = int(np.nansum(rr > 22))
        feat["map_lt65"]   = int(np.nansum(map_est < 65))
        feat["shock_gt1"]  = int(np.nansum(shock_index > 1.0))
        feat["pp_lt30"]    = int(np.nansum(pulse_pressure < 30))

        # keyword counts (all + last2)
        for kw, pat in KW_PATTERNS:
            key = kw.replace(" ", "_").replace("/", "_")
            feat[f"all_kw_{key}"] = len(pat.findall(note_all))
            feat[f"last2_kw_{key}"] = len(pat.findall(note_last2))

        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    out = base.merge(feat_df, on="stay_id", how="left")
    return out

print("=== [FE] Building features ===")
train_df = build_features(stays_train)
test_df  = build_features(stays_test)

# -----------------------------
# 5) Schema / cols
# -----------------------------
LABEL_COL = "discharge_ready_day11"
ID_COLS = ["stay_id", "patient_id"]
TEXT_COLS = ["note_text_all", "note_text_last2"]

# Base categorical cols (dynamic: include if present)
CAT_CANDIDATES = ["unit_type", "sex", "insurance", "zip3", "admission_reason", "primary_chronic"]
CAT_COLS = [c for c in CAT_CANDIDATES if c in train_df.columns]

# numeric cols = all except ids, label, text, cats
DROP_FOR_NUM = set(ID_COLS + [LABEL_COL] + TEXT_COLS + CAT_COLS)
NUM_COLS = [c for c in train_df.columns if c not in DROP_FOR_NUM]

print(f"[SCHEMA] CAT_COLS={CAT_COLS} | NUM_COLS={len(NUM_COLS)} | TEXT_COLS={TEXT_COLS}")

X_train = train_df.drop(columns=[LABEL_COL])
X_test  = test_df.copy()

# -----------------------------
# 6) Fixed CV folds
# -----------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_id = np.zeros(len(y), dtype=int)
for f, (_, va) in enumerate(skf.split(np.zeros(len(y)), y)):
    fold_id[va] = f

# -----------------------------
# 7) Load P* anchor
# -----------------------------
PSTAR_PATH = CKPT_ROOT / "PSTAR.json"
pstar = None
if PSTAR_PATH.exists():
    try:
        pstar = json.loads(PSTAR_PATH.read_text(encoding="utf-8"))
    except Exception:
        pstar = None

if pstar is None:
    # fallback if missing
    pstar = dict(iteration_tag="NONE", oof_macro_f1=-1.0, fold_std=999.0, threshold=0.5, pos_rate=train_prev)

print("=== P* (loaded) ===")
print(json.dumps(pstar, indent=2))

PSTAR_MACRO = float(pstar.get("oof_macro_f1", -1.0))
PSTAR_STD   = float(pstar.get("fold_std", 999.0))
PSTAR_POS   = float(pstar.get("pos_rate", train_prev))

# -----------------------------
# 8) Helpers: threshold sweep + gate
# -----------------------------
def threshold_sweep(oof_proba: np.ndarray, y_true: np.ndarray, fold_id: np.ndarray, thresholds=None) -> pd.DataFrame:
    if thresholds is None:
        thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)

    rows = []
    for t in thresholds:
        pred = (oof_proba >= t).astype(int)
        macro = f1_score(y_true, pred, average="macro")
        f10 = f1_score(y_true, pred, pos_label=0)
        f11 = f1_score(y_true, pred, pos_label=1)
        pos_rate = float(pred.mean())
        pos_gap = float(abs(pos_rate - float(y_true.mean())))

        fold_scores = []
        for f in range(N_SPLITS):
            idx = np.where(fold_id == f)[0]
            if len(idx) == 0:
                continue
            fold_scores.append(f1_score(y_true[idx], pred[idx], average="macro"))
        fold_mean = float(np.mean(fold_scores)) if fold_scores else np.nan
        fold_std  = float(np.std(fold_scores)) if fold_scores else np.nan

        rows.append(dict(
            threshold=float(t),
            macro_f1=float(macro),
            f1_0=float(f10),
            f1_1=float(f11),
            pos_rate=pos_rate,
            pos_gap=pos_gap,
            fold_mean=fold_mean,
            fold_std=fold_std
        ))
    return pd.DataFrame(rows)

def pick_threshold(sweep_df: pd.DataFrame, policy: str) -> pd.Series:
    df = sweep_df.copy()
    if policy == "prev_first":
        # prioritize prevalence alignment (pos_gap), then macro
        df = df.sort_values(by=["pos_gap", "macro_f1", "fold_std", "threshold"],
                            ascending=[True, False, True, True])
    else:
        # macro_first
        df = df.sort_values(by=["macro_f1", "fold_std", "pos_gap", "threshold"],
                            ascending=[False, True, True, True])
    return df.iloc[0]

def passes_gate(candidate_row: dict) -> bool:
    if PSTAR_MACRO < 0:
        return True
    if candidate_row["macro_f1"] < PSTAR_MACRO + MIN_DELTA_MACRO:
        return False
    if candidate_row["fold_std"] > PSTAR_STD + MAX_STD_INCREASE:
        return False
    if candidate_row["pos_gap"] > MAX_POS_GAP:
        return False
    return True

# -----------------------------
# 9) Model OOF generators
# -----------------------------
def build_base_preprocessor():
    num_pipe = Pipeline(steps=[
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ])
    cat_pipe = OneHotEncoder(handle_unknown="ignore")
    pre = ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUM_COLS),
            ("cat", cat_pipe, CAT_COLS),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )
    return pre

def build_fold_cache_for_preprocessor(pre: ColumnTransformer, X: pd.DataFrame, y: np.ndarray):
    cache = []
    for fold, (tr, va) in enumerate(skf.split(np.zeros(len(y)), y)):
        pre_f = clone(pre)
        X_tr = pre_f.fit_transform(X.iloc[tr], y[tr])
        X_va = pre_f.transform(X.iloc[va])
        cache.append(dict(fold=fold, tr_idx=tr, va_idx=va, pre=pre_f, X_tr=X_tr, y_tr=y[tr], X_va=X_va))
    return cache

def oof_base_lr_from_cache(cache, C: float) -> np.ndarray:
    oof = np.zeros(len(y), dtype=float)
    for item in cache:
        clf = LogisticRegression(
            solver="liblinear",
            penalty="l1",
            C=float(C),
            max_iter=5000,
            random_state=SEED
        )
        clf.fit(item["X_tr"], item["y_tr"])
        oof[item["va_idx"]] = clf.predict_proba(item["X_va"])[:, 1]
    return oof

def fit_full_base_lr(X: pd.DataFrame, y: np.ndarray, C: float):
    pre = build_base_preprocessor()
    Xtr = pre.fit_transform(X, y)
    clf = LogisticRegression(
        solver="liblinear",
        penalty="l1",
        C=float(C),
        max_iter=5000,
        random_state=SEED
    )
    clf.fit(Xtr, y)
    return dict(kind="base_lr", C=float(C), pre=pre, clf=clf)

def predict_full_base_lr(bundle, X: pd.DataFrame) -> np.ndarray:
    Xte = bundle["pre"].transform(X)
    return bundle["clf"].predict_proba(Xte)[:, 1]

def fit_te_maps(df: pd.DataFrame, y: np.ndarray, cat_cols, alpha: float):
    global_mean = float(np.mean(y))
    maps = {}
    for c in cat_cols:
        tmp = pd.DataFrame({c: df[c].astype(str).values, "y": y})
        stats = tmp.groupby(c)["y"].agg(["mean", "count"])
        enc = (stats["mean"] * stats["count"] + alpha * global_mean) / (stats["count"] + alpha)
        maps[c] = enc.to_dict()
    return maps, global_mean

def apply_te(df: pd.DataFrame, maps, global_mean: float, cat_cols):
    out = pd.DataFrame(index=df.index)
    for c in cat_cols:
        out[c + "_te"] = df[c].astype(str).map(maps.get(c, {})).fillna(global_mean).astype(float)
    return out

def oof_hgb_te(X: pd.DataFrame, y: np.ndarray, params: dict, alpha: float) -> np.ndarray:
    oof = np.zeros(len(y), dtype=float)
    for fold, (tr, va) in enumerate(skf.split(np.zeros(len(y)), y)):
        X_tr = X.iloc[tr].copy()
        X_va = X.iloc[va].copy()

        maps, gmean = fit_te_maps(X_tr, y[tr], CAT_COLS, alpha=alpha)
        Xtr_te = apply_te(X_tr, maps, gmean, CAT_COLS)
        Xva_te = apply_te(X_va, maps, gmean, CAT_COLS)

        feats = NUM_COLS + list(Xtr_te.columns)
        Xtr_mat = pd.concat([X_tr[NUM_COLS].reset_index(drop=True), Xtr_te.reset_index(drop=True)], axis=1)
        Xva_mat = pd.concat([X_va[NUM_COLS].reset_index(drop=True), Xva_te.reset_index(drop=True)], axis=1)

        imp = SimpleImputer(strategy="median")
        Xtr_imp = imp.fit_transform(Xtr_mat)
        Xva_imp = imp.transform(Xva_mat)

        model = HistGradientBoostingClassifier(
            random_state=SEED,
            **params
        )
        model.fit(Xtr_imp, y[tr])
        oof[va] = model.predict_proba(Xva_imp)[:, 1]
    return oof

def fit_full_hgb_te(X: pd.DataFrame, y: np.ndarray, params: dict, alpha: float):
    maps, gmean = fit_te_maps(X, y, CAT_COLS, alpha=alpha)
    X_te = apply_te(X, maps, gmean, CAT_COLS)
    X_mat = pd.concat([X[NUM_COLS].reset_index(drop=True), X_te.reset_index(drop=True)], axis=1)
    imp = SimpleImputer(strategy="median")
    X_imp = imp.fit_transform(X_mat)
    model = HistGradientBoostingClassifier(random_state=SEED, **params)
    model.fit(X_imp, y)
    return dict(kind="hgb_te", params=params, alpha=float(alpha), maps=maps, global_mean=gmean, imp=imp, model=model)

def predict_full_hgb_te(bundle, X: pd.DataFrame) -> np.ndarray:
    X_te = apply_te(X, bundle["maps"], bundle["global_mean"], CAT_COLS)
    X_mat = pd.concat([X[NUM_COLS].reset_index(drop=True), X_te.reset_index(drop=True)], axis=1)
    X_imp = bundle["imp"].transform(X_mat)
    return bundle["model"].predict_proba(X_imp)[:, 1]

def oof_text_lr(text_series: pd.Series, y: np.ndarray, C: float) -> np.ndarray:
    oof = np.zeros(len(y), dtype=float)

    word_params = dict(
        ngram_range=(1,2),
        max_features=int(TEXT_WORD_MAX_FEATS),
        min_df=2,
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b"
    )
    char_params = dict(
        analyzer="char_wb",
        ngram_range=(3,5),
        max_features=int(TEXT_CHAR_MAX_FEATS),
        min_df=2,
        lowercase=True
    )

    for fold, (tr, va) in enumerate(skf.split(np.zeros(len(y)), y)):
        tr_text = text_series.iloc[tr].fillna("").astype(str).values
        va_text = text_series.iloc[va].fillna("").astype(str).values

        vw = TfidfVectorizer(**word_params)
        vc = TfidfVectorizer(**char_params)

        Xtr_w = vw.fit_transform(tr_text)
        Xva_w = vw.transform(va_text)
        Xtr_c = vc.fit_transform(tr_text)
        Xva_c = vc.transform(va_text)

        Xtr = sparse.hstack([Xtr_w, Xtr_c]).tocsr()
        Xva = sparse.hstack([Xva_w, Xva_c]).tocsr()

        clf = LogisticRegression(
            solver="liblinear",
            penalty="l2",
            C=float(C),
            max_iter=5000,
            random_state=SEED
        )
        clf.fit(Xtr, y[tr])
        oof[va] = clf.predict_proba(Xva)[:, 1]

    return oof

def fit_full_text_lr(text_series: pd.Series, y: np.ndarray, C: float):
    word_params = dict(
        ngram_range=(1,2),
        max_features=int(TEXT_WORD_MAX_FEATS),
        min_df=2,
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b"
    )
    char_params = dict(
        analyzer="char_wb",
        ngram_range=(3,5),
        max_features=int(TEXT_CHAR_MAX_FEATS),
        min_df=2,
        lowercase=True
    )
    vw = TfidfVectorizer(**word_params)
    vc = TfidfVectorizer(**char_params)
    txt = text_series.fillna("").astype(str).values
    Xw = vw.fit_transform(txt)
    Xc = vc.fit_transform(txt)
    X = sparse.hstack([Xw, Xc]).tocsr()

    clf = LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=float(C),
        max_iter=5000,
        random_state=SEED
    )
    clf.fit(X, y)
    return dict(kind="text_lr", C=float(C), vw=vw, vc=vc, clf=clf)

def predict_full_text_lr(bundle, text_series: pd.Series) -> np.ndarray:
    txt = text_series.fillna("").astype(str).values
    Xw = bundle["vw"].transform(txt)
    Xc = bundle["vc"].transform(txt)
    X = sparse.hstack([Xw, Xc]).tocsr()
    return bundle["clf"].predict_proba(X)[:, 1]

# -----------------------------
# 10) Train OOF base learners (fixed CV)
# -----------------------------
print("=== [OOF] Training base learners (fixed CV) ===")

model_oof = {}   # name -> oof_proba
model_spec = {}  # name -> spec for refit

# BASE LR
base_pre = build_base_preprocessor()
base_cache = build_fold_cache_for_preprocessor(base_pre, X_train, y)
for C in BASE_LR_C_GRID:
    name = f"BASE_LR_C{C:.2f}"
    oof = oof_base_lr_from_cache(base_cache, C)
    model_oof[name] = oof
    model_spec[name] = dict(kind="base_lr", C=float(C))

# HGB TE
for i, params in enumerate(HGB_PARAM_GRID):
    name = f"HGB_TE_v{i}_d{params['max_depth']}_lr{params['learning_rate']}"
    oof = oof_hgb_te(X_train, y, params=params, alpha=TE_ALPHA)
    model_oof[name] = oof
    model_spec[name] = dict(kind="hgb_te", params=params, alpha=float(TE_ALPHA))

# TEXT model (true 3rd signal)
text_candidates = []
if ENABLE_TEXT_MODEL:
    if TEXT_USE_LAST2:
        text_candidates.append(("TEXT_last2", X_train["note_text_last2"]))
    if TEXT_USE_ALL:
        text_candidates.append(("TEXT_all", X_train["note_text_all"]))

    for prefix, txt in text_candidates:
        for C in TEXT_LR_C_GRID:
            name = f"{prefix}_TFIDF_LR_C{C:.1f}"
            oof = oof_text_lr(txt, y, C=C)
            model_oof[name] = oof
            model_spec[name] = dict(kind="text_lr", C=float(C), text_source=prefix)

# -----------------------------
# 11) Build candidate leaderboard (single + blends)
# -----------------------------
print("\n=== [EVAL] Candidate search (single / blends) ===")

def eval_candidate(name: str, oof_proba: np.ndarray, policy: str, extra=None) -> dict:
    sw = threshold_sweep(oof_proba, y, fold_id)
    best = pick_threshold(sw, policy=policy)
    pred = (oof_proba >= float(best["threshold"])).astype(int)

    cm = confusion_matrix(y, pred).tolist()
    auc = float(roc_auc_score(y, oof_proba)) if len(np.unique(y)) > 1 else float("nan")

    row = dict(
        name=name,
        type=extra.get("type", "single") if extra else "single",
        policy=policy,
        macro_f1=float(best["macro_f1"]),
        threshold=float(best["threshold"]),
        pos_rate=float(best["pos_rate"]),
        pos_gap=float(best["pos_gap"]),
        fold_mean=float(best["fold_mean"]),
        fold_std=float(best["fold_std"]),
        f1_0=float(best["f1_0"]),
        f1_1=float(best["f1_1"]),
        auc=auc,
        confusion_matrix=cm,
    )
    if extra:
        row.update(extra)
    return row

candidates = []

# Singles
for mname, oof in model_oof.items():
    for policy in ["macro_first", "prev_first"]:
        candidates.append(eval_candidate(mname, oof, policy=policy, extra=dict(type="single", models=mname)))

# Pick best base/hgb/text by macro_first (for blend generation)
def best_model_by_prefix(prefix: str):
    rows = [r for r in candidates if r["type"] == "single" and r["policy"] == "macro_first" and r["name"].startswith(prefix)]
    if not rows:
        return None
    rows = sorted(rows, key=lambda r: (-r["macro_f1"], r["fold_std"], r["pos_gap"]))
    return rows[0]["name"]

best_base = best_model_by_prefix("BASE_LR")
best_hgb  = best_model_by_prefix("HGB_TE")
best_text = best_model_by_prefix("TEXT_") if ENABLE_TEXT_MODEL else None

# Blends (2-model)
def add_blend2(a, b, w_a, policy):
    oof = w_a * model_oof[a] + (1.0 - w_a) * model_oof[b]
    name = f"ENS2[{a}+{b}]_w{w_a:.2f}"
    candidates.append(eval_candidate(
        name, oof, policy=policy,
        extra=dict(type="blend2", models=f"{a}+{b}", w=f"{w_a:.2f},{1.0-w_a:.2f}", w_a=float(w_a), w_b=float(1.0-w_a))
    ))

if best_base and best_hgb:
    for w in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        for policy in ["macro_first", "prev_first"]:
            add_blend2(best_base, best_hgb, w, policy)

if best_text:
    if best_base:
        for w in [0.2, 0.35, 0.5, 0.65, 0.8]:
            for policy in ["macro_first", "prev_first"]:
                add_blend2(best_base, best_text, w, policy)
    if best_hgb:
        for w in [0.2, 0.35, 0.5, 0.65, 0.8]:
            for policy in ["macro_first", "prev_first"]:
                add_blend2(best_hgb, best_text, w, policy)

# Blends (3-model)
def add_blend3(a, b, c, w_a, w_b, w_c, policy):
    oof = w_a * model_oof[a] + w_b * model_oof[b] + w_c * model_oof[c]
    name = f"ENS3[{a}+{b}+{c}]_w{w_a:.2f}-{w_b:.2f}-{w_c:.2f}"
    candidates.append(eval_candidate(
        name, oof, policy=policy,
        extra=dict(type="blend3", models=f"{a}+{b}+{c}", w=f"{w_a:.2f},{w_b:.2f},{w_c:.2f}", w_a=float(w_a), w_b=float(w_b), w_c=float(w_c))
    ))

if best_base and best_hgb and best_text:
    # coarse but useful grid
    for w_a in [0.2, 0.35, 0.5]:
        for w_c in [0.15, 0.25, 0.35]:
            w_b = 1.0 - w_a - w_c
            if w_b <= 0:
                continue
            for policy in ["macro_first", "prev_first"]:
                add_blend3(best_base, best_hgb, best_text, w_a, w_b, w_c, policy)

cand_df = pd.DataFrame(candidates)
cand_df["delta_vs_pstar_macro"] = cand_df["macro_f1"] - PSTAR_MACRO if PSTAR_MACRO >= 0 else np.nan
cand_df = cand_df.sort_values(by=["macro_f1", "fold_std", "pos_gap"], ascending=[False, True, True]).reset_index(drop=True)

print("\n=== Candidate leaderboard (top 15) ===")
show_cols = ["name","type","macro_f1","delta_vs_pstar_macro","threshold","pos_rate","pos_gap","fold_std","f1_0","f1_1","auc","policy"]
print(cand_df[show_cols].head(15).to_string(index=False))

# Save ablation summary
ABL_PATH = ART_ROOT / f"{ITER_TAG}_ablation_summary.csv"
cand_df.to_csv(ABL_PATH, index=False)
print(f"\n✅ Saved ablation summary -> {ABL_PATH}")

# -----------------------------
# 12) Selection w/ hard gate + SAFE_MODE fallback
# -----------------------------
eligible = []
for _, r in cand_df.iterrows():
    row = r.to_dict()
    if passes_gate(row):
        eligible.append(row)

if eligible:
    # pick best eligible by macro_f1 then stability
    eligible = sorted(eligible, key=lambda r: (-r["macro_f1"], r["fold_std"], r["pos_gap"]))
    selected = eligible[0]
    decision = "SELECT_CANDIDATE (passes gate vs P*)"
    submission_mode = "use_candidate"
else:
    selected = cand_df.iloc[0].to_dict()
    decision = "NO_CANDIDATE_PASSES_GATE -> SAFE_MODE uses P* candidate"
    submission_mode = "SAFE_MODE_copy_PSTAR_candidate"

print("\n=== Selection decision ===")
print("Decision:", decision)
print("Selected:", selected["name"], "| type=", selected["type"], "| macro_f1=", round(selected["macro_f1"],6),
      "| thr=", selected["threshold"], "| pos_rate=", round(selected["pos_rate"],3), "| policy=", selected["policy"])

# -----------------------------
# 13) Build selected OOF + group diagnostics
# -----------------------------
def build_selected_oof(selected_row: dict) -> np.ndarray:
    name = selected_row["name"]
    if selected_row["type"] == "single":
        return model_oof[selected_row["models"]]
    # blend2 / blend3 names store weights in row
    if selected_row["type"] == "blend2":
        a, b = selected_row["models"].split("+")
        w = [float(x) for x in str(selected_row["w"]).split(",")]
        return w[0]*model_oof[a] + w[1]*model_oof[b]
    if selected_row["type"] == "blend3":
        a, b, c = selected_row["models"].split("+")
        w = [float(x) for x in str(selected_row["w"]).split(",")]
        return w[0]*model_oof[a] + w[1]*model_oof[b] + w[2]*model_oof[c]
    raise ValueError("Unknown candidate type")

sel_oof = build_selected_oof(selected)
sel_thr = float(selected["threshold"])
sel_pred = (sel_oof >= sel_thr).astype(int)

# Group diagnostics
group_cols = [c for c in ["unit_type","sex","insurance","zip3","admission_reason","primary_chronic"] if c in train_df.columns]
grp_rows = []
for gc in group_cols:
    for gval, sub in train_df.assign(y_true=y, y_pred=sel_pred).groupby(gc, dropna=False):
        yt = sub["y_true"].values
        yp = sub["y_pred"].values
        cm = confusion_matrix(yt, yp, labels=[0,1])
        tn, fp, fn, tp = cm.ravel()
        grp_rows.append(dict(
            group_col=gc,
            group_val=str(gval),
            n=int(len(sub)),
            prevalence=float(yt.mean()),
            pred_pos_rate=float(yp.mean()),
            fp=int(fp),
            fn=int(fn),
            macro_f1=float(f1_score(yt, yp, average="macro")) if len(np.unique(yt))>1 else float("nan")
        ))
GROUP_PATH = ART_ROOT / f"{ITER_TAG}_group_error_summary.csv"
pd.DataFrame(grp_rows).sort_values(["group_col","n"], ascending=[True, False]).to_csv(GROUP_PATH, index=False)
print(f"✅ Saved group diagnostics -> {GROUP_PATH}")

# Save OOF predictions + threshold sweep of selected
OOF_PATH = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
pd.DataFrame({
    "stay_id": train_df["stay_id"].values,
    "y_true": y,
    "oof_proba": sel_oof,
    "oof_pred": sel_pred,
    "fold": fold_id
}).to_csv(OOF_PATH, index=False)
print(f"✅ Saved OOF -> {OOF_PATH}")

SWEEP_PATH = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"
threshold_sweep(sel_oof, y, fold_id).to_csv(SWEEP_PATH, index=False)
print(f"✅ Saved sweep -> {SWEEP_PATH}")

# -----------------------------
# 14) Fit selected candidate on full train + predict test
# -----------------------------
def fit_and_predict_component(model_name: str):
    spec = model_spec[model_name]
    if spec["kind"] == "base_lr":
        bundle = fit_full_base_lr(X_train, y, C=spec["C"])
        test_proba = predict_full_base_lr(bundle, X_test)
        return bundle, test_proba
    if spec["kind"] == "hgb_te":
        bundle = fit_full_hgb_te(X_train, y, params=spec["params"], alpha=spec["alpha"])
        test_proba = predict_full_hgb_te(bundle, X_test)
        return bundle, test_proba
    if spec["kind"] == "text_lr":
        # choose correct text source
        src = spec.get("text_source", "TEXT_last2")
        if src == "TEXT_all":
            train_txt = X_train["note_text_all"]
            test_txt  = X_test["note_text_all"]
        else:
            train_txt = X_train["note_text_last2"]
            test_txt  = X_test["note_text_last2"]
        bundle = fit_full_text_lr(train_txt, y, C=spec["C"])
        test_proba = predict_full_text_lr(bundle, test_txt)
        return bundle, test_proba
    raise ValueError("Unknown model kind")

def build_selected_test_proba(selected_row: dict):
    bundles = {}
    if selected_row["type"] == "single":
        m = selected_row["models"]
        b, p = fit_and_predict_component(m)
        bundles[m] = b
        return p, bundles

    if selected_row["type"] == "blend2":
        a, bname = selected_row["models"].split("+")
        w = [float(x) for x in str(selected_row["w"]).split(",")]
        ba, pa = fit_and_predict_component(a)
        bb, pb = fit_and_predict_component(bname)
        bundles[a] = ba
        bundles[bname] = bb
        return w[0]*pa + w[1]*pb, bundles

    if selected_row["type"] == "blend3":
        a, bname, cname = selected_row["models"].split("+")
        w = [float(x) for x in str(selected_row["w"]).split(",")]
        ba, pa = fit_and_predict_component(a)
        bb, pb = fit_and_predict_component(bname)
        bc, pc = fit_and_predict_component(cname)
        bundles[a] = ba
        bundles[bname] = bb
        bundles[cname] = bc
        return w[0]*pa + w[1]*pb + w[2]*pc, bundles

    raise ValueError("Unknown candidate type")

# Always fit the selected candidate snapshot for audit
test_proba_candidate, bundles = build_selected_test_proba(selected)
test_pred_candidate = (test_proba_candidate >= sel_thr).astype(int)

CAND_PATH = ART_ROOT / f"{TAG}_{BRANCH}_block{BLOCK_ID}_{ITER_TAG}_candidate.csv"
pd.DataFrame({
    "stay_id": test_df["stay_id"].values,
    "discharge_ready_day11": test_pred_candidate,
    "proba": test_proba_candidate
}).to_csv(CAND_PATH, index=False)
print(f"✅ Wrote candidate snapshot -> {CAND_PATH} | pos_rate: {float(test_pred_candidate.mean()):.3f}")

# SAFE_MODE submission: copy P* candidate if gate failed
def find_pstar_candidate_csv():
    # prefer latest P* iteration tag in PSTAR.json
    it = pstar.get("iteration_tag", None)
    if isinstance(it, str) and re.match(r"iter\d{4}$", it):
        # search artifact dir for a candidate file containing that iter tag
        for p in ART_ROOT.glob(f"*{it}*candidate.csv"):
            return p
    # fallback: any candidate file (best effort)
    cands = sorted(ART_ROOT.glob("*candidate.csv"))
    return cands[-1] if cands else None

if submission_mode == "use_candidate":
    sub_df = pd.DataFrame({
        "stay_id": test_df["stay_id"].values,
        "discharge_ready_day11": test_pred_candidate
    })
    threshold_used_submit = sel_thr
    submission_pos_rate = float(test_pred_candidate.mean())
else:
    pstar_cand = find_pstar_candidate_csv()
    if pstar_cand is None:
        print("[SAFE_MODE] Could not find P* candidate csv. Fallback: submit candidate anyway (still logged).")
        sub_df = pd.DataFrame({
            "stay_id": test_df["stay_id"].values,
            "discharge_ready_day11": test_pred_candidate
        })
        threshold_used_submit = sel_thr
        submission_pos_rate = float(test_pred_candidate.mean())
    else:
        print(f"[SAFE_MODE] Copy P* candidate -> {pstar_cand.name}")
        tmp = pd.read_csv(pstar_cand)
        # standardize col names
        if "discharge_ready_day11" not in tmp.columns and tmp.shape[1] >= 2:
            tmp.columns = ["stay_id", "discharge_ready_day11"] + list(tmp.columns[2:])
        sub_df = tmp[["stay_id","discharge_ready_day11"]].copy()
        threshold_used_submit = float(pstar.get("threshold", 0.5))
        submission_pos_rate = float(sub_df["discharge_ready_day11"].mean())

# Write submission
sub_df.to_csv(SUB_PATH, index=False)
print("\n=== Submission written ===")
print("SUB_PATH:", str(SUB_PATH))
print("submission_mode:", submission_mode)
print("Threshold used (submit):", threshold_used_submit)
print("Submission pos_rate:", round(submission_pos_rate, 3))

# -----------------------------
# 15) Save checkpoint (auditable & reversible)
# -----------------------------
ckpt_payload = {
    "iter_tag": ITER_TAG,
    "branch": BRANCH,
    "selected": {k: selected[k] for k in selected.keys() if k not in ["confusion_matrix"]},
    "selected_confusion_matrix": selected.get("confusion_matrix", None),
    "schema": {
        "cat_cols": CAT_COLS,
        "num_cols_n": len(NUM_COLS),
        "text_cols": TEXT_COLS,
        "id_cols": ID_COLS
    },
    "gate": {
        "pstar_macro": PSTAR_MACRO,
        "pstar_std": PSTAR_STD,
        "pstar_pos_rate": PSTAR_POS,
        "min_delta_macro": MIN_DELTA_MACRO,
        "max_std_increase": MAX_STD_INCREASE,
        "max_pos_gap": MAX_POS_GAP,
        "passed": bool(passes_gate(selected))
    },
    "timestamp": utc_ts(),
    "seed": SEED
}

# Save bundles for reproducibility (even if SAFE_MODE)
try:
    joblib.dump({"bundles": bundles, "selected": selected, "threshold": sel_thr}, ITER_CKPT_DIR / "model_bundle.joblib")
except Exception as e:
    print("[WARN] Failed to save model_bundle.joblib:", repr(e))

(ITER_CKPT_DIR / "ckpt.json").write_text(json.dumps(ckpt_payload, indent=2), encoding="utf-8")
print(f"✅ Saved checkpoint -> {ITER_CKPT_DIR}")

# -----------------------------
# 16) Decision note + PSTAR update (only if gate passes)
# -----------------------------
DECISION_PATH = ART_ROOT / f"{ITER_TAG}_DECISION.md"
note_lines = []
note_lines.append(f"# {TAG} {ITER_TAG} Decision\n")
note_lines.append(f"- timestamp: {ckpt_payload['timestamp']}")
note_lines.append(f"- branch: {BRANCH}")
note_lines.append(f"- decision: {decision}")
note_lines.append(f"- selected: `{selected['name']}`")
note_lines.append(f"- type: {selected['type']} | policy: {selected['policy']}")
note_lines.append(f"- oof_macro_f1: {selected['macro_f1']:.6f} (P*: {PSTAR_MACRO:.6f})")
note_lines.append(f"- threshold(oof): {selected['threshold']:.2f} | oof_pos_rate: {selected['pos_rate']:.3f} | pos_gap: {selected['pos_gap']:.3f}")
note_lines.append(f"- fold_std: {selected['fold_std']:.6f} (P*: {PSTAR_STD:.6f})")
note_lines.append(f"- safe_submission_mode: {submission_mode}")
note_lines.append(f"\n## What changed\n- Added SAFE patient history features from ed_cost + admissions (no cross-task labels; no los_days/discharge_weekday).\n- Added real text model: TFIDF(word+char) on stay notes and tri-blend search.\n- Hard gate + Safe Mode enforced (never worse than P* submission).\n")
note_lines.append("## Risks\n- Text features can overfit; mitigated by strict CV-in-loop vectorizer fitting + hard gate.\n- Patient history aggregation may be noisy; mitigated by hard gate and Safe Mode.\n")
DECISION_PATH.write_text("\n".join(note_lines), encoding="utf-8")
print(f"✅ Wrote decision note -> {DECISION_PATH}")

if passes_gate(selected):
    new_pstar = {
        "iteration_tag": ITER_TAG,
        "oof_macro_f1": float(selected["macro_f1"]),
        "threshold": float(selected["threshold"]),
        "fold_mean": float(selected["fold_mean"]),
        "fold_std": float(selected["fold_std"]),
        "pos_rate": float(selected["pos_rate"]),
        "f1_0": float(selected["f1_0"]),
        "f1_1": float(selected["f1_1"]),
        "confusion_matrix": selected.get("confusion_matrix", None),
        "branch": BRANCH,
        "selected_name": selected["name"],
        "timestamp": utc_ts()
    }
    PSTAR_PATH.write_text(json.dumps(new_pstar, indent=2), encoding="utf-8")
    print(f"✅ Updated PSTAR.json -> {ITER_TAG} macro= {new_pstar['oof_macro_f1']:.6f}")
else:
    print("ℹ️ PSTAR.json unchanged (no candidate passed hard gate).")

# -----------------------------
# 17) Append iteration log + consultant packet
# -----------------------------
log_rec = {
    "iter_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": utc_ts(),
    "phase": "Modeling",
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "pstar_loaded": pstar,
    "selected": {k: selected[k] for k in selected.keys() if k != "confusion_matrix"},
    "selected_confusion_matrix": selected.get("confusion_matrix", None),
    "decision": decision,
    "submission_mode": submission_mode,
    "submission_pos_rate": submission_pos_rate,
    "threshold_submit": threshold_used_submit,
    "paths": {
        "submission": str(SUB_PATH),
        "candidate_snapshot": str(CAND_PATH),
        "ablation": str(ABL_PATH),
        "oof": str(OOF_PATH),
        "sweep": str(SWEEP_PATH),
        "group_diag": str(GROUP_PATH),
        "decision_note": str(DECISION_PATH),
        "checkpoint_dir": str(ITER_CKPT_DIR),
        "pstar": str(PSTAR_PATH),
    }
}

try:
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(log_rec, ensure_ascii=False) + "\n")
    print(f"✅ Appended iteration log -> {LOG_PATH}")
except Exception as e:
    print("[WARN] Failed to append log:", repr(e))

packet_path = CONSULT_ROOT / f"{TAG}_{ITER_TAG}_packet.json"
try:
    packet_path.write_text(json.dumps(log_rec, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"✅ Wrote consultant packet -> {packet_path}")
except Exception as e:
    print("[WARN] Failed to write consultant packet:", repr(e))

print("\nDONE ✅")
print("Key outputs:")
print(" - Submission       :", SUB_PATH, "| mode:", submission_mode)
print(" - Candidate        :", CAND_PATH)
print(" - Ablation         :", ABL_PATH)
print(" - OOF              :", OOF_PATH)
print(" - Sweep            :", SWEEP_PATH)
print(" - Group diag       :", GROUP_PATH)
print(" - Decision note    :", DECISION_PATH)
print(" - PSTAR            :", PSTAR_PATH)
print(" - Checkpoint dir   :", ITER_CKPT_DIR)

=== PREFLIGHT ===
PROJECT_ROOT: D:\AgentDs\agent_ds_healthcare
ART_ROOT    : D:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT   : D:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
CONSULT_ROOT: D:\AgentDs\agent_ds_healthcare\consultant_packets
LOG_PATH    : D:\AgentDs\agent_ds_healthcare\clai_ch3_v3_iteration_detail.jsonl
ITER_ID=22  ITER_TAG=iter0022  BLOCK_ID=5  BLOCK_POS=0  BRANCH=W6_patientHistory+TextTriBlend
[DATA] Train prevalence(y=1): 0.656 | n=1000
[LEAKAGE_CHECK] Max day in vitals_timeseries: 10 (OK <= 10)
=== [FE] Building features ===
[SCHEMA] CAT_COLS=['unit_type', 'sex', 'insurance', 'zip3', 'admission_reason', 'primary_chronic'] | NUM_COLS=255 | TEXT_COLS=['note_text_all', 'note_text_last2']
=== P* (loaded) ===
{
  "tag": "clai_ch3_v3",
  "pstar": {
    "iteration_tag": "iter0020",
    "oof_macro_f1": 0.8249641287951847,
    "threshold": 0.63,
    "fold_mean": 0.8248897805164553,
    "fold_std": 0.024420082144040903,
    "pos_rate": 0.665,
    "f1_0":

# Iteration 15
- 0.8168

In [58]:
# =========================================================
# clai_ch3_v3 — Iteration Runner (single cell)
# Branch: W7_EDHistory_ablation + robust P* recovery + hard gate
#
# GOALS (behavior-first):
# - Reversible: this iter ONLY adds ED-history features (primary_chronic + prior ED utilization/cost)
# - Auditable: writes OOF + sweep + ablation + decision note + checkpoint + consultant packet + append-only log
# - Attributable: explicit "what_changed / evidence / risks / fallback" captured in DECISION.md + jsonl
# - Safe: never worse than best-known P* (SAFE MODE copies P* candidate or uses P* pipeline)
#
# CRITICAL GUARDRAILS:
# - Fixed CV: StratifiedKFold(5, shuffle=True, random_state=42)
# - Leakage: vitals_timeseries max(day) must be <= 10 (no day11 vitals)
# - Avoid obvious future-info sources (e.g. discharge_notes, admissions los/discharge_weekday); NOT used here
# - Hard gate MUST reject NaN / missing P* / schema bugs (prevents iter0022-type disaster)
# =========================================================

# -----------------------------
# 0) Thread limits (stability)
# -----------------------------
import os
for _v in ["OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_v, "1")

import json, re, random, shutil, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier

import joblib

warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice")
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------------
# 1) Config + paths
# ---------------------
TEAM, TASK, VERSION = "clai", "ch3", "v3"
TAG = f"{TEAM}_{TASK}_{VERSION}"
SEED = 42
N_SPLITS = 5

np.random.seed(SEED)
random.seed(SEED)

BRANCH = "W7_EDHistory_ablation"   # ✅ single new info source (ED history)
CHANGESET = [
    "NEW INFO SOURCE: join ED history features from ed_cost_{train,test}.csv: primary_chronic + prior_ed_visits_5y + prior_ed_cost_5y_usd (+ log1p).",
    "Everything else stays in the proven iter0020 direction: vitals-derived indices + LR(base) + HGB(TE dense) + blend + prevalence-aware thresholding + gate.",
    "Robust P* recovery + strict NaN-safe gate to prevent iter0022-like PSTAR corruption."
]
RISKS = [
    "ED history features might be weakly correlated or noisy; could add variance → must pass hard gate vs effective P*.",
    "Target encoding must stay inside CV folds to avoid leakage; implemented per-fold.",
]
FALLBACK = "If NO candidate passes hard gate vs effective P*, SAFE MODE copies P* candidate (or uses P* pipeline) so submission will NOT be worse."

def utc_ts() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def _find_project_root() -> Path:
    cand = Path.cwd()
    if (cand / "stays_train.csv").exists():
        return cand
    cand2 = Path("/mnt/data")
    if (cand2 / "stays_train.csv").exists():
        return cand2
    return cand

PROJECT_ROOT = _find_project_root()
ART_ROOT = PROJECT_ROOT / "artifacts" / TAG
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / TAG
CONSULT_ROOT = PROJECT_ROOT / "consultant_packets"

ART_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
CONSULT_ROOT.mkdir(parents=True, exist_ok=True)

print("=== PREFLIGHT ===")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ART_ROOT    :", ART_ROOT)
print("CKPT_ROOT   :", CKPT_ROOT)
print("CONSULT_ROOT:", CONSULT_ROOT)

def dp(name: str) -> Path:
    p = PROJECT_ROOT / name
    if p.exists():
        return p
    p2 = Path("/mnt/data") / name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find {name} under {PROJECT_ROOT} or /mnt/data")

# ---------------------
# 2) Scoped artifact bootstrap (idempotent)
# ---------------------
def _copy_if_missing(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists() and (not dst.exists()):
        shutil.copy2(src, dst)

for pat in ["iter????_oof_predictions.csv", "iter????_threshold_sweep.csv", "iter????_ablation_summary.csv", "iter????_group_error_summary.csv", "iter????_DECISION.md"]:
    for src in sorted(PROJECT_ROOT.glob(pat)):
        _copy_if_missing(src, ART_ROOT / src.name)
for src in sorted(PROJECT_ROOT.glob(f"{TAG}_*candidate*.csv")):
    _copy_if_missing(src, ART_ROOT / src.name)

# ---------------------
# 3) Append-only log path (with fallback)
# ---------------------
RAW_LOG_PATH = PROJECT_ROOT / f"{TAG}_iteration_detail.jsonl"

def ensure_appendable(path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if os.access(path, os.W_OK):
            return path
        alt = path.with_name(path.stem + "_appended" + path.suffix)
        if not alt.exists():
            try:
                shutil.copy2(path, alt)
            except Exception:
                pass
        return alt
    else:
        try:
            with open(path, "a", encoding="utf-8") as f:
                pass
            return path
        except Exception:
            alt = path.with_name(path.stem + "_appended" + path.suffix)
            with open(alt, "a", encoding="utf-8") as f:
                pass
            return alt

LOG_PATH = ensure_appendable(RAW_LOG_PATH)
print("LOG_PATH    :", LOG_PATH)

# ---------------------
# 4) Iteration id (robust): max(log, artifacts, ckpts)+1
# ---------------------
def _extract_iter_num(name: str):
    m = re.search(r"iter(\d{4})", name)
    return int(m.group(1)) if m else None

def max_iteration_id_from_log(log_path: Path) -> int:
    if not log_path.exists():
        return -1
    mx = -1
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                mx = max(mx, int(obj.get("iteration_id", -1)))
            except Exception:
                continue
    return mx

def max_iter_num_from_artifacts(art_root: Path) -> int:
    mx = -1
    for p in art_root.glob("iter????_*"):
        n = _extract_iter_num(p.name)
        if n is not None:
            mx = max(mx, n)
    return mx

def max_iter_num_from_ckpt(ckpt_root: Path) -> int:
    mx = -1
    for d in ckpt_root.glob("iter????"):
        if d.is_dir():
            n = _extract_iter_num(d.name)
            if n is not None:
                mx = max(mx, n)
    return mx

max_log = max_iteration_id_from_log(LOG_PATH)
max_art = max_iter_num_from_artifacts(ART_ROOT)
max_ckpt = max_iter_num_from_ckpt(CKPT_ROOT)

ITER_ID = max(max_log, max_art, max_ckpt) + 1
ITER_TAG = f"iter{ITER_ID:04d}"
BLOCK_ID = ITER_ID // 5
BLOCK_POS = ITER_ID % 5

print(f"ITER_ID={ITER_ID}  ITER_TAG={ITER_TAG}  BLOCK_ID={BLOCK_ID}  BLOCK_POS={BLOCK_POS}  BRANCH={BRANCH}")

# ---------------------
# 5) Load core data + leakage check
# ---------------------
stays_train = pd.read_csv(dp("stays_train.csv"))
stays_test  = pd.read_csv(dp("stays_test.csv"))
patients    = pd.read_csv(dp("patients.csv"))

with open(dp("vitals_timeseries.json"), "r", encoding="utf-8") as f:
    vit_list = json.load(f)

max_day = max(d.get("day", 0) for item in vit_list for d in item.get("days", []))
assert max_day <= 10, f"Leakage risk: found vitals day={max_day} (>10). Abort."
print(f"[LEAKAGE_CHECK] Max day in vitals_timeseries: {max_day} (OK <= 10)")

vit_map = {int(item["stay_id"]): item.get("days", []) for item in vit_list}

y_all = stays_train["discharge_ready_day11"].astype(int).values
train_prev = float(y_all.mean())
print(f"[DATA] Train prevalence(y=1): {train_prev:.3f} | n={len(y_all)}")

# ED history data (SAFE subset only)
# - ed_cost_train has label ed_cost_next3y_usd (future) => DROP
# - keep: primary_chronic, prior_ed_visits_5y, prior_ed_cost_5y_usd
ed_hist = None
try:
    ed_train = pd.read_csv(dp("ed_cost_train.csv"))
    ed_test  = pd.read_csv(dp("ed_cost_test.csv"))
    drop_cols = [c for c in ["ed_cost_next3y_usd"] if c in ed_train.columns]
    ed_train = ed_train.drop(columns=drop_cols, errors="ignore")
    ed_hist = pd.concat([ed_train, ed_test], ignore_index=True)
    ed_hist = ed_hist[["patient_id", "primary_chronic", "prior_ed_visits_5y", "prior_ed_cost_5y_usd"]].copy()
    # deduplicate (should be 1 row per patient_id; keep last if any issue)
    ed_hist = ed_hist.drop_duplicates(subset=["patient_id"], keep="last").reset_index(drop=True)
    print("[ED_HISTORY] Loaded:", ed_hist.shape, "| patients covered:", ed_hist["patient_id"].nunique())
except Exception as e:
    print("⚠️ [ED_HISTORY] Could not load ed_cost_{train,test}.csv ->", repr(e))
    ed_hist = None

# ---------------------
# 6) Deterministic folds
# ---------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_ids = np.empty(len(y_all), dtype=int)
for f, (_, va_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
    fold_ids[va_idx] = f

# ---------------------
# 7) Feature engineering (baseline + ED-history ablation)
# ---------------------
VITAL_COLS = ["hr", "sbp", "dbp", "temp_c", "rr"]

KEYWORDS = [
    "stable","improving","worsening","afebrile","fever","chills","hypotensive","tachy",
    "culture","antibiotic","abx","iv","oxygen","weaned","dyspnea","cough",
    "pain","nausea","vomit","n/v","ambulated","pt","ot","telemetry","fluids","room air","sats",
    "discharge","transfer"
]
KW_PATTERNS = [(kw, re.compile(re.escape(kw), re.IGNORECASE)) for kw in KEYWORDS]

def safe_nan(fn, arr, default=np.nan):
    try:
        with np.errstate(all="ignore"):
            return fn(arr)
    except Exception:
        return default

def build_features(stays_df: pd.DataFrame, include_ed_history: bool) -> pd.DataFrame:
    base = stays_df.merge(patients, on="patient_id", how="left")

    rows = []
    for sid in base["stay_id"].astype(int).tolist():
        days = sorted(vit_map.get(int(sid), []), key=lambda x: x.get("day", 0))
        day_idx = np.array([d.get("day", np.nan) for d in days], dtype=float)

        vit = {c: np.array([d.get(c, np.nan) for d in days], dtype=float) for c in VITAL_COLS}

        # derived per-day indices (proven strong in iter0016+)
        pulse_pressure = vit["sbp"] - vit["dbp"]
        map_est = (vit["sbp"] + 2.0 * vit["dbp"]) / 3.0
        shock_index = vit["hr"] / vit["sbp"]
        shock_index = np.where(np.isfinite(shock_index), shock_index, np.nan)
        temp_dev = vit["temp_c"] - 37.0

        series = {
            "hr": vit["hr"],
            "sbp": vit["sbp"],
            "dbp": vit["dbp"],
            "rr": vit["rr"],
            "temp_c": vit["temp_c"],
            "pulse_pressure": pulse_pressure,
            "map": map_est,
            "shock_index": shock_index,
            "temp_dev": temp_dev,
        }

        notes = [(d.get("note") or "") for d in days]
        note_all = " ".join(notes).strip()
        note_last2 = " ".join(notes[-2:]).strip()

        feat = {
            "stay_id": int(sid),
            "note_len_all": len(note_all),
            "note_len_last2": len(note_last2),
            "note_days": int(sum(1 for n in notes if str(n).strip() != "")),
            "days_recorded": int(len(days)),
            "day_first": float(day_idx[0]) if len(day_idx) else np.nan,
            "day_last": float(day_idx[-1]) if len(day_idx) else np.nan,
        }

        for name, vals in series.items():
            vals = np.array(vals, dtype=float)
            diffs = np.diff(vals) if len(vals) >= 2 else np.array([], dtype=float)

            feat[f"{name}_mean"] = safe_nan(np.nanmean, vals)
            feat[f"{name}_std"]  = safe_nan(np.nanstd,  vals)
            feat[f"{name}_min"]  = safe_nan(np.nanmin,  vals)
            feat[f"{name}_max"]  = safe_nan(np.nanmax,  vals)
            feat[f"{name}_median"] = safe_nan(np.nanmedian, vals)
            feat[f"{name}_first"] = vals[0] if len(vals) else np.nan
            feat[f"{name}_last"]  = vals[-1] if len(vals) else np.nan

            last2 = vals[-2:] if len(vals) >= 2 else vals
            early = vals[:8] if len(vals) >= 8 else (vals[:-2] if len(vals) > 2 else vals)
            feat[f"{name}_last2_mean"] = safe_nan(np.nanmean, last2)
            feat[f"{name}_early_mean"] = safe_nan(np.nanmean, early)
            feat[f"{name}_delta_last2_early"] = feat[f"{name}_last2_mean"] - feat[f"{name}_early_mean"]
            feat[f"{name}_delta_last_first"] = feat[f"{name}_last"] - feat[f"{name}_first"]

            mask = (~np.isnan(vals)) & (~np.isnan(day_idx))
            feat[f"{name}_slope"] = np.polyfit(day_idx[mask], vals[mask], 1)[0] if mask.sum() >= 2 else np.nan

            feat[f"{name}_diff_mean"] = safe_nan(np.nanmean, diffs) if len(diffs) else np.nan
            feat[f"{name}_diff_std"]  = safe_nan(np.nanstd,  diffs) if len(diffs) else np.nan
            feat[f"{name}_last3_std"] = safe_nan(np.nanstd, vals[-3:]) if len(vals) >= 3 else safe_nan(np.nanstd, vals)

            feat[f"{name}_missing"] = int(np.isnan(vals).sum())
            feat[f"{name}_n_obs"]   = int(np.sum(~np.isnan(vals)))

        # simple clinical threshold counts
        hr, sbp, dbp, temp, rr = vit["hr"], vit["sbp"], vit["dbp"], vit["temp_c"], vit["rr"]
        feat["hr_gt100"] = int(np.nansum(hr > 100))
        feat["hr_lt60"]  = int(np.nansum(hr < 60))
        feat["sbp_lt90"] = int(np.nansum(sbp < 90))
        feat["sbp_gt160"] = int(np.nansum(sbp > 160))
        feat["dbp_lt60"] = int(np.nansum(dbp < 60))
        feat["temp_gt38"] = int(np.nansum(temp > 38))
        feat["temp_lt36"] = int(np.nansum(temp < 36))
        feat["rr_gt22"] = int(np.nansum(rr > 22))
        feat["map_lt65"] = int(np.nansum(map_est < 65))
        feat["shock_gt1"] = int(np.nansum(shock_index > 1.0))
        feat["pp_lt30"] = int(np.nansum(pulse_pressure < 30))

        # keyword counts (cheap text signal, controlled dimensionality)
        for kw, pat in KW_PATTERNS:
            key = kw.replace(" ", "_").replace("/", "_")
            feat[f"all_kw_{key}"] = len(pat.findall(note_all))
            feat[f"last2_kw_{key}"] = len(pat.findall(note_last2))

        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    out = base.merge(feat_df, on="stay_id", how="left")

    # ED history ablation (NEW info source)
    if include_ed_history and (ed_hist is not None):
        out = out.merge(ed_hist, on="patient_id", how="left")
        out["prior_ed_cost_5y_log1p"] = np.log1p(pd.to_numeric(out["prior_ed_cost_5y_usd"], errors="coerce"))
        out["has_ed_history_row"] = out["primary_chronic"].notna().astype(int)
    return out

print("=== [FE] Building features (BASE vs EDHIST ablation) ===")
train_base = build_features(stays_train, include_ed_history=False)
test_base  = build_features(stays_test,  include_ed_history=False)

train_ed = build_features(stays_train, include_ed_history=True)
test_ed  = build_features(stays_test,  include_ed_history=True)

# Sanity: text nonempty rate (last2 keywords-only; raw text not used in this iter)
# we keep this print to monitor unexpected empties.
print("[FE] note_days nonzero rate:", float((train_base["note_days"] > 0).mean()))

# ---------------------
# 8) Robust P* loader + recovery (fix iter0022-type schema bug)
# ---------------------
PSTAR_PATH = CKPT_ROOT / "PSTAR.json"

def _to_float(x, default=np.nan):
    try:
        if x is None:
            return default
        v = float(x)
        if np.isnan(v) or np.isinf(v):
            return default
        return v
    except Exception:
        return default

def load_pstar_file(p: Path):
    if not p.exists():
        return None
    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        return None
    # schema variants:
    # - {"pstar": {...}} (most common)
    # - {"tag":..., "pstar": {...}, ...}
    # - directly {...} with "iteration_tag"
    if isinstance(obj, dict) and "pstar" in obj and isinstance(obj["pstar"], dict) and "iteration_tag" in obj["pstar"]:
        ps = obj["pstar"]
    elif isinstance(obj, dict) and "iteration_tag" in obj:
        ps = obj
    else:
        return None

    out = {
        "iteration_tag": str(ps.get("iteration_tag", "NONE")),
        "oof_macro_f1": _to_float(ps.get("oof_macro_f1"), np.nan),
        "threshold": _to_float(ps.get("threshold"), 0.5),
        "fold_mean": _to_float(ps.get("fold_mean"), np.nan),
        "fold_std": _to_float(ps.get("fold_std"), np.nan),
        "pos_rate": _to_float(ps.get("pos_rate"), np.nan),
        "f1_0": _to_float(ps.get("f1_0"), np.nan),
        "f1_1": _to_float(ps.get("f1_1"), np.nan),
        "confusion_matrix": ps.get("confusion_matrix", None),
    }
    # must have numeric macro
    if not np.isfinite(out["oof_macro_f1"]):
        return None
    return out

def load_best_checkpoint_pstar(ckpt_root: Path):
    best = None
    for d in sorted(ckpt_root.glob("iter????")):
        if not d.is_dir():
            continue
        metrics_p = d / "metrics.json"
        config_p  = d / "config.json"
        if not metrics_p.exists():
            continue
        try:
            m = json.loads(metrics_p.read_text(encoding="utf-8"))
        except Exception:
            continue

        # normalize metrics schema
        macro = None
        thr = None
        fold_std = None
        fold_mean = None
        pos_rate = None
        f1_0 = None
        f1_1 = None
        cm = None

        if isinstance(m, dict):
            if "oof_macro_f1" in m:
                macro = m.get("oof_macro_f1")
                thr = m.get("threshold")
                fold_std = m.get("fold_std")
                fold_mean = m.get("fold_mean")
                pos_rate = m.get("pos_rate")
                pc = m.get("per_class_f1", {})
                f1_0 = pc.get("f1_0")
                f1_1 = pc.get("f1_1")
                cm = m.get("confusion_matrix")
            elif "selected_metrics" in m and isinstance(m["selected_metrics"], dict):
                sm = m["selected_metrics"]
                macro = sm.get("oof_macro_f1") or sm.get("macro_f1") or sm.get("oof_macro_f1_num")
                thr = sm.get("threshold")
                fold_std = sm.get("fold_std")
                fold_mean = sm.get("fold_mean")
                pos_rate = sm.get("pos_rate")
                f1_0 = sm.get("f1_0")
                f1_1 = sm.get("f1_1")
                cm = sm.get("confusion_matrix")
            elif "pstar" in m and isinstance(m["pstar"], dict):
                sm = m["pstar"]
                macro = sm.get("oof_macro_f1")
                thr = sm.get("threshold")
                fold_std = sm.get("fold_std")
                fold_mean = sm.get("fold_mean")
                pos_rate = sm.get("pos_rate")
                f1_0 = sm.get("f1_0")
                f1_1 = sm.get("f1_1")
                cm = sm.get("confusion_matrix")

        macro = _to_float(macro, np.nan)
        if not np.isfinite(macro):
            continue

        iter_tag = d.name
        # threshold might be in config if missing
        if thr is None and config_p.exists():
            try:
                cfg = json.loads(config_p.read_text(encoding="utf-8"))
                thr = cfg.get("selected", {}).get("threshold", None)
            except Exception:
                pass

        cand = {
            "iteration_tag": iter_tag,
            "oof_macro_f1": macro,
            "threshold": _to_float(thr, 0.5),
            "fold_mean": _to_float(fold_mean, np.nan),
            "fold_std": _to_float(fold_std, np.nan),
            "pos_rate": _to_float(pos_rate, np.nan),
            "f1_0": _to_float(f1_0, np.nan),
            "f1_1": _to_float(f1_1, np.nan),
            "confusion_matrix": cm,
        }
        if (best is None) or (cand["oof_macro_f1"] > best["oof_macro_f1"]):
            best = cand
    return best

pstar_file = load_pstar_file(PSTAR_PATH)
pstar_ckpt_best = load_best_checkpoint_pstar(CKPT_ROOT)
effective_pstar = None
if pstar_file is None:
    effective_pstar = pstar_ckpt_best
else:
    effective_pstar = pstar_file
    if (pstar_ckpt_best is not None) and (pstar_ckpt_best["oof_macro_f1"] > pstar_file["oof_macro_f1"] + 1e-6):
        effective_pstar = pstar_ckpt_best

# If PSTAR is missing/corrupted or worse than best checkpoint, recover it.
if (effective_pstar is not None) and ((pstar_file is None) or (effective_pstar["iteration_tag"] != pstar_file.get("iteration_tag"))):
    try:
        with open(PSTAR_PATH, "w", encoding="utf-8") as f:
            json.dump({
                "tag": TAG,
                "pstar": effective_pstar,
                "train_prevalence": train_prev,
                "timestamp": utc_ts(),
                "schema_version": 7,
                "notes": "Recovered/normalized P* from best available checkpoint to prevent schema/NaN gate bugs."
            }, f, indent=2, ensure_ascii=False)
        print("✅ [PSTAR RECOVERY] PSTAR.json rewritten to best checkpoint:", effective_pstar["iteration_tag"], "macro=", round(effective_pstar["oof_macro_f1"], 6))
    except Exception as e:
        print("⚠️ [PSTAR RECOVERY] Could not rewrite PSTAR.json:", repr(e))

print("=== P* (effective, used for gating) ===")
print(json.dumps(effective_pstar, indent=2, ensure_ascii=False))

assert effective_pstar is not None and np.isfinite(effective_pstar["oof_macro_f1"]), "P* is missing/invalid even after recovery. Abort."

# ---------------------
# 9) Modeling utilities (LR base + HGB(TE dense))
# ---------------------
def threshold_sweep(oof_proba: np.ndarray, y_true: np.ndarray, fold_ids: np.ndarray):
    thresholds = np.round(np.linspace(0.05, 0.95, 91), 2)
    rows = []
    for t in thresholds:
        pred = (oof_proba >= t).astype(int)
        cm = confusion_matrix(y_true, pred)
        f1s = f1_score(y_true, pred, average=None)
        macro = f1_score(y_true, pred, average="macro")
        fold_scores = []
        for f in range(N_SPLITS):
            idx = fold_ids == f
            fold_scores.append(float(f1_score(y_true[idx], pred[idx], average="macro")))
        rows.append({
            "threshold": float(t),
            "macro_f1": float(macro),
            "f1_0": float(f1s[0]), "f1_1": float(f1s[1]),
            "pos_rate": float(pred.mean()),
            "fold_mean": float(np.mean(fold_scores)),
            "fold_std": float(np.std(fold_scores)),
            "tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        })
    return pd.DataFrame(rows)

def choose_threshold_prev_first(df_sweep: pd.DataFrame, prevalence: float, pos_tol: float = 0.07, macro_drop_tol: float = 0.005):
    best_macro = float(df_sweep["macro_f1"].max())
    df_pos_ok = df_sweep.loc[(df_sweep["pos_rate"] - prevalence).abs() <= pos_tol].copy()
    pool = df_pos_ok if len(df_pos_ok) else df_sweep.copy()
    pool_best = float(pool["macro_f1"].max())
    df_near = pool.loc[pool["macro_f1"] >= (pool_best - macro_drop_tol)].copy()
    df_near["pos_gap"] = (df_near["pos_rate"] - prevalence).abs()
    df_near = df_near.sort_values(["pos_gap", "fold_std", "macro_f1", "threshold"],
                                  ascending=[True, True, False, True]).reset_index(drop=True)
    chosen = df_near.iloc[0].to_dict()
    chosen["best_macro_overall"] = best_macro
    chosen["policy"] = "prev_first"
    return chosen

def choose_threshold_macro_first(df_sweep: pd.DataFrame, prevalence: float):
    df = df_sweep.copy()
    df["pos_gap"] = (df["pos_rate"] - prevalence).abs()
    df = df.sort_values(["macro_f1", "fold_std", "pos_gap", "threshold"],
                        ascending=[False, True, True, True]).reset_index(drop=True)
    chosen = df.iloc[0].to_dict()
    chosen["best_macro_overall"] = float(df_sweep["macro_f1"].max())
    chosen["policy"] = "macro_first"
    return chosen

def make_preprocessor_lr(num_cols, cat_cols):
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler(with_mean=False))]), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )

def oof_lr(df_X: pd.DataFrame, y: np.ndarray, num_cols, cat_cols, C: float):
    pre = make_preprocessor_lr(num_cols, cat_cols)
    oof = np.zeros(len(df_X), dtype=float)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(df_X, y)):
        pre_f = clone(pre)
        X_tr = pre_f.fit_transform(df_X.iloc[tr_idx], y[tr_idx])
        X_va = pre_f.transform(df_X.iloc[va_idx])
        clf = LogisticRegression(
            solver="liblinear",
            penalty="l1",
            C=float(C),
            class_weight=None,
            max_iter=5000,
            random_state=SEED
        )
        clf.fit(X_tr, y[tr_idx])
        oof[va_idx] = clf.predict_proba(X_va)[:, 1]
    return oof

def fit_full_lr(df_X: pd.DataFrame, y: np.ndarray, num_cols, cat_cols, C: float):
    pre = make_preprocessor_lr(num_cols, cat_cols)
    clf = LogisticRegression(
        solver="liblinear",
        penalty="l1",
        C=float(C),
        class_weight=None,
        max_iter=5000,
        random_state=SEED
    )
    pipe = Pipeline([("preprocess", pre), ("clf", clf)])
    pipe.fit(df_X, y)
    return pipe

# Target encoding for HGB (per-fold, leakage-safe)
def _fit_median_imputer(train_num: pd.DataFrame) -> pd.Series:
    return train_num.median(axis=0, skipna=True)

def _apply_median_imputer(df_num: pd.DataFrame, med: pd.Series) -> np.ndarray:
    return df_num.fillna(med).astype(float).values

def _fit_target_encoders(train_cat: pd.DataFrame, y_train: np.ndarray, smoothing: float = 30.0):
    global_mean = float(np.mean(y_train))
    encoders = {}
    for col in train_cat.columns:
        tmp = pd.DataFrame({"y": y_train, col: train_cat[col].values})
        grp = tmp.groupby(col)["y"].agg(["mean", "count"])
        enc = (grp["mean"] * grp["count"] + global_mean * smoothing) / (grp["count"] + smoothing)
        encoders[col] = enc.to_dict()
    return encoders, global_mean

def _apply_target_encoders(cat_df: pd.DataFrame, encoders: dict, global_mean: float) -> np.ndarray:
    out = np.zeros((len(cat_df), len(cat_df.columns)), dtype=float)
    for j, col in enumerate(cat_df.columns):
        mp = encoders.get(col, {})
        out[:, j] = cat_df[col].map(mp).fillna(global_mean).astype(float).values
    return out

def oof_hgb_te(df_X: pd.DataFrame, y: np.ndarray, num_cols, cat_cols,
               max_depth: int, learning_rate: float, max_leaf_nodes: int, min_samples_leaf: int, l2_regularization: float):
    oof = np.zeros(len(df_X), dtype=float)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(df_X, y)):
        X_tr_df = df_X.iloc[tr_idx]
        X_va_df = df_X.iloc[va_idx]

        med = _fit_median_imputer(X_tr_df[num_cols])
        Xtr_num = _apply_median_imputer(X_tr_df[num_cols], med)
        Xva_num = _apply_median_imputer(X_va_df[num_cols], med)

        encoders, gmean = _fit_target_encoders(X_tr_df[cat_cols], y[tr_idx], smoothing=30.0)
        Xtr_te = _apply_target_encoders(X_tr_df[cat_cols], encoders, gmean)
        Xva_te = _apply_target_encoders(X_va_df[cat_cols], encoders, gmean)

        Xtr = np.hstack([Xtr_num, Xtr_te])
        Xva = np.hstack([Xva_num, Xva_te])

        hgb = HistGradientBoostingClassifier(
            max_depth=int(max_depth),
            learning_rate=float(learning_rate),
            max_leaf_nodes=int(max_leaf_nodes),
            min_samples_leaf=int(min_samples_leaf),
            l2_regularization=float(l2_regularization),
            random_state=SEED
        )
        hgb.fit(Xtr, y[tr_idx])
        oof[va_idx] = hgb.predict_proba(Xva)[:, 1]
    return oof

def fit_full_hgb_te(df_X: pd.DataFrame, y: np.ndarray, num_cols, cat_cols,
                    max_depth: int, learning_rate: float, max_leaf_nodes: int, min_samples_leaf: int, l2_regularization: float):
    med = _fit_median_imputer(df_X[num_cols])
    X_num = _apply_median_imputer(df_X[num_cols], med)
    encoders, gmean = _fit_target_encoders(df_X[cat_cols], y, smoothing=30.0)
    X_te = _apply_target_encoders(df_X[cat_cols], encoders, gmean)
    X = np.hstack([X_num, X_te])

    hgb = HistGradientBoostingClassifier(
        max_depth=int(max_depth),
        learning_rate=float(learning_rate),
        max_leaf_nodes=int(max_leaf_nodes),
        min_samples_leaf=int(min_samples_leaf),
        l2_regularization=float(l2_regularization),
        random_state=SEED
    )
    hgb.fit(X, y)
    state = {
        "num_cols": list(num_cols),
        "cat_cols": list(cat_cols),
        "median": med.to_dict(),
        "encoders": encoders,
        "global_mean": gmean,
    }
    return hgb, state

def predict_hgb_te(hgb, state: dict, df_X: pd.DataFrame) -> np.ndarray:
    num_cols = state["num_cols"]
    cat_cols = state["cat_cols"]
    med = pd.Series(state["median"])
    X_num = _apply_median_imputer(df_X[num_cols], med)
    X_te = _apply_target_encoders(df_X[cat_cols], state["encoders"], float(state["global_mean"]))
    X = np.hstack([X_num, X_te])
    return hgb.predict_proba(X)[:, 1]

# ---------------------
# 10) Build modeling tables (BASE vs EDHIST)
# ---------------------
def prep_xy(train_df: pd.DataFrame, test_df: pd.DataFrame, y: np.ndarray, cat_cols: list):
    X_tr = train_df.drop(columns=["discharge_ready_day11"]).copy()
    X_te = test_df.copy()
    for c in cat_cols:
        if c in X_tr.columns:
            X_tr[c] = X_tr[c].astype(str)
            X_te[c] = X_te[c].astype(str)
    text_cols = [c for c in X_tr.columns if c.startswith("note_text_")]  # not used in this iter
    id_cols = ["stay_id", "patient_id"]
    num_cols = [c for c in X_tr.columns if c not in (id_cols + cat_cols + text_cols)]
    return X_tr, X_te, num_cols

CAT_BASE = ["unit_type", "sex", "insurance", "zip3", "admission_reason"]
CAT_ED   = ["unit_type", "sex", "insurance", "zip3", "admission_reason", "primary_chronic"] if ("primary_chronic" in train_ed.columns) else CAT_BASE

Xtr_base, Xte_base, NUM_BASE = prep_xy(train_base, test_base, y_all, CAT_BASE)
Xtr_ed,   Xte_ed,   NUM_ED   = prep_xy(train_ed,   test_ed,   y_all, CAT_ED)

print(f"[SCHEMA] BASE: CAT={len(CAT_BASE)} NUM={len(NUM_BASE)} | EDHIST: CAT={len(CAT_ED)} NUM={len(NUM_ED)}")

# ---------------------
# 11) Candidate search (single models + blends) — ablation focus
# ---------------------
print("\n=== [OOF] Training base learners (fixed CV) ===")

# Model grids (kept small; focus is NEW signal, not micro-tuning)
C_GRID = [0.06, 0.08, 0.10]
HGB_GRID = [
    {"max_depth": 3, "learning_rate": 0.05, "max_leaf_nodes": 31, "min_samples_leaf": 20, "l2_regularization": 0.1},
    {"max_depth": 4, "learning_rate": 0.03, "max_leaf_nodes": 63, "min_samples_leaf": 30, "l2_regularization": 0.2},
]

def eval_one_proba(name: str, oof_proba: np.ndarray, y_true: np.ndarray, fold_ids: np.ndarray, prevalence: float):
    sw = threshold_sweep(oof_proba, y_true, fold_ids)
    ch_prev = choose_threshold_prev_first(sw, prevalence=prevalence)
    ch_macro = choose_threshold_macro_first(sw, prevalence=prevalence)

    out = []
    for ch in [ch_prev, ch_macro]:
        out.append({
            "name": f"{name}_{ch['policy']}",
            "base_name": name,
            "macro_f1": float(ch["macro_f1"]),
            "threshold": float(ch["threshold"]),
            "pos_rate": float(ch["pos_rate"]),
            "pos_gap": float(abs(float(ch["pos_rate"]) - prevalence)),
            "fold_mean": float(ch["fold_mean"]),
            "fold_std": float(ch["fold_std"]),
            "f1_0": float(ch["f1_0"]),
            "f1_1": float(ch["f1_1"]),
            "tn": int(ch["tn"]), "fp": int(ch["fp"]), "fn": int(ch["fn"]), "tp": int(ch["tp"]),
            "policy": ch["policy"],
            "best_macro_overall": float(ch["best_macro_overall"]),
        })
    return out, sw

def auc_safe(y, p):
    try:
        return float(roc_auc_score(y, p))
    except Exception:
        return float("nan")

# OOF stores (so blends are consistent)
oof_store = {}        # key -> oof_proba
sweep_store = {}      # key -> sweep df

rows = []

def add_model_family(prefix: str, Xtr: pd.DataFrame, num_cols: list, cat_cols: list):
    # LR
    for C in C_GRID:
        key = f"{prefix}LR_C{C:.2f}"
        oof = oof_lr(Xtr, y_all, num_cols, cat_cols, C=float(C))
        oof_store[key] = oof
        auc = auc_safe(y_all, oof)
        eval_rows, sw = eval_one_proba(key, oof, y_all, fold_ids, train_prev)
        for r in eval_rows:
            r.update({"type":"single","family":prefix,"model":"LR","C":float(C),"hgb_cfg":None,"auc":auc})
            rows.append(r)
        sweep_store[key] = sw

    # HGB(TE dense)
    for i, cfg in enumerate(HGB_GRID):
        key = f"{prefix}HGB_TE_g{i}"
        oof = oof_hgb_te(
            Xtr, y_all, num_cols, cat_cols,
            max_depth=cfg["max_depth"], learning_rate=cfg["learning_rate"],
            max_leaf_nodes=cfg["max_leaf_nodes"], min_samples_leaf=cfg["min_samples_leaf"],
            l2_regularization=cfg["l2_regularization"]
        )
        oof_store[key] = oof
        auc = auc_safe(y_all, oof)
        eval_rows, sw = eval_one_proba(key, oof, y_all, fold_ids, train_prev)
        for r in eval_rows:
            r.update({"type":"single","family":prefix,"model":"HGB_TE","C":None,"hgb_cfg":cfg,"auc":auc})
            rows.append(r)
        sweep_store[key] = sw

# Train BASE and EDHIST families
add_model_family(prefix="BASE_", Xtr=Xtr_base, num_cols=NUM_BASE, cat_cols=CAT_BASE)
add_model_family(prefix="EDH_",  Xtr=Xtr_ed,   num_cols=NUM_ED,   cat_cols=CAT_ED)

# Blends: LR + HGB within same family (kept like iter0020 direction)
def add_blends(prefix: str):
    lr_keys = [k for k in oof_store.keys() if k.startswith(prefix+"LR_")]
    hgb_keys = [k for k in oof_store.keys() if k.startswith(prefix+"HGB_TE_")]
    weights = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]  # weight for LR; HGB weight = 1-w
    for lk in lr_keys:
        for hk in hgb_keys:
            for w in weights:
                oof = float(w) * oof_store[lk] + (1.0 - float(w)) * oof_store[hk]
                key = f"{prefix}ENS2[{lk.split(prefix)[1]}+{hk.split(prefix)[1]}]_w{w:.2f}"
                oof_store[key] = oof
                eval_rows, sw = eval_one_proba(key, oof, y_all, fold_ids, train_prev)
                for r in eval_rows:
                    r.update({
                        "type":"blend2","family":prefix,
                        "model":"ENS2",
                        "models":[lk, hk],
                        "weights":[float(w), float(1.0-w)],
                        "auc":float("nan")
                    })
                    rows.append(r)
                sweep_store[key] = sw

add_blends("BASE_")
add_blends("EDH_")

abl_df = pd.DataFrame(rows)
# Add deltas vs effective P*
pstar_macro = float(effective_pstar["oof_macro_f1"])
pstar_std = _to_float(effective_pstar.get("fold_std"), np.nan)
abl_df["delta_vs_pstar_macro"] = abl_df["macro_f1"] - pstar_macro
abl_df["delta_vs_pstar_std"] = abl_df["fold_std"] - pstar_std if np.isfinite(pstar_std) else np.nan

abl_df = abl_df.sort_values(["macro_f1","fold_std","pos_gap"], ascending=[False, True, True]).reset_index(drop=True)

print("\n=== Candidate leaderboard (top 15) ===")
show_cols = ["name","type","family","macro_f1","delta_vs_pstar_macro","threshold","pos_rate","pos_gap","fold_std","delta_vs_pstar_std","f1_0","f1_1","auc","policy"]
print(abl_df[show_cols].head(15).to_string(index=False))

ABL_PATH = ART_ROOT / f"{ITER_TAG}_ablation_summary.csv"
abl_df.to_csv(ABL_PATH, index=False)
print("\n✅ Saved ablation summary ->", ABL_PATH)

# ---------------------
# 12) Hard gate (NaN-safe) + selection
# ---------------------
MIN_DELTA = 0.001
STD_TOL  = 0.01
POS_TOL  = 0.07

def passes_gate(row: dict) -> (bool, list):
    reasons = []
    m = _to_float(row.get("macro_f1"), np.nan)
    fs = _to_float(row.get("fold_std"), np.nan)
    pr = _to_float(row.get("pos_rate"), np.nan)

    if not np.isfinite(m):
        reasons.append("macro_f1 is NaN/inf")
        return False, reasons

    if m < pstar_macro + MIN_DELTA:
        reasons.append(f"macro_f1<{pstar_macro + MIN_DELTA:.6f} (need +{MIN_DELTA})")

    if np.isfinite(pstar_std) and np.isfinite(fs):
        if fs > pstar_std + STD_TOL:
            reasons.append(f"fold_std>{pstar_std + STD_TOL:.6f} (p* {pstar_std:.6f} + tol {STD_TOL})")

    if np.isfinite(pr):
        if abs(pr - train_prev) > POS_TOL:
            reasons.append(f"pos_rate drift |{pr:.3f}-{train_prev:.3f}|>{POS_TOL:.3f}")
    else:
        reasons.append("pos_rate is NaN")

    return (len(reasons) == 0), reasons

eligible = []
for r in abl_df.to_dict("records"):
    ok, reasons = passes_gate(r)
    if ok:
        eligible.append(r)

if eligible:
    # pick best macro_f1; tie-breaker: lower fold_std, lower pos_gap, lower threshold
    eligible = sorted(eligible, key=lambda d: (float(d["macro_f1"]), -float(d["fold_std"]), -float(d["pos_gap"]), -float(d["threshold"])), reverse=True)
    selected = eligible[0]
    decision = "SELECT_CANDIDATE (passes hard gate vs effective P*)"
else:
    selected = abl_df.iloc[0].to_dict()
    ok, reasons = passes_gate(selected)
    decision = "NO_CANDIDATE_PASSES_GATE -> SAFE MODE uses effective P*"

print("\n=== Selection decision ===")
print("Decision:", decision)
print("Selected:", selected["name"], "| type=", selected["type"], "| macro_f1=", round(float(selected["macro_f1"]),6),
      "| thr=", float(selected["threshold"]), "| pos_rate=", round(float(selected["pos_rate"]),3),
      "| family=", selected.get("family","?"), "| policy=", selected.get("policy","?"))

sel_key = selected["base_name"]
sel_thr = float(selected["threshold"])
sel_policy = str(selected["policy"])

sel_oof = oof_store[sel_key]
sel_sweep = sweep_store[sel_key]

# ---------------------
# 13) SAFE submission utilities (never worse than effective P*)
# ---------------------
SUB_PATH = PROJECT_ROOT / f"{TAG}_submission.csv"

def pick_candidate_file(iter_tag: str):
    cands = sorted(ART_ROOT.glob(f"{TAG}_*candidate*.csv"))
    if not cands:
        cands = sorted(PROJECT_ROOT.glob(f"{TAG}_*candidate*.csv"))
    if not cands:
        return None
    matched = [p for p in cands if iter_tag in p.name]
    return matched[-1] if matched else cands[-1]

def safe_mode_write_submission_from_pstar():
    # prefer copying P* candidate snapshot
    pstar_iter = str(effective_pstar["iteration_tag"])
    p = pick_candidate_file(pstar_iter)
    if p and p.exists():
        df = pd.read_csv(p)
        df = df[["stay_id","discharge_ready_day11"]].copy()
        df["stay_id"] = df["stay_id"].astype(int)
        df["discharge_ready_day11"] = df["discharge_ready_day11"].astype(int)
        df.to_csv(SUB_PATH, index=False)
        return f"SAFE_MODE_copy_PSTAR_candidate({pstar_iter})", p.name, float(df["discharge_ready_day11"].mean())

    # fallback: use P* pipeline if available
    pipe_path = CKPT_ROOT / pstar_iter / "pipeline.joblib"
    if pipe_path.exists():
        pipe = joblib.load(pipe_path)
        thr = float(effective_pstar.get("threshold", 0.5))
        # Need to rebuild features consistent with that pipeline? If mismatch, abort to dummy.
        # We try to predict using the pipe on EDHIST feature table (most superset).
        try:
            testX = Xte_ed.copy() if "EDH_" in str(selected.get("family","")) else Xte_base.copy()
            proba = pipe.predict_proba(testX)[:,1]
            pred = (proba >= thr).astype(int)
            sub = pd.DataFrame({"stay_id": stays_test["stay_id"].astype(int).values, "discharge_ready_day11": pred.astype(int)})
            sub.to_csv(SUB_PATH, index=False)
            return f"SAFE_MODE_pipeline({pstar_iter})", pipe_path.name, float(sub["discharge_ready_day11"].mean())
        except Exception:
            pass

    # last resort: dummy all-1 (should not happen)
    dummy = pd.DataFrame({"stay_id": stays_test["stay_id"].astype(int).values, "discharge_ready_day11": np.ones(len(stays_test), dtype=int)})
    dummy.to_csv(SUB_PATH, index=False)
    return "SAFE_MODE_dummy_all1", None, float(dummy["discharge_ready_day11"].mean())

# ---------------------
# 14) Fit selected candidate on full train (only if passes gate)
# ---------------------
# Decide family -> choose feature tables
use_ed_family = str(selected.get("family","")).startswith("EDH_")

Xtr = Xtr_ed if use_ed_family else Xtr_base
Xte = Xte_ed if use_ed_family else Xte_base
NUM = NUM_ED if use_ed_family else NUM_BASE
CAT = CAT_ED if use_ed_family else CAT_BASE

submission_mode = None
candidate_used = None

# Build full blended proba on test if candidate is accepted; else safe mode
if eligible:
    # Identify selected underlying components if it's an ensemble
    # - if single LR/HGB: sel_key is that model key
    # - if ENS2: parse weights stored in selected row
    # We'll reconstruct by checking selected['type']
    def parse_components(row: dict):
        if row.get("type") == "blend2" and isinstance(row.get("models"), list) and isinstance(row.get("weights"), list):
            return row["models"], row["weights"]
        # single
        return [row["base_name"]], [1.0]

    models, weights = parse_components(selected)

    # Fit components on full train
    fitted = {}
    fitted_meta = {}

    for mk in models:
        # mk is like "EDH_LR_C0.08" or "EDH_HGB_TE_g0"
        if mk.endswith("LR_C0.06") or mk.endswith("LR_C0.08") or mk.endswith("LR_C0.10"):
            m = re.search(r"LR_C(\d+\.\d+)", mk)
            C = float(m.group(1)) if m else 0.08
            pipe = fit_full_lr(Xtr, y_all, NUM, CAT, C=C)
            fitted[mk] = ("lr", pipe, {"C": C})
        elif "HGB_TE" in mk:
            # map grid index
            g = re.search(r"_g(\d+)$", mk)
            gidx = int(g.group(1)) if g else 0
            cfg = HGB_GRID[gidx] if gidx < len(HGB_GRID) else HGB_GRID[0]
            hgb, state = fit_full_hgb_te(
                Xtr, y_all, NUM, CAT,
                max_depth=cfg["max_depth"], learning_rate=cfg["learning_rate"],
                max_leaf_nodes=cfg["max_leaf_nodes"], min_samples_leaf=cfg["min_samples_leaf"],
                l2_regularization=cfg["l2_regularization"]
            )
            fitted[mk] = ("hgb_te", hgb, state)
            fitted_meta[mk] = cfg
        else:
            raise RuntimeError(f"Unknown model key: {mk}")

    # Predict test proba for each component
    test_probas = []
    for mk, w in zip(models, weights):
        kind, model_obj, state = fitted[mk]
        if kind == "lr":
            proba = model_obj.predict_proba(Xte)[:,1]
        elif kind == "hgb_te":
            proba = predict_hgb_te(model_obj, state, Xte)
        else:
            raise RuntimeError("Unknown fitted model kind")
        test_probas.append(float(w) * proba)

    test_proba_blend = np.sum(test_probas, axis=0)
    test_pred = (test_proba_blend >= sel_thr).astype(int)

    sub = pd.DataFrame({"stay_id": stays_test["stay_id"].astype(int).values,
                        "discharge_ready_day11": test_pred.astype(int)})
    sub.to_csv(SUB_PATH, index=False)

    submission_mode = "use_candidate"
    candidate_used = "blended_fit_full_train"
    print("\n=== Submission written ===")
    print("SUB_PATH:", SUB_PATH)
    print("submission_mode:", submission_mode)
    print("Threshold used (submit):", sel_thr)
    print("Submission pos_rate:", float(sub["discharge_ready_day11"].mean()))
else:
    submission_mode, candidate_used, posr = safe_mode_write_submission_from_pstar()
    print("\n=== Submission written ===")
    print("SUB_PATH:", SUB_PATH)
    print("submission_mode:", submission_mode)
    print("Candidate used:", candidate_used)
    print("Submission pos_rate:", posr)

# Candidate snapshot always written (even if SAFE MODE copied P*)
CAND_PATH = ART_ROOT / f"{TAG}_{BRANCH}_block{BLOCK_ID}_iter{ITER_ID:04d}_candidate.csv"
try:
    shutil.copy2(SUB_PATH, CAND_PATH)
except Exception:
    # fallback: rewrite from reading SUB_PATH
    pd.read_csv(SUB_PATH).to_csv(CAND_PATH, index=False)

print("CAND_PATH:", CAND_PATH)

# ---------------------
# 15) Save artifacts: OOF + sweep + group diagnostics + checkpoint + decision note
# ---------------------
# OOF predictions saved for selected candidate (even if rejected; for audit)
oof_df = pd.DataFrame({
    "stay_id": stays_train["stay_id"].astype(int).values,
    "y_true": y_all.astype(int),
    "oof_proba": sel_oof.astype(float),
    "fold": fold_ids.astype(int),
})
OOF_PATH = ART_ROOT / f"{ITER_TAG}_oof_predictions.csv"
oof_df.to_csv(OOF_PATH, index=False)

SWEEP_PATH = ART_ROOT / f"{ITER_TAG}_threshold_sweep.csv"
sel_sweep.to_csv(SWEEP_PATH, index=False)

# diagnostics
pred_oof = (sel_oof >= sel_thr).astype(int)
diag_df = (train_ed if use_ed_family else train_base).copy()
diag_cols = [c for c in ["unit_type","insurance","admission_reason","primary_chronic"] if c in diag_df.columns]
diag = diag_df[["stay_id"] + diag_cols].copy()
diag["y"] = y_all
diag["pred"] = pred_oof
diag["proba"] = sel_oof
diag["is_fp"] = (diag["y"] == 0) & (diag["pred"] == 1)
diag["is_fn"] = (diag["y"] == 1) & (diag["pred"] == 0)

def group_err(df: pd.DataFrame, col: str) -> pd.DataFrame:
    g = df.groupby(col).agg(
        n=("stay_id", "count"),
        y_prev=("y", "mean"),
        pred_rate=("pred", "mean"),
        fp=("is_fp", "sum"),
        fn=("is_fn", "sum"),
    ).reset_index()
    neg = np.maximum(1.0, g["n"] - g["y_prev"] * g["n"])
    pos = np.maximum(1.0, g["y_prev"] * g["n"])
    g["fp_rate_in_neg"] = g["fp"] / neg
    g["fn_rate_in_pos"] = g["fn"] / pos
    return g.sort_values(["fp_rate_in_neg", "fn_rate_in_pos"], ascending=False)

ges = []
for col in diag_cols:
    ge = group_err(diag, col)
    ge["group_col"] = col
    ges.append(ge)

GES_PATH = ART_ROOT / f"{ITER_TAG}_group_error_summary.csv"
if ges:
    pd.concat(ges, ignore_index=True).to_csv(GES_PATH, index=False)
else:
    pd.DataFrame().to_csv(GES_PATH, index=False)

# checkpoint
iter_ckpt_dir = CKPT_ROOT / ITER_TAG
iter_ckpt_dir.mkdir(parents=True, exist_ok=True)

# compute OOF metrics (selected threshold)
cm = confusion_matrix(y_all, pred_oof).tolist()
f1s = f1_score(y_all, pred_oof, average=None)
macro = f1_score(y_all, pred_oof, average="macro")
fold_scores = [float(f1_score(y_all[fold_ids==f], pred_oof[fold_ids==f], average="macro")) for f in range(N_SPLITS)]

metrics = {
    "oof_macro_f1": float(macro),
    "per_class_f1": {"f1_0": float(f1s[0]), "f1_1": float(f1s[1])},
    "confusion_matrix": cm,
    "threshold": float(sel_thr),
    "pos_rate": float(pred_oof.mean()),
    "fold_scores": fold_scores,
    "fold_mean": float(np.mean(fold_scores)),
    "fold_std": float(np.std(fold_scores)),
    "auc": auc_safe(y_all, sel_oof),
    "train_prevalence": train_prev,
    "pstar_reference_effective": effective_pstar,
    "gate": {"min_delta": MIN_DELTA, "std_tol": STD_TOL, "pos_tol": POS_TOL, "decision": decision},
}

config = {
    "team": TEAM, "task": TASK, "version": VERSION,
    "iteration_id": ITER_ID, "iteration_tag": ITER_TAG,
    "branch": BRANCH, "block_id": BLOCK_ID, "block_pos": BLOCK_POS,
    "seed": SEED,
    "validation": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "what_changed": CHANGESET,
    "risk": RISKS,
    "fallback": FALLBACK,
    "selected": {
        "name": selected["name"],
        "base_key": sel_key,
        "family": selected.get("family",""),
        "type": selected.get("type",""),
        "threshold": float(sel_thr),
        "policy": sel_policy,
        "spec": {k: selected.get(k) for k in ["model","models","weights","C","hgb_cfg"] if k in selected},
    },
    "schema": {
        "BASE": {"cat_cols": CAT_BASE, "num_cols": len(NUM_BASE)},
        "EDHIST": {"cat_cols": CAT_ED, "num_cols": len(NUM_ED)},
        "derived_indices": {
            "pulse_pressure": "sbp - dbp",
            "map": "(sbp + 2*dbp) / 3",
            "shock_index": "hr / sbp",
            "temp_dev": "temp_c - 37",
        },
        "keywords_count": len(KEYWORDS),
        "vitals_days_used": "1-10",
        "ed_history_used": bool(ed_hist is not None),
        "ed_history_fields": ["primary_chronic","prior_ed_visits_5y","prior_ed_cost_5y_usd","prior_ed_cost_5y_log1p","has_ed_history_row"],
        "explicitly_not_used": ["discharge_notes.json (leak risk)", "admissions los_days/discharge_weekday/readmit_30d (future/outcome for this task)"],
    },
}

with open(iter_ckpt_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
with open(iter_ckpt_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

# save selected sweep + oof snapshot for convenience
shutil.copy2(OOF_PATH, iter_ckpt_dir / OOF_PATH.name)
shutil.copy2(SWEEP_PATH, iter_ckpt_dir / SWEEP_PATH.name)
shutil.copy2(ABL_PATH, iter_ckpt_dir / ABL_PATH.name)
shutil.copy2(GES_PATH, iter_ckpt_dir / GES_PATH.name)

print("✅ Saved checkpoint ->", iter_ckpt_dir)

# Decision note (human-readable)
DEC_PATH = ART_ROOT / f"{ITER_TAG}_DECISION.md"
gate_ok, gate_reasons = passes_gate(selected)
dec = []
dec.append(f"# {TAG} — {ITER_TAG} Decision Note\nGenerated: {utc_ts()}\n")
dec.append("## 1) Objective\nPredict discharge_ready_day11 with Macro-F1; strict CV reproducibility.\n")
dec.append("## 2) What changed (single new signal)\n" + "\n".join([f"- {x}" for x in CHANGESET]) + "\n")
dec.append("## 3) P* used for gating (effective)\n```json\n" + json.dumps(effective_pstar, indent=2, ensure_ascii=False) + "\n```\n")
dec.append("## 4) Candidate selected (by OOF leaderboard)\n```json\n" + json.dumps({
    "name": selected["name"],
    "type": selected.get("type"),
    "family": selected.get("family"),
    "macro_f1": float(selected["macro_f1"]),
    "threshold": float(selected["threshold"]),
    "pos_rate": float(selected["pos_rate"]),
    "fold_std": float(selected["fold_std"]),
    "policy": selected.get("policy"),
    "delta_vs_pstar_macro": float(selected["macro_f1"] - pstar_macro),
}, indent=2, ensure_ascii=False) + "\n```\n")
dec.append("## 5) Gate decision\n"
           f"- Decision: **{decision}**\n"
           f"- Gate pass: **{bool(gate_ok and eligible)}**\n"
           f"- If failed, reasons: {gate_reasons if gate_reasons else 'N/A'}\n")
dec.append("## 6) Evidence (OOF)\n"
           "- Saved ablation CSV: "
           f"`{ABL_PATH.name}`\n"
           "- Saved OOF predictions: "
           f"`{OOF_PATH.name}`\n"
           "- Saved threshold sweep: "
           f"`{SWEEP_PATH.name}`\n")
dec.append("## 7) Risks\n" + "\n".join([f"- {x}" for x in RISKS]) + "\n")
dec.append("## 8) Fallback\n" + FALLBACK + "\n")
dec.append("## 9) Why iter0022 failed (institutional lesson)\n"
           "- A schema bug made P* parse as NaN / -1, so the gate incorrectly accepted a far-worse model.\n"
           "- This iter hardens P* parsing + rejects NaN so that cannot happen again.\n"
           "- Also avoid discharge_notes/admissions outcome-like fields.\n")

DEC_PATH.write_text("\n".join(dec), encoding="utf-8")
print("✅ Wrote decision note ->", DEC_PATH)

# ---------------------
# 16) Update PSTAR.json ONLY if candidate truly passes gate
# ---------------------
if eligible:
    new_pstar = {
        "iteration_tag": ITER_TAG,
        "oof_macro_f1": float(metrics["oof_macro_f1"]),
        "threshold": float(metrics["threshold"]),
        "fold_mean": float(metrics["fold_mean"]),
        "fold_std": float(metrics["fold_std"]),
        "pos_rate": float(metrics["pos_rate"]),
        "f1_0": float(metrics["per_class_f1"]["f1_0"]),
        "f1_1": float(metrics["per_class_f1"]["f1_1"]),
        "confusion_matrix": metrics["confusion_matrix"],
    }
    # Extra safety: never allow PSTAR to drop below recovered effective P*
    if new_pstar["oof_macro_f1"] >= pstar_macro + MIN_DELTA:
        with open(PSTAR_PATH, "w", encoding="utf-8") as f:
            json.dump({
                "tag": TAG,
                "pstar": new_pstar,
                "train_prevalence": train_prev,
                "timestamp": utc_ts(),
                "schema_version": 7,
                "notes": "Updated because candidate passed hard gate vs effective P* (NaN-safe)."
            }, f, indent=2, ensure_ascii=False)
        print("✅ Updated PSTAR.json ->", new_pstar["iteration_tag"], "macro=", round(new_pstar["oof_macro_f1"], 6))
    else:
        print("⚠️ Gate logic discrepancy: eligible=True but macro insufficient; PSTAR not updated.")
else:
    print("ℹ️ PSTAR.json unchanged (no candidate passed hard gate).")

# ---------------------
# 17) Append iteration log (append-only; NaN-safe)
# ---------------------
iteration_record = {
    "version": VERSION,
    "iteration_id": ITER_ID,
    "timestamp": utc_ts(),
    "phase": "Modeling",
    "branch": BRANCH,
    "block_id": BLOCK_ID,
    "block_pos": BLOCK_POS,
    "data_paths_used": {
        "stays_train": str(dp("stays_train.csv")),
        "stays_test": str(dp("stays_test.csv")),
        "patients": str(dp("patients.csv")),
        "vitals_timeseries": str(dp("vitals_timeseries.json")),
        "ed_cost_train": str(dp("ed_cost_train.csv")) if (PROJECT_ROOT / "ed_cost_train.csv").exists() or (Path("/mnt/data") / "ed_cost_train.csv").exists() else None,
        "ed_cost_test": str(dp("ed_cost_test.csv")) if (PROJECT_ROOT / "ed_cost_test.csv").exists() or (Path("/mnt/data") / "ed_cost_test.csv").exists() else None,
    },
    "join_keys": ["stays.patient_id -> patients.patient_id", "stays.stay_id -> vitals_timeseries.stay_id", "stays.patient_id -> ed_cost.patient_id (ED history)"],
    "leakage_checks_performed": [
        "Asserted vitals max(day) <= 10 (no day11 leakage).",
        "Target encoding fit inside each CV fold only.",
        "Explicitly avoided discharge_notes/admissions outcome-like columns.",
        "Gate rejects NaN/invalid P* and uses recovered effective P* from checkpoints.",
    ],
    "what_changed": CHANGESET,
    "risk": RISKS,
    "fallback": FALLBACK,
    "validation_protocol": {"type": "StratifiedKFold", "n_splits": N_SPLITS, "shuffle": True, "random_state": SEED},
    "effective_pstar_used": effective_pstar,
    "selected_model": {
        "name": selected["name"],
        "type": selected.get("type"),
        "family": selected.get("family"),
        "base_key": sel_key,
        "threshold": float(sel_thr),
        "policy": sel_policy,
        "decision": decision,
    },
    "metrics": metrics,
    "artifacts_written": {
        "submission_csv": str(SUB_PATH),
        "candidate_snapshot": str(CAND_PATH),
        "oof_predictions": str(OOF_PATH),
        "threshold_sweep": str(SWEEP_PATH),
        "ablation_summary": str(ABL_PATH),
        "group_error_summary": str(GES_PATH),
        "decision_note": str(DEC_PATH),
        "checkpoint_dir": str(iter_ckpt_dir),
        "pstar_path": str(PSTAR_PATH),
    },
}

try:
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
    print("✅ Appended iteration log ->", LOG_PATH)
except PermissionError:
    alt_log = CKPT_ROOT / f"{TAG}_iteration_detail.jsonl"
    with open(alt_log, "a", encoding="utf-8") as f:
        f.write(json.dumps(iteration_record, ensure_ascii=False) + "\n")
    print("⚠️ Could not append to LOG_PATH; wrote to:", alt_log)

# ---------------------
# 18) Consultant packet (every iter)
# ---------------------
consult_packet = {
    "tag": TAG,
    "iteration_id": ITER_ID,
    "iteration_tag": ITER_TAG,
    "timestamp": iteration_record["timestamp"],
    "what_changed": CHANGESET,
    "metric_changes": {
        "effective_pstar": effective_pstar,
        "selected_metrics": metrics,
        "gate_decision": decision,
        "submission_mode": submission_mode,
    },
    "suspected_leakage_risks": [
        "Vitals day>10 would leak: enforced assert max(day)<=10.",
        "Outcome-like external fields (e.g., discharge_notes, admissions los/discharge_weekday) are excluded.",
        "Target encoding is fit only on fold-train split."
    ],
    "complexity_drift_risks": [
        "This iter adds only 1 small patient-history source (ED history); no TF-IDF expansion."
    ],
    "evaluation_alignment_risks": [
        "Threshold overfit risk controlled by grid sweep + fold_std tracking + pos_rate vs prevalence."
    ],
    "unknown_unknowns": [
        "ED history distribution shift between train/test could affect generalization.",
    ],
    "recommendations_for_next_iter": [
        "If ED history helps: consider a SECOND ablation adding a small text model (max_features<=2k, min_df>=2) as a third learner.",
        "If ED history does not pass gate: keep P* and instead try a new physiology characterization (e.g., stability/variability segments) as the ONLY change next.",
    ],
}

consult_path = CONSULT_ROOT / f"{TAG}_iter{ITER_ID}_packet.json"
with open(consult_path, "w", encoding="utf-8") as f:
    json.dump(consult_packet, f, indent=2, ensure_ascii=False)
print("✅ Wrote consultant packet ->", consult_path)

print("\nDONE ✅")
print("Key outputs:")
print(" - Submission       :", SUB_PATH, "| mode:", submission_mode)
print(" - Candidate        :", CAND_PATH)
print(" - Ablation         :", ABL_PATH)
print(" - OOF              :", OOF_PATH)
print(" - Sweep            :", SWEEP_PATH)
print(" - Group diag       :", GES_PATH)
print(" - Decision note    :", DEC_PATH)
print(" - PSTAR            :", PSTAR_PATH)
print(" - Checkpoint dir   :", iter_ckpt_dir)

=== PREFLIGHT ===
PROJECT_ROOT: d:\AgentDs\agent_ds_healthcare
ART_ROOT    : d:\AgentDs\agent_ds_healthcare\artifacts\clai_ch3_v3
CKPT_ROOT   : d:\AgentDs\agent_ds_healthcare\checkpoints\clai_ch3_v3
CONSULT_ROOT: d:\AgentDs\agent_ds_healthcare\consultant_packets
LOG_PATH    : d:\AgentDs\agent_ds_healthcare\clai_ch3_v3_iteration_detail.jsonl
ITER_ID=23  ITER_TAG=iter0023  BLOCK_ID=4  BLOCK_POS=3  BRANCH=W7_EDHistory_ablation
[LEAKAGE_CHECK] Max day in vitals_timeseries: 10 (OK <= 10)
[DATA] Train prevalence(y=1): 0.656 | n=1000
[ED_HISTORY] Loaded: (4000, 4) | patients covered: 4000
=== [FE] Building features (BASE vs EDHIST ablation) ===
[FE] note_days nonzero rate: 1.0
✅ [PSTAR RECOVERY] PSTAR.json rewritten to best checkpoint: iter0020 macro= 0.824964
=== P* (effective, used for gating) ===
{
  "iteration_tag": "iter0020",
  "oof_macro_f1": 0.8249641287951847,
  "threshold": 0.63,
  "fold_mean": 0.8248897805164553,
  "fold_std": 0.024420082144040903,
  "pos_rate": NaN,
  "f1_0": 0.76

# New Iter

Iterations: 0.7589, 0.7041, 0.7041, 0.7564m 0.7746, 0.6770, 0.7739, 0.7903, 0.7906, 0.7965, 0.7965, 0.8168, 0.8168, 0.7321, 0.8168

# Submission

In [59]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 3, "clai_ch3_v3_submission.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. You've completed all Healthcare challenges!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 0.8168 (Macro-F1)
✅ Validation passed
✅ Submission successful!
   📊 Score: 0.8168
   📏 Metric: Macro-F1
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. You've completed all Healthcare challenges!
